In [ ]:
import nbimporter
import os
from pathlib import Path
from math import sqrt
import bw2data as bd
import bw2io as bi
import bw2calc as bc
from premise import NewDatabase
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Border, Side

In [ ]:
print("bw2data version", bd.__version__)
print("bw2io version", bi.__version__)
print("bw2calc version", bi.__version__)

# 0. brightway project, import ecoinvent and update ecoinvent according to your background scenario

## 0.1 Project Definition

In [ ]:
# 1. Get project name from user input
project_name = input("Enter the Brightway project name: ").strip()
    
if not project_name:
        raise SystemExit("Project name cannot be empty.")

# 2. Check if project already exists in Brightway
if project_name in bd.projects:
    print(f"Project '{project_name}' already exists.")
        
    # 3. Give choice to use another name
    choice = input(f"Project '{project_name}' already exists. Do you want to use another name and start a new project? (y/n): ").lower()
        
    if choice == 'y':
        try:
            print(f"Existing projects: {sorted(bd.projects)}")
            new_project_name = input("Enter a new project name: ").strip()
            if not new_project_name:
                raise SystemExit("Project name cannot be empty.")
            else:
                print(f"New project name set to: {new_project_name}")
                bd.projects.set_current(new_project_name)
        except Exception as e:
            print(f"Error: {e}")
            raise SystemExit("Failed to set new project name.")
    else:
            print(f"Using existing project: {project_name}")
            bd.projects.set_current(project_name)
    
# Create/Set the new project
# In Brightway, setting the current project creates it if it doesn't exist.
bd.projects.set_current(project_name)
print(f"Current project set to: {bd.projects.current}")

In [ ]:
# If this is your first time using this BW25 project, this should be an empty dictionary!
bd.databases

## 0.2 Ecoinvent import

In [ ]:
# This tool was built allowing the user to use any premise-modified version of ecoinvent 3.11.
# To use other versions, make sure that the datasets imported by ImportEcoinventDatasets.ipynb 
# are present in the version of ecoinvent you want to use. 

# Define the version and system model of ecoinvent to import
print("You can use any version of ecoinvent, but make sure that the datasets imported by ImportEcoinventDatasets.ipynb are present in the version you choose.\n")
ei_version = input("Enter ecoinvent version (e.g., 3.11): ")
ei_model = input("Enter system model (cutoff / apos / consequential / EN15804): ")

if f'ecoinvent-{ei_version}-{ei_model}' in bd.databases:
    print(f'ecoinvent {ei_version}-{ei_model} is already present in the project')
else:
    bi.import_ecoinvent_release(
        version=ei_version,
        system_model=ei_model, # can be cutoff / apos / consequential / EN15804
        username='username', # replace with your actual ecoinvent username
        password='password'  # replace with your actual ecoinventpassword
    )

bd.databases

## 0.3 Generation of premise-modified futurized version of ecoinvent

In [ ]:
# --- 1. User Inputs ---
print("### PREMISE CONFIGURATION ###")
print("Please provide the following details to check for an existing premise-generated database or to generate a new one.")
print("If it is the first time you are generating a premise database with these specific model and pathway, the process may take a while as it needs to download the necessary IAM data and perform the transformations.\n")
print("Note: The source database (e.g., ecoinvent-3.11-cutoff) must already be imported into the Brightway project.\n")

source_db = input("Enter source database name for premise transformation (e.g., ecoinvent-3.11-cutoff): ").strip()
source_version = input("Enter source ecoinvent version for premise transformation (e.g., 3.11): ").strip()
model = input("Enter IAM model (e.g., remind): ").strip()
pathway = input("Enter pathway (e.g., SSP2-PkBudg1000): ").strip()
year_input = input("Enter year (e.g., 2030): ").strip()

if all([source_db, source_version, model, pathway, year_input]):
    year = int(year_input)
    
    # --- 2. Check for existing database ---
    target_db_found = False
    existing_db_name = ""

    print("\nChecking existing databases for a match...")
    # Iterate through existing databases and check for key components
    for db in bd.databases:
        # Check if version, model, pathway, and year appear in the database name
        if (source_version in db and 
            model in db and 
            pathway in db and 
            str(year) in db):
            
            target_db_found = True
            existing_db_name = db
            break

    # --- 3. Execute Premise (or skip) ---
    if target_db_found:
        print(f"✅ SKIPPING premise generation.")
        print(f"Found existing database matching parameters: '{existing_db_name}'")

    else:
        print(f"❌ No match found. Starting premise generation for {model} - {pathway} - {year}...")
        
        # Initialize Premise
        ei_fut = NewDatabase(
            scenarios=[{"model": model, "pathway": pathway, "year": year}],
            source_db=source_db,
            source_version=source_version, 
            key='tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo=', # Provided decryption key
        )
        
        # Run transformation
        ei_fut.update()
        
        # Write to Brightway 2.5
        ei_fut.write_db_to_brightway()
        
        print("Database creation complete.")
else:
    print("One or more inputs are empty. Please run the cell again and provide all values.")

bd.databases

In [ ]:
ei_bg = input("To store it as a variable and make it accessible to the other scripts, enter the full name of the futurized ecoinvent database you want to use as background database (e.g., ecoinvent-3.11-remind-SSP2-PkBudg1000-2030-cutoff yyyy-mm-dd):").strip()
bio_bg = input("To store it as a variable and make it accessible to the other scripts, enter the name of the biosphere database. It must be the same biosphere database used as source in the premise transformation (e.g., ecoinvent-3.11-biosphere):").strip()

ei = bd.Database(ei_bg)      # Do NOT change the name of this variable, as it is used by the linked notebooks to access the background database
bio = bd.Database(bio_bg)    # Do NOT change the name of this variable, as it is used by the linked notebooks to access the background database

# 1. Import of energy systems activity data

In [ ]:
# Import libraries 
current_dir = Path().resolve()

Libraries_path = (current_dir / "Libraries_pd.ipynb").resolve()

# Use get_ipython to run magic commands safely
get_ipython().run_line_magic('run', str(Libraries_path))

In [ ]:
file_path = generatorsConfig_path
file_path

## 1.1 Import boilers (natural gas or other blends) activity data

In [ ]:
try:
    # File Excel upload
    file_path = generatorsConfig_path
    sheet_name = "Boiler"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        # Check if Boiler exists as a Jupyter Notebook file
        if os.path.exists(boiler_path):
            print(f"📁 Found file '{boiler_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                
                get_ipython().run_line_magic('run', f'-i "{boiler_path}"')
                print("✅ Module 'Boiler' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook 'Boiler.ipynb' not found. Assigning zero matrices.")

    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")

except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
   

## 1.2 Import (solid) biomass boilers activity data

In [ ]:
try:
    # Upload the Excel sheet
    file_path = generatorsConfig_path    
    sheet_name = "Biomass boiler"
    
    # Verify is the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        # Check if Biomass boiler exists as Jupyter Notebook file
        if os.path.exists(Biomassboiler_path):
            print(f"📁 Found '{Biomassboiler_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{Biomassboiler_path}"')
                print("✅ Module 'Biomassboiler' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
        
        else:
            print("⚠️ Notebook 'Biomass boiler.ipynb' not found. Assigning zero matrices.")

    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")

except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")


## 1.3 Import fuel oil boilers activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Fuel oil boiler"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(Fueloilboiler_path):
            print(f"📁 Found file '{Fueloilboiler_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{Fueloilboiler_path}"')
                print("✅ Module 'Fueloilboiler' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook 'Fuel oil boiler.ipynb' not found. Assigning zero matrices.")

    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")

except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")

## 1.4 Import Combined Heat and Power (CHP) - Reciprocating internal combustion engines activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "CHP-Internal Combustion Engine"
    
    # Verify if the Excel sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(RICE_path):
            print(f"📁 Found file '{RICE_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{RICE_path}"')
                print("✅ Module 'CombinedHeatPower_InternalCombustionEngine' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
        else:
            print("⚠️ Notebook 'CombinedHeatPower_InternalCombustionEngine.ipynb' not found. Assigning zero matrices.")
    
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")

except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    

## 1.5 Import Combined Heat and Power (CHP) - Organic Rankine Cycle plants activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "CHP-Organic Rankine Cycle"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        #Check if ORC.ipynb exists as Notebook file
        if os.path.exists(ORC_path):
            print(f"📁 Found file '{ORC_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{ORC_path}"')
                print("✅ Module 'CombinedHeatPower_OrganicRankineCycle' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook 'CombinedHeatPower_OrganicRankineCycle.ipynb' not found. Assigning zero matrices.")
        
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")

## 1.8 Import Combined Heat and Power (CHP) - Waste-to-Energy plants activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "CHP-Waste-to-Energy"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        # Check if CombinedHeatPower_WastetoEnergy file Exists as Notebook file
        if os.path.exists(WtE_path):
            print(f"📁 Found file '{WtE_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{WtE_path}"')
                print("✅ Module 'CombinedHeatPower_WasteToEnergy' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
        else:
            print("⚠️ Notebook 'CombinedHeatPower_WasteToEnergy.ipynb' not found. Assigning zero matrices.")

    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
       

except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")


## 1.9 Import Combined Cycles activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "CHP-Combined Cycle"
    
    # Verifyif the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        # Check if Combined Cycle exists as notebook file
        if os.path.exists(CombinedCycle_path):
            print(f"📁 Found file '{CombinedCycle_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{CombinedCycle_path}"')
                print("✅ Module 'CombinedHeatPower_CombinedCycle' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook 'CombinedHeatPower_CombinedCycle.ipynb' not found. Assigning zero matrices.")
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")

except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    

## 1.10 Import Combined Heat and Power (CHP) - Gas turbinies activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "CHP-Turbogas"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        # Check if Turbogas exists as notebook
        if os.path.exists(GasTurbine_path):
            print(f"📁 Found file '{GasTurbine_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{GasTurbine_path}"')
                print("✅ Module 'CombinedHeatPower_GasTurbine' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
               
        else:
            print("⚠️ Notebook 'CombinedHeatPower_GasTurbine.ipynb' not found. Assigning zero matrices.")
        

    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")

except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    

## 1.11 Import heat recoveries activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Heat recovery"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(HeatRecovery_path):
            print(f"📁 Found file '{HeatRecovery_path}'attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{HeatRecovery_path}"')
                print("✅ Module 'HeatRecovery.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()

        else:
            print("⚠️ Notebook 'HeatRecovery.ipynb' not found. Assigning zero matrices.")
           
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")


except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")


## 1.13 Import solar thermals activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Solar thermal system"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(SolarThermal_path):
            print(f"📁 Found file '{SolarThermal_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{SolarThermal_path}"')
                print("✅ Module 'SolarThermal.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook'Solar thermal.ipynb'  not found. Assigning zero matrices.")
                       

    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    

## 1.14 Import electric heat pumps (HPs) activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Electric Heat Pump"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(EHP_path):
            print(f"📁 Found file '{EHP_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{EHP_path}"')
                print("✅ Module 'ElectricHeatPump.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
               
        else:
            print("⚠️ Notebook'ElectricHeatPump.ipynb'  not found. Assigning zero matrices.")
            
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")


## 1.15 Import Gas absorption heat pump activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Gas Absorption Heat Pump"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(GAHP_path):
            print(f"📁 Found file '{GAHP_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{GAHP_path}"')
                print("✅ Module 'GasAbsorptionHeatPump.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook'GasAbsorptionHeatPump.ipynb'  not found. Assigning zero matrices.")

    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")


## 1.16 Import Gas Engine heat pump

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Gas Engine Heat Pump"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(GEHP_path):
            print(f"📁 Found file '{GEHP_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{GEHP_path}"')
                print("✅ Module 'GasEngineHeatPump.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook'GasEngineHeatPump.ipynb'  not found. Assigning zero matrices.")
           
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    

## 1.17 Import thermal concrete storage activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Concrete thermal storage"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(ThermalConcreteStorage_path):
            print(f"📁 Found file '{ThermalConcreteStorage_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{ThermalConcreteStorage_path}"')
                print("✅ Module 'ThermalConcreteStorage.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook'ThermalConcreteStorage.ipynb'  not found. Assigning zero matrices.")
            
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    

## 1.18 Import steel thermal storage activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Steel thermal storage"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(ThermalSteelStorage_path):
            print(f"📁 Found file '{ThermalSteelStorage_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{ThermalSteelStorage_path}"')
                print("✅ Module 'ThermalSteelStorage.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook'ThermalSteelStorage.ipynb' not found. Assigning zero matrices.")
            
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
   

## 1.19 Import geothermal activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Geothermal plant"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(Geothermal_path):
            print(f"📁 Found file '{Geothermal_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{Geothermal_path}"')
                print("✅ Module 'GeothermalPlant.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook'GeothermalPlant.ipynb'  not found. Assigning zero matrices.")
            
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")

## 1.20 Import Cooling Absorption chiller activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Cooling - Absorption chiller"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(AbsorptionChiller_path):
            print(f"📁 Found file '{AbsorptionChiller_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{AbsorptionChiller_path}"')
                print("✅ Module 'CoolingAbsorptionChiller.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
        else:
            print("⚠️ Notebook 'CoolingAbsorptionChiller.ipynb' not found. Assigning zero matrices.")
           
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
       
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    

## 1.21 Import Cooling Compression chiller activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Cooling - Compression chiller"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(CompressionChiller_path):
            print(f"📁 Found file '{CompressionChiller_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{CompressionChiller_path}"')
                print("✅ Module 'CoolingCompressionChiller.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                
        else:
            print("⚠️ Notebook 'CoolingCompressionChiller.ipynb' not found. Assigning zero matrices.")
            
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
                 
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    

## 1.22 Import Cooling dry cooler activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Cooling - Dry cooler"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(DryCooler_path):
            print(f"📁 Found file '{DryCooler_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{DryCooler_path}"')
                print("✅ Module 'CoolingDryCooler.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
                      
        else:
            print("⚠️ Notebook 'CoolingDryCooler.ipynb' not found. Assigning zero matrices.")
            
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")

## 1.23 Import Cooling Evaporative tower activity data

In [ ]:
try:
    # Upload the Excel file
    file_path = generatorsConfig_path    
    sheet_name = "Cooling - Cooling tower"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        if os.path.exists(CoolingTower_path):
            print(f"📁 Found file '{CoolingTower_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{CoolingTower_path}"')
                print("✅ Module 'CoolingEvaporativeTower.ipynb' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
        else:
            print("⚠️ Notebook 'CoolingEvaporativeTower.ipynb' not found. Assigning zero matrices.")
            
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")
        
except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")

## 1.24 Import Appliances activity data

In [ ]:
try:
    # File Excel upload
    file_path = generatorsConfig_path
    sheet_name = "Appliances demand"
    
    # Verify if the sheet exists
    xls = pd.ExcelFile(file_path)
    if sheet_name in xls.sheet_names:
        print(f"✅ The sheet '{sheet_name}' exists, proceeding with import.")
        
        # Check if Appliances exists as a Jupyter Notebook file
        if os.path.exists(Appliances_path):
            print(f"📁 Found file '{Appliances_path}', attempting to execute it.")
            try:
                RUN_DIR = Path().resolve()
                
                print(f"🏃 Main execution from: {RUN_DIR}")
                get_ipython().run_line_magic('run', f'-i "{Appliances_path}"')
                print("✅ Module 'Appliances' executed successfully!")
            except Exception as e:
                print("⚠️ Error occurred while executing the notebook!")
                traceback.print_exc()
        else:
            print("⚠️ Notebook 'Appliances.ipynb' not found. Assigning zero matrices.")
    else:
        print(f"⚠️ Sheet '{sheet_name}' does not exist. Assigning zero matrices.")

except FileNotFoundError:
    print("❌ Excel file not found! Assigning zero matrices for safety.")
    Appliances_usephaseelectricity = Appliances_usephaseelectricityPV = 0

# 3. Import ecoinvent datasets

In [ ]:
# Datasets are imported from ei and bio in ImportEcoinventDatasets_pd.ipynb and stored as variables to be used in the linked notebooks.

In [ ]:
current_dir = Path().resolve()
EcoinventDatasets_path = (current_dir / "ImportEcoinventDatasets_pd.ipynb").resolve()

get_ipython().run_line_magic('run', f'"{EcoinventDatasets_path}"')
print("✅ Module 'EcoinventDatasets' executed successfully!")

In [ ]:
bd.databases

In [ ]:
# LCA "models" are built in brightway as trees. A database is a collection of one or more trees.
scenario_db_name = input("Enter the name of the database in which you want to store the model of your neighbourhood scenario: ").strip()

if scenario_db_name in bd.databases:
    print(f"⚠️ Database '{scenario_db_name}' already exists. Deleting it to avoid conflicts. It will be recreated as an empty database to store the new scenario model.")
    del bd.databases[scenario_db_name]
else:
    print(f'{scenario_db_name} is not present in the project')

In [ ]:
print(f"Creating and registering database '{scenario_db_name}' for the neighbourhood scenario model...")
scenario_db = bd.Database(scenario_db_name)
scenario_db.register()
bd.databases

# 5. Boilers (natural gas or other blends)

## 5.1 Activities

In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_Boiler_{bd_name}',
        'name': f'A13_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            A13_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_Boiler_{bd_name}',
        'name': f'A4_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            A4_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_Boiler_{bd_name}',
        'name': f'A5_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            A5_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_Boiler_{bd_name}',
        'name': f'B2_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            B2_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_Boiler_{bd_name}',
        'name': f'B6_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            B6_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")


In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_Boiler_{bd_name}',
        'name': f'C2_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            C2_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_Boiler_{bd_name}',
        'name': f'C3_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            C3_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_Boiler_{bd_name}',
        'name': f'C4_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            C4_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:

# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_Boiler_{bd_name}',
        'name': f'D1_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            D1_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
# Check for the sheet BEFORE starting the heavy work
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_Boiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in Boiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_Boiler_{bd_name}',
        'name': f'D3_Boiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_Boiler_acts[bd_name] = scenario_db.new_activity(**data)
            D3_Boiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 5.2 Exchanges

### 5.2.1 Modules A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_Boiler_exchange_configs = [
        (EF_StainlessSteel.key, Boiler_stainlessSteel, 'technosphere'),
        (EF_steel.key, Boiler_steel, 'technosphere'),
        (EF_Aluminium.key, Boiler_aluminium, 'technosphere'),
        (EF_Elastomer.key, Boiler_elastomer, 'technosphere'),
        (EF_PVC.key, Boiler_PVC, 'technosphere'),
        (EF_refrigerantgas.key, Boiler_refrigerantgas, 'technosphere'),
        (EF_Copper.key, Boiler_copper, 'technosphere'),
        (EF_Electronic.key, Boiler_electronic, 'technosphere'),
        (EF_Welding.key, Boiler_welding, 'technosphere'),
        (EF_waterconsumption.key, Boiler_waterconsumption, 'technosphere'),
        (EF_waterair.key, Boiler_emission_water_air, 'biosphere'),
        (EF_waterwater.key, Boiler_emission_water_water, 'biosphere'),
        (EF_Wastewater_treatment.key, Boiler_wastewater_treatment, 'technosphere'),
        (EF_Lubricating_oil.key, Boiler_lubricating_oil, 'technosphere'),
        (EF_Electricity.key, Boiler_A13electricity, 'technosphere'),
        (EF_Heat.key, Boiler_heat, 'technosphere'),
        (EF_Hazardous.key, Boiler_hazardous, 'technosphere')
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in A13_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_Boiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_Boiler = A13_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Boiler_BdName:
        A13_Boiler_acts[bd_name].new_exchange(
            input=A13_Boiler_acts[bd_name].key, 
            output=A13_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_Boiler.lci()

    mlca_A13_Boiler.lcia()
    mlca_A13_Boiler.scores

    dfresults_A13_Boiler = pd.DataFrame.from_dict(mlca_A13_Boiler.scores, orient='index')
    dfresults_A13_Boiler.index = pd.MultiIndex.from_tuples(dfresults_A13_Boiler.index, names=['Column', 'Row'])
    dfresults_A13_Boiler = dfresults_A13_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_Boiler.columns = dfresults_A13_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_Boiler = dfresults_A13_Boiler.mul(SFBoiler, axis=0)

In [ ]:
dfresults_A13_Boiler

### 5.2.2 Module A4

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_Boiler_exchange_configs = [
        (EF_Truk_16_32.key, Boiler_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, Boiler_Light_Truk, 'technosphere'),
        
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in A4_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_Boiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_Boiler = A4_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Boiler_BdName:
        A4_Boiler_acts[bd_name].new_exchange(
            input=A4_Boiler_acts[bd_name].key, 
            output=A4_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_Boiler.lci()

    mlca_A4_Boiler.lcia()
    mlca_A4_Boiler.scores

    dfresults_A4_Boiler = pd.DataFrame.from_dict(mlca_A4_Boiler.scores, orient='index')
    dfresults_A4_Boiler.index = pd.MultiIndex.from_tuples(dfresults_A4_Boiler.index, names=['Column', 'Row'])
    dfresults_A4_Boiler = dfresults_A4_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_Boiler.columns = dfresults_A4_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_Boiler = dfresults_A4_Boiler.mul(SFBoiler, axis=0)

### 5.2.3 Module A5

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    for bd_name in Boiler_BdName:
        A5_Boiler_acts[bd_name].new_exchange(
            input=A5_Boiler_acts[bd_name].key, 
            output=A5_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_Boiler.lci()

    mlca_A5_Boiler.lcia()
    mlca_A5_Boiler.scores

    dfresults_A5_Boiler = pd.DataFrame.from_dict(mlca_A5_Boiler.scores, orient='index')
    dfresults_A5_Boiler.index = pd.MultiIndex.from_tuples(dfresults_A5_Boiler.index, names=['Column', 'Row'])
    dfresults_A5_Boiler = dfresults_A5_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_Boiler.columns = dfresults_A5_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_Boiler = dfresults_A5_Boiler.mul(SFBoiler, axis=0)

### 5.2.4 Module B2

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_Boiler_exchange_configs = [
        (EF_Light_Truk.key, Boiler_Light_Truk_B2, 'technosphere'),
        
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in B2_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_Boiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_Boiler = B2_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
        
    for bd_name in Boiler_BdName:
        B2_Boiler_acts[bd_name].new_exchange(
            input=B2_Boiler_acts[bd_name].key, 
            output=B2_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_Boiler.lci()

    mlca_B2_Boiler.lcia()
    mlca_B2_Boiler.scores

    dfresults_B2_Boiler = pd.DataFrame.from_dict(mlca_B2_Boiler.scores, orient='index')
    dfresults_B2_Boiler.index = pd.MultiIndex.from_tuples(dfresults_B2_Boiler.index, names=['Column', 'Row'])
    dfresults_B2_Boiler = dfresults_B2_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_Boiler.columns = dfresults_B2_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_Boiler = dfresults_B2_Boiler.mul(SFBoiler, axis=0)

### 5.2.5 Module B6

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    
    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_Boiler_exchange_configs = [
        (EF_energyvectorNG.key, Boiler_EnergyVectorNG, 'technosphere'),
        (EF_energyvectorBiom.key, Boiler_EnergyVectorBiomethane, 'technosphere'),
        (EF_energyvectorHydrogen.key, Boiler_EnergyVectorHydrogen, 'technosphere'),
        (EF_energyvectorBiom.key, Boiler_EnergyVectorBiomethaneInput, 'technosphere'),
        (EF_LPG.key, Boiler_EnergyVectorLPGinput, 'technosphere'),
        (EF_energyvectorHydrogen.key, Boiler_EnergyVectorHydrogeninput, 'technosphere'),
        (EF_Acetaldehyde.key, Boiler_Acetaldehyde, 'biosphere'),
        (EF_Aceticacid.key, Boiler_AceticAcid, 'biosphere'),
        (EF_Benzene.key, Boiler_Benzene, 'biosphere'),
        (EF_Benzoapyrene.key, Boiler_Benzoapyrene, 'biosphere'),
        (EF_Buthane.key, Boiler_Butane, 'biosphere'),
        (EF_CarbondioxideFossil.key, Boiler_CarbonDioxideFossil, 'biosphere'),
        (EF_CarbonmonoxideFossil.key, Boiler_CarbonMonoxideFossil, 'biosphere'),
        (EF_DinitrogenMonoxide.key, Boiler_DinitrogenMonoxide, 'biosphere'),
        (EF_Formaldehyde.key, Boiler_Formaldehyde, 'biosphere'),
        (EF_Mercury.key, Boiler_Mercury, 'biosphere'),
        (EF_MethaneFossil.key, Boiler_MethaneFossil, 'biosphere'),
        (EF_NitrogenOxides.key, Boiler_NitrogenOxides, 'biosphere'),
        (EF_PAH.key, Boiler_PAH, 'biosphere'),
        (EF_Particulates025.key, Boiler_Particulates, 'biosphere'),
        (EF_Pentane.key, Boiler_Pentane, 'biosphere'),
        (EF_PropaneB6.key, Boiler_Propane, 'biosphere'),
        (EF_PropionicAcid.key, Boiler_PropionicAcid, 'biosphere'),
        (EF_SulfurDioxide.key, Boiler_SulfurDioxide, 'biosphere'),
        (EF_Toluene.key, Boiler_Toluene, 'biosphere'),
        (EF_Nitrate.key, Boiler_Nitrate, 'biosphere'),
        (EF_Nitrite.key, Boiler_Nitrite, 'biosphere'),
        (EF_Sulfate.key, Boiler_Sulfate, 'biosphere'),
        (EF_Sulfite.key, Boiler_Sulfite, 'biosphere'),
        (EF_WatervaporH.key, Boiler_WatervaporH, 'biosphere'),
        (EF_NitrogenOxides.key, Boiler_NitrogenH, 'biosphere'),
        (EF_OxygenH.key, Boiler_OxygenH, 'biosphere'),
        (EF_NOXH.key, Boiler_NOXH, 'biosphere'),
        (EF_WatercondensateH.key, Boiler_WatercondensateH, 'biosphere'),
        (EF_electricity.key, Boiler_electricity, 'technosphere'),
        (EF_electricityPV.key, Boiler_electricityPV, 'technosphere'),
        # (EF_sun.key, Boiler_electricityRECSPV, 'technosphere'),
        # (EF_hydro.key, Boiler_electricityRECSHydro, 'technosphere'),
        # (EF_wind.key, Boiler_electricityRECSEolic, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in B6_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_Boiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_Boiler = B6_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    for bd_name in Boiler_BdName:
        B6_Boiler_acts[bd_name].new_exchange(
            input=B6_Boiler_acts[bd_name].key, 
            output=B6_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_Boiler.lci()

    mlca_B6_Boiler.lcia()
    mlca_B6_Boiler.scores

    dfresults_B6_Boiler = pd.DataFrame.from_dict(mlca_B6_Boiler.scores, orient='index')
    dfresults_B6_Boiler.index = pd.MultiIndex.from_tuples(dfresults_B6_Boiler.index, names=['Column', 'Row'])
    dfresults_B6_Boiler = dfresults_B6_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_Boiler.columns = dfresults_B6_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_Boiler.rename(columns=method_lookup, inplace=True)

### 5.2.6 Module C2

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_Boiler_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, Boiler_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in C2_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_Boiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_Boiler = C2_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
        
    for bd_name in Boiler_BdName:
        C2_Boiler_acts[bd_name].new_exchange(
            input=C2_Boiler_acts[bd_name].key, 
            output=C2_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_Boiler.lci()

    mlca_C2_Boiler.lcia()
    mlca_C2_Boiler.scores

    dfresults_C2_Boiler = pd.DataFrame.from_dict(mlca_C2_Boiler.scores, orient='index')
    dfresults_C2_Boiler.index = pd.MultiIndex.from_tuples(dfresults_C2_Boiler.index, names=['Column', 'Row'])
    dfresults_C2_Boiler = dfresults_C2_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_Boiler.columns = dfresults_C2_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_Boiler = dfresults_C2_Boiler.mul(SFBoiler, axis=0)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_Boiler = dfresults_C2_Boiler.mul(SFBoiler, axis=0)

### 5.2.7 Module C3

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_Boiler_exchange_configs = [
        (EF_EOL_metal.key, Boiler_EOL_stainlessSteel, 'technosphere'),
        (EF_EOL_metal.key, Boiler_EOL_steel, 'technosphere'),
        (EF_EOL_Aluminium.key, Boiler_EOL_aluminium, 'technosphere'),
        (EF_EOL_Elastomer.key, Boiler_EOL_elastomer, 'technosphere'),
        (EF_EOL_PVC.key, Boiler_EOL_PVC, 'technosphere'),
        (EF_EOL_refrigerantgas.key, Boiler_EOL_refrigerantgas, 'technosphere'),
        (EF_EOL_metal.key, Boiler_EOL_copper, 'technosphere'),
        (EF_EOL_electronics.key, Boiler_EOL_electronic, 'technosphere'),
        (EF_EOL_inert.key, Boiler_EOL_lubricating_oil, 'technosphere'),
        (EF_EOL_inert.key, Boiler_EOL_hazardous, 'technosphere')
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in C3_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_Boiler_exchange_list:
                # Create a new exchange for each item in the list
            new_exc_C3_Boiler = C3_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
        
    for bd_name in Boiler_BdName:
        C3_Boiler_acts[bd_name].new_exchange(
            input=C3_Boiler_acts[bd_name].key, 
            output=C3_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()


In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_Boiler.lci()

    mlca_C3_Boiler.lcia()
    mlca_C3_Boiler.scores

    dfresults_C3_Boiler = pd.DataFrame.from_dict(mlca_C3_Boiler.scores, orient='index')
    dfresults_C3_Boiler.index = pd.MultiIndex.from_tuples(dfresults_C3_Boiler.index, names=['Column', 'Row'])
    dfresults_C3_Boiler = dfresults_C3_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_Boiler.columns = dfresults_C3_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_Boiler = dfresults_C3_Boiler.mul(SFBoiler, axis=0)

### 5.2.8 Module C4

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_Boiler_exchange_configs = [
        (EF_EOL_landfill_metal.key, Boiler_EOL_landfill_stainlessSteel, 'technosphere'),
        (EF_EOL_landfill_metal.key, Boiler_EOL_landfill_steel, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, Boiler_EOL_landfill_aluminium, 'technosphere'),
        (EF_EOL_inc_elastomer.key, Boiler_EOL_inc_elastomer, 'technosphere'),
        (EF_EOL_landfill_plastic.key, Boiler_EOL_landfill_elastomer, 'technosphere'),
        (EF_EOL_inc_plastic.key, Boiler_EOL_inc_PVC, 'technosphere'),
        (EF_EOL_landfill_plastic.key, Boiler_EOL_landfill_PVC, 'technosphere'),
        (EF_EOL_landfill_refrigerantgas.key, Boiler_EOL_landfill_refrigerantgas, 'technosphere'),
        (EF_EOL_landfill_metal.key, Boiler_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_inc_electronic.key, Boiler_EOL_inc_electronic, 'technosphere'),
        (EF_EOL_landfill_electronic.key, Boiler_EOL_landfill_electronic, 'technosphere'),
        (EF_EOL_inc_lubricatingoil.key, Boiler_EOL_inc_lubricatingoil, 'technosphere'),
        (EF_EOL_inc_hazardouswaste.key, Boiler_EOL_inc_hazardouswaste, 'technosphere')
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in C4_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_Boiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_Boiler = C4_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
        
    for bd_name in Boiler_BdName:
        C4_Boiler_acts[bd_name].new_exchange(
            input=C4_Boiler_acts[bd_name].key, 
            output=C4_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_Boiler.lci()

    mlca_C4_Boiler.lcia()
    mlca_C4_Boiler.scores

    dfresults_C4_Boiler = pd.DataFrame.from_dict(mlca_C4_Boiler.scores, orient='index')
    dfresults_C4_Boiler.index = pd.MultiIndex.from_tuples(dfresults_C4_Boiler.index, names=['Column', 'Row'])
    dfresults_C4_Boiler = dfresults_C4_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_Boiler.columns = dfresults_C4_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_Boiler = dfresults_C4_Boiler.mul(SFBoiler, axis=0)

### 5.2.9 Module B4

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_Boiler = pd.DataFrame(index=dfresults_A13_Boiler.index)  # Start with the index from A13 for consistency   
    dfresults_B4_Boiler = (
        dfresults_A13_Boiler.add(dfresults_A4_Boiler, fill_value=0)
        .add(dfresults_C3_Boiler, fill_value=0)
        .add(dfresults_C4_Boiler, fill_value=0)
    ).mul(BoilerNumberReplacements, axis=0)

### 5.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_Boiler_exchange_configs = [
        (EF_recycling_steel.key, Boiler_EOL_recycling_stainlessSteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, Boiler_EOL_primary_stainlessSteel, 'technosphere'),
        (EF_recycling_steel.key, Boiler_EOL_recycling_steel, 'technosphere'),
        (EF_primary_steel.key, Boiler_EOL_primary_steel, 'technosphere'),
        (EF_recycling_aluminium.key, Boiler_EOL_recycling_aluminium, 'technosphere'),
        (EF_primary_aluminium.key, Boiler_EOL_primary_aluminium, 'technosphere'),
        (EF_recycling_copper.key, Boiler_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, Boiler_EOL_primary_copper, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in D1_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_Boiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_Boiler = D1_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
        
    for bd_name in Boiler_BdName:
        D1_Boiler_acts[bd_name].new_exchange(
            input=D1_Boiler_acts[bd_name].key, 
            output=D1_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_Boiler.lci()

    mlca_D1_Boiler.lcia()
    mlca_D1_Boiler.scores

    dfresults_D1_Boiler = pd.DataFrame.from_dict(mlca_D1_Boiler.scores, orient='index')
    dfresults_D1_Boiler.index = pd.MultiIndex.from_tuples(dfresults_D1_Boiler.index, names=['Column', 'Row'])
    dfresults_D1_Boiler = dfresults_D1_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_Boiler.columns = dfresults_D1_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_Boiler = dfresults_D1_Boiler.mul(SFBoiler, axis=0)

### 5.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_Boiler = dfresults_D1_Boiler.add(dfresults_D1_Boiler.mul(BoilerNumberReplacements, axis=0))

### 5.2.12 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_Boiler_exchange_configs = [
        (EF_heat_energyrecovery.key, Boiler_EOL_energyrecoveryheat_elastomer, 'technosphere'),
        (EF_electricity_energyrecovery.key, Boiler_EOL_energyrecoveryele_elastomer, 'technosphere'),
        (EF_heat_energyrecovery.key, Boiler_EOL_energyrecoveryheat_PVC, 'technosphere'),
        (EF_electricity_energyrecovery.key, Boiler_EOL_energyrecoveryele_PVC, 'technosphere'),
        (EF_heat_energyrecovery.key, Boiler_EOL_energyrecoveryheat_electronics, 'technosphere'),
        (EF_electricity_energyrecovery.key, Boiler_EOL_energyrecoveryele_electronics, 'technosphere'),
        (EF_heat_energyrecovery.key, Boiler_EOL_energyrecoveryheat_lubricatingoil, 'technosphere'),
        (EF_electricity_energyrecovery.key, Boiler_EOL_energyrecoveryele_lubricatingoil, 'technosphere'),
        (EF_heat_energyrecovery.key, Boiler_EOL_energyrecoveryheat_hazardouswaste, 'technosphere'),
        (EF_electricity_energyrecovery.key, Boiler_EOL_energyrecoveryele_hazardouswaste, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Boiler_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_Boiler_exchange_list = []
        
        for input_key, data_array, exc_type in D3_Boiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_Boiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_Boiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_Boiler = D3_Boiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_Boiler.save()

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
        
    for bd_name in Boiler_BdName:
        D3_Boiler_acts[bd_name].new_exchange(
            input=D3_Boiler_acts[bd_name].key, 
            output=D3_Boiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_Boiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_Boiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_Boiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Boiler_BdName, Boiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_Boiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_Boiler.lci()

    mlca_D3_Boiler.lcia()
    mlca_D3_Boiler.scores

    dfresults_D3_Boiler = pd.DataFrame.from_dict(mlca_D3_Boiler.scores, orient='index')
    dfresults_D3_Boiler.index = pd.MultiIndex.from_tuples(dfresults_D3_Boiler.index, names=['Column', 'Row'])
    dfresults_D3_Boiler = dfresults_D3_Boiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_Boiler.columns = dfresults_D3_Boiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_Boiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_Boiler = dfresults_D3_Boiler.mul(SFBoiler, axis=0)

### 5.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "Boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_Boiler = dfresults_D3_Boiler.add(dfresults_D3_Boiler.mul(BoilerNumberReplacements, axis=0))

# 6. Biomass (solid) boilers

## 6.1 Activities

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_Biomassboiler_{bd_name}',
        'name': f'A13_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            A13_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_Biomassboiler_{bd_name}',
        'name': f'A4_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            A4_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_Biomassboiler_{bd_name}',
        'name': f'A5_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            A5_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_Biomassboiler_{bd_name}',
        'name': f'B2_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            B2_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_Biomassboiler_{bd_name}',
        'name': f'B6_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            B6_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_Biomassboiler_{bd_name}',
        'name': f'C2_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            C2_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_Biomassboiler_{bd_name}',
        'name': f'C3_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            C3_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_Biomassboiler_{bd_name}',
        'name': f'C4_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            C4_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_Biomassboiler_{bd_name}',
        'name': f'D1_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            D1_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Biomass boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_Biomassboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in BiomassBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_Biomassboiler_{bd_name}',
        'name': f'D3_Biomassboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_Biomassboiler_acts[bd_name] = scenario_db.new_activity(**data)
            D3_Biomassboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 6.2 Exchanges

### 6.2.1 Modules A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_Biomassboiler_exchange_configs = [
        (EF_polyethylene.key, np.atleast_1d(Biomassboiler_HDPE), 'technosphere'),
        (EF_alkydpaint.key, np.atleast_1d(Biomassboiler_alkydpaint), 'technosphere'),
        (EF_Aluminium.key, np.atleast_1d(Biomassboiler_aluminium), 'technosphere'),
        (EF_StainlessSteel.key, np.atleast_1d(Biomassboiler_chromiumsteel), 'technosphere'),
        (EF_chamotte.key, np.atleast_1d(Biomassboiler_claybrick), 'technosphere'),
        (EF_castiron.key, np.atleast_1d(Biomassboiler_castiron), 'technosphere'),
        (EF_ceramictile.key, np.atleast_1d(Biomassboiler_ceramictile), 'technosphere'),
        (EF_concreteCH.key, np.atleast_1d(Biomassboiler_concrete), 'technosphere'),
        (EF_Copper.key, np.atleast_1d(Biomassboiler_copper), 'technosphere'),
        (EF_Electricity.key, np.atleast_1d(Biomassboiler_electricityA13), 'technosphere'),
        (EF_Electronic.key, np.atleast_1d(Biomassboiler_electronics), 'technosphere'),
        (EF_flatglasscoated.key, np.atleast_1d(Biomassboiler_flatglasscoated), 'technosphere'),
        (EF_Heat.key, np.atleast_1d(Biomassboiler_heat), 'technosphere'),
        (EF_steel.key, np.atleast_1d(Biomassboiler_lowalloyedsteel), 'technosphere'),
        (EF_Lubricating_oil.key, np.atleast_1d(Biomassboiler_lubricatingoil), 'technosphere'),
        (EF_polyurethane.key, np.atleast_1d(Biomassboiler_polystyrene), 'technosphere'),
        (EF_sheetrollingsteel.key, np.atleast_1d(Biomassboiler_sheetrollingsteel), 'technosphere'),
        (EF_mineralwool.key, np.atleast_1d(Biomassboiler_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in A13_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in A13_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_Biomassboiler = A13_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        A13_Biomassboiler_acts[bd_name].new_exchange(
            input=A13_Biomassboiler_acts[bd_name].key, 
            output=A13_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_Biomassboiler.lci()

    mlca_A13_Biomassboiler.lcia()
    mlca_A13_Biomassboiler.scores

    dfresults_A13_Biomassboiler = pd.DataFrame.from_dict(mlca_A13_Biomassboiler.scores, orient='index')
    dfresults_A13_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_A13_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_A13_Biomassboiler = dfresults_A13_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_Biomassboiler.columns = dfresults_A13_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_Biomassboiler = dfresults_A13_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_Biomassboiler_exchange_configs = [
        (EF_Truk_16_32.key, Biomassboiler_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, Biomassboiler_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in A4_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_Biomassboiler = A4_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        A4_Biomassboiler_acts[bd_name].new_exchange(
            input=A4_Biomassboiler_acts[bd_name].key, 
            output=A4_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_Biomassboiler.lci()

    mlca_A4_Biomassboiler.lcia()
    mlca_A4_Biomassboiler.scores

    dfresults_A4_Biomassboiler = pd.DataFrame.from_dict(mlca_A4_Biomassboiler.scores, orient='index')
    dfresults_A4_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_A4_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_A4_Biomassboiler = dfresults_A4_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_Biomassboiler.columns = dfresults_A4_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_Biomassboiler = dfresults_A4_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.3 Module A5

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        A5_Biomassboiler_acts[bd_name].new_exchange(
            input=A5_Biomassboiler_acts[bd_name].key, 
            output=A5_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_Biomassboiler.lci()

    mlca_A5_Biomassboiler.lcia()
    mlca_A5_Biomassboiler.scores

    dfresults_A5_Biomassboiler = pd.DataFrame.from_dict(mlca_A5_Biomassboiler.scores, orient='index')
    dfresults_A5_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_A5_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_A5_Biomassboiler = dfresults_A5_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_Biomassboiler.columns = dfresults_A5_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_Biomassboiler = dfresults_A5_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_Biomassboiler_exchange_configs = [
        (EF_LightTruk.key, Biomassboiler_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in B2_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_Biomassboiler = B2_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        B2_Biomassboiler_acts[bd_name].new_exchange(
            input=B2_Biomassboiler_acts[bd_name].key, 
            output=B2_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_Biomassboiler.lci()

    mlca_B2_Biomassboiler.lcia()
    mlca_B2_Biomassboiler.scores

    dfresults_B2_Biomassboiler = pd.DataFrame.from_dict(mlca_B2_Biomassboiler.scores, orient='index')
    dfresults_B2_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_B2_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_B2_Biomassboiler = dfresults_B2_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_Biomassboiler.columns = dfresults_B2_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_Biomassboiler = dfresults_B2_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_Biomassboiler_exchange_configs = [
        (EF_Wood.key, Biomassboiler_Energyvector, 'technosphere'),
        (EF_Acetaldehyde.key, Biomassboiler_acetaldehyde, 'biosphere'),
        (EF_Ammonia.key, Biomassboiler_ammonia, 'biosphere'),
        (EF_Benzene.key, Biomassboiler_benzene, 'biosphere'),
        (EF_Benzoapyrene.key, Biomassboiler_benzoapyrene, 'biosphere'),
        (EF_Bromine.key, Biomassboiler_bromine, 'biosphere'),
        (EF_Cadmium.key, Biomassboiler_cadmium, 'biosphere'),
        (EF_Calcium.key, Biomassboiler_calcium, 'biosphere'),
        (EF_CarbondioxideNonFossil.key, Biomassboiler_carbondioxide_biogenic, 'biosphere'),
        (EF_CarbonmonoxideNonFossil.key, Biomassboiler_carbonmonoxide_biogenic, 'biosphere'),
        (EF_ChromiumVI.key, Biomassboiler_chromiumVI, 'biosphere'),
        (EF_CopperIon.key, Biomassboiler_copperB6, 'biosphere'),
        (EF_DinitrogenMonoxide.key, Biomassboiler_dinitrogenmonoxide, 'biosphere'),
        (EF_Dioxin.key, Biomassboiler_dioxine, 'biosphere'),
        (EF_Fluorine.key, Biomassboiler_fluorine, 'biosphere'),
        (EF_Formaldehyde.key, Biomassboiler_formaldeide, 'biosphere'),
        (EF_HydrocarbonsUnsaturated.key, Biomassboiler_hydrocarbons_aliphatic_unsaturated, 'biosphere'),
        (EF_LeadB6.key, Biomassboiler_lead, 'biosphere'),
        (EF_Magnesium.key, Biomassboiler_magnesium, 'biosphere'),
        (EF_Manganese.key, Biomassboiler_manganese, 'biosphere'),
        (EF_Mercury.key, Biomassboiler_mercury, 'biosphere'),
        (EF_Mxylene.key, Biomassboiler_mxylene, 'biosphere'),
        (EF_NickelB6.key, Biomassboiler_nickel, 'biosphere'),
        (EF_NitrogenOxides.key, Biomassboiler_nitrogenoxides, 'biosphere'),
        (EF_NMVOC.key, Biomassboiler_NMVOC, 'biosphere'),
        (EF_PAH.key, Biomassboiler_PAH, 'biosphere'),
        (EF_Particulates025.key, Biomassboiler_particulates_25, 'biosphere'),
        (EF_Particulates100.key, Biomassboiler_particulates_10, 'biosphere'),
        (EF_Phosphorus.key, Biomassboiler_posphorus, 'biosphere'),
        (EF_Potassium.key, Biomassboiler_potassium, 'biosphere'),
        (EF_Sodium.key, Biomassboiler_sodium, 'biosphere'),
        (EF_SulfurDioxide.key, Biomassboiler_sulfurdioxide, 'biosphere'),
        (EF_Toluene.key, Biomassboiler_toluene, 'biosphere'),
        (EF_WaterB6.key, Biomassboiler_water, 'biosphere'),
        (EF_Zinc.key, Biomassboiler_zinc, 'biosphere'),
        (EF_electricity.key, Biomassboiler_electricityB6, 'technosphere'),
        (EF_electricityPV.key, Biomassboiler_electricityB6PV, 'technosphere'),
        # (EF_sun.key, Biomassboiler_electricityB6RECSPV, 'technosphere'),
        # (EF_hydro.key, Biomassboiler_electricityB6RECSHydro, 'technosphere'),
        # (EF_wind.key, Biomassboiler_electricityB6RECSEolic, 'technosphere')
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in B6_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_Biomassboiler = B6_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        B6_Biomassboiler_acts[bd_name].new_exchange(
            input=B6_Biomassboiler_acts[bd_name].key, 
            output=B6_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_Biomassboiler.lci()

    mlca_B6_Biomassboiler.lcia()
    mlca_B6_Biomassboiler.scores

    dfresults_B6_Biomassboiler = pd.DataFrame.from_dict(mlca_B6_Biomassboiler.scores, orient='index')
    dfresults_B6_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_B6_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_B6_Biomassboiler = dfresults_B6_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_Biomassboiler.columns = dfresults_B6_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_Biomassboiler.rename(columns=method_lookup, inplace=True)

### 6.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_Biomassboiler_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, Biomassboiler_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in C2_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in C2_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_Biomassboiler = C2_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        C2_Biomassboiler_acts[bd_name].new_exchange(
            input=C2_Biomassboiler_acts[bd_name].key, 
            output=C2_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_Biomassboiler.lci()

    mlca_C2_Biomassboiler.lcia()
    mlca_C2_Biomassboiler.scores

    dfresults_C2_Biomassboiler = pd.DataFrame.from_dict(mlca_C2_Biomassboiler.scores, orient='index')
    dfresults_C2_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_C2_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_C2_Biomassboiler = dfresults_C2_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_Biomassboiler.columns = dfresults_C2_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_Biomassboiler = dfresults_C2_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_Biomassboiler_exchange_configs = [
        (EF_EOL_plastic.key, np.atleast_1d(Biomassboiler_EOL_HDPE), 'technosphere'),
        (EF_EOL_alkydpaint.key, np.atleast_1d(Biomassboiler_EOL_alkydpaint), 'technosphere'),
        (EF_EOL_Aluminium.key, np.atleast_1d(Biomassboiler_EOL_aluminium), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Biomassboiler_EOL_chromiumsteel), 'technosphere'),
        (EF_EOL_claybrick.key, np.atleast_1d(Biomassboiler_EOL_claybrick), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Biomassboiler_EOL_castiron), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Biomassboiler_EOL_ceramictile), 'technosphere'),
        (EF_EOL_concrete.key, np.atleast_1d(Biomassboiler_EOL_concrete), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Biomassboiler_EOL_copper), 'technosphere'),
        (EF_EOL_electronics.key, np.atleast_1d(Biomassboiler_EOL_electronics), 'technosphere'),
        (EF_EOL_inert.key, np.atleast_1d(Biomassboiler_EOL_flatglasscoated), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Biomassboiler_EOL_lowalloyedsteel), 'technosphere'),
        (EF_EOL_plastic.key, np.atleast_1d(Biomassboiler_EOL_polystyrene), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Biomassboiler_EOL_sheetrollingsteel), 'technosphere'),
        (EF_EOL_stonewool.key, np.atleast_1d(Biomassboiler_EOL_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in C3_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_Biomassboiler = C3_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        C3_Biomassboiler_acts[bd_name].new_exchange(
            input=C3_Biomassboiler_acts[bd_name].key, 
            output=C3_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_Biomassboiler.lci()

    mlca_C3_Biomassboiler.lcia()
    mlca_C3_Biomassboiler.scores

    dfresults_C3_Biomassboiler = pd.DataFrame.from_dict(mlca_C3_Biomassboiler.scores, orient='index')
    dfresults_C3_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_C3_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_C3_Biomassboiler = dfresults_C3_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_Biomassboiler.columns = dfresults_C3_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_Biomassboiler = dfresults_C3_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_Biomassboiler_exchange_configs = [
        (EF_EOL_inc_PEHD.key, np.atleast_1d(Biomassboiler_EOL_inc_PEHD), 'technosphere'),
        (EF_EOL_landfill_PEHD.key, np.atleast_1d(Biomassboiler_EOL_landfill_PEHD), 'technosphere'),
        (EF_EOL_inc_alkydpaint.key, np.atleast_1d(Biomassboiler_EOL_inc_alkydpaint), 'technosphere'),
        (EF_EOL_landfill_aluminium.key, np.atleast_1d(Biomassboiler_EOL_landfill_aluminium), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Biomassboiler_EOL_landfill_chromiumsteel), 'technosphere'),
        (EF_EOL_landfill_inert.key, np.atleast_1d(Biomassboiler_EOL_landfill_claybrick), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Biomassboiler_EOL_landfill_castiron), 'technosphere'),
        (EF_EOL_landfill_inert.key, np.atleast_1d(Biomassboiler_EOL_landfill_ceramictile), 'technosphere'),
        (EF_EOL_landfill_concrete.key, np.atleast_1d(Biomassboiler_EOL_landfill_concrete), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Biomassboiler_EOL_landfill_copper), 'technosphere'),
        (EF_EOL_inc_electronic.key, np.atleast_1d(Biomassboiler_EOL_inc_electronics), 'technosphere'),
        (EF_EOL_landfill_electronic.key, np.atleast_1d(Biomassboiler_EOL_landfill_electronics), 'technosphere'),
        (EF_EOL_landfill_glass.key, np.atleast_1d(Biomassboiler_EOL_landfill_flatglasscoated), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Biomassboiler_EOL_landfill_lowalloyedsteel), 'technosphere'),
        (EF_EOL_inc_lubricatingoil.key, np.atleast_1d(Biomassboiler_EOL_inc_lubricatingoil), 'technosphere'),
        (EF_EOL_inc_plastic.key, np.atleast_1d(Biomassboiler_EOL_inc_polystyrene), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Biomassboiler_EOL_landfill_sheetrollingsteel), 'technosphere'),
        (EF_EOL_landfill_stonewool.key, np.atleast_1d(Biomassboiler_EOL_landfill_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in C4_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in C4_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_Biomassboiler = C4_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        C4_Biomassboiler_acts[bd_name].new_exchange(
            input=C4_Biomassboiler_acts[bd_name].key, 
            output=C4_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_Biomassboiler.lci()

    mlca_C4_Biomassboiler.lcia()
    mlca_C4_Biomassboiler.scores

    dfresults_C4_Biomassboiler = pd.DataFrame.from_dict(mlca_C4_Biomassboiler.scores, orient='index')
    dfresults_C4_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_C4_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_C4_Biomassboiler = dfresults_C4_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_Biomassboiler.columns = dfresults_C4_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_Biomassboiler = dfresults_C4_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.9 Module B4

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_Biomassboiler = pd.DataFrame(index=dfresults_A13_Biomassboiler.index)  # Start with the index from A13 for consistency   
    dfresults_B4_Biomassboiler = (
        dfresults_A13_Biomassboiler.add(dfresults_A4_Biomassboiler, fill_value=0)
        .add(dfresults_C3_Biomassboiler, fill_value=0)
        .add(dfresults_C4_Biomassboiler, fill_value=0)
        ).mul(BiomassboilerNumberReplacements, axis=0)

### 6.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_Biomassboiler_exchange_configs = [
        (EF_recycling_aluminium.key, np.atleast_1d(Biomassboiler_EOL_recycling_aluminium), 'technosphere'),
        (EF_primary_aluminium.key, np.atleast_1d(Biomassboiler_EOL_primary_aluminium), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(Biomassboiler_EOL_recycling_castiron), 'technosphere'),
        (EF_primary_castiron.key, np.atleast_1d(Biomassboiler_EOL_primary_castiron), 'technosphere'),
        (EF_recycling_copper.key, np.atleast_1d(Biomassboiler_EOL_recycling_copper), 'technosphere'),
        (EF_primary_copper.key, np.atleast_1d(Biomassboiler_EOL_primary_copper), 'technosphere'),
        (EF_recycling_PEHD.key, np.atleast_1d(Biomassboiler_EOL_recycling_HDPE), 'technosphere'),
        (EF_primary_PEHD.key, np.atleast_1d(Biomassboiler_EOL_primary_HDPE), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(Biomassboiler_EOL_recycling_sheetrollingsteel), 'technosphere'),
        (EF_primary_steel.key, np.atleast_1d(Biomassboiler_EOL_primary_sheetrollingsteel), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(Biomassboiler_EOL_recycling_chromiumsteel), 'technosphere'),
        (EF_primary_stainlessSteel.key, np.atleast_1d(Biomassboiler_EOL_primary_chromiumsteel), 'technosphere'),
        (EF_recycling_claybrick.key, np.atleast_1d(Biomassboiler_EOL_recycling_claybrick), 'technosphere'),
        (EF_primary_claybrick.key, np.atleast_1d(Biomassboiler_EOL_primary_claybrick), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(Biomassboiler_EOL_recycling_lowalloyedsteel), 'technosphere'),
        (EF_primary_steel.key, np.atleast_1d(Biomassboiler_EOL_primary_lowalloyedsteel), 'technosphere'),
        (EF_recycling_concrete.key, np.atleast_1d(Biomassboiler_EOL_recycling_concrete), 'technosphere'),
        (EF_primary_concrete.key, np.atleast_1d(Biomassboiler_EOL_primary_concrete), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in D1_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in D1_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_Biomassboiler = D1_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        D1_Biomassboiler_acts[bd_name].new_exchange(
            input=D1_Biomassboiler_acts[bd_name].key, 
            output=D1_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_Biomassboiler.lci()

    mlca_D1_Biomassboiler.lcia()
    mlca_D1_Biomassboiler.scores

    dfresults_D1_Biomassboiler = pd.DataFrame.from_dict(mlca_D1_Biomassboiler.scores, orient='index')
    dfresults_D1_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_D1_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_D1_Biomassboiler = dfresults_D1_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_Biomassboiler.columns = dfresults_D1_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_Biomassboiler = dfresults_D1_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_Biomassboiler = dfresults_D1_Biomassboiler.add(dfresults_D1_Biomassboiler.mul(BiomassboilerNumberReplacements, axis=0))

### 6.2.12 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_Biomassboiler_exchange_configs = [
        (EF_heat_energyrecovery, np.atleast_1d(Biomassboiler_EOL_energyrecoveryheat_alkydpaint), 'technosphere'),
        (EF_electricity_energyrecovery, np.atleast_1d(Biomassboiler_EOL_energyrecoveryele_alkydpaint), 'technosphere'),
        (EF_heat_energyrecovery, np.atleast_1d(Biomassboiler_EOL_energyrecoveryheat_electronics), 'technosphere'),
        (EF_electricity_energyrecovery, np.atleast_1d(Biomassboiler_EOL_energyrecoveryele_electronics), 'technosphere'),
        (EF_heat_energyrecovery, np.atleast_1d(Biomassboiler_EOL_energyrecoveryheat_PEHD), 'technosphere'),
        (EF_electricity_energyrecovery, np.atleast_1d(Biomassboiler_EOL_energyrecoveryele_PEHD), 'technosphere'),
        (EF_heat_energyrecovery, np.atleast_1d(Biomassboiler_EOL_energyrecoveryheat_polystyrene), 'technosphere'),
        (EF_electricity_energyrecovery, np.atleast_1d(Biomassboiler_EOL_energyrecoveryele_polystyrene), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(BiomassBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_Biomassboiler_exchange_list = []
        
        for input_key, data_array, exc_type in D3_Biomassboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_Biomassboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_Biomassboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_Biomassboiler = D3_Biomassboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_Biomassboiler.save()

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in BiomassBoiler_BdName:
        D3_Biomassboiler_acts[bd_name].new_exchange(
            input=D3_Biomassboiler_acts[bd_name].key, 
            output=D3_Biomassboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_Biomassboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_Biomassboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_Biomassboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(BiomassBoiler_BdName, BiomassBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_Biomassboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_Biomassboiler.lci()

    mlca_D3_Biomassboiler.lcia()
    mlca_D3_Biomassboiler.scores

    dfresults_D3_Biomassboiler = pd.DataFrame.from_dict(mlca_D3_Biomassboiler.scores, orient='index')
    dfresults_D3_Biomassboiler.index = pd.MultiIndex.from_tuples(dfresults_D3_Biomassboiler.index, names=['Column', 'Row'])
    dfresults_D3_Biomassboiler = dfresults_D3_Biomassboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_Biomassboiler.columns = dfresults_D3_Biomassboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_Biomassboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_Biomassboiler = dfresults_D3_Biomassboiler.mul(SFBiomassboiler, axis=0)

### 6.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "Biomass boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_Biomassboiler = dfresults_D3_Biomassboiler.add(dfresults_D3_Biomassboiler.mul(BiomassboilerNumberReplacements, axis=0))

# 7. Fuel oil boilers

## 7.1 Activities

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_Fueloilboiler_{bd_name}',
        'name': f'A13_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            A13_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_Fueloilboiler_{bd_name}',
        'name': f'A4_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            A4_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_Fueloilboiler_{bd_name}',
        'name': f'A5_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            A5_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_Fueloilboiler_{bd_name}',
        'name': f'B2_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            B2_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_Fueloilboiler_{bd_name}',
        'name': f'B6_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            B6_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_Fueloilboiler_{bd_name}',
        'name': f'C2_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            C2_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_Fueloilboiler_{bd_name}',
        'name': f'C3_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            C3_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_Fueloilboiler_{bd_name}',
        'name': f'C4_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            C4_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_Fueloilboiler_{bd_name}',
        'name': f'D1_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            D1_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Fuel oil boiler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_Fueloilboiler_acts = {}  # Dictionary to store activities by building name
    for bd_name in FuelOilBoiler_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_Fueloilboiler_{bd_name}',
        'name': f'D3_Fueloilboiler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_Fueloilboiler_acts[bd_name] = scenario_db.new_activity(**data)
            D3_Fueloilboiler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 7.2 Exchanges

### 7.2.1 Modules A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_Fueloilboiler_exchange_configs = [
        (EF_steel.key, np.atleast_1d(Fueloilboiler_steellowalloyed), 'technosphere'),
        (EF_StainlessSteel.key, np.atleast_1d(Fueloilboiler_stainlesssteel), 'technosphere'),
        (EF_castiron.key, np.atleast_1d(Fueloilboiler_castiron), 'technosphere'),
        (EF_chamotte.key, np.atleast_1d(Fueloilboiler_chamotte), 'technosphere'),
        (EF_Aluminium.key, np.atleast_1d(Fueloilboiler_aluminium), 'technosphere'),
        (EF_mineralwool.key, np.atleast_1d(Fueloilboiler_mineralwool), 'technosphere'),
        (EF_Copper.key, np.atleast_1d(Fueloilboiler_copperA13), 'technosphere'),
        (EF_Electronic.key, np.atleast_1d(Fueloilboiler_electronics), 'technosphere'),
        (EF_brass.key, np.atleast_1d(Fueloilboiler_tank_brass), 'technosphere'),
        (EF_castiron.key, np.atleast_1d(Fueloilboiler_tank_castiron), 'technosphere'),
        (EF_concreteCH.key, np.atleast_1d(Fueloilboiler_tank_concrete), 'technosphere'),
        (EF_Copper.key, np.atleast_1d(Fueloilboiler_tank_copperA13), 'technosphere'),
        (EF_polyethylene.key, np.atleast_1d(Fueloilboiler_tank_PEHD), 'technosphere'),
        (EF_steel.key, np.atleast_1d(Fueloilboiler_tank_steellowalloyed), 'technosphere'),
        (EF_concreteCH.key, np.atleast_1d(Fueloilboiler_chimney_concrete), 'technosphere'),
        (EF_mineralwool.key, np.atleast_1d(Fueloilboiler_chimney_stonewool), 'technosphere'),
        (EF_Welding.key, np.atleast_1d(Fueloilboiler_welding), 'technosphere'),
        (EF_waterconsumption.key, np.atleast_1d(Fueloilboiler_waterconsumption), 'technosphere'),
        (EF_Electricity.key, np.atleast_1d(Fueloilboiler_electricityA13), 'technosphere'),
        (EF_Heat.key, np.atleast_1d(Fueloilboiler_heatfromnaturalgas), 'technosphere'),
        (EF_Heatfromoil.key, np.atleast_1d(Fueloilboiler_heatfromoil), 'technosphere'),
        (EF_Hazardous.key, np.atleast_1d(Fueloilboiler_hazardouswaste), 'technosphere'),
        (EF_waterair.key, np.atleast_1d(Fueloilboiler_watertoair), 'biosphere'),
        (EF_waterwater.key, np.atleast_1d(Fueloilboiler_watertowater), 'biosphere'),
        (EF_Wastewater_treatment.key, np.atleast_1d(Fueloilboiler_wasterwater), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in A13_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_Fueloilboiler = A13_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        A13_Fueloilboiler_acts[bd_name].new_exchange(
            input=A13_Fueloilboiler_acts[bd_name].key, 
            output=A13_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_Fueloilboiler.lci()
    mlca_A13_Fueloilboiler.lcia()
    mlca_A13_Fueloilboiler.scores

    dfresults_A13_Fueloilboiler = pd.DataFrame.from_dict(mlca_A13_Fueloilboiler.scores, orient='index')
    dfresults_A13_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_A13_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_A13_Fueloilboiler = dfresults_A13_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_Fueloilboiler.columns = dfresults_A13_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_Fueloilboiler = dfresults_A13_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.2 Module A4 

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_Fueloilboiler_exchange_configs = [
        (EF_Truk_16_32.key, Fueloilboiler_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, Fueloilboiler_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in A4_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in A4_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_Fueloilboiler = A4_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        A4_Fueloilboiler_acts[bd_name].new_exchange(
            input=A4_Fueloilboiler_acts[bd_name].key, 
            output=A4_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_Fueloilboiler.lci()
    mlca_A4_Fueloilboiler.lcia()
    mlca_A4_Fueloilboiler.scores

    dfresults_A4_Fueloilboiler = pd.DataFrame.from_dict(mlca_A4_Fueloilboiler.scores, orient='index')
    dfresults_A4_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_A4_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_A4_Fueloilboiler = dfresults_A4_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_Fueloilboiler.columns = dfresults_A4_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_Fueloilboiler = dfresults_A4_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.3 Module A5

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        A5_Fueloilboiler_acts[bd_name].new_exchange(
            input=A5_Fueloilboiler_acts[bd_name].key, 
            output=A5_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_Fueloilboiler.lci()
    mlca_A5_Fueloilboiler.lcia()
    mlca_A5_Fueloilboiler.scores

    dfresults_A5_Fueloilboiler = pd.DataFrame.from_dict(mlca_A5_Fueloilboiler.scores, orient='index')
    dfresults_A5_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_A5_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_A5_Fueloilboiler = dfresults_A5_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_Fueloilboiler.columns = dfresults_A5_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_Fueloilboiler = dfresults_A5_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_Fueloilboiler_exchange_configs = [
        (EF_LightTruk.key, Fueloilboiler_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in B2_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_Fueloilboiler = B2_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        B2_Fueloilboiler_acts[bd_name].new_exchange(
            input=B2_Fueloilboiler_acts[bd_name].key, 
            output=B2_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_Fueloilboiler.lci()
    mlca_B2_Fueloilboiler.lcia()
    mlca_B2_Fueloilboiler.scores

    dfresults_B2_Fueloilboiler = pd.DataFrame.from_dict(mlca_B2_Fueloilboiler.scores, orient='index')
    dfresults_B2_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_B2_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_B2_Fueloilboiler = dfresults_B2_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_Fueloilboiler.columns = dfresults_B2_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_Fueloilboiler = dfresults_B2_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_Fueloilboiler_exchange_configs = [
        (EF_diesel.key, Fueloilboiler_Energyvector, 'technosphere'),
        (EF_Acetaldehyde.key, Fueloilboiler_acetaldehyde, 'biosphere'),
        (EF_Acetone.key, Fueloilboiler_acetone, 'biosphere'),
        (EF_Acrolein.key, Fueloilboiler_acrolein, 'biosphere'),
        (EF_Benzaldehyde.key, Fueloilboiler_benzaldehyde, 'biosphere'),
        (EF_Benzene.key, Fueloilboiler_benzene, 'biosphere'),
        (EF_Buthane.key, Fueloilboiler_butane, 'biosphere'),
        (EF_CarbondioxideFossil.key, Fueloilboiler_carbondioxide_fossil, 'biosphere'),
        (EF_CarbonmonoxideFossil.key, Fueloilboiler_carbonmonoxide_fossil, 'biosphere'),
        (EF_CopperIon.key, Fueloilboiler_copperB6, 'biosphere'),
        (EF_DinitrogenMonoxide.key, Fueloilboiler_dinitrogenmonoxide, 'biosphere'),
        (EF_Ethane.key, Fueloilboiler_ethane, 'biosphere'),
        (EF_Formaldehyde.key, Fueloilboiler_formaldehyde, 'biosphere'),
        (EF_HydrocarbonsAlkanes.key, Fueloilboiler_hydrocarbons_alkanes, 'biosphere'),
        (EF_HydrocarbonsUnsaturated.key, Fueloilboiler_hydrocarbons_unsatured, 'biosphere'),
        (EF_HydrocarbonsUnsaturated.key, Fueloilboiler_hydrocarbons_aromatic, 'biosphere'),
        (EF_Hydrogenchloride.key, Fueloilboiler_hydrogenchloride, 'biosphere'),
        (EF_Hydrogenfluoride.key, Fueloilboiler_hydrogenfluoride, 'biosphere'),
        (EF_Mercury.key, Fueloilboiler_mercury, 'biosphere'),
        (EF_MethaneFossil.key, Fueloilboiler_methanefossil, 'biosphere'),
        (EF_NitrogenOxides.key, Fueloilboiler_nitrogenoxyde, 'biosphere'),
        (EF_PAH.key, Fueloilboiler_PAH, 'biosphere'),
        (EF_Particulates025.key, Fueloilboiler_particulate_025, 'biosphere'),
        (EF_Pentane.key, Fueloilboiler_pentane, 'biosphere'),
        (EF_Propanal.key, Fueloilboiler_propanal, 'biosphere'),
        (EF_propane.key, Fueloilboiler_propane, 'biosphere'),
        (EF_Propene.key, Fueloilboiler_propene, 'biosphere'),
        (EF_SulfurDioxide.key, Fueloilboiler_sulfurdioxyde, 'biosphere'),
        (EF_Toluene.key, Fueloilboiler_toluene, 'biosphere'),
        (EF_Zinc.key, Fueloilboiler_zinc, 'biosphere'),
        (EF_electricity.key, Fueloilboiler_electricityB6, 'technosphere'),
        (EF_electricityPV.key, Fueloilboiler_electricityB6PV, 'technosphere'),
        # (EF_sun.key, Fueloilboiler_electricityB6RECSPV, 'technosphere'),
        # (EF_hydro.key, Fueloilboiler_electricityB6RECSHydro, 'technosphere'),
        # (EF_wind.key, Fueloilboiler_electricityB6RECSEolic, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in B6_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_Fueloilboiler = B6_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        B6_Fueloilboiler_acts[bd_name].new_exchange(
            input=B6_Fueloilboiler_acts[bd_name].key, 
            output=B6_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_Fueloilboiler.lci()
    mlca_B6_Fueloilboiler.lcia()
    mlca_B6_Fueloilboiler.scores

    dfresults_B6_Fueloilboiler = pd.DataFrame.from_dict(mlca_B6_Fueloilboiler.scores, orient='index')
    dfresults_B6_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_B6_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_B6_Fueloilboiler = dfresults_B6_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_Fueloilboiler.columns = dfresults_B6_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_Fueloilboiler.rename(columns=method_lookup, inplace=True)

### 7.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_Fueloilboiler_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, Fueloilboiler_Transport_to_treatment_plant, 'technosphere'),
        
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in C2_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_Fueloilboiler = C2_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        C2_Fueloilboiler_acts[bd_name].new_exchange(
            input=C2_Fueloilboiler_acts[bd_name].key, 
            output=C2_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_Fueloilboiler.lci()
    mlca_C2_Fueloilboiler.lcia()
    mlca_C2_Fueloilboiler.scores

    dfresults_C2_Fueloilboiler = pd.DataFrame.from_dict(mlca_C2_Fueloilboiler.scores, orient='index')
    dfresults_C2_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_C2_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_C2_Fueloilboiler = dfresults_C2_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_Fueloilboiler.columns = dfresults_C2_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_Fueloilboiler = dfresults_C2_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_Fueloilboiler_exchange_configs = [
        (EF_EOL_metal.key, np.atleast_1d(Fueloilboiler_EOL_steellowalloyed), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Fueloilboiler_EOL_stainlesssteel), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Fueloilboiler_EOL_castiron), 'technosphere'),
        (EF_EOL_inert.key, np.atleast_1d(Fueloilboiler_EOL_chamotte), 'technosphere'),
        (EF_EOL_Aluminium.key, np.atleast_1d(Fueloilboiler_EOL_aluminium), 'technosphere'),
        (EF_EOL_stonewool.key, np.atleast_1d(Fueloilboiler_EOL_mineralwool), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Fueloilboiler_EOL_copper), 'technosphere'),
        (EF_EOL_electronics.key, np.atleast_1d(Fueloilboiler_EOL_electronics), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Fueloilboiler_EOL_tank_brass), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Fueloilboiler_EOL_tank_castiron), 'technosphere'),
        (EF_EOL_concrete.key, np.atleast_1d(Fueloilboiler_EOL_tank_concrete), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Fueloilboiler_EOL_tank_copper), 'technosphere'),
        (EF_EOL_plastic.key, np.atleast_1d(Fueloilboiler_EOL_tank_PEHD), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(Fueloilboiler_EOL_tank_steellowalloyed), 'technosphere'),
        (EF_EOL_concrete.key, np.atleast_1d(Fueloilboiler_EOL_chimney_concrete), 'technosphere'),
        (EF_EOL_stonewool.key, np.atleast_1d(Fueloilboiler_EOL_chimney_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in C3_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_Fueloilboiler = C3_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        C3_Fueloilboiler_acts[bd_name].new_exchange(
            input=C3_Fueloilboiler_acts[bd_name].key, 
            output=C3_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_Fueloilboiler.lci()
    mlca_C3_Fueloilboiler.lcia()
    mlca_C3_Fueloilboiler.scores

    dfresults_C3_Fueloilboiler = pd.DataFrame.from_dict(mlca_C3_Fueloilboiler.scores, orient='index')
    dfresults_C3_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_C3_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_C3_Fueloilboiler = dfresults_C3_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_Fueloilboiler.columns = dfresults_C3_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_Fueloilboiler = dfresults_C3_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_Fueloilboiler_exchange_configs = [
        (EF_EOL_landfill_metal.key, np.atleast_1d(Fueloilboiler_EOL_landfill_steel), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Fueloilboiler_EOL_landfill_stainlessSteel), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Fueloilboiler_EOL_landfill_castiron), 'technosphere'),
        (EF_EOL_landfill_inert.key, np.atleast_1d(Fueloilboiler_EOL_landfill_chamotte), 'technosphere'),
        (EF_EOL_landfill_aluminium.key, np.atleast_1d(Fueloilboiler_EOL_landfill_aluminium), 'technosphere'),
        (EF_EOL_landfill_stonewool.key, np.atleast_1d(Fueloilboiler_EOL_landfill_mineralwool), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Fueloilboiler_EOL_landfill_copper), 'technosphere'),
        (EF_EOL_inc_electronic.key, np.atleast_1d(Fueloilboiler_EOL_inc_electronics), 'technosphere'),
        (EF_EOL_landfill_electronic.key, np.atleast_1d(Fueloilboiler_EOL_landfill_electronics), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Fueloilboiler_EOL_landfill_tank_brass), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Fueloilboiler_EOL_landfill_tank_castiron), 'technosphere'),
        (EF_EOL_landfill_concrete.key, np.atleast_1d(Fueloilboiler_EOL_landfill_tank_concrete), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Fueloilboiler_EOL_landfill_tank_copper), 'technosphere'),
        (EF_EOL_inc_PEHD.key, np.atleast_1d(Fueloilboiler_EOL_inc_tank_PEHD), 'technosphere'),
        (EF_EOL_landfill_PEHD.key, np.atleast_1d(Fueloilboiler_EOL_landfill_tank_PEHD), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(Fueloilboiler_EOL_landfill_tank_steellowalloyed), 'technosphere'),
        (EF_EOL_landfill_concrete.key, np.atleast_1d(Fueloilboiler_EOL_landfill_chimney_concrete), 'technosphere'),
        (EF_EOL_landfill_stonewool.key, np.atleast_1d(Fueloilboiler_EOL_landfill_chimney_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in C4_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_Fueloilboiler = C4_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        C4_Fueloilboiler_acts[bd_name].new_exchange(
            input=C4_Fueloilboiler_acts[bd_name].key, 
            output=C4_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_Fueloilboiler.lci()
    mlca_C4_Fueloilboiler.lcia()
    mlca_C4_Fueloilboiler.scores

    dfresults_C4_Fueloilboiler = pd.DataFrame.from_dict(mlca_C4_Fueloilboiler.scores, orient='index')
    dfresults_C4_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_C4_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_C4_Fueloilboiler = dfresults_C4_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_Fueloilboiler.columns = dfresults_C4_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_Fueloilboiler = dfresults_C4_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.9 Module B4

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_Fueloilboiler = pd.DataFrame(index=dfresults_A13_Fueloilboiler.index)  # Start with the index from A13 for consistency   
    dfresults_B4_Fueloilboiler = (
        dfresults_A13_Fueloilboiler.add(dfresults_A4_Fueloilboiler, fill_value=0)
        .add(dfresults_C3_Fueloilboiler, fill_value=0)
        .add(dfresults_C4_Fueloilboiler, fill_value=0)
        ).mul(FueloilboilerNumberReplacements, axis=0)
    

### 7.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_Fueloilboiler_exchange_configs = [
        (EF_recycling_steel.key, np.atleast_1d(Fueloilboiler_EOL_recycling_steel), 'technosphere'),
        (EF_primary_steel.key, np.atleast_1d(Fueloilboiler_EOL_primary_steel), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(Fueloilboiler_EOL_recycling_stainlessSteel), 'technosphere'),
        (EF_primary_stainlessSteel.key, np.atleast_1d(Fueloilboiler_EOL_primary_stainlessSteel), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(Fueloilboiler_EOL_recycling_castiron), 'technosphere'),
        (EF_primary_castiron.key, np.atleast_1d(Fueloilboiler_EOL_primary_castiron), 'technosphere'),
        (EF_recycling_aluminium.key, np.atleast_1d(Fueloilboiler_EOL_recycling_aluminium), 'technosphere'),
        (EF_primary_aluminium.key, np.atleast_1d(Fueloilboiler_EOL_primary_aluminium), 'technosphere'),
        (EF_recycling_copper.key, np.atleast_1d(Fueloilboiler_EOL_recycling_copper), 'technosphere'),
        (EF_primary_copper.key, np.atleast_1d(Fueloilboiler_EOL_primary_copper), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(Fueloilboiler_EOL_recycling_tank_castiron), 'technosphere'),
        (EF_primary_castiron.key, np.atleast_1d(Fueloilboiler_EOL_primary_tank_castiron), 'technosphere'),
        (EF_recycling_concrete.key, np.atleast_1d(Fueloilboiler_EOL_recycling_tank_concrete), 'technosphere'),
        (EF_primary_concrete.key, np.atleast_1d(Fueloilboiler_EOL_primary_tank_concrete), 'technosphere'),
        (EF_recycling_copper.key, np.atleast_1d(Fueloilboiler_EOL_recycling_tank_copper), 'technosphere'),
        (EF_primary_copper.key, np.atleast_1d(Fueloilboiler_EOL_primary_tank_copper), 'technosphere'),
        (EF_recycling_PEHD.key, np.atleast_1d(Fueloilboiler_EOL_recycling_tank_PEHD), 'technosphere'),
        (EF_primary_PEHD.key, np.atleast_1d(Fueloilboiler_EOL_primary_tank_PEHD), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(Fueloilboiler_EOL_recycling_tank_steellowalloyed), 'technosphere'),
        (EF_primary_steel.key, np.atleast_1d(Fueloilboiler_EOL_primary_tank_steellowalloyed), 'technosphere'),
        (EF_recycling_concrete.key, np.atleast_1d(Fueloilboiler_EOL_recycling_chimney_concrete), 'technosphere'),
        (EF_primary_concrete.key, np.atleast_1d(Fueloilboiler_EOL_primary_chimney_concrete), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in D1_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_Fueloilboiler = D1_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        D1_Fueloilboiler_acts[bd_name].new_exchange(
            input=D1_Fueloilboiler_acts[bd_name].key, 
            output=D1_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_Fueloilboiler.lci()
    mlca_D1_Fueloilboiler.lcia()
    mlca_D1_Fueloilboiler.scores

    dfresults_D1_Fueloilboiler = pd.DataFrame.from_dict(mlca_D1_Fueloilboiler.scores, orient='index')
    dfresults_D1_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_D1_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_D1_Fueloilboiler = dfresults_D1_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_Fueloilboiler.columns = dfresults_D1_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_Fueloilboiler = dfresults_D1_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_Fueloilboiler = dfresults_D1_Fueloilboiler.add(dfresults_D1_Fueloilboiler.mul(FueloilboilerNumberReplacements, axis=0))

### 7.2.12 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_Fueloilboiler_exchange_configs = [
        (EF_heat_energyrecovery.key, np.atleast_1d(Fueloilboiler_EOL_energyrecoveryheat_electronics), 'technosphere'),
        (EF_electricity_energyrecovery.key, np.atleast_1d(Fueloilboiler_EOL_energyrecoveryele_electronics), 'technosphere'),
        (EF_heat_energyrecovery.key, np.atleast_1d(Fueloilboiler_EOL_energyrecoveryheat_PEHD), 'technosphere'),
        (EF_electricity_energyrecovery.key, np.atleast_1d(Fueloilboiler_EOL_energyrecoveryele_PEHD), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(FuelOilBoiler_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_Fueloilboiler_exchange_list = []
        
        for input_key, data_array, exc_type in D3_Fueloilboiler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_Fueloilboiler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_Fueloilboiler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_Fueloilboiler = D3_Fueloilboiler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_Fueloilboiler.save()

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in FuelOilBoiler_BdName:
        D3_Fueloilboiler_acts[bd_name].new_exchange(
            input=D3_Fueloilboiler_acts[bd_name].key, 
            output=D3_Fueloilboiler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_Fueloilboiler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_Fueloilboiler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_Fueloilboiler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(FuelOilBoiler_BdName, FuelOilBoiler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_Fueloilboiler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_Fueloilboiler.lci()
    mlca_D3_Fueloilboiler.lcia()
    mlca_D3_Fueloilboiler.scores

    dfresults_D3_Fueloilboiler = pd.DataFrame.from_dict(mlca_D3_Fueloilboiler.scores, orient='index')
    dfresults_D3_Fueloilboiler.index = pd.MultiIndex.from_tuples(dfresults_D3_Fueloilboiler.index, names=['Column', 'Row'])
    dfresults_D3_Fueloilboiler = dfresults_D3_Fueloilboiler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_Fueloilboiler.columns = dfresults_D3_Fueloilboiler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_Fueloilboiler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_Fueloilboiler = dfresults_D3_Fueloilboiler.mul(SFFueloilboiler, axis=0)

### 7.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "Fuel oil boiler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_Fueloilboiler = dfresults_D3_Fueloilboiler.add(dfresults_D3_Fueloilboiler.mul(FueloilboilerNumberReplacements, axis=0))

# 8. CHP - Reciprocating internal combustion engines

## 8.1 Activities

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_RICE_{bd_name}',
        'name': f'A13_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            A13_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_RICE_{bd_name}',
        'name': f'A4_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            A4_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_RICE_{bd_name}',
        'name': f'A5_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            A5_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_RICE_{bd_name}',
        'name': f'B2_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            B2_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_RICE_{bd_name}',
        'name': f'B6_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            B6_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_RICE_{bd_name}',
        'name': f'C2_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            C2_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_RICE_{bd_name}',
        'name': f'C3_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            C3_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_RICE_{bd_name}',
        'name': f'C4_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            C4_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_RICE_{bd_name}',
        'name': f'D1_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            D1_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_RICE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_RICE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_RICE_{bd_name}',
        'name': f'D3_RICE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_RICE_acts[bd_name] = scenario_db.new_activity(**data)
            D3_RICE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 8.2 Exchanges

### 8.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_RICE_exchange_configs = [
        (EF_Air.key, RICE_Air, 'technosphere'),
        (EF_Control.key, RICE_Control, 'technosphere'),
        (EF_Motor.key, RICE_Motor, 'technosphere'),
        (EF_Insulation.key, RICE_Insulation, 'technosphere'),
        (EF_Catalytic.key, RICE_Catalytic, 'technosphere'),
        (EF_Electric.key, RICE_Electric, 'technosphere'),
        (EF_Generator.key, RICE_Generator, 'technosphere'),
        (EF_Sanitary.key, RICE_Sanitary, 'technosphere'),
        (EF_storage.key, RICE_Storage, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in A13_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_RICE = A13_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        A13_RICE_acts[bd_name].new_exchange(
            input=A13_RICE_acts[bd_name].key, 
            output=A13_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_RICE.lci()
    mlca_A13_RICE.lcia()
    mlca_A13_RICE.scores

    dfresults_A13_RICE = pd.DataFrame.from_dict(mlca_A13_RICE.scores, orient='index')
    dfresults_A13_RICE.index = pd.MultiIndex.from_tuples(dfresults_A13_RICE.index, names=['Column', 'Row'])
    dfresults_A13_RICE = dfresults_A13_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_RICE.columns = dfresults_A13_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_RICE = dfresults_A13_RICE.mul(SFRICE, axis=0)

### 8.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_RICE_exchange_configs = [
        (EF_Truk_16_32.key, RICE_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, RICE_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in A4_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_RICE = A4_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        A4_RICE_acts[bd_name].new_exchange(
            input=A4_RICE_acts[bd_name].key, 
            output=A4_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_RICE.lci()
    mlca_A4_RICE.lcia()
    mlca_A4_RICE.scores

    dfresults_A4_RICE = pd.DataFrame.from_dict(mlca_A4_RICE.scores, orient='index')
    dfresults_A4_RICE.index = pd.MultiIndex.from_tuples(dfresults_A4_RICE.index, names=['Column', 'Row'])
    dfresults_A4_RICE = dfresults_A4_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_RICE.columns = dfresults_A4_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_RICE = dfresults_A4_RICE.mul(SFRICE, axis=0)

### 8.2.3 Module A5

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A5_RICE_exchange_configs = [
        (EF_energy.key, RICE_energy, 'technosphere'),
        (EF_construction.key, RICE_construction, 'technosphere'),
        (EF_planning.key, RICE_planning, 'technosphere'),
        (EF_startup.key, RICE_startup, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        A5_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in A5_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A5_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A5_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A5_RICE = A5_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A5_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        A5_RICE_acts[bd_name].new_exchange(
            input=A5_RICE_acts[bd_name].key, 
            output=A5_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_RICE.lci()
    mlca_A5_RICE.lcia()
    mlca_A5_RICE.scores

    dfresults_A5_RICE = pd.DataFrame.from_dict(mlca_A5_RICE.scores, orient='index')
    dfresults_A5_RICE.index = pd.MultiIndex.from_tuples(dfresults_A5_RICE.index, names=['Column', 'Row'])
    dfresults_A5_RICE = dfresults_A5_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_RICE.columns = dfresults_A5_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_RICE = dfresults_A5_RICE.mul(SFRICE, axis=0)

### 8.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_RICE_exchange_configs = [
        (EF_maintenance.key, RICE_maintenance, 'technosphere'),
        (EF_lubricatingOilB2.key, RICE_lubricatingoil, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in B2_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_RICE = B2_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        B2_RICE_acts[bd_name].new_exchange(
            input=B2_RICE_acts[bd_name].key, 
            output=B2_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_RICE.lci()
    mlca_B2_RICE.lcia()
    mlca_B2_RICE.scores

    dfresults_B2_RICE = pd.DataFrame.from_dict(mlca_B2_RICE.scores, orient='index')
    dfresults_B2_RICE.index = pd.MultiIndex.from_tuples(dfresults_B2_RICE.index, names=['Column', 'Row'])
    dfresults_B2_RICE = dfresults_B2_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_RICE.columns = dfresults_B2_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_RICE = dfresults_B2_RICE.mul(SFRICE, axis=0)

### 8.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_RICE_exchange_configs = [
        (EF_energyvectorNG.key, RICE_EnergyvectorNG, 'technosphere'),
        (EF_energyvectorBiom.key, RICE_EnergyvectorBiomethane, 'technosphere'),
        (EF_energyvectorHydrogen.key, RICE_EnergyvectorHydrogen, 'technosphere'),
        (EF_biogas.key, RICE_EnergyvectorInputBiogas, 'technosphere'),
        (EF_energyvectorHydrogen.key, RICE_EnergyVectorInputHydrogen, 'technosphere'),
        (EF_Acetaldehyde.key, RICE_Acetaldehyde, 'biosphere'),
        (EF_Aceticacid.key, RICE_AceticAcid, 'biosphere'),
        (EF_Benzene.key, RICE_Benzene, 'biosphere'),
        (EF_Benzoapyrene.key, RICE_Benzoapyrene, 'biosphere'),
        (EF_Buthane.key, RICE_Butane, 'biosphere'),
        (EF_CarbondioxideFossil.key, RICE_CarbonDioxideFossil, 'biosphere'),
        (EF_CarbonmonoxideFossil.key, RICE_CarbonMonoxideFossil, 'biosphere'),
        (EF_DinitrogenMonoxide.key, RICE_DinitrogenMonoxide, 'biosphere'),
        (EF_Ethane.key, RICE_Ethane, 'biosphere'),
        (EF_Formaldehyde.key, RICE_Formaldehyde, 'biosphere'),
        (EF_Hexane.key, RICE_Hexane, 'biosphere'),
        (EF_Mercury.key, RICE_Mercury, 'biosphere'),
        (EF_MethaneFossil.key, RICE_MethaneFossil, 'biosphere'),
        (EF_NitrogenOxides.key, RICE_NitrogenOxides, 'biosphere'),
        (EF_PAH.key, RICE_PAH, 'biosphere'),
        (EF_Particulates025.key, RICE_Particulates, 'biosphere'),
        (EF_Pentane.key, RICE_Pentane, 'biosphere'),
        (EF_propane.key, RICE_Propane, 'biosphere'),
        (EF_PropionicAcid.key, RICE_PropionicAcid, 'biosphere'),
        (EF_SulfurDioxide.key, RICE_SulfurDioxide, 'biosphere'),
        (EF_Toluene.key, RICE_Toluene, 'biosphere'),
        (EF_Dioxin.key, RICE_Dioxins, 'biosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in B6_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_RICE = B6_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        B6_RICE_acts[bd_name].new_exchange(
            input=B6_RICE_acts[bd_name].key, 
            output=B6_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_RICE.lci()
    mlca_B6_RICE.lcia()
    mlca_B6_RICE.scores

    dfresults_B6_RICE = pd.DataFrame.from_dict(mlca_B6_RICE.scores, orient='index')
    dfresults_B6_RICE.index = pd.MultiIndex.from_tuples(dfresults_B6_RICE.index, names=['Column', 'Row'])
    dfresults_B6_RICE = dfresults_B6_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_RICE.columns = dfresults_B6_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_RICE.rename(columns=method_lookup, inplace=True)

### 8.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_RICE_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, RICE_Transport_to_treatment_plant, 'technosphere'), 
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in C2_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_RICE = C2_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        C2_RICE_acts[bd_name].new_exchange(
            input=C2_RICE_acts[bd_name].key, 
            output=C2_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_RICE.lci()
    mlca_C2_RICE.lcia()
    mlca_C2_RICE.scores

    dfresults_C2_RICE = pd.DataFrame.from_dict(mlca_C2_RICE.scores, orient='index')
    dfresults_C2_RICE.index = pd.MultiIndex.from_tuples(dfresults_C2_RICE.index, names=['Column', 'Row'])
    dfresults_C2_RICE = dfresults_C2_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_RICE.columns = dfresults_C2_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_RICE = dfresults_C2_RICE.mul(SFRICE, axis=0)

### 8.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_RICE_exchange_configs = [
        (EF_EOL_metal.key, RICE_reinforcingsteelAirinput, 'technosphere'),
        (EF_EOL_Aluminium.key, RICE_aluminiumControlcabinet, 'technosphere'),
        (EF_EOL_metal.key, RICE_copperControlcabinet, 'technosphere'),
        (EF_EOL_metal.key, RICE_leadControlcabinet, 'technosphere'),
        (EF_EOL_metal.key, RICE_nickelControlcabinet, 'technosphere'),
        (EF_EOL_metal.key, RICE_platinumControlcabinet, 'technosphere'),
        (EF_EOL_plastic.key, RICE_HDPEControlcabinet, 'technosphere'),
        (EF_EOL_PVC.key, RICE_PVCemulsionatedControlcabinet, 'technosphere'),
        (EF_EOL_PVC.key, RICE_PVCpolimerizedControlcabinet, 'technosphere'),
        (EF_EOL_metal.key, RICE_reinforcingsteelControlcabinet, 'technosphere'),
        (EF_EOL_metal.key, RICE_tinControlcabinet, 'technosphere'),
        (EF_EOL_metal.key, RICE_zincControlcabinet, 'technosphere'),
        (EF_EOL_metal.key, RICE_castironGasmotor, 'technosphere'),
        (EF_EOL_metal.key, RICE_reinforcingsteelGasmotor, 'technosphere'),
        (EF_EOL_metal.key, RICE_chromiumsteelGasmotor, 'technosphere'),
        (EF_EOL_metal.key, RICE_lowalloyedsteelGasmotor, 'technosphere'),
        (EF_EOL_metal.key, RICE_reinforcingsteelSoundinsulation, 'technosphere'),
        (EF_EOL_stonewool.key, RICE_stonewoolSoundinsulation, 'technosphere'),
        (EF_EOL_corrugatedboxboard.key, RICE_corrugatedboxboardConverter, 'technosphere'),
        (EF_EOL_metal.key, RICE_palladiumConverter, 'technosphere'),
        (EF_EOL_metal.key, RICE_platinumConverter, 'technosphere'),
        (EF_EOL_metal.key, RICE_rhodiumConverter, 'technosphere'),
        (EF_EOL_metal.key, RICE_chromiumsteelConverter, 'technosphere'),
        (EF_EOL_inert.key, RICE_zeoliteConverter, 'technosphere'),
        (EF_EOL_Aluminium.key, RICE_aluminiumEle, 'technosphere'),
        (EF_EOL_metal.key, RICE_copperEle, 'technosphere'),
        (EF_EOL_metal.key, RICE_leadEle, 'technosphere'),
        (EF_EOL_metal.key, RICE_nickelEle, 'technosphere'),
        (EF_EOL_metal.key, RICE_platinumEle, 'technosphere'),
        (EF_EOL_plastic.key, RICE_HDPEEle, 'technosphere'),
        (EF_EOL_PVC.key, RICE_PVCpolimerizedEle, 'technosphere'),
        (EF_EOL_metal.key, RICE_reinforcingsteelEle, 'technosphere'),
        (EF_EOL_metal.key, RICE_tinEle, 'technosphere'),
        (EF_EOL_metal.key, RICE_zincEle, 'technosphere'),
        (EF_EOL_metal.key, RICE_castironGen, 'technosphere'),
        (EF_EOL_metal.key, RICE_copperGen, 'technosphere'),
        (EF_EOL_metal.key, RICE_reinforcingsteelSan, 'technosphere'),
        (EF_EOL_stonewool.key, RICE_stonewoolSan, 'technosphere'),
        (EF_EOL_alkydpaint.key, RICE_alkydpaintStorage, 'technosphere'),
        (EF_EOL_Aluminium.key, RICE_aluminiumStorage, 'technosphere'),
        (EF_EOL_metal.key, RICE_reinforcingsteelStorage, 'technosphere'),
        (EF_EOL_Aluminium.key, RICE_sheetrollingaluminiumStorage, 'technosphere'),
        (EF_EOL_metal.key, RICE_sheetrollingsteelStorage, 'technosphere'),
        (EF_EOL_metal.key, RICE_chromiumsteelStorage, 'technosphere'),
        (EF_EOL_stonewool.key, RICE_stonewoolStorage, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in C3_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_RICE = C3_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        C3_RICE_acts[bd_name].new_exchange(
            input=C3_RICE_acts[bd_name].key, 
            output=C3_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_RICE.lci()
    mlca_C3_RICE.lcia()
    mlca_C3_RICE.scores

    dfresults_C3_RICE = pd.DataFrame.from_dict(mlca_C3_RICE.scores, orient='index')
    dfresults_C3_RICE.index = pd.MultiIndex.from_tuples(dfresults_C3_RICE.index, names=['Column', 'Row'])
    dfresults_C3_RICE = dfresults_C3_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_RICE.columns = dfresults_C3_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_RICE = dfresults_C3_RICE.mul(SFRICE, axis=0)

### 8.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_RICE_exchange_configs = [
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_reinforcingsteelAirinput, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, RICE_EOL_landfill_aluminiumControlcabinet, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_copperControlcabinet, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_leadControlcabinet, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_nickelControlcabinet, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_platinumControlcabinet, 'technosphere'),
        (EF_EOL_inc_PEHD.key, RICE_EOL_inc_HDPEControlcabinet, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, RICE_EOL_landfill_HDPEControlcabinet, 'technosphere'),
        (EF_EOL_inc_plastic.key, RICE_EOL_inc_PVCemulsionatedControlcabinet, 'technosphere'),
        (EF_EOL_landfill_plastic.key, RICE_EOL_landfill_PVCemulsionatedControlcabinet, 'technosphere'),
        (EF_EOL_inc_plastic.key, RICE_EOL_inc_PVCpolimerizedControlcabinet, 'technosphere'),
        (EF_EOL_landfill_plastic.key, RICE_EOL_landfill_PVCpolimerizedControlcabinet, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_reinforcingsteelControlcabinet, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_tinControlcabinet, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_zincControlcabinet, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_castironGasmotor, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_reinforcingsteelGasmotor, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_chromiumsteelGasmotor, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_lowalloyedsteelGasmotor, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_reinforcingsteelSoundinsulation, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, RICE_EOL_landfill_stonewoolSoundinsulation, 'technosphere'),
        (EF_EOL_inc_paperboard.key, RICE_EOL_inc_corrugatedboxboardConverter, 'technosphere'),
        (EF_EOL_landfill_paperboard.key, RICE_EOL_landfill_corrugatedboxboardConverter, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_palladiumConverter, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_platinumConverter, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_rhodiumConverter, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_chromiumsteelConverter, 'technosphere'),
        (EF_EOL_inc_hazardouswaste.key, RICE_EOL_inc_zeoliteConverter, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, RICE_EOL_landfill_aluminiumEle, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_copperEle, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_leadEle, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_nickelEle, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_platinumEle, 'technosphere'),
        (EF_EOL_inc_PEHD.key, RICE_EOL_inc_HDPEEle, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, RICE_EOL_landfill_HDPEEle, 'technosphere'),
        (EF_EOL_inc_plastic.key, RICE_EOL_inc_PVCpolimerizedEle, 'technosphere'),
        (EF_EOL_landfill_plastic.key, RICE_EOL_landfill_PVCpolimerizedEle, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_reinforcingsteelEle, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_tinEle, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_zincEle, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_castironGen, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_copperGen, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_reinforcingsteelSan, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, RICE_EOL_landfill_stonewoolSan, 'technosphere'),
        (EF_EOL_inc_alkydpaint.key, RICE_EOL_inc_alkydpaintStorage, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, RICE_EOL_landfill_aluminiumStorage, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_reinforcingsteelStorage, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, RICE_EOL_landfill_sheetrollingaluminiumStorage, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_sheetrollingsteelStorage, 'technosphere'),
        (EF_EOL_landfill_metal.key, RICE_EOL_landfill_chromiumsteelStorage, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, RICE_EOL_landfill_stonewoolStorage, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in C4_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_RICE = C4_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        C4_RICE_acts[bd_name].new_exchange(
            input=C4_RICE_acts[bd_name].key, 
            output=C4_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_RICE.lci()
    mlca_C4_RICE.lcia()
    mlca_C4_RICE.scores

    dfresults_C4_RICE = pd.DataFrame.from_dict(mlca_C4_RICE.scores, orient='index')
    dfresults_C4_RICE.index = pd.MultiIndex.from_tuples(dfresults_C4_RICE.index, names=['Column', 'Row'])
    dfresults_C4_RICE = dfresults_C4_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_RICE.columns = dfresults_C4_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_RICE = dfresults_C4_RICE.mul(SFRICE, axis=0)

### 8.2.9 Module B4

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_RICE = pd.DataFrame(index=dfresults_A13_RICE.index)  # Start with the index from A13 for consistency   
    dfresults_B4_RICE = (
        dfresults_A13_RICE.add(dfresults_A4_RICE, fill_value=0)
        .add(dfresults_C3_RICE, fill_value=0)
        .add(dfresults_C4_RICE, fill_value=0)
        ).mul(RICENumberReplacements, axis=0)
    

### 8.2.10 Module D1  (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_RICE_exchange_configs = [
        (EF_recycling_steel.key, RICE_recycling_reinforcingsteelAirinput, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_reinforcingsteelAirinput, 'technosphere'),
        (EF_recycling_aluminium.key, RICE_recycling_aluminiumControlcabinet, 'technosphere'),
        (EF_primary_aluminium.key, RICE_primary_aluminiumControlcabinet, 'technosphere'),
        (EF_recycling_copper.key, RICE_recycling_copperControlcabinet, 'technosphere'),
        (EF_primary_copper.key, RICE_primary_copperControlcabinet, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_leadControlcabinet, 'technosphere'),
        (EF_primary_lead.key, RICE_primary_leadControlcabinet, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_nickelControlcabinet, 'technosphere'),
        (EF_primary_nickel.key, RICE_primary_nickelControlcabinet, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_platinumControlcabinet, 'technosphere'),
        (EF_primary_platinum.key, RICE_primary_platinumControlcabinet, 'technosphere'),
        (EF_recycling_PEHD.key, RICE_recycling_HDPEControlcabinet, 'technosphere'),
        (EF_primary_PEHD.key, RICE_primary_HDPEControlcabinet, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_reinforcingsteelControlcabinet, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_reinforcingsteelControlcabinet, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_tinControlcabinet, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_tinControlcabinet, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_zincControlcabinet, 'technosphere'),
        (EF_primary_zinc.key, RICE_primary_zincControlcabinet, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_castironGasmotor, 'technosphere'),
        (EF_primary_castiron.key, RICE_primary_castironGasmotor, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_reinforcingsteelGasmotor, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_reinforcingsteelGasmotor, 'technosphere'),
        (EF_primary_steel.key, RICE_recycling_chromiumsteelGasmotor, 'technosphere'),
        (EF_recycling_steel.key, RICE_primary_chromiumsteelGasmotor, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_lowalloyedsteelGasmotor, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_lowalloyedsteelGasmotor, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_reinforcingsteelSoundinsulation, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_reinforcingsteelSoundinsulation, 'technosphere'),
        (EF_recycling_cardboard.key, RICE_recycling_corrugatedboxboardConverter, 'technosphere'),
        (EF_primary_cardboard.key, RICE_primary_corrugatedboxboardConverter, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_palladiumConverter, 'technosphere'),
        (EF_primary_palladium.key, RICE_primary_palladiumConverter, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_platinumConverter, 'technosphere'),
        (EF_primary_platinum.key, RICE_primary_platinumConverter, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_rhodiumConverter, 'technosphere'),
        (EF_primary_rhodium.key, RICE_primary_rhodiumConverter, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_chromiumsteelConverter, 'technosphere'),
        (EF_primary_stainlessSteel.key, RICE_primary_chromiumsteelConverter, 'technosphere'),
        (EF_recycling_aluminium.key, RICE_recycling_aluminiumEle, 'technosphere'),
        (EF_primary_aluminium.key, RICE_primary_aluminiumEle, 'technosphere'),
        (EF_recycling_copper.key, RICE_recycling_copperEle, 'technosphere'),
        (EF_primary_copper.key, RICE_primary_copperEle, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_leadEle, 'technosphere'),
        (EF_primary_lead.key, RICE_primary_leadEle, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_nickelEle, 'technosphere'),
        (EF_primary_nickel.key, RICE_primary_nickelEle, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_platinumEle, 'technosphere'),
        (EF_primary_platinum.key, RICE_primary_platinumEle, 'technosphere'),
        (EF_recycling_PEHD.key, RICE_recycling_HDPEEle, 'technosphere'),
        (EF_primary_PEHD.key, RICE_primary_HDPEEle, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_reinforcingsteelEle, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_reinforcingsteelEle, 'technosphere'),
        (EF_recycling_iron.key, RICE_recycling_zincEle, 'technosphere'),
        (EF_primary_zinc.key, RICE_primary_zincEle, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_castironGen, 'technosphere'),
        (EF_primary_castiron.key, RICE_primary_castironGen, 'technosphere'),
        (EF_recycling_copper.key, RICE_recycling_copperGen, 'technosphere'),
        (EF_primary_copper.key, RICE_primary_copperGen, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_reinforcingsteelSan, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_reinforcingsteelSan, 'technosphere'),
        (EF_recycling_aluminium.key, RICE_recycling_aluminiumStorage, 'technosphere'),
        (EF_primary_aluminium.key, RICE_primary_aluminiumStorage, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_reinforcingsteelStorage, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_reinforcingsteelStorage, 'technosphere'),
        (EF_recycling_aluminium.key, RICE_recycling_sheetrollingaluminiumStorage, 'technosphere'),
        (EF_primary_aluminium.key, RICE_primary_sheetrollingaluminiumStorage, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_sheetrollingsteelStorage, 'technosphere'),
        (EF_primary_steel.key, RICE_primary_sheetrollingsteelStorage, 'technosphere'),
        (EF_recycling_steel.key, RICE_recycling_chromiumsteelStorage, 'technosphere'),
        (EF_primary_stainlessSteel.key, RICE_primary_chromiumsteelStorage, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in D1_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_RICE = D1_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        D1_RICE_acts[bd_name].new_exchange(
            input=D1_RICE_acts[bd_name].key, 
            output=D1_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_RICE.lci()
    mlca_D1_RICE.lcia()
    mlca_D1_RICE.scores

    dfresults_D1_RICE = pd.DataFrame.from_dict(mlca_D1_RICE.scores, orient='index')
    dfresults_D1_RICE.index = pd.MultiIndex.from_tuples(dfresults_D1_RICE.index, names=['Column', 'Row'])
    dfresults_D1_RICE = dfresults_D1_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_RICE.columns = dfresults_D1_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_RICE = dfresults_D1_RICE.mul(SFRICE, axis=0)

### 8.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_RICE = dfresults_D1_RICE.add(dfresults_D1_RICE.mul(RICENumberReplacements, axis=0))

### 8.2.12 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_RICE_exchange_configs = [
        (EF_heat_energyrecovery.key, RICE_energyrecoveryheat_HDPEControlcabinet, 'technosphere'),
        (EF_electricity_energyrecovery.key, RICE_energyrecoveryele_HDPEControlcabinet, 'technosphere'),
        (EF_heat_energyrecovery.key, RICE_energyrecoveryheat_PVCemulsionatedControlcabinet, 'technosphere'),
        (EF_electricity_energyrecovery.key, RICE_energyrecoveryele_PVCemulsionatedControlcabinet, 'technosphere'),
        (EF_heat_energyrecovery.key, RICE_energyrecoveryheat_PVCpolimerizedControlcabinet, 'technosphere'),
        (EF_electricity_energyrecovery.key, RICE_energyrecoveryele_PVCpolimerizedControlcabinet, 'technosphere'),
        (EF_heat_energyrecovery.key, RICE_energyrecoveryheat_corrugatedboxboardConverter, 'technosphere'),
        (EF_electricity_energyrecovery.key, RICE_energyrecoveryele_corrugatedboxboardConverter, 'technosphere'),
        (EF_heat_energyrecovery.key, RICE_energyrecoveryheat_HDPEEle, 'technosphere'),
        (EF_electricity_energyrecovery.key, RICE_energyrecoveryele_HDPEEle, 'technosphere'),
        (EF_heat_energyrecovery.key, RICE_energyrecoveryheat_PVCpolimerizedEle, 'technosphere'),
        (EF_electricity_energyrecovery.key, RICE_energyrecoveryele_PVCpolimerizedEle, 'technosphere'),
        (EF_heat_energyrecovery.key, RICE_energyrecoveryheat_alkydpaintStorage, 'technosphere'),
        (EF_electricity_energyrecovery.key, RICE_energyrecoveryele_alkydpaintStorage, 'technosphere'),
        (EF_heat_energyrecovery.key, RICE_energyrecoveryheat_zeoliteConverter, 'technosphere'),
        (EF_electricity_energyrecovery.key, RICE_energyrecoveryele_zeoliteConverter, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_RICE_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_RICE_exchange_list = []
        
        for input_key, data_array, exc_type in D3_RICE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_RICE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_RICE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_RICE = D3_RICE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_RICE.save()

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_RICE_BdName:
        D3_RICE_acts[bd_name].new_exchange(
            input=D3_RICE_acts[bd_name].key, 
            output=D3_RICE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_RICE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_RICE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_RICE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_RICE_BdName, CHP_RICE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_RICE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_RICE.lci()
    mlca_D3_RICE.lcia()
    mlca_D3_RICE.scores

    dfresults_D3_RICE = pd.DataFrame.from_dict(mlca_D3_RICE.scores, orient='index')
    dfresults_D3_RICE.index = pd.MultiIndex.from_tuples(dfresults_D3_RICE.index, names=['Column', 'Row'])
    dfresults_D3_RICE = dfresults_D3_RICE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_RICE.columns = dfresults_D3_RICE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_RICE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_RICE = dfresults_D3_RICE.mul(SFRICE, axis=0)

### 8.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "CHP-Internal Combustion Engine"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_RICE = dfresults_D3_RICE.add(dfresults_D3_RICE.mul(RICENumberReplacements, axis=0))

# 9. Organic Rankine Cycle plants activities

## 9.1 Activities

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_ORC_{bd_name}',
        'name': f'A13_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            A13_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_ORC_{bd_name}',
        'name': f'A4_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            A4_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_ORC_{bd_name}',
        'name': f'A5_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            A5_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_ORC_{bd_name}',
        'name': f'B2_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            B2_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_ORC_{bd_name}',
        'name': f'B6_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            B6_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_ORC_{bd_name}',
        'name': f'C2_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            C2_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_ORC_{bd_name}',
        'name': f'C3_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            C3_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_ORC_{bd_name}',
        'name': f'C4_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            C4_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_ORC_{bd_name}',
        'name': f'D1_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            D1_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_ORC_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_ORC_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_ORC_{bd_name}',
        'name': f'D3_ORC_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_ORC_acts[bd_name] = scenario_db.new_activity(**data)
            D3_ORC_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 9.2 Exchanges

### 9.2.1 Modules A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_ORC_exchange_configs = [
        (EF_dustCollector.key, ORC_dustCollector, 'technosphere'),
        (EF_furnace.key, ORC_furnace, 'technosphere'),
        (EF_CHP.key, ORC_CHP, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in A13_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_ORC = A13_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        A13_ORC_acts[bd_name].new_exchange(
            input=A13_ORC_acts[bd_name].key, 
            output=A13_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_ORC.lci()
    mlca_A13_ORC.lcia()
    mlca_A13_ORC.scores

    dfresults_A13_ORC = pd.DataFrame.from_dict(mlca_A13_ORC.scores, orient='index')
    dfresults_A13_ORC.index = pd.MultiIndex.from_tuples(dfresults_A13_ORC.index, names=['Column', 'Row'])
    dfresults_A13_ORC = dfresults_A13_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_ORC.columns = dfresults_A13_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_ORC = dfresults_A13_ORC.mul(SFORC, axis=0)

### 9.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_ORC_exchange_configs = [
        (EF_Truk_16_32.key, ORC_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, ORC_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in A4_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_ORC = A4_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        A4_ORC_acts[bd_name].new_exchange(
            input=A4_ORC_acts[bd_name].key, 
            output=A4_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_ORC.lci()
    mlca_A4_ORC.lcia()
    mlca_A4_ORC.scores

    dfresults_A4_ORC = pd.DataFrame.from_dict(mlca_A4_ORC.scores, orient='index')
    dfresults_A4_ORC.index = pd.MultiIndex.from_tuples(dfresults_A4_ORC.index, names=['Column', 'Row'])
    dfresults_A4_ORC = dfresults_A4_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_ORC.columns = dfresults_A4_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_ORC = dfresults_A4_ORC.mul(SFORC, axis=0)

### 9.2.3 Module A5

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        A5_ORC_acts[bd_name].new_exchange(
            input=A5_ORC_acts[bd_name].key, 
            output=A5_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_ORC.lci()
    mlca_A5_ORC.lcia()
    mlca_A5_ORC.scores

    dfresults_A5_ORC = pd.DataFrame.from_dict(mlca_A5_ORC.scores, orient='index')
    dfresults_A5_ORC.index = pd.MultiIndex.from_tuples(dfresults_A5_ORC.index, names=['Column', 'Row'])
    dfresults_A5_ORC = dfresults_A5_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_ORC.columns = dfresults_A5_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_ORC = dfresults_A5_ORC.mul(SFORC, axis=0)

### 9.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_ORC_exchange_configs = [
        (EF_LightTruk.key, ORC_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in B2_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_ORC = B2_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        B2_ORC_acts[bd_name].new_exchange(
            input=B2_ORC_acts[bd_name].key, 
            output=B2_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_ORC.lci()
    mlca_B2_ORC.lcia()
    mlca_B2_ORC.scores

    dfresults_B2_ORC = pd.DataFrame.from_dict(mlca_B2_ORC.scores, orient='index')
    dfresults_B2_ORC.index = pd.MultiIndex.from_tuples(dfresults_B2_ORC.index, names=['Column', 'Row'])
    dfresults_B2_ORC = dfresults_B2_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_ORC.columns = dfresults_B2_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_ORC = dfresults_B2_ORC.mul(SFORC, axis=0)

### 9.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_ORC_exchange_configs = [
        (EF_Ammonia.key, ORC_Ammonia1, 'biosphere'),
        (EF_Chemical.key, ORC_Chemical, 'technosphere'),
        (EF_Chlorine.key, ORC_Chlorine, 'technosphere'),
        (EF_Mineraloilwaste.key, ORC_Oil, 'technosphere'),
        (EF_NOx.key, ORC_NOx, 'technosphere'),
        (EF_Sodium.key, ORC_Sodium, 'biosphere'),
        (EF_WaterB6.key, ORC_Water, 'biosphere'),
        (EF_Wood.key, ORC_Wood, 'technosphere'),
        (EF_Acetaldehyde.key, ORC_Acetaldehyde, 'biosphere'),
        (EF_Ammonia.key, ORC_Ammonia2, 'biosphere'),
        (EF_Benzene.key, ORC_Benzene, 'biosphere'),
        (EF_Benzenethyl.key, ORC_Benzeneethyl, 'biosphere'),
        (EF_Benzenehexachloro.key, ORC_Benzenehexachloro, 'biosphere'),
        (EF_Benzoapyrene.key, ORC_Benzoapyrene, 'biosphere'),
        (EF_Bromine.key, ORC_Bromine, 'biosphere'),
        (EF_Cadmium.key, ORC_Cadmium, 'biosphere'),
        (EF_Calcium.key, ORC_Calcium, 'biosphere'),
        (EF_CarbondioxideNonFossil.key, ORC_Carbondioxidebiogenic, 'biosphere'),
        (EF_CarbonmonoxideNonFossil.key, ORC_Carbonmonoxidebiogenic, 'biosphere'),
        (EF_Chlorine.key, ORC_Chlorine, 'technosphere'),
        (EF_ChromiumVI.key, ORC_Chromium, 'biosphere'),
        (EF_ChromiumVI.key, ORC_ChromiumVI, 'biosphere'),
        (EF_CopperIon.key, ORC_Copper, 'biosphere'),
        (EF_DinitrogenMonoxide.key, ORC_Dinitrogenmonoxide, 'biosphere'),
        (EF_Dioxin.key, ORC_Dioxin, 'biosphere'),
        (EF_Fluorine.key, ORC_Fluorine, 'biosphere'),
        (EF_Formaldehyde.key, ORC_Formaldehyde, 'biosphere'),
        (EF_HydrocarbonsAlkanes.key, ORC_Hydrocarbonsaliphaticalkanesunspecified, 'biosphere'),
        (EF_HydrocarbonsUnsaturated.key, ORC_Hydrocarbonsaliphaticunsaturated, 'biosphere'),
        (EF_LeadB6.key, ORC_Lead, 'biosphere'),
        (EF_Magnesium.key, ORC_Magnesium, 'biosphere'),
        (EF_Manganese.key, ORC_Manganese, 'biosphere'),
        (EF_Mercury.key, ORC_Mercury, 'biosphere'),
        (EF_Mxylene.key, ORC_mXylene, 'biosphere'),
        (EF_NickelB6.key, ORC_Nickel, 'biosphere'),
        (EF_NitrogenOxides.key, ORC_Nitrogenoxides, 'biosphere'),
        (EF_NMVOC.key, ORC_NMVOC, 'biosphere'),
        (EF_PAH.key, ORC_PAH, 'biosphere'),
        (EF_Particulates025.key, ORC_Particulates, 'biosphere'),
        (EF_Phosphorus.key, ORC_Phosphorus, 'biosphere'),
        (EF_Potassium.key, ORC_Potassium, 'biosphere'),
        (EF_Sodium.key, ORC_Sodium, 'biosphere'),
        (EF_SulfurDioxide.key, ORC_Sulfurdioxide, 'biosphere'),
        (EF_Toluene.key, ORC_Toluene, 'biosphere'),
        (EF_WaterB6.key, ORC_Watermc, 'biosphere'),
        (EF_Zinc.key, ORC_Zinc, 'biosphere'),
        (EF_Solidwaste.key, ORC_Solidwaste, 'technosphere'),
        (EF_Mineraloilwaste.key, ORC_Mineraloilwaste, 'technosphere'),
        (EF_Wastewater.key, ORC_Wastewater, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in B6_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_ORC = B6_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        B6_ORC_acts[bd_name].new_exchange(
            input=B6_ORC_acts[bd_name].key, 
            output=B6_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_ORC.lci()
    mlca_B6_ORC.lcia()
    mlca_B6_ORC.scores

    dfresults_B6_ORC = pd.DataFrame.from_dict(mlca_B6_ORC.scores, orient='index')
    dfresults_B6_ORC.index = pd.MultiIndex.from_tuples(dfresults_B6_ORC.index, names=['Column', 'Row'])
    dfresults_B6_ORC = dfresults_B6_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_ORC.columns = dfresults_B6_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_ORC.rename(columns=method_lookup, inplace=True)

### 9.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_ORC_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, ORC_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in C2_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_ORC = C2_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        C2_ORC_acts[bd_name].new_exchange(
            input=C2_ORC_acts[bd_name].key, 
            output=C2_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_ORC.lci()
    mlca_C2_ORC.lcia()
    mlca_C2_ORC.scores

    dfresults_C2_ORC = pd.DataFrame.from_dict(mlca_C2_ORC.scores, orient='index')
    dfresults_C2_ORC.index = pd.MultiIndex.from_tuples(dfresults_C2_ORC.index, names=['Column', 'Row'])
    dfresults_C2_ORC = dfresults_C2_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_ORC.columns = dfresults_C2_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_ORC = dfresults_C2_ORC.mul(SFORC, axis=0)

### 9.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_ORC_exchange_configs = [
        (EF_EOL_Aluminium.key, ORC_EOL_aluminiumDustcollector, 'technosphere'),
        (EF_EOL_inert.key, ORC_EOL_ceramictileDustcollector, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_copperDustcollector, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_drawingpipeSteelDustcollector, 'technosphere'),
        (EF_EOL_electronics.key, ORC_EOL_electronicsDustcollector, 'technosphere'),
        (EF_EOL_inert.key, ORC_EOL_glasswoolmatDustcollector, 'technosphere'),
        (EF_EOL_plastic.key, ORC_EOL_HDPEDustcollector, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_sheetrollingsteelDustcollector, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_chromiumsteelDustcollector, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_lowalloyedsteelDustcollector, 'technosphere'),
        (EF_EOL_alkydpaint.key, ORC_EOL_alkydpaintFurnace, 'technosphere'),
        (EF_EOL_Aluminium.key, ORC_EOL_aluminiumFurnace, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_castironFurnace, 'technosphere'),
        (EF_EOL_concrete.key, ORC_EOL_concreteFurnace, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_copperFurnace, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_drawingpipeFurnace, 'technosphere'),
        (EF_EOL_electronics.key, ORC_EOL_electronicsFurnace, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_ironnickelchromiumalloyFurnace, 'technosphere'),
        (EF_EOL_inert.key, ORC_EOL_lubricatingoilFurnace, 'technosphere'),
        (EF_EOL_plastic.key, ORC_EOL_HDPEFurnace, 'technosphere'),
        (EF_EOL_plastic.key, ORC_EOL_polystyreneFurnace, 'technosphere'),
        (EF_EOL_claybrick.key, ORC_EOL_claybrickFurnace, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_sheetrollingsteelFurnace, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_chromiumsteelFurnace, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_lowalloyedsteelFurnace, 'technosphere'),
        (EF_EOL_stonewool.key, ORC_EOL_stonewoolFurnace, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_copperCHP, 'technosphere'),
        (EF_EOL_electronics.key, ORC_EOL_electronicsCHP, 'technosphere'),
        (EF_EOL_inert.key, ORC_EOL_perfluoropentaneCHP, 'technosphere'),
        (EF_EOL_plastic.key, ORC_EOL_HDPECHP, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_chromiumsteelCHP, 'technosphere'),
        (EF_EOL_metal.key, ORC_EOL_lowalloyedsteelCHP, 'technosphere'),
        (EF_EOL_stonewool.key, ORC_EOL_stonewoolCHP, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in C3_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_ORC = C3_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        C3_ORC_acts[bd_name].new_exchange(
            input=C3_ORC_acts[bd_name].key, 
            output=C3_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_ORC.lci()
    mlca_C3_ORC.lcia()
    mlca_C3_ORC.scores

    dfresults_C3_ORC = pd.DataFrame.from_dict(mlca_C3_ORC.scores, orient='index')
    dfresults_C3_ORC.index = pd.MultiIndex.from_tuples(dfresults_C3_ORC.index, names=['Column', 'Row'])
    dfresults_C3_ORC = dfresults_C3_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_ORC.columns = dfresults_C3_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_ORC = dfresults_C3_ORC.mul(SFORC, axis=0)

### 9.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_ORC_exchange_configs = [
        (EF_EOL_landfill_aluminium.key, ORC_EOL_landfill_aluminiumDustcollector, 'technosphere'),
        (EF_EOL_landfill_inert.key, ORC_EOL_landfill_ceramictileDustcollector, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_copperDustcollector, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_drawingpipesteelDustcollector, 'technosphere'),
        (EF_EOL_inc_electronic.key, ORC_EOL_inc_electronicsDustcollector, 'technosphere'),
        (EF_EOL_landfill_electronic.key, ORC_EOL_landfill_electronicsDustcollector, 'technosphere'),
        (EF_EOL_landfill_glass.key, ORC_EOL_landfill_glasswoolmatDustcollector, 'technosphere'),
        (EF_EOL_inc_PEHD.key, ORC_EOL_inc_HDPEDustcollector, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, ORC_EOL_landfill_HDPEDustcollector, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_sheetrollingsteelDustcollector, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_chromiumsteelDustcollector, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_lowalloyedsteelDustcollector, 'technosphere'),
        (EF_EOL_inc_alkydpaint.key, ORC_EOL_inc_alkydpaintFurnace, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, ORC_EOL_landfill_aluminiumFurnace, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_castironFurnace, 'technosphere'),
        (EF_EOL_landfill_concrete.key, ORC_EOL_landfill_concreteFurnace, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_copperFurnace, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_drawingpipesteelFurnace, 'technosphere'),
        (EF_EOL_inc_electronic.key, ORC_EOL_inc_electronicsFurnace, 'technosphere'),
        (EF_EOL_landfill_electronic.key, ORC_EOL_landfill_electronicsFurnace, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_ironnickelchromiumalloyFurnace, 'technosphere'),
        (EF_EOL_inc_lubricatingoil.key, ORC_EOL_inc_lubricatingoilFurnace, 'technosphere'),
        (EF_EOL_inc_PEHD.key, ORC_EOL_inc_HDPEFurnace, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, ORC_EOL_landfill_HDPEFurnace, 'technosphere'),
        (EF_EOL_inc_plastic.key, ORC_EOL_inc_polystyreneFurnace, 'technosphere'),
        (EF_EOL_landfill_inert.key, ORC_EOL_landfill_claybrickFurnace, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_sheetrollingsteelFurnace, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_chromiumsteelFurnace, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_lowalloyedsteelFurnace, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, ORC_EOL_landfill_stonewoolFurnace, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_copperCHP, 'technosphere'),
        (EF_EOL_inc_electronic.key, ORC_EOL_inc_electronicsCHP, 'technosphere'),
        (EF_EOL_landfill_electronic.key, ORC_EOL_landfill_electronicsCHP, 'technosphere'),
        (EF_EOL_inc_hazardouswaste.key, ORC_EOL_inc_perfluoropentaneCHP, 'technosphere'),
        (EF_EOL_inc_PEHD.key, ORC_EOL_inc_HDPECHP, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, ORC_EOL_landfill_HDPEDCHP, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_chromiumsteelCHP, 'technosphere'),
        (EF_EOL_landfill_metal.key, ORC_EOL_landfill_lowalloyedsteelCHP, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, ORC_EOL_landfill_stonewoolCHP, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in C4_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_ORC = C4_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        C4_ORC_acts[bd_name].new_exchange(
            input=C4_ORC_acts[bd_name].key, 
            output=C4_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_ORC.lci()
    mlca_C4_ORC.lcia()
    mlca_C4_ORC.scores

    dfresults_C4_ORC = pd.DataFrame.from_dict(mlca_C4_ORC.scores, orient='index')
    dfresults_C4_ORC.index = pd.MultiIndex.from_tuples(dfresults_C4_ORC.index, names=['Column', 'Row'])
    dfresults_C4_ORC = dfresults_C4_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_ORC.columns = dfresults_C4_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_ORC = dfresults_C4_ORC.mul(SFORC, axis=0)

### 9.2.9 Module B4

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_ORC = pd.DataFrame(index=dfresults_A13_ORC.index)  # Start with the index from A13 for consistency   
    dfresults_B4_ORC = (
        dfresults_A13_ORC.add(dfresults_A4_ORC, fill_value=0)
        .add(dfresults_C3_ORC, fill_value=0)
        .add(dfresults_C4_ORC, fill_value=0)
        ).mul(ORCNumberReplacements, axis=0)
    

### 9.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_ORC_exchange_configs = [
        (EF_recycling_aluminium.key, ORC_EOL_recycling_aluminiumDustcollector, 'technosphere'),
        (EF_primary_aluminium.key, ORC_EOL_primary_aluminiumDustcollector, 'technosphere'),
        (EF_recycling_copper.key, ORC_EOL_recycling_copperDustcollector, 'technosphere'),
        (EF_primary_copper.key, ORC_EOL_primary_copperDustcollector, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_drawingpipeDustcollector, 'technosphere'),
        (EF_primary_steel.key, ORC_EOL_primary_drawingpipeDustcollector, 'technosphere'),
        (EF_recycling_PEHD.key, ORC_EOL_recycling_HDPEDustcollector, 'technosphere'),
        (EF_primary_PEHD.key, ORC_EOL_primary_HDPEDustcollector, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_sheetrollingsteelDustcollector, 'technosphere'),
        (EF_primary_steel.key, ORC_EOL_primary_sheetrollingsteelDustcollector, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_chromiumsteelDustcollector, 'technosphere'),
        (EF_primary_stainlessSteel.key, ORC_EOL_primary_chromiumsteelDustcollector, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_lowalloyedsteelDustcollector, 'technosphere'),
        (EF_primary_steel.key, ORC_EOL_primary_lowalloyedsteelDustcollector, 'technosphere'),
        (EF_recycling_aluminium.key, ORC_EOL_recycling_aluminiumFurnace, 'technosphere'),
        (EF_primary_aluminium.key, ORC_EOL_primary_aluminiumFurnace, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_castironFurnace, 'technosphere'),
        (EF_primary_castiron.key, ORC_EOL_primary_castironFurnace, 'technosphere'),
        (EF_recycling_concrete.key, ORC_EOL_recycling_concreteFurnace, 'technosphere'),
        (EF_primary_concrete.key, ORC_EOL_primary_concreteFurnace, 'technosphere'),
        (EF_recycling_copper.key, ORC_EOL_recycling_copperFurnace, 'technosphere'),
        (EF_primary_copper.key, ORC_EOL_primary_copperFurnace, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_drawingpipesteelFurnace, 'technosphere'),
        (EF_primary_steel.key, ORC_EOL_primary_drawingpipesteelFurnace, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_ironickelchromiumsteelFurnace, 'technosphere'),
        (EF_primary_stainlessSteel.key, ORC_EOL_primary_ironickelchromiumsteelFurnace, 'technosphere'),
        (EF_recycling_PEHD.key, ORC_EOL_recycling_HDPEFurnace, 'technosphere'),
        (EF_primary_PEHD.key, ORC_EOL_primary_HDPEFurnace, 'technosphere'),
        (EF_recycling_claybrick.key, ORC_EOL_recycling_claybrickFurnace, 'technosphere'),
        (EF_primary_claybrick.key, ORC_EOL_primary_claybrickFurnace, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_sheetrollingsteelFurnace, 'technosphere'),
        (EF_primary_steel.key, ORC_EOL_primary_sheetrollingsteelFurnace, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_chromiumsteelFurnace, 'technosphere'),
        (EF_primary_stainlessSteel.key, ORC_EOL_primary_chromiumsteelFurnace, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_lowalloyedsteelFurnace, 'technosphere'),
        (EF_primary_steel.key, ORC_EOL_primary_lowalloyedsteelFurnace, 'technosphere'),
        (EF_recycling_copper.key, ORC_EOL_recycling_copperCHP, 'technosphere'),
        (EF_primary_copper.key, ORC_EOL_primary_copperCHP, 'technosphere'),
        (EF_recycling_PEHD.key, ORC_EOL_recycling_HDPECHP, 'technosphere'),
        (EF_primary_PEHD.key, ORC_EOL_primary_HDPECHP, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_chromiumsteelCHP, 'technosphere'),
        (EF_primary_stainlessSteel.key, ORC_EOL_primary_chromiumsteelCHP, 'technosphere'),
        (EF_recycling_steel.key, ORC_EOL_recycling_lowalloyedsteelCHP, 'technosphere'),
        (EF_primary_steel.key, ORC_EOL_primary_lowalloyedsteelCHP, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in D1_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_ORC = D1_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        D1_ORC_acts[bd_name].new_exchange(
            input=D1_ORC_acts[bd_name].key, 
            output=D1_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_ORC.lci()
    mlca_D1_ORC.lcia()
    mlca_D1_ORC.scores

    dfresults_D1_ORC = pd.DataFrame.from_dict(mlca_D1_ORC.scores, orient='index')
    dfresults_D1_ORC.index = pd.MultiIndex.from_tuples(dfresults_D1_ORC.index, names=['Column', 'Row'])
    dfresults_D1_ORC = dfresults_D1_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_ORC.columns = dfresults_D1_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_ORC = dfresults_D1_ORC.mul(SFORC, axis=0)

### 9.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_ORC = dfresults_D1_ORC.add(dfresults_D1_ORC.mul(ORCNumberReplacements, axis=0))

### 9.2.12 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_ORC_exchange_configs = [
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_electronicsDustcollector, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_electronicsDustcollector, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_HDPEDustcollector, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_HDPEDustcollector, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_alkydpaintFurnace, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_alkydpaintFurnace, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_electronicsFurnace, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_electronicsFurnace, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_lubricatingoilFurnace, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_lubricatingoilFurnace, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_HDPEFurnace, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_HDPEFurnace, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_polystyreneFurnace, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_polystyreneFurnace, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_electronics, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_electronics, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_perfluoropentaneCHP, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_perfluoropentaneCHP, 'technosphere'),
        (EF_heat_energyrecovery.key, ORC_EOL_energyrecoveryheat_HDPECHP, 'technosphere'),
        (EF_electricity_energyrecovery.key, ORC_EOL_energyrecoveryele_HDPECHP, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_ORC_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_ORC_exchange_list = []
        
        for input_key, data_array, exc_type in D3_ORC_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_ORC_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_ORC_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_ORC = D3_ORC_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_ORC.save()

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_ORC_BdName:
        D3_ORC_acts[bd_name].new_exchange(
            input=D3_ORC_acts[bd_name].key, 
            output=D3_ORC_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_ORC = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_ORC['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_ORC_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_ORC_BdName, CHP_ORC_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_ORC = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_ORC.lci()
    mlca_D3_ORC.lcia()
    mlca_D3_ORC.scores

    dfresults_D3_ORC = pd.DataFrame.from_dict(mlca_D3_ORC.scores, orient='index')
    dfresults_D3_ORC.index = pd.MultiIndex.from_tuples(dfresults_D3_ORC.index, names=['Column', 'Row'])
    dfresults_D3_ORC = dfresults_D3_ORC.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_ORC.columns = dfresults_D3_ORC.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_ORC.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_ORC = dfresults_D3_ORC.mul(SFORC, axis=0)

### 9.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "CHP-Organic Rankine Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_ORC = dfresults_D3_ORC.add(dfresults_D3_ORC.mul(ORCNumberReplacements, axis=0))

# 10. Waste to energy plants activity

## 10.1 Activities

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_WtE_{bd_name}',
        'name': f'A13_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            A13_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_WtE_{bd_name}',
        'name': f'A4_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            A4_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_WtE_{bd_name}',
        'name': f'A5_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            A5_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_WtE_{bd_name}',
        'name': f'B2_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            B2_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_WtE_{bd_name}',
        'name': f'B6_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            B6_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_WtE_{bd_name}',
        'name': f'C2_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            C2_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_WtE_{bd_name}',
        'name': f'C3_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            C3_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_WtE_{bd_name}',
        'name': f'C4_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            C4_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_WtE_{bd_name}',
        'name': f'D1_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            D1_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Waste-to-Energy"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_WtE_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_WtE_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_WtE_{bd_name}',
        'name': f'D3_WtE_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_WtE_acts[bd_name] = scenario_db.new_activity(**data)
            D3_WtE_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 10.2 Exchanges

### 10.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_WtE_exchange_configs = [
        (EF_cement.key, WtE_cement, 'technosphere'),
        (EF_gravel.key, WtE_gravel, 'technosphere'),
        (EF_pitch.key, WtE_pitch, 'technosphere'),
        (EF_reinforcingsteel.key, WtE_reinforcingsteel, 'technosphere'),
        (EF_sand.key, WtE_sand, 'technosphere'),
        (EF_StainlessSteel.key, WtE_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_WtE_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_WtE_exchange_list = []
        
        for input_key, data_array, exc_type in A13_WtE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_WtE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_WtE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_WtE = A13_WtE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_WtE.save()

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        A13_WtE_acts[bd_name].new_exchange(
            input=A13_WtE_acts[bd_name].key, 
            output=A13_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_WtE.lci()
    mlca_A13_WtE.lcia()
    mlca_A13_WtE.scores

    dfresults_A13_WtE = pd.DataFrame.from_dict(mlca_A13_WtE.scores, orient='index')
    dfresults_A13_WtE.index = pd.MultiIndex.from_tuples(dfresults_A13_WtE.index, names=['Column', 'Row'])
    dfresults_A13_WtE = dfresults_A13_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_WtE.columns = dfresults_A13_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_WtE = dfresults_A13_WtE.mul(SFWtE, axis=0)

### 10.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_WtE_exchange_configs = [
        (EF_Truk_16_32.key, WtE_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, WtE_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_WtE_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_WtE_exchange_list = []
        
        for input_key, data_array, exc_type in A4_WtE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_WtE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_WtE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_WtE = A4_WtE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_WtE.save()

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        A4_WtE_acts[bd_name].new_exchange(
            input=A4_WtE_acts[bd_name].key, 
            output=A4_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_WtE.lci()
    mlca_A4_WtE.lcia()
    mlca_A4_WtE.scores

    dfresults_A4_WtE = pd.DataFrame.from_dict(mlca_A4_WtE.scores, orient='index')
    dfresults_A4_WtE.index = pd.MultiIndex.from_tuples(dfresults_A4_WtE.index, names=['Column', 'Row'])
    dfresults_A4_WtE = dfresults_A4_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_WtE.columns = dfresults_A4_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_WtE = dfresults_A4_WtE.mul(SFWtE, axis=0)

### 10.2.3 Module A5

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        A5_WtE_acts[bd_name].new_exchange(
            input=A5_WtE_acts[bd_name].key, 
            output=A5_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_WtE.lci()
    mlca_A5_WtE.lcia()
    mlca_A5_WtE.scores

    dfresults_A5_WtE = pd.DataFrame.from_dict(mlca_A5_WtE.scores, orient='index')
    dfresults_A5_WtE.index = pd.MultiIndex.from_tuples(dfresults_A5_WtE.index, names=['Column', 'Row'])
    dfresults_A5_WtE = dfresults_A5_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_WtE.columns = dfresults_A5_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_WtE = dfresults_A5_WtE.mul(SFWtE, axis=0)

### 10.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_WtE_exchange_configs = [
        (EF_LightTruk.key, WtE_LightTruk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_WtE_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_WtE_exchange_list = []
        
        for input_key, data_array, exc_type in B2_WtE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_WtE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_WtE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_WtE = B2_WtE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_WtE.save()

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        B2_WtE_acts[bd_name].new_exchange(
            input=B2_WtE_acts[bd_name].key, 
            output=B2_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_WtE.lci()
    mlca_B2_WtE.lcia()
    mlca_B2_WtE.scores

    dfresults_B2_WtE = pd.DataFrame.from_dict(mlca_B2_WtE.scores, orient='index')
    dfresults_B2_WtE.index = pd.MultiIndex.from_tuples(dfresults_B2_WtE.index, names=['Column', 'Row'])
    dfresults_B2_WtE = dfresults_B2_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_WtE.columns = dfresults_B2_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_WtE = dfresults_B2_WtE.mul(SFWtE, axis=0)

### 10.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_WtE_exchange_configs = [
        (EF_SolidWasteTreatment.key, WtE_Waste, 'technosphere'),
        (EF_municipalsolidwaste.key, WtE_municipalsolidwaste, 'technosphere'),
        (EF_energyvectorNG.key, WtE_EnergyVectorNG, 'technosphere'),
        (EF_energyvectorBiom.key, WtE_EnergyVectorBiomethane, 'technosphere'),
        (EF_energyvectorHydrogen.key, WtE_EnergyVectorHydrogen, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_WtE_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_WtE_exchange_list = []
        
        for input_key, data_array, exc_type in B6_WtE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_WtE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_WtE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_WtE = B6_WtE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_WtE.save()

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        B6_WtE_acts[bd_name].new_exchange(
            input=B6_WtE_acts[bd_name].key, 
            output=B6_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_WtE.lci()
    mlca_B6_WtE.lcia()
    mlca_B6_WtE.scores

    dfresults_B6_WtE = pd.DataFrame.from_dict(mlca_B6_WtE.scores, orient='index')
    dfresults_B6_WtE.index = pd.MultiIndex.from_tuples(dfresults_B6_WtE.index, names=['Column', 'Row'])
    dfresults_B6_WtE = dfresults_B6_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_WtE.columns = dfresults_B6_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_WtE.rename(columns=method_lookup, inplace=True)

### 10.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_WtE_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, WtE_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_WtE_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_WtE_exchange_list = []
        
        for input_key, data_array, exc_type in C2_WtE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_WtE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_WtE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_WtE = C2_WtE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_WtE.save()

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        C2_WtE_acts[bd_name].new_exchange(
            input=C2_WtE_acts[bd_name].key, 
            output=C2_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_WtE.lci()
    mlca_C2_WtE.lcia()
    mlca_C2_WtE.scores

    dfresults_C2_WtE = pd.DataFrame.from_dict(mlca_C2_WtE.scores, orient='index')
    dfresults_C2_WtE.index = pd.MultiIndex.from_tuples(dfresults_C2_WtE.index, names=['Column', 'Row'])
    dfresults_C2_WtE = dfresults_C2_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_WtE.columns = dfresults_C2_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_WtE = dfresults_C2_WtE.mul(SFWtE, axis=0)

### 10.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_WtE_exchange_configs = [
        (EF_EOL_cement.key, WtE_EOL_cement, 'technosphere'),
        (EF_EOL_inert.key, WtE_EOL_gravel, 'technosphere'),
        (EF_EOL_inert.key, WtE_EOL_pitch, 'technosphere'),
        (EF_EOL_metal.key, WtE_EOL_reinforcingsteel, 'technosphere'),
        (EF_EOL_inert.key, WtE_EOL_sand, 'technosphere'),
        (EF_EOL_metal.key, WtE_EOL_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_WtE_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_WtE_exchange_list = []
        
        for input_key, data_array, exc_type in C3_WtE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_WtE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_WtE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_WtE = C3_WtE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_WtE.save()

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        C3_WtE_acts[bd_name].new_exchange(
            input=C3_WtE_acts[bd_name].key, 
            output=C3_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_WtE.lci()
    mlca_C3_WtE.lcia()
    mlca_C3_WtE.scores

    dfresults_C3_WtE = pd.DataFrame.from_dict(mlca_C3_WtE.scores, orient='index')
    dfresults_C3_WtE.index = pd.MultiIndex.from_tuples(dfresults_C3_WtE.index, names=['Column', 'Row'])
    dfresults_C3_WtE = dfresults_C3_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_WtE.columns = dfresults_C3_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_WtE = dfresults_C3_WtE.mul(SFWtE, axis=0)

### 10.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_WtE_exchange_configs = [
        (EF_EOL_landfill_inert.key, WtE_EOL_landfill_cement, 'technosphere'),
        (EF_EOL_landfill_inert.key, WtE_EOL_landfill_gravel, 'technosphere'),
        (EF_EOL_landfill_inert.key, WtE_EOL_landfill_pitch, 'technosphere'),
        (EF_EOL_landfill_metal.key, WtE_EOL_landfill_reinforcingsteel, 'technosphere'),
        (EF_EOL_landfill_inert.key, WtE_EOL_landfill_sand, 'technosphere'),
        (EF_EOL_landfill_metal.key, WtE_EOL_landfill_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_WtE_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_WtE_exchange_list = []
        
        for input_key, data_array, exc_type in C4_WtE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_WtE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_WtE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_WtE = C4_WtE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_WtE.save()

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        C4_WtE_acts[bd_name].new_exchange(
            input=C4_WtE_acts[bd_name].key, 
            output=C4_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_WtE.lci()
    mlca_C4_WtE.lcia()
    mlca_C4_WtE.scores

    dfresults_C4_WtE = pd.DataFrame.from_dict(mlca_C4_WtE.scores, orient='index')
    dfresults_C4_WtE.index = pd.MultiIndex.from_tuples(dfresults_C4_WtE.index, names=['Column', 'Row'])
    dfresults_C4_WtE = dfresults_C4_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_WtE.columns = dfresults_C4_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_WtE = dfresults_C4_WtE.mul(SFWtE, axis=0)

### 10.2.9 Module B4

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_WtE = pd.DataFrame(index=dfresults_A13_WtE.index)  # Start with the index from A13 for consistency   
    dfresults_B4_WtE = (
        dfresults_A13_WtE.add(dfresults_A4_WtE, fill_value=0)
        .add(dfresults_C3_WtE, fill_value=0)
        .add(dfresults_C4_WtE, fill_value=0)
        ).mul(WtENumberReplacements, axis=0)
    

### 10.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_WtE_exchange_configs = [
        (EF_recycling_steel.key, WtE_EOL_recycling_reinforcingsteel, 'technosphere'),
        (EF_primary_steel.key, WtE_EOL_primary_reinforcingsteel, 'technosphere'),
        (EF_recycling_steel.key, WtE_EOL_recycling_chromiumsteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, WtE_EOL_primary_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_WtE_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_WtE_exchange_list = []
        
        for input_key, data_array, exc_type in D1_WtE_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_WtE_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_WtE_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_WtE = D1_WtE_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_WtE.save()

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        D1_WtE_acts[bd_name].new_exchange(
            input=D1_WtE_acts[bd_name].key, 
            output=D1_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_WtE.lci()
    mlca_D1_WtE.lcia()
    mlca_D1_WtE.scores

    dfresults_D1_WtE = pd.DataFrame.from_dict(mlca_D1_WtE.scores, orient='index')
    dfresults_D1_WtE.index = pd.MultiIndex.from_tuples(dfresults_D1_WtE.index, names=['Column', 'Row'])
    dfresults_D1_WtE = dfresults_D1_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_WtE.columns = dfresults_D1_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_WtE = dfresults_D1_WtE.mul(SFWtE, axis=0)

### 10.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_WtE = dfresults_D1_WtE.add(dfresults_D1_WtE.mul(WtENumberReplacements, axis=0))

### 10.2.12 Module D3 (related to the export of energy as a results of waste incineration)

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_WtE_BdName:
        D3_WtE_acts[bd_name].new_exchange(
            input=D3_WtE_acts[bd_name].key, 
            output=D3_WtE_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_WtE = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_WtE['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_WtE_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_WtE_BdName, CHP_WtE_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_WtE = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_WtE.lci()
    mlca_D3_WtE.lcia()
    mlca_D3_WtE.scores

    dfresults_D3_WtE = pd.DataFrame.from_dict(mlca_D3_WtE.scores, orient='index')
    dfresults_D3_WtE.index = pd.MultiIndex.from_tuples(dfresults_D3_WtE.index, names=['Column', 'Row'])
    dfresults_D3_WtE = dfresults_D3_WtE.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_WtE.columns = dfresults_D3_WtE.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_WtE.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_WtE = dfresults_D3_WtE.mul(SFWtE, axis=0)

### 10.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "CHP-Waste-to-Energy"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_WtE = dfresults_D3_WtE.add(dfresults_D3_WtE.mul(WtENumberReplacements, axis=0))

# 11. Combined Cycle plants

## 11.1 Activities

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_CombinedCycle_{bd_name}',
        'name': f'A13_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            A13_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_CombinedCycle_{bd_name}',
        'name': f'A4_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            A4_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_CombinedCycle_{bd_name}',
        'name': f'A5_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            A5_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_CombinedCycle_{bd_name}',
        'name': f'B2_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            B2_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_CombinedCycle_{bd_name}',
        'name': f'B6_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            B6_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_CombinedCycle_{bd_name}',
        'name': f'C2_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            C2_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_CombinedCycle_{bd_name}',
        'name': f'C3_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            C3_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_CombinedCycle_{bd_name}',
        'name': f'C4_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            C4_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_CombinedCycle_{bd_name}',
        'name': f'D1_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            D1_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Combined Cycle"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_CombinedCycle_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_CombCycle_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_CombinedCycle_{bd_name}',
        'name': f'D3_CombinedCycle_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_CombinedCycle_acts[bd_name] = scenario_db.new_activity(**data)
            D3_CombinedCycle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 11.2 Exchanges

### 11.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_CombinedCycle_exchange_configs = [
        (EF_AluminiumCast.key, CombinedCycle_aluminiumCastalloy, 'technosphere'),
        (EF_Aluminium.key, CombinedCycle_aluminiumWroughtalloy, 'technosphere'),
        (EF_ceramictile.key, CombinedCycle_ceramictile, 'technosphere'),
        (EF_StainlessSteel.key, CombinedCycle_chromium, 'technosphere'),
        (EF_StainlessSteel.key, CombinedCycle_cobalt, 'technosphere'),
        (EF_concreteAT.key, CombinedCycle_concreteAT, 'technosphere'),
        (EF_concreteCH.key, CombinedCycle_concreteCH, 'technosphere'),
        (EF_Copper.key, CombinedCycle_copper, 'technosphere'),
        (EF_diesel.key, CombinedCycle_diesel, 'technosphere'),
        (EF_Electricity.key, CombinedCycle_electricityA13, 'technosphere'),
        (EF_Heat.key, CombinedCycle_heat, 'technosphere'),
        (EF_StainlessSteel.key, CombinedCycle_nickel, 'technosphere'),
        (EF_polyethylene.key, CombinedCycle_PEHD, 'technosphere'),
        (EF_steel.key, CombinedCycle_reinforcingsteel, 'technosphere'),
        (EF_StainlessSteel.key, CombinedCycle_chromiumsteel, 'technosphere'),
        (EF_mineralwool.key, CombinedCycle_stonewool, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in A13_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in A13_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_CombinedCycle = A13_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        A13_CombinedCycle_acts[bd_name].new_exchange(
            input=A13_CombinedCycle_acts[bd_name].key, 
            output=A13_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_CombinedCycle.lci()
    mlca_A13_CombinedCycle.lcia()
    mlca_A13_CombinedCycle.scores

    dfresults_A13_CombinedCycle = pd.DataFrame.from_dict(mlca_A13_CombinedCycle.scores, orient='index')
    dfresults_A13_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_A13_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_A13_CombinedCycle = dfresults_A13_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_CombinedCycle.columns = dfresults_A13_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_CombinedCycle = dfresults_A13_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.2 Module A4 

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_CombinedCycle_exchange_configs = [
        (EF_Truk_16_32.key, CombinedCycle_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, CombinedCycle_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in A4_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_CombinedCycle = A4_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        A4_CombinedCycle_acts[bd_name].new_exchange(
            input=A4_CombinedCycle_acts[bd_name].key, 
            output=A4_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_CombinedCycle.lci()
    mlca_A4_CombinedCycle.lcia()
    mlca_A4_CombinedCycle.scores

    dfresults_A4_CombinedCycle = pd.DataFrame.from_dict(mlca_A4_CombinedCycle.scores, orient='index')
    dfresults_A4_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_A4_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_A4_CombinedCycle = dfresults_A4_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_CombinedCycle.columns = dfresults_A4_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_CombinedCycle = dfresults_A4_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.3 Module A5

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        A5_CombinedCycle_acts[bd_name].new_exchange(
            input=A5_CombinedCycle_acts[bd_name].key, 
            output=A5_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_CombinedCycle.lci()
    mlca_A5_CombinedCycle.lcia()
    mlca_A5_CombinedCycle.scores

    dfresults_A5_CombinedCycle = pd.DataFrame.from_dict(mlca_A5_CombinedCycle.scores, orient='index')
    dfresults_A5_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_A5_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_A5_CombinedCycle = dfresults_A5_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_CombinedCycle.columns = dfresults_A5_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_CombinedCycle = dfresults_A5_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_CombinedCycle_exchange_configs = [
        (EF_LightTruk.key, CombinedCycle_LightTruk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in B2_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_CombinedCycle = B2_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        B2_CombinedCycle_acts[bd_name].new_exchange(
            input=B2_CombinedCycle_acts[bd_name].key, 
            output=B2_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_CombinedCycle.lci()
    mlca_B2_CombinedCycle.lcia()
    mlca_B2_CombinedCycle.scores

    dfresults_B2_CombinedCycle = pd.DataFrame.from_dict(mlca_B2_CombinedCycle.scores, orient='index')
    dfresults_B2_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_B2_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_B2_CombinedCycle = dfresults_B2_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_CombinedCycle.columns = dfresults_B2_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_CombinedCycle = dfresults_B2_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_CombinedCycle_exchange_configs = [
        (EF_energyvectorNG.key, CombinedCycle_EnergyVectorNG, 'technosphere'),
        (EF_energyvectorBiom.key, CombinedCycle_EnergyVectorBiomethane, 'technosphere'),
        (EF_energyvectorHydrogen.key, CombinedCycle_EnergyVectorHydrogen, 'technosphere'),
        (EF_biogas.key, CombinedCycle_EnergyVectorInputBiogas, 'technosphere'),
        (EF_energyvectorHydrogen.key, CombinedCycle_EnergyVectorInputHydrogen, 'technosphere'),
        (EF_watercoolingtower.key, CombinedCycle_WaterCooling, 'technosphere'),
        (EF_waterIn2.key, CombinedCycle_WaterIn2, 'technosphere'),
        (EF_Water.key, CombinedCycle_WaterIn3, 'technosphere'),
        (EF_WaterB6.key, CombinedCycle_WaterOut4, 'biosphere'),
        (EF_Acenaphtene.key, CombinedCycle_Acenaphthene, 'biosphere'),
        (EF_Acetaldehyde.key, CombinedCycle_Acetaldehyde, 'biosphere'),
        (EF_Aceticacid.key, CombinedCycle_AceticAcid, 'biosphere'),
        (EF_Benzene.key, CombinedCycle_Benzene, 'biosphere'),
        (EF_Benzoapyrene.key, CombinedCycle_Benzoapyrene, 'biosphere'),
        (EF_Beryllium.key, CombinedCycle_Beryllium, 'biosphere'),
        (EF_Buthane.key, CombinedCycle_Butane, 'biosphere'),
        (EF_Cadmium.key, CombinedCycle_Cadmium, 'biosphere'),
        (EF_CarbondioxideFossil.key, CombinedCycle_CarbonDioxideFossil, 'biosphere'),
        (EF_CarbonmonoxideFossil.key, CombinedCycle_CarbonMonoxideFossil, 'biosphere'),
        (EF_ChromiumVI.key, CombinedCycle_Chromium, 'biosphere'),
        (EF_CobaltB6.key, CombinedCycle_Cobalt, 'biosphere'),
        (EF_DinitrogenMonoxide.key, CombinedCycle_DinitrogenMonoxide, 'biosphere'),
        (EF_Ethane.key, CombinedCycle_Ethane, 'biosphere'),
        (EF_Formaldehyde.key, CombinedCycle_Formaldehyde, 'biosphere'),
        (EF_Hexane.key, CombinedCycle_Hexane, 'biosphere'),
        (EF_LeadB6.key, CombinedCycle_Lead, 'biosphere'),
        (EF_Manganese.key, CombinedCycle_Manganese, 'biosphere'),
        (EF_Mercury.key, CombinedCycle_Mercury, 'biosphere'),
        (EF_MethaneFossil.key, CombinedCycle_MethaneFossil, 'biosphere'),
        (EF_NickelB6.key, CombinedCycle_Nickel, 'biosphere'),
        (EF_NitrogenOxides.key, CombinedCycle_NitrogenOxides, 'biosphere'),
        (EF_PAH.key, CombinedCycle_PAH, 'biosphere'),
        (EF_Particulates025.key, CombinedCycle_Particulates, 'biosphere'),
        (EF_Pentane.key, CombinedCycle_Pentane, 'biosphere'),
        (EF_PropaneB6.key, CombinedCycle_Propane, 'biosphere'),
        (EF_PropionicAcid.key, CombinedCycle_PropionicAcid, 'biosphere'),
        (EF_Selenium.key, CombinedCycle_Selenium, 'biosphere'),
        (EF_SulfurDioxide.key, CombinedCycle_SulfurDioxide, 'biosphere'),
        (EF_Toluene.key, CombinedCycle_Toluene, 'biosphere'),
        (EF_WaterairB6.key, CombinedCycle_WaterAir, 'biosphere'),
        (EF_WaterB6.key, CombinedCycle_WaterWater, 'biosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in B6_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_CombinedCycle = B6_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        B6_CombinedCycle_acts[bd_name].new_exchange(
            input=B6_CombinedCycle_acts[bd_name].key, 
            output=B6_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_CombinedCycle.lci()
    mlca_B6_CombinedCycle.lcia()
    mlca_B6_CombinedCycle.scores

    dfresults_B6_CombinedCycle = pd.DataFrame.from_dict(mlca_B6_CombinedCycle.scores, orient='index')
    dfresults_B6_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_B6_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_B6_CombinedCycle = dfresults_B6_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_CombinedCycle.columns = dfresults_B6_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_CombinedCycle.rename(columns=method_lookup, inplace=True)

### 11.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_CombinedCycle_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, CombinedCycle_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in C2_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_CombinedCycle = C2_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        C2_CombinedCycle_acts[bd_name].new_exchange(
            input=C2_CombinedCycle_acts[bd_name].key, 
            output=C2_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_CombinedCycle.lci()
    mlca_C2_CombinedCycle.lcia()
    mlca_C2_CombinedCycle.scores

    dfresults_C2_CombinedCycle = pd.DataFrame.from_dict(mlca_C2_CombinedCycle.scores, orient='index')
    dfresults_C2_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_C2_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_C2_CombinedCycle = dfresults_C2_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_CombinedCycle.columns = dfresults_C2_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_CombinedCycle = dfresults_C2_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_CombinedCycle_exchange_configs = [
        (EF_EOL_Aluminium.key, CombinedCycle_EOL_aluminiumCastalloy, 'technosphere'),
        (EF_EOL_Aluminium.key, CombinedCycle_EOL_aluminiumWroughtalloy, 'technosphere'),
        (EF_EOL_inert.key, CombinedCycle_EOL_ceramictile, 'technosphere'),
        (EF_EOL_metal.key, CombinedCycle_EOL_chromium, 'technosphere'),
        (EF_EOL_metal.key, CombinedCycle_EOL_cobalt, 'technosphere'),
        (EF_EOL_concrete.key, CombinedCycle_EOL_concreteAT, 'technosphere'),
        (EF_EOL_concrete.key, CombinedCycle_EOL_concreteCH, 'technosphere'),
        (EF_EOL_metal.key, CombinedCycle_EOL_copper, 'technosphere'),
        (EF_EOL_metal.key, CombinedCycle_EOL_nickel, 'technosphere'),
        (EF_EOL_plastic.key, CombinedCycle_EOL_PEHD, 'technosphere'),
        (EF_EOL_metal.key, CombinedCycle_EOL_reinforcingsteel, 'technosphere'),
        (EF_EOL_metal.key, CombinedCycle_EOL_chromiumsteel, 'technosphere'),
        (EF_EOL_stonewool.key, CombinedCycle_EOL_stonewool, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in C3_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_CombinedCycle = C3_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        C3_CombinedCycle_acts[bd_name].new_exchange(
            input=C3_CombinedCycle_acts[bd_name].key, 
            output=C3_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_CombinedCycle.lci()
    mlca_C3_CombinedCycle.lcia()
    mlca_C3_CombinedCycle.scores

    dfresults_C3_CombinedCycle = pd.DataFrame.from_dict(mlca_C3_CombinedCycle.scores, orient='index')
    dfresults_C3_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_C3_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_C3_CombinedCycle = dfresults_C3_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_CombinedCycle.columns = dfresults_C3_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_CombinedCycle = dfresults_C3_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_CombinedCycle_exchange_configs = [
        (EF_EOL_landfill_aluminium.key, CombinedCycle_EOL_landfill_aluminium, 'technosphere'),
        (EF_EOL_landfill_inert.key, CombinedCycle_EOL_landfill_ceramictile, 'technosphere'),
        (EF_EOL_landfill_metal.key, CombinedCycle_EOL_landfill_chromium, 'technosphere'),
        (EF_EOL_landfill_metal.key, CombinedCycle_EOL_landfill_cobalt, 'technosphere'),
        (EF_EOL_landfill_concrete.key, CombinedCycle_EOL_landfill_concreteAT, 'technosphere'),
        (EF_EOL_landfill_concrete.key, CombinedCycle_EOL_landfill_concreteCH, 'technosphere'),
        (EF_EOL_landfill_metal.key, CombinedCycle_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_landfill_metal.key, CombinedCycle_EOL_landfill_nickel, 'technosphere'),
        (EF_EOL_inc_PEHD.key, CombinedCycle_EOL_inc_PEHD, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, CombinedCycle_EOL_landfill_PEHD, 'technosphere'),
        (EF_EOL_landfill_metal.key, CombinedCycle_EOL_landfill_reinforcingsteel, 'technosphere'),
        (EF_EOL_landfill_metal.key, CombinedCycle_EOL_landfill_chromiumsteel, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, CombinedCycle_EOL_landfill_stonewool, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in C4_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_CombinedCycle = C4_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        C4_CombinedCycle_acts[bd_name].new_exchange(
            input=C4_CombinedCycle_acts[bd_name].key, 
            output=C4_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_CombinedCycle.lci()
    mlca_C4_CombinedCycle.lcia()
    mlca_C4_CombinedCycle.scores

    dfresults_C4_CombinedCycle = pd.DataFrame.from_dict(mlca_C4_CombinedCycle.scores, orient='index')
    dfresults_C4_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_C4_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_C4_CombinedCycle = dfresults_C4_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_CombinedCycle.columns = dfresults_C4_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_CombinedCycle = dfresults_C4_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.9 Module B4

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_CombinedCycle = pd.DataFrame(index=dfresults_A13_CombinedCycle.index)  # Start with the index from A13 for consistency   
    dfresults_B4_CombinedCycle = (
        dfresults_A13_CombinedCycle.add(dfresults_A4_CombinedCycle, fill_value=0)
        .add(dfresults_C3_CombinedCycle, fill_value=0)
        .add(dfresults_C4_CombinedCycle, fill_value=0)
        ).mul(CombinedCycleNumberReplacements, axis=0)
    

### 11.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_CombinedCycle_exchange_configs = [
        (EF_recycling_aluminium.key, CombinedCycle_EOL_recycling_aluminiumCastalloy, 'technosphere'),
        (EF_primary_aluminium.key, CombinedCycle_EOL_primary_aluminiumCastalloy, 'technosphere'),
        (EF_recycling_aluminium.key, CombinedCycle_EOL_recycling_aluminiumWroughtalloy, 'technosphere'),
        (EF_primary_aluminium.key, CombinedCycle_EOL_primary_aluminiumWroughtalloy, 'technosphere'),
        (EF_recycling_iron.key, CombinedCycle_EOL_recycling_chromium, 'technosphere'),
        (EF_primary_chromium.key, CombinedCycle_EOL_primary_chromium, 'technosphere'),
        (EF_recycling_iron.key, CombinedCycle_EOL_recycling_cobalt, 'technosphere'),
        (EF_primary_cobalt.key, CombinedCycle_EOL_primary_cobalt, 'technosphere'),
        (EF_recycling_concrete.key, CombinedCycle_EOL_recycling_concreteAT, 'technosphere'),
        (EF_primary_concrete.key, CombinedCycle_EOL_primary_concreteAT, 'technosphere'),
        (EF_recycling_concrete.key, CombinedCycle_EOL_recycling_concreteCH, 'technosphere'),
        (EF_primary_concrete.key, CombinedCycle_EOL_primary_concreteCH, 'technosphere'),
        (EF_recycling_copper.key, CombinedCycle_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, CombinedCycle_EOL_primary_copper, 'technosphere'),
        (EF_recycling_iron.key, CombinedCycle_EOL_recycling_nickel, 'technosphere'),
        (EF_primary_nickel.key, CombinedCycle_EOL_primary_nickel, 'technosphere'),
        (EF_recycling_PEHD.key, CombinedCycle_EOL_recycling_HDPE, 'technosphere'),
        (EF_primary_PEHD.key, CombinedCycle_EOL_primary_HDPE, 'technosphere'),
        (EF_recycling_steel.key, CombinedCycle_EOL_recycling_reinforcingsteel, 'technosphere'),
        (EF_primary_steel.key, CombinedCycle_EOL_primary_reinforcingsteel, 'technosphere'),
        (EF_recycling_steel.key, CombinedCycle_EOL_recycling_chromiumsteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, CombinedCycle_EOL_primary_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in D1_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_CombinedCycle = D1_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        D1_CombinedCycle_acts[bd_name].new_exchange(
            input=D1_CombinedCycle_acts[bd_name].key, 
            output=D1_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_CombinedCycle.lci()
    mlca_D1_CombinedCycle.lcia()
    mlca_D1_CombinedCycle.scores

    dfresults_D1_CombinedCycle = pd.DataFrame.from_dict(mlca_D1_CombinedCycle.scores, orient='index')
    dfresults_D1_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_D1_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_D1_CombinedCycle = dfresults_D1_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_CombinedCycle.columns = dfresults_D1_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_CombinedCycle = dfresults_D1_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_CombinedCycle = dfresults_D1_CombinedCycle.add(dfresults_D1_CombinedCycle.mul(CombinedCycleNumberReplacements, axis=0))

### 11.2.12 Module D3 (related to the export of energy as a results of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_CombinedCycle_exchange_configs = [
        (EF_heat_energyrecovery.key, CombinedCycle_EOL_energyrecoveryheat_PEHD, 'technosphere'),
        (EF_electricity_energyrecovery.key, CombinedCycle_EOL_energyrecoveryele_PEHD, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_CombCycle_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_CombinedCycle_exchange_list = []
        
        for input_key, data_array, exc_type in D3_CombinedCycle_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_CombinedCycle_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_CombinedCycle_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_CombinedCycle = D3_CombinedCycle_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_CombinedCycle.save()

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_CombCycle_BdName:
        D3_CombinedCycle_acts[bd_name].new_exchange(
            input=D3_CombinedCycle_acts[bd_name].key, 
            output=D3_CombinedCycle_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_CombinedCycle = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_CombinedCycle['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_CombinedCycle_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_CombCycle_BdName, CHP_CombCycle_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_CombinedCycle = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_CombinedCycle.lci()
    mlca_D3_CombinedCycle.lcia()
    mlca_D3_CombinedCycle.scores

    dfresults_D3_CombinedCycle = pd.DataFrame.from_dict(mlca_D3_CombinedCycle.scores, orient='index')
    dfresults_D3_CombinedCycle.index = pd.MultiIndex.from_tuples(dfresults_D3_CombinedCycle.index, names=['Column', 'Row'])
    dfresults_D3_CombinedCycle = dfresults_D3_CombinedCycle.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_CombinedCycle.columns = dfresults_D3_CombinedCycle.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_CombinedCycle.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_CombinedCycle = dfresults_D3_CombinedCycle.mul(SFCombinedCycle, axis=0)

### 11.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "CHP-Combined Cycle"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_CombinedCycle = dfresults_D3_CombinedCycle.add(dfresults_D3_CombinedCycle.mul(CombinedCycleNumberReplacements, axis=0))

# 12. Gas Turbine

## 12.1 Activities

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_GasTurbine_{bd_name}',
        'name': f'A13_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            A13_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_GasTurbine_{bd_name}',
        'name': f'A4_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            A4_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_GasTurbine_{bd_name}',
        'name': f'A5_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            A5_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_GasTurbine_{bd_name}',
        'name': f'B2_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            B2_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_GasTurbine_{bd_name}',
        'name': f'B6_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            B6_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_GasTurbine_{bd_name}',
        'name': f'C2_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            C2_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_GasTurbine_{bd_name}',
        'name': f'C3_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            C3_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_GasTurbine_{bd_name}',
        'name': f'C4_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            C4_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_GasTurbine_{bd_name}',
        'name': f'D1_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            D1_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "CHP-Turbogas"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_GasTurbine_acts = {}  # Dictionary to store activities by building name
    for bd_name in CHP_Turbogas_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_GasTurbine_{bd_name}',
        'name': f'D3_GasTurbine_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_GasTurbine_acts[bd_name] = scenario_db.new_activity(**data)
            D3_GasTurbine_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 12.2 Exchanges

### 12.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_GasTurbine_exchange_configs = [
        (EF_concreteAT.key, GasTurbine_concreteAT, 'technosphere'),
        (EF_concreteCH.key, GasTurbine_concreteCH, 'technosphere'),
        (EF_Copper.key, GasTurbine_copper, 'technosphere'),
        (EF_diesel.key, GasTurbine_diesel, 'technosphere'),
        (EF_Electricity.key, GasTurbine_electricity, 'technosphere'),
        (EF_Heat.key, GasTurbine_heat, 'technosphere'),
        (EF_polyethylene.key, GasTurbine_PEHD, 'technosphere'),
        (EF_steel.key, GasTurbine_reinforcingsteel, 'technosphere'),
        (EF_StainlessSteel.key, GasTurbine_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in A13_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_GasTurbine = A13_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        A13_GasTurbine_acts[bd_name].new_exchange(
            input=A13_GasTurbine_acts[bd_name].key, 
            output=A13_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_GasTurbine.lci()
    mlca_A13_GasTurbine.lcia()
    mlca_A13_GasTurbine.scores

    dfresults_A13_GasTurbine = pd.DataFrame.from_dict(mlca_A13_GasTurbine.scores, orient='index')
    dfresults_A13_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_A13_GasTurbine.index, names=['Column', 'Row'])
    dfresults_A13_GasTurbine = dfresults_A13_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_GasTurbine.columns = dfresults_A13_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_GasTurbine = dfresults_A13_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_GasTurbine_exchange_configs = [
        (EF_Truk_16_32.key, GasTurbine_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, GasTurbine_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in A4_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_GasTurbine = A4_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        A4_GasTurbine_acts[bd_name].new_exchange(
            input=A4_GasTurbine_acts[bd_name].key, 
            output=A4_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_GasTurbine.lci()
    mlca_A4_GasTurbine.lcia()
    mlca_A4_GasTurbine.scores

    dfresults_A4_GasTurbine = pd.DataFrame.from_dict(mlca_A4_GasTurbine.scores, orient='index')
    dfresults_A4_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_A4_GasTurbine.index, names=['Column', 'Row'])
    dfresults_A4_GasTurbine = dfresults_A4_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_GasTurbine.columns = dfresults_A4_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_GasTurbine = dfresults_A4_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.3 Module A5

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        A5_GasTurbine_acts[bd_name].new_exchange(
            input=A5_GasTurbine_acts[bd_name].key, 
            output=A5_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_GasTurbine.lci()
    mlca_A5_GasTurbine.lcia()
    mlca_A5_GasTurbine.scores

    dfresults_A5_GasTurbine = pd.DataFrame.from_dict(mlca_A5_GasTurbine.scores, orient='index')
    dfresults_A5_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_A5_GasTurbine.index, names=['Column', 'Row'])
    dfresults_A5_GasTurbine = dfresults_A5_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_GasTurbine.columns = dfresults_A5_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_GasTurbine = dfresults_A5_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_GasTurbine_exchange_configs = [
        (EF_LightTruk.key, GasTurbine_LightTruk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in B2_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_GasTurbine = B2_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        B2_GasTurbine_acts[bd_name].new_exchange(
            input=B2_GasTurbine_acts[bd_name].key, 
            output=B2_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_GasTurbine.lci()
    mlca_B2_GasTurbine.lcia()
    mlca_B2_GasTurbine.scores

    dfresults_B2_GasTurbine = pd.DataFrame.from_dict(mlca_B2_GasTurbine.scores, orient='index')
    dfresults_B2_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_B2_GasTurbine.index, names=['Column', 'Row'])
    dfresults_B2_GasTurbine = dfresults_B2_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_GasTurbine.columns = dfresults_B2_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_GasTurbine = dfresults_B2_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_GasTurbine_exchange_configs = [
        (EF_energyvectorNG.key, GasTurbine_EnergyVectorNG, 'technosphere'),
        (EF_energyvectorBiom.key, GasTurbine_EnergyVectorBiomethane, 'technosphere'),
        (EF_energyvectorHydrogen.key, GasTurbine_EnergyVectorHydrogen, 'technosphere'),
        (EF_biogas.key, GasTurbine_EnergyVectorInputBiogas, 'technosphere'),
        (EF_energyvectorHydrogen.key, GasTurbine_EnergyVectorInputHydrogen, 'technosphere'),
        (EF_watercoolingtower.key, GasTurbine_WaterCooling, 'technosphere'),
        (EF_Acetaldehyde.key, GasTurbine_Acetaldehyde, 'biosphere'),
        (EF_Aceticacid.key, GasTurbine_AceticAcid, 'biosphere'),
        (EF_Benzene.key, GasTurbine_Benzene, 'biosphere'),
        (EF_Benzoapyrene.key, GasTurbine_Benzoapyrene, 'biosphere'),
        (EF_Buthane.key, GasTurbine_Butane, 'biosphere'),
        (EF_CarbondioxideFossil.key, GasTurbine_CarbonDioxideFossil, 'biosphere'),
        (EF_CarbonmonoxideFossil.key, GasTurbine_CarbonMonoxideFossil, 'biosphere'),
        (EF_DinitrogenMonoxide.key, GasTurbine_DinitrogenMonoxide, 'biosphere'),
        (EF_Formaldehyde.key, GasTurbine_Formaldehyde, 'biosphere'),
        (EF_Mercury.key, GasTurbine_Mercury, 'biosphere'),
        (EF_MethaneFossil.key, GasTurbine_MethaneFossil, 'biosphere'),
        (EF_NitrogenOxides.key, GasTurbine_NitrogenOxides, 'biosphere'),
        (EF_PAH.key, GasTurbine_PAH, 'biosphere'),
        (EF_Particulates025.key, GasTurbine_Particulates, 'biosphere'),
        (EF_Pentane.key, GasTurbine_Pentane, 'biosphere'),
        (EF_PropaneB6.key, GasTurbine_Propane, 'biosphere'),
        (EF_PropionicAcid.key, GasTurbine_PropionicAcid, 'biosphere'),
        (EF_SulfurDioxide.key, GasTurbine_SulfurDioxide, 'biosphere'),
        (EF_Toluene.key, GasTurbine_Toluene, 'biosphere'),
        (EF_WaterairB6.key, GasTurbine_WaterAir, 'biosphere'),
        (EF_WaterB6.key, GasTurbine_WaterWater, 'biosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in B6_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_GasTurbine = B6_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        B6_GasTurbine_acts[bd_name].new_exchange(
            input=B6_GasTurbine_acts[bd_name].key, 
            output=B6_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_GasTurbine.lci()
    mlca_B6_GasTurbine.lcia()
    mlca_B6_GasTurbine.scores

    dfresults_B6_GasTurbine = pd.DataFrame.from_dict(mlca_B6_GasTurbine.scores, orient='index')
    dfresults_B6_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_B6_GasTurbine.index, names=['Column', 'Row'])
    dfresults_B6_GasTurbine = dfresults_B6_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_GasTurbine.columns = dfresults_B6_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_GasTurbine.rename(columns=method_lookup, inplace=True)

### 12.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_GasTurbine_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, GasTurbine_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in C2_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_GasTurbine = C2_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        C2_GasTurbine_acts[bd_name].new_exchange(
            input=C2_GasTurbine_acts[bd_name].key, 
            output=C2_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_GasTurbine.lci()
    mlca_C2_GasTurbine.lcia()
    mlca_C2_GasTurbine.scores

    dfresults_C2_GasTurbine = pd.DataFrame.from_dict(mlca_C2_GasTurbine.scores, orient='index')
    dfresults_C2_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_C2_GasTurbine.index, names=['Column', 'Row'])
    dfresults_C2_GasTurbine = dfresults_C2_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_GasTurbine.columns = dfresults_C2_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_GasTurbine = dfresults_C2_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_GasTurbine_exchange_configs = [
        (EF_EOL_concrete.key, GasTurbine_EOL_concreteAT, 'technosphere'),
        (EF_EOL_concrete.key, GasTurbine_EOL_concreteCH, 'technosphere'),
        (EF_EOL_metal.key, GasTurbine_EOL_copper, 'technosphere'),
        (EF_EOL_plastic.key, GasTurbine_EOL_PEHD, 'technosphere'),
        (EF_EOL_metal.key, GasTurbine_EOL_reinforcingsteel, 'technosphere'),
        (EF_EOL_metal.key, GasTurbine_EOL_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in C3_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_GasTurbine = C3_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        C3_GasTurbine_acts[bd_name].new_exchange(
            input=C3_GasTurbine_acts[bd_name].key, 
            output=C3_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_GasTurbine.lci()
    mlca_C3_GasTurbine.lcia()
    mlca_C3_GasTurbine.scores

    dfresults_C3_GasTurbine = pd.DataFrame.from_dict(mlca_C3_GasTurbine.scores, orient='index')
    dfresults_C3_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_C3_GasTurbine.index, names=['Column', 'Row'])
    dfresults_C3_GasTurbine = dfresults_C3_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_GasTurbine.columns = dfresults_C3_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_GasTurbine = dfresults_C3_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_GasTurbine_exchange_configs = [
        (EF_EOL_landfill_concrete.key, GasTurbine_EOL_landfill_concreteAT, 'technosphere'),
        (EF_EOL_landfill_concrete.key, GasTurbine_EOL_landfill_concreteCH, 'technosphere'),
        (EF_EOL_landfill_metal.key, GasTurbine_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_inc_PEHD.key, GasTurbine_EOL_inc_PEHD, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, GasTurbine_EOL_landfill_PEHD, 'technosphere'),
        (EF_EOL_landfill_metal.key, GasTurbine_EOL_landfill_reinforcingsteel, 'technosphere'),
        (EF_EOL_landfill_metal.key, GasTurbine_EOL_landfill_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in C4_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in C4_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_GasTurbine = C4_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        C4_GasTurbine_acts[bd_name].new_exchange(
            input=C4_GasTurbine_acts[bd_name].key, 
            output=C4_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_GasTurbine.lci()
    mlca_C4_GasTurbine.lcia()
    mlca_C4_GasTurbine.scores

    dfresults_C4_GasTurbine = pd.DataFrame.from_dict(mlca_C4_GasTurbine.scores, orient='index')
    dfresults_C4_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_C4_GasTurbine.index, names=['Column', 'Row'])
    dfresults_C4_GasTurbine = dfresults_C4_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_GasTurbine.columns = dfresults_C4_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_GasTurbine = dfresults_C4_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.9 Module B4

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_GasTurbine = pd.DataFrame(index=dfresults_A13_GasTurbine.index)  # Start with the index from A13 for consistency   
    dfresults_B4_GasTurbine = (
        dfresults_A13_GasTurbine.add(dfresults_A4_GasTurbine, fill_value=0)
        .add(dfresults_C3_GasTurbine, fill_value=0)
        .add(dfresults_C4_GasTurbine, fill_value=0)
        ).mul(GasTurbineNumberReplacements, axis=0)
    

### 12.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_GasTurbine_exchange_configs = [
        (EF_recycling_concrete.key, GasTurbine_EOL_recycling_concreteAT, 'technosphere'),
        (EF_primary_concrete.key, GasTurbine_EOL_primary_concreteAT, 'technosphere'),
        (EF_recycling_concrete.key, GasTurbine_EOL_recycling_concreteCH, 'technosphere'),
        (EF_primary_concrete.key, GasTurbine_EOL_primary_concreteCH, 'technosphere'),
        (EF_recycling_copper.key, GasTurbine_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, GasTurbine_EOL_primary_copper, 'technosphere'),
        (EF_recycling_PEHD.key, GasTurbine_EOL_recycling_HDPE, 'technosphere'),
        (EF_primary_PEHD.key, GasTurbine_EOL_primary_HDPE, 'technosphere'),
        (EF_recycling_steel.key, GasTurbine_EOL_recycling_reinforcingsteel, 'technosphere'),
        (EF_primary_steel.key, GasTurbine_EOL_primary_reinforcingsteel, 'technosphere'),
        (EF_recycling_steel.key, GasTurbine_EOL_recycling_chromiumsteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, GasTurbine_EOL_primary_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in D1_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_GasTurbine = D1_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        D1_GasTurbine_acts[bd_name].new_exchange(
            input=D1_GasTurbine_acts[bd_name].key, 
            output=D1_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_GasTurbine.lci()
    mlca_D1_GasTurbine.lcia()
    mlca_D1_GasTurbine.scores

    dfresults_D1_GasTurbine = pd.DataFrame.from_dict(mlca_D1_GasTurbine.scores, orient='index')
    dfresults_D1_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_D1_GasTurbine.index, names=['Column', 'Row'])
    dfresults_D1_GasTurbine = dfresults_D1_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_GasTurbine.columns = dfresults_D1_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_GasTurbine = dfresults_D1_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_GasTurbine = dfresults_D1_GasTurbine.add(dfresults_D1_GasTurbine.mul(GasTurbineNumberReplacements, axis=0))

### 12.2.12 Module D3 (related to the export of energy as a results of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_GasTurbine_exchange_configs = [
        (EF_heat_energyrecovery.key, GasTurbine_EOL_energyrecoveryheat_PEHD, 'technosphere'),
        (EF_electricity_energyrecovery.key, GasTurbine_EOL_energyrecoveryele_PEHD, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CHP_Turbogas_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_GasTurbine_exchange_list = []
        
        for input_key, data_array, exc_type in D3_GasTurbine_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_GasTurbine_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_GasTurbine_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_GasTurbine = D3_GasTurbine_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_GasTurbine.save()

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CHP_Turbogas_BdName:
        D3_GasTurbine_acts[bd_name].new_exchange(
            input=D3_GasTurbine_acts[bd_name].key, 
            output=D3_GasTurbine_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_GasTurbine = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_GasTurbine['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_GasTurbine_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CHP_Turbogas_BdName, CHP_Turbogas_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_GasTurbine = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_GasTurbine.lci()
    mlca_D3_GasTurbine.lcia()
    mlca_D3_GasTurbine.scores

    dfresults_D3_GasTurbine = pd.DataFrame.from_dict(mlca_D3_GasTurbine.scores, orient='index')
    dfresults_D3_GasTurbine.index = pd.MultiIndex.from_tuples(dfresults_D3_GasTurbine.index, names=['Column', 'Row'])
    dfresults_D3_GasTurbine = dfresults_D3_GasTurbine.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_GasTurbine.columns = dfresults_D3_GasTurbine.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_GasTurbine.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_GasTurbine = dfresults_D3_GasTurbine.mul(SFGasTurbine, axis=0)

### 12.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "CHP-Turbogas"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_GasTurbine = dfresults_D3_GasTurbine.add(dfresults_D3_GasTurbine.mul(GasTurbineNumberReplacements, axis=0))

# 13. Heat recoveries

## 13.1 Activities

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_HeatRecovery_{bd_name}',
        'name': f'A13_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            A13_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_HeatRecovery_{bd_name}',
        'name': f'A4_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            A4_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_HeatRecovery_{bd_name}',
        'name': f'A5_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            A5_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_HeatRecovery_{bd_name}',
        'name': f'B2_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            B2_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_HeatRecovery_{bd_name}',
        'name': f'B6_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            B6_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_HeatRecovery_{bd_name}',
        'name': f'C2_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            C2_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_HeatRecovery_{bd_name}',
        'name': f'C3_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            C3_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_HeatRecovery_{bd_name}',
        'name': f'C4_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            C4_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_HeatRecovery_{bd_name}',
        'name': f'D1_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            D1_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Heat recovery"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_HeatRecovery_acts = {}  # Dictionary to store activities by building name
    for bd_name in HeatRecovery_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_HeatRecovery_{bd_name}',
        'name': f'D3_HeatRecovery_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_HeatRecovery_acts[bd_name] = scenario_db.new_activity(**data)
            D3_HeatRecovery_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 13.2 Exchanges

### 13.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_HeatRecovery_exchange_configs = [
        (EF_StainlessSteel.key, HeatRecovery_stainlessSteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(HeatRecovery_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_HeatRecovery_exchange_list = []
        
        for input_key, data_array, exc_type in A13_HeatRecovery_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_HeatRecovery_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_HeatRecovery_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_HeatRecovery = A13_HeatRecovery_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_HeatRecovery.save()

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        A13_HeatRecovery_acts[bd_name].new_exchange(
            input=A13_HeatRecovery_acts[bd_name].key, 
            output=A13_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_HeatRecovery.lci()
    mlca_A13_HeatRecovery.lcia()
    mlca_A13_HeatRecovery.scores

    dfresults_A13_HeatRecovery = pd.DataFrame.from_dict(mlca_A13_HeatRecovery.scores, orient='index')
    dfresults_A13_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_A13_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_A13_HeatRecovery = dfresults_A13_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_HeatRecovery.columns = dfresults_A13_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_HeatRecovery = dfresults_A13_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_HeatRecovery_exchange_configs = [
        (EF_Truk_16_32.key, HeatRecovery_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, HeatRecovery_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(HeatRecovery_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_HeatRecovery_exchange_list = []
        
        for input_key, data_array, exc_type in A4_HeatRecovery_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_HeatRecovery_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_HeatRecovery_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_HeatRecovery = A4_HeatRecovery_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_HeatRecovery.save()

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        A4_HeatRecovery_acts[bd_name].new_exchange(
            input=A4_HeatRecovery_acts[bd_name].key, 
            output=A4_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_HeatRecovery.lci()
    mlca_A4_HeatRecovery.lcia()
    mlca_A4_HeatRecovery.scores

    dfresults_A4_HeatRecovery = pd.DataFrame.from_dict(mlca_A4_HeatRecovery.scores, orient='index')
    dfresults_A4_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_A4_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_A4_HeatRecovery = dfresults_A4_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_HeatRecovery.columns = dfresults_A4_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_HeatRecovery = dfresults_A4_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.3 Module A5

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        A5_HeatRecovery_acts[bd_name].new_exchange(
            input=A5_HeatRecovery_acts[bd_name].key, 
            output=A5_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_HeatRecovery.lci()
    mlca_A5_HeatRecovery.lcia()
    mlca_A5_HeatRecovery.scores

    dfresults_A5_HeatRecovery = pd.DataFrame.from_dict(mlca_A5_HeatRecovery.scores, orient='index')
    dfresults_A5_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_A5_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_A5_HeatRecovery = dfresults_A5_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_HeatRecovery.columns = dfresults_A5_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_HeatRecovery = dfresults_A5_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_HeatRecovery_exchange_configs = [
        (EF_Light_Truk.key, HeatRecovery_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(HeatRecovery_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_HeatRecovery_exchange_list = []
        
        for input_key, data_array, exc_type in B2_HeatRecovery_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_HeatRecovery_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_HeatRecovery_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_HeatRecovery = B2_HeatRecovery_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_HeatRecovery.save()

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        B2_HeatRecovery_acts[bd_name].new_exchange(
            input=B2_HeatRecovery_acts[bd_name].key, 
            output=B2_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_HeatRecovery.lci()
    mlca_B2_HeatRecovery.lcia()
    mlca_B2_HeatRecovery.scores

    dfresults_B2_HeatRecovery = pd.DataFrame.from_dict(mlca_B2_HeatRecovery.scores, orient='index')
    dfresults_B2_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_B2_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_B2_HeatRecovery = dfresults_B2_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_HeatRecovery.columns = dfresults_B2_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_HeatRecovery = dfresults_B2_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.5 Module B6

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        B6_HeatRecovery_acts[bd_name].new_exchange(
            input=B6_HeatRecovery_acts[bd_name].key, 
            output=B6_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_HeatRecovery.lci()
    mlca_B6_HeatRecovery.lcia()
    mlca_B6_HeatRecovery.scores

    dfresults_B6_HeatRecovery = pd.DataFrame.from_dict(mlca_B6_HeatRecovery.scores, orient='index')
    dfresults_B6_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_B6_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_B6_HeatRecovery = dfresults_B6_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_HeatRecovery.columns = dfresults_B6_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_HeatRecovery.rename(columns=method_lookup, inplace=True)

### 13.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_HeatRecovery_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, HeatRecovery_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(HeatRecovery_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_HeatRecovery_exchange_list = []
        
        for input_key, data_array, exc_type in C2_HeatRecovery_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_HeatRecovery_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_HeatRecovery_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_HeatRecovery = C2_HeatRecovery_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_HeatRecovery.save()

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        C2_HeatRecovery_acts[bd_name].new_exchange(
            input=C2_HeatRecovery_acts[bd_name].key, 
            output=C2_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_HeatRecovery.lci()
    mlca_C2_HeatRecovery.lcia()
    mlca_C2_HeatRecovery.scores

    dfresults_C2_HeatRecovery = pd.DataFrame.from_dict(mlca_C2_HeatRecovery.scores, orient='index')
    dfresults_C2_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_C2_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_C2_HeatRecovery = dfresults_C2_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_HeatRecovery.columns = dfresults_C2_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_HeatRecovery = dfresults_C2_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_HeatRecovery_exchange_configs = [
        (EF_EOL_metal.key, HeatRecovery_EOL_stainlessSteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(HeatRecovery_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_HeatRecovery_exchange_list = []
        
        for input_key, data_array, exc_type in C3_HeatRecovery_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_HeatRecovery_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_HeatRecovery_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_HeatRecovery = C3_HeatRecovery_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_HeatRecovery.save()

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        C3_HeatRecovery_acts[bd_name].new_exchange(
            input=C3_HeatRecovery_acts[bd_name].key, 
            output=C3_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_HeatRecovery.lci()
    mlca_C3_HeatRecovery.lcia()
    mlca_C3_HeatRecovery.scores

    dfresults_C3_HeatRecovery = pd.DataFrame.from_dict(mlca_C3_HeatRecovery.scores, orient='index')
    dfresults_C3_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_C3_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_C3_HeatRecovery = dfresults_C3_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_HeatRecovery.columns = dfresults_C3_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_HeatRecovery = dfresults_C3_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_HeatRecovery_exchange_configs = [
        (EF_EOL_landfill_metal.key, HeatRecovery_EOL_landfill_stainlessSteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(HeatRecovery_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_HeatRecovery_exchange_list = []
        
        for input_key, data_array, exc_type in C4_HeatRecovery_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_HeatRecovery_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_HeatRecovery_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_HeatRecovery = C4_HeatRecovery_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_HeatRecovery.save()

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        C4_HeatRecovery_acts[bd_name].new_exchange(
            input=C4_HeatRecovery_acts[bd_name].key, 
            output=C4_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_HeatRecovery.lci()
    mlca_C4_HeatRecovery.lcia()
    mlca_C4_HeatRecovery.scores

    dfresults_C4_HeatRecovery = pd.DataFrame.from_dict(mlca_C4_HeatRecovery.scores, orient='index')
    dfresults_C4_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_C4_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_C4_HeatRecovery = dfresults_C4_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_HeatRecovery.columns = dfresults_C4_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_HeatRecovery = dfresults_C4_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.9 Module B4

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_HeatRecovery = pd.DataFrame(index=dfresults_A13_HeatRecovery.index)  # Start with the index from A13 for consistency   
    dfresults_B4_HeatRecovery = (
        dfresults_A13_HeatRecovery.add(dfresults_A4_HeatRecovery, fill_value=0)
        .add(dfresults_C3_HeatRecovery, fill_value=0)
        .add(dfresults_C4_HeatRecovery, fill_value=0)
    ).mul(HeatRecoveryNumberReplacements, axis=0)

### 13.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_HeatRecovery_exchange_configs = [
        (EF_recycling_steel.key, HeatRecovery_EOL_recycling_stainlessSteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, HeatRecovery_EOL_primary_stainlessSteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(HeatRecovery_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_HeatRecovery_exchange_list = []
        
        for input_key, data_array, exc_type in D1_HeatRecovery_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_HeatRecovery_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_HeatRecovery_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_HeatRecovery = D1_HeatRecovery_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_HeatRecovery.save()

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        D1_HeatRecovery_acts[bd_name].new_exchange(
            input=D1_HeatRecovery_acts[bd_name].key, 
            output=D1_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_HeatRecovery.lci()
    mlca_D1_HeatRecovery.lcia()
    mlca_D1_HeatRecovery.scores

    dfresults_D1_HeatRecovery = pd.DataFrame.from_dict(mlca_D1_HeatRecovery.scores, orient='index')
    dfresults_D1_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_D1_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_D1_HeatRecovery = dfresults_D1_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_HeatRecovery.columns = dfresults_D1_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_HeatRecovery = dfresults_D1_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_HeatRecovery = dfresults_D1_HeatRecovery.add(dfresults_D1_HeatRecovery.mul(HeatRecoveryNumberReplacements, axis=0))

### 13.2.12 Module D3 (related to the export of energy as results of waste incineration)

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in HeatRecovery_BdName:
        D3_HeatRecovery_acts[bd_name].new_exchange(
            input=D3_HeatRecovery_acts[bd_name].key, 
            output=D3_HeatRecovery_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_HeatRecovery = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_HeatRecovery['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_HeatRecovery_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(HeatRecovery_BdName, HeatRecovery_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_HeatRecovery = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_HeatRecovery.lci()
    mlca_D3_HeatRecovery.lcia()
    mlca_D3_HeatRecovery.scores

    dfresults_D3_HeatRecovery = pd.DataFrame.from_dict(mlca_D3_HeatRecovery.scores, orient='index')
    dfresults_D3_HeatRecovery.index = pd.MultiIndex.from_tuples(dfresults_D3_HeatRecovery.index, names=['Column', 'Row'])
    dfresults_D3_HeatRecovery = dfresults_D3_HeatRecovery.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_HeatRecovery.columns = dfresults_D3_HeatRecovery.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_HeatRecovery.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_HeatRecovery = dfresults_D3_HeatRecovery.mul(SFHeatRecovery, axis=0)

### 13.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "Heat recovery"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_HeatRecovery = dfresults_D3_HeatRecovery.add(dfresults_D3_HeatRecovery.mul(HeatRecoveryNumberReplacements, axis=0))

# 14. Solar thermal arraies

## 14.1 Activities

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_SolarThermal_{bd_name}',
        'name': f'A13_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            A13_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_SolarThermal_{bd_name}',
        'name': f'A4_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            A4_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_SolarThermal_{bd_name}',
        'name': f'A5_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            A5_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_SolarThermal_{bd_name}',
        'name': f'B2_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            B2_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_SolarThermal_{bd_name}',
        'name': f'B6_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            B6_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_SolarThermal_{bd_name}',
        'name': f'C2_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            C2_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_SolarThermal_{bd_name}',
        'name': f'C3_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            C3_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_SolarThermal_{bd_name}',
        'name': f'C4_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            C4_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_SolarThermal_{bd_name}',
        'name': f'D1_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            D1_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Solar thermal system"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_SolarThermal_acts = {}  # Dictionary to store activities by building name
    for bd_name in SolarThermal_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_SolarThermal_{bd_name}',
        'name': f'D3_SolarThermal_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_SolarThermal_acts[bd_name] = scenario_db.new_activity(**data)
            D3_SolarThermal_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 14.2 Exchanges

### 14.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_SolarThermal_exchange_configs = [
        (EF_solarcollector.key, SolarThermal_solarcollector, 'technosphere'),
        (EF_storage.key, SolarThermal_storage, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in A13_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_SolarThermal = A13_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        A13_SolarThermal_acts[bd_name].new_exchange(
            input=A13_SolarThermal_acts[bd_name].key, 
            output=A13_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_SolarThermal.lci()
    mlca_A13_SolarThermal.lcia()
    mlca_A13_SolarThermal.scores

    dfresults_A13_SolarThermal = pd.DataFrame.from_dict(mlca_A13_SolarThermal.scores, orient='index')
    dfresults_A13_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_A13_SolarThermal.index, names=['Column', 'Row'])
    dfresults_A13_SolarThermal = dfresults_A13_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_SolarThermal.columns = dfresults_A13_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_SolarThermal = dfresults_A13_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_SolarThermal_exchange_configs = [
        (EF_Truk_16_32.key, SolarThermal_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, SolarThermal_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in A4_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_SolarThermal = A4_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        A4_SolarThermal_acts[bd_name].new_exchange(
            input=A4_SolarThermal_acts[bd_name].key, 
            output=A4_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_SolarThermal.lci()
    mlca_A4_SolarThermal.lcia()
    mlca_A4_SolarThermal.scores

    dfresults_A4_SolarThermal = pd.DataFrame.from_dict(mlca_A4_SolarThermal.scores, orient='index')
    dfresults_A4_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_A4_SolarThermal.index, names=['Column', 'Row'])
    dfresults_A4_SolarThermal = dfresults_A4_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_SolarThermal.columns = dfresults_A4_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_SolarThermal = dfresults_A4_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.3 Module A5

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        A5_SolarThermal_acts[bd_name].new_exchange(
            input=A5_SolarThermal_acts[bd_name].key, 
            output=A5_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_SolarThermal.lci()
    mlca_A5_SolarThermal.lcia()
    mlca_A5_SolarThermal.scores

    dfresults_A5_SolarThermal = pd.DataFrame.from_dict(mlca_A5_SolarThermal.scores, orient='index')
    dfresults_A5_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_A5_SolarThermal.index, names=['Column', 'Row'])
    dfresults_A5_SolarThermal = dfresults_A5_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_SolarThermal.columns = dfresults_A5_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_SolarThermal = dfresults_A5_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_SolarThermal_exchange_configs = [
        (EF_LightTruk.key, SolarThermal_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in B2_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_SolarThermal = B2_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        B2_SolarThermal_acts[bd_name].new_exchange(
            input=B2_SolarThermal_acts[bd_name].key, 
            output=B2_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_SolarThermal.lci()
    mlca_B2_SolarThermal.lcia()
    mlca_B2_SolarThermal.scores

    dfresults_B2_SolarThermal = pd.DataFrame.from_dict(mlca_B2_SolarThermal.scores, orient='index')
    dfresults_B2_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_B2_SolarThermal.index, names=['Column', 'Row'])
    dfresults_B2_SolarThermal = dfresults_B2_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_SolarThermal.columns = dfresults_B2_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_SolarThermal = dfresults_B2_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.5 Module B6 

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_SolarThermal_exchange_configs = [
        (EF_electricity.key, SolarThermal_electricity, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in B6_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in B6_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_SolarThermal = B6_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        B6_SolarThermal_acts[bd_name].new_exchange(
            input=B6_SolarThermal_acts[bd_name].key, 
            output=B6_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_SolarThermal.lci()
    mlca_B6_SolarThermal.lcia()
    mlca_B6_SolarThermal.scores

    dfresults_B6_SolarThermal = pd.DataFrame.from_dict(mlca_B6_SolarThermal.scores, orient='index')
    dfresults_B6_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_B6_SolarThermal.index, names=['Column', 'Row'])
    dfresults_B6_SolarThermal = dfresults_B6_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_SolarThermal.columns = dfresults_B6_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_SolarThermal.rename(columns=method_lookup, inplace=True)

### 14.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_SolarThermal_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, SolarThermal_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in C2_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_SolarThermal = C2_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        C2_SolarThermal_acts[bd_name].new_exchange(
            input=C2_SolarThermal_acts[bd_name].key, 
            output=C2_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_SolarThermal.lci()
    mlca_C2_SolarThermal.lcia()
    mlca_C2_SolarThermal.scores

    dfresults_C2_SolarThermal = pd.DataFrame.from_dict(mlca_C2_SolarThermal.scores, orient='index')
    dfresults_C2_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_C2_SolarThermal.index, names=['Column', 'Row'])
    dfresults_C2_SolarThermal = dfresults_C2_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_SolarThermal.columns = dfresults_C2_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_SolarThermal = dfresults_C2_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_SolarThermal_exchange_configs = [
        (EF_EOL_metal.key, SolarThermal_EOL_copper, 'technosphere'),
        (EF_EOL_metal.key, SolarThermal_EOL_steel, 'technosphere'),
        (EF_EOL_metal.key, SolarThermal_EOL_stainlesssteel, 'technosphere'),
        (EF_EOL_inert.key, SolarThermal_EOL_glass, 'technosphere'),
        (EF_EOL_stonewool.key, SolarThermal_EOL_mineralwool, 'technosphere'),
        (EF_EOL_paperboard.key, SolarThermal_EOL_paperboard, 'technosphere'),
        (EF_EOL_plastic.key, SolarThermal_EOL_plastic, 'technosphere'),
        (EF_EOL_Elastomer.key, SolarThermal_EOL_elastomer, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in C3_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_SolarThermal = C3_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        C3_SolarThermal_acts[bd_name].new_exchange(
            input=C3_SolarThermal_acts[bd_name].key, 
            output=C3_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_SolarThermal.lci()
    mlca_C3_SolarThermal.lcia()
    mlca_C3_SolarThermal.scores

    dfresults_C3_SolarThermal = pd.DataFrame.from_dict(mlca_C3_SolarThermal.scores, orient='index')
    dfresults_C3_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_C3_SolarThermal.index, names=['Column', 'Row'])
    dfresults_C3_SolarThermal = dfresults_C3_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_SolarThermal.columns = dfresults_C3_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_SolarThermal = dfresults_C3_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_SolarThermal_exchange_configs = [
        (EF_EOL_landfill_metal.key, SolarThermal_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_landfill_metal.key, SolarThermal_EOL_landfill_steel, 'technosphere'),
        (EF_EOL_landfill_metal.key, SolarThermal_EOL_landfill_stainlessSteel, 'technosphere'),
        (EF_EOL_landfill_glass.key, SolarThermal_EOL_landfill_glass, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, SolarThermal_EOL_landfill_mineralwool, 'technosphere'),
        (EF_EOL_inc_paperboard.key, SolarThermal_EOL_inc_paperboard, 'technosphere'),
        (EF_EOL_landfill_paperboard.key, SolarThermal_EOL_landfill_paperboard, 'technosphere'),
        (EF_EOL_inc_plastic.key, SolarThermal_EOL_inc_plastic, 'technosphere'),
        (EF_EOL_landfill_plastic.key, SolarThermal_EOL_landfill_plastic, 'technosphere'),
        (EF_EOL_inc_elastomer.key, SolarThermal_EOL_inc_elastomer, 'technosphere'),
        (EF_EOL_landfill_plastic.key, SolarThermal_EOL_landfill_elastomer, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in C4_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_SolarThermal = C4_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        C4_SolarThermal_acts[bd_name].new_exchange(
            input=C4_SolarThermal_acts[bd_name].key, 
            output=C4_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_SolarThermal.lci()
    mlca_C4_SolarThermal.lcia()
    mlca_C4_SolarThermal.scores

    dfresults_C4_SolarThermal = pd.DataFrame.from_dict(mlca_C4_SolarThermal.scores, orient='index')
    dfresults_C4_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_C4_SolarThermal.index, names=['Column', 'Row'])
    dfresults_C4_SolarThermal = dfresults_C4_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_SolarThermal.columns = dfresults_C4_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_SolarThermal = dfresults_C4_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.9 Module B4

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_SolarThermal = pd.DataFrame(index=dfresults_A13_SolarThermal.index)  # Start with the index from A13 for consistency   
    dfresults_B4_SolarThermal = (
        dfresults_A13_SolarThermal.add(dfresults_A4_SolarThermal, fill_value=0)
        .add(dfresults_C3_SolarThermal, fill_value=0)
        .add(dfresults_C4_SolarThermal, fill_value=0)
        ).mul(SolarThermalNumberReplacements, axis=0)

### 14.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_SolarThermal_exchange_configs = [
        (EF_recycling_copper.key, SolarThermal_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, SolarThermal_EOL_primary_copper, 'technosphere'),
        (EF_recycling_steel.key, SolarThermal_EOL_recycling_steel, 'technosphere'),
        (EF_primary_steel.key, SolarThermal_EOL_primary_steel, 'technosphere'),
        (EF_recycling_steel.key, SolarThermal_EOL_recycling_stainlessSteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, SolarThermal_EOL_primary_stainlessSteel, 'technosphere'),
        (EF_recycling_cardboard.key, SolarThermal_EOL_recycling_paperboard, 'technosphere'),
        (EF_primary_cardboard.key, SolarThermal_EOL_primary_paperboard, 'technosphere'),
        (EF_recycling_PEHD.key, SolarThermal_EOL_recycling_plastic, 'technosphere'),
        (EF_primary_PEHD.key, SolarThermal_EOL_primary_plastic, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in D1_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in D1_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_SolarThermal = D1_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        D1_SolarThermal_acts[bd_name].new_exchange(
            input=D1_SolarThermal_acts[bd_name].key, 
            output=D1_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_SolarThermal.lci()
    mlca_D1_SolarThermal.lcia()
    mlca_D1_SolarThermal.scores

    dfresults_D1_SolarThermal = pd.DataFrame.from_dict(mlca_D1_SolarThermal.scores, orient='index')
    dfresults_D1_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_D1_SolarThermal.index, names=['Column', 'Row'])
    dfresults_D1_SolarThermal = dfresults_D1_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_SolarThermal.columns = dfresults_D1_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_SolarThermal = dfresults_D1_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_SolarThermal = dfresults_D1_SolarThermal.add(dfresults_D1_SolarThermal.mul(SolarThermalNumberReplacements, axis=0))

### 14.2.12 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_SolarThermal_exchange_configs = [
        (EF_heat_energyrecovery.key, SolarThermal_EOL_energyrecoveryheat_paperboard, 'technosphere'),
        (EF_electricity_energyrecovery.key, SolarThermal_EOL_energyrecoveryele_paperboard, 'technosphere'),
        (EF_heat_energyrecovery.key, SolarThermal_EOL_energyrecoveryheat_plastic, 'technosphere'),
        (EF_electricity_energyrecovery.key, SolarThermal_EOL_energyrecoveryele_plastic, 'technosphere'),
        (EF_heat_energyrecovery.key, SolarThermal_EOL_energyrecoveryheat_elastomer, 'technosphere'),
        (EF_electricity_energyrecovery.key, SolarThermal_EOL_energyrecoveryele_elastomer, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SolarThermal_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_SolarThermal_exchange_list = []
        
        for input_key, data_array, exc_type in D3_SolarThermal_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_SolarThermal_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_SolarThermal_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_SolarThermal = D3_SolarThermal_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_SolarThermal.save()

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SolarThermal_BdName:
        D3_SolarThermal_acts[bd_name].new_exchange(
            input=D3_SolarThermal_acts[bd_name].key, 
            output=D3_SolarThermal_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_SolarThermal = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_SolarThermal['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_SolarThermal_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SolarThermal_BdName, SolarThermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_SolarThermal = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_SolarThermal.lci()
    mlca_D3_SolarThermal.lcia()
    mlca_D3_SolarThermal.scores

    dfresults_D3_SolarThermal = pd.DataFrame.from_dict(mlca_D3_SolarThermal.scores, orient='index')
    dfresults_D3_SolarThermal.index = pd.MultiIndex.from_tuples(dfresults_D3_SolarThermal.index, names=['Column', 'Row'])
    dfresults_D3_SolarThermal = dfresults_D3_SolarThermal.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_SolarThermal.columns = dfresults_D3_SolarThermal.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_SolarThermal.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_SolarThermal = dfresults_D3_SolarThermal.mul(SFSolarThermal, axis=0)

### 14.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "Solar thermal system"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_SolarThermal = dfresults_D3_SolarThermal.add(dfresults_D3_SolarThermal.mul(SolarThermalNumberReplacements, axis=0))

# 15. Electric-driven HPs activity

## 15.1 Activities

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_EHP_{bd_name}',
        'name': f'A13_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            A13_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")
            

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_EHP_{bd_name}',
        'name': f'A4_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            A4_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_EHP_{bd_name}',
        'name': f'A5_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            A5_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B1_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B1_EHP_{bd_name}',
        'name': f'B1_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B1_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            B1_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_EHP_{bd_name}',
        'name': f'B2_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            B2_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_EHP_{bd_name}',
        'name': f'B6_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            B6_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_EHP_{bd_name}',
        'name': f'C2_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            C2_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_EHP_{bd_name}',
        'name': f'C3_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            C3_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_EHP_{bd_name}',
        'name': f'C4_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            C4_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_EHP_{bd_name}',
        'name': f'D1_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            D1_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Electric Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_EHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in ElectricHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_EHP_{bd_name}',
        'name': f'D3_EHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_EHP_acts[bd_name] = scenario_db.new_activity(**data)
            D3_EHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 15.2 Exchanges

### 15.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_EHP_exchange_configs = [
        (EF_reinforcingsteel.key, EHP_reinforcingsteel, 'technosphere'),
        (EF_steel.key, EHP_lowalloyedsteel, 'technosphere'),
        (EF_Copper.key, EHP_copper, 'technosphere'),
        (EF_Elastomer.key, EHP_elastomer, 'technosphere'),
        (EF_PVC.key, EHP_PVC, 'technosphere'),
        (EF_airfun.key, EHP_airfun, 'technosphere'),
        (EF_Electronic.key, EHP_electronics, 'technosphere'),
        (EF_waterconsumption.key, EHP_waterconsumption, 'technosphere'),
        (EF_waterair.key, EHP_emissionwaterair, 'biosphere'),
        (EF_waterwater.key, EHP_emissionwaterwater, 'biosphere'),
        (EF_Wastewater_treatment.key, EHP_wastewatertreatment, 'technosphere'),
        (EF_Lubricating_oil.key, EHP_lubricatingoil, 'technosphere'),
        (EF_Electricity.key, EHP_electricity, 'technosphere'),
        (EF_Heat.key, EHP_heat, 'technosphere'),
        (EF_refrigerantA13.key, EHP_ethanepentafluoro_methanedifluoro, 'technosphere'),
        (EF_steel.key, EHP_steelwell, 'technosphere'),
        (EF_polyethylene.key, EHP_PEwell, 'technosphere'),
        (EF_bentonite.key, EHP_bentonitewell, 'technosphere'),
        (EF_cement.key, EHP_cementwell, 'technosphere'),
        (EF_glycole.key, EHP_glycolethylenewell, 'technosphere'),
        (EF_waterconsumption.key, EHP_resourcewaterwell, 'technosphere'),
        (EF_waterair.key, EHP_emissionwaterairwell, 'biosphere'),
        (EF_waterwater.key, EHP_emissionwaterwaterwell, 'biosphere'),
        (EF_ironsulfate.key, EHP_ironwell, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in A13_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_EHP = A13_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        A13_EHP_acts[bd_name].new_exchange(
            input=A13_EHP_acts[bd_name].key, 
            output=A13_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_EHP.lci()
    mlca_A13_EHP.lcia()
    mlca_A13_EHP.scores

    dfresults_A13_EHP = pd.DataFrame.from_dict(mlca_A13_EHP.scores, orient='index')
    dfresults_A13_EHP.index = pd.MultiIndex.from_tuples(dfresults_A13_EHP.index, names=['Column', 'Row'])
    dfresults_A13_EHP = dfresults_A13_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_EHP.columns = dfresults_A13_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_EHP = dfresults_A13_EHP.mul(SFEHP, axis=0)

### 15.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_EHP_exchange_configs = [
        (EF_Truk_16_32.key, EHP_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, EHP_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in A4_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_EHP = A4_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        A4_EHP_acts[bd_name].new_exchange(
            input=A4_EHP_acts[bd_name].key, 
            output=A4_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_EHP.lci()
    mlca_A4_EHP.lcia()
    mlca_A4_EHP.scores

    dfresults_A4_EHP = pd.DataFrame.from_dict(mlca_A4_EHP.scores, orient='index')
    dfresults_A4_EHP.index = pd.MultiIndex.from_tuples(dfresults_A4_EHP.index, names=['Column', 'Row'])
    dfresults_A4_EHP = dfresults_A4_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_EHP.columns = dfresults_A4_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_EHP = dfresults_A4_EHP.mul(SFEHP, axis=0)

### 15.2.3 Module A5

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A5_EHP_exchange_configs = [
        (EF_diesel.key, EHP_dieselA5, 'technosphere'),
        (EF_Light_Truk.key, EHP_Light_TrukA5, 'technosphere'),
        (EF_Truk_16_32.key, EHP_Truk_16_32A5, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        A5_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in A5_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A5_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A5_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A5_EHP = A5_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A5_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        A5_EHP_acts[bd_name].new_exchange(
            input=A5_EHP_acts[bd_name].key, 
            output=A5_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_EHP.lci()
    mlca_A5_EHP.lcia()
    mlca_A5_EHP.scores

    dfresults_A5_EHP = pd.DataFrame.from_dict(mlca_A5_EHP.scores, orient='index')
    dfresults_A5_EHP.index = pd.MultiIndex.from_tuples(dfresults_A5_EHP.index, names=['Column', 'Row'])
    dfresults_A5_EHP = dfresults_A5_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_EHP.columns = dfresults_A5_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_EHP = dfresults_A5_EHP.mul(SFEHP, axis=0)

### 15.2.4 Module B1

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B1_EHP_exchange_configs = [
        (EF_refrigerantA13.key, EHP_maintenancerefrigerantgas, 'technosphere'),
        (EF_ethanepentafluoro.key, EHP_maintenanceethanepentafluoro_methanedifluoro, 'biosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        B1_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in B1_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B1_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B1_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B1_EHP = B1_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B1_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        B1_EHP_acts[bd_name].new_exchange(
            input=B1_EHP_acts[bd_name].key, 
            output=B1_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B1_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B1_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B1_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B1_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B1_EHP.lci()
    mlca_B1_EHP.lcia()
    mlca_B1_EHP.scores

    dfresults_B1_EHP = pd.DataFrame.from_dict(mlca_B1_EHP.scores, orient='index')
    dfresults_B1_EHP.index = pd.MultiIndex.from_tuples(dfresults_B1_EHP.index, names=['Column', 'Row'])
    dfresults_B1_EHP = dfresults_B1_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B1_EHP.columns = dfresults_B1_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B1_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B1_EHP = dfresults_B1_EHP.mul(SFEHP, axis=0)

### 15.2.5 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_EHP_exchange_configs = [
        (EF_Light_Truk.key, EHP_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in B2_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in B2_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_EHP = B2_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        B2_EHP_acts[bd_name].new_exchange(
            input=B2_EHP_acts[bd_name].key, 
            output=B2_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_EHP.lci()
    mlca_B2_EHP.lcia()
    mlca_B2_EHP.scores

    dfresults_B2_EHP = pd.DataFrame.from_dict(mlca_B2_EHP.scores, orient='index')
    dfresults_B2_EHP.index = pd.MultiIndex.from_tuples(dfresults_B2_EHP.index, names=['Column', 'Row'])
    dfresults_B2_EHP = dfresults_B2_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_EHP.columns = dfresults_B2_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_EHP = dfresults_B2_EHP.mul(SFEHP, axis=0)

### 15.2.6 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_EHP_exchange_configs = [
        (EF_ethanepentafluoroB6.key, EHP_usephaserefrigerantgas, 'biosphere'),
        (EF_methanedifluoroB6.key, EHP_usephaseethanepentafluoro_methanedifluoro, 'biosphere'),
        (EF_electricity.key, EHP_usephaseelectricity, 'technosphere'),
        (EF_electricityPV.key, EHP_usephaseelectricityPV, 'technosphere'),
        # (EF_sun.key, EHP_usephaseelectricityRECSPV, 'technosphere'),
        # (EF_hydro.key, EHP_usephaseelectricityRECSHydro, 'technosphere'),
        # (EF_wind.key, EHP_usephaseelectricityRECSEolic, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in B6_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_EHP = B6_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        B6_EHP_acts[bd_name].new_exchange(
            input=B6_EHP_acts[bd_name].key, 
            output=B6_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_EHP.lci()
    mlca_B6_EHP.lcia()
    mlca_B6_EHP.scores

    dfresults_B6_EHP = pd.DataFrame.from_dict(mlca_B6_EHP.scores, orient='index')
    dfresults_B6_EHP.index = pd.MultiIndex.from_tuples(dfresults_B6_EHP.index, names=['Column', 'Row'])
    dfresults_B6_EHP = dfresults_B6_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_EHP.columns = dfresults_B6_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_EHP.rename(columns=method_lookup, inplace=True)

### 15.2.7 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_EHP_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, EHP_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in C2_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_EHP = C2_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        C2_EHP_acts[bd_name].new_exchange(
            input=C2_EHP_acts[bd_name].key, 
            output=C2_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_EHP.lci()
    mlca_C2_EHP.lcia()
    mlca_C2_EHP.scores

    dfresults_C2_EHP = pd.DataFrame.from_dict(mlca_C2_EHP.scores, orient='index')
    dfresults_C2_EHP.index = pd.MultiIndex.from_tuples(dfresults_C2_EHP.index, names=['Column', 'Row'])
    dfresults_C2_EHP = dfresults_C2_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_EHP.columns = dfresults_C2_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_EHP = dfresults_C2_EHP.mul(SFEHP, axis=0)

### 15.2.8 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_EHP_exchange_configs = [
        (EF_EOL_metal.key, EHP_EOL_reinforcingsteel, 'technosphere'),
        (EF_EOL_metal.key, EHP_EOL_lowalloyedsteel, 'technosphere'),
        (EF_EOL_metal.key, EHP_EOL_copper, 'technosphere'),
        (EF_EOL_plastic.key, EHP_EOL_elastomer, 'technosphere'),
        (EF_EOL_PVC.key, EHP_EOL_PVC, 'technosphere'),
        (EF_EOL_metal.key, EHP_EOL_airfun, 'technosphere'),
        (EF_EOL_electronics.key, EHP_EOL_electronics, 'technosphere'),
        (EF_EOL_refrigerantgas.key, EHP_EOL_refrigerantgas, 'technosphere'),
        (EF_EOL_metal.key, EHP_EOL_steelwell, 'technosphere'),
        (EF_EOL_plastic.key, EHP_EOL_PEwell, 'technosphere'),
        (EF_EOL_inert.key, EHP_EOL_bentonitewell, 'technosphere'),
        (EF_EOL_cement.key, EHP_EOL_cementwell, 'technosphere'),
        (EF_EOL_metal.key, EHP_EOL_ironwell, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in C3_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_EHP = C3_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        C3_EHP_acts[bd_name].new_exchange(
            input=C3_EHP_acts[bd_name].key, 
            output=C3_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_EHP.lci()
    mlca_C3_EHP.lcia()
    mlca_C3_EHP.scores

    dfresults_C3_EHP = pd.DataFrame.from_dict(mlca_C3_EHP.scores, orient='index')
    dfresults_C3_EHP.index = pd.MultiIndex.from_tuples(dfresults_C3_EHP.index, names=['Column', 'Row'])
    dfresults_C3_EHP = dfresults_C3_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_EHP.columns = dfresults_C3_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_EHP = dfresults_C3_EHP.mul(SFEHP, axis=0)

### 15.2.9 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_EHP_exchange_configs = [
        (EF_EOL_landfill_metal.key, EHP_EOL_landfill_reinforcingsteel, 'technosphere'),
        (EF_EOL_landfill_metal.key, EHP_EOL_landfill_lowalloyedsteel, 'technosphere'),
        (EF_EOL_landfill_metal.key, EHP_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_landfill_plastic.key, EHP_EOL_landfill_elastomer, 'technosphere'),
        (EF_EOL_landfill_plastic.key, EHP_EOL_landfill_PVC, 'technosphere'),
        (EF_EOL_landfill_metal.key, EHP_EOL_landfill_airfun, 'technosphere'),
        (EF_EOL_landfill_electronic.key, EHP_EOL_landfill_electronics, 'technosphere'),
        (EF_EOL_landfill_refrigerantgas.key, EHP_EOL_landfill_refrigerantgas, 'technosphere'),
        (EF_EOL_landfill_metal.key, EHP_EOL_landfill_steelwell, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, EHP_EOL_landfill_PEwell, 'technosphere'),
        (EF_EOL_landfill_inert.key, EHP_EOL_landfill_cementwell, 'technosphere'),
        (EF_EOL_landfill_metal.key, EHP_EOL_landfill_ironwell, 'technosphere'),
        (EF_EOL_inc_elastomer.key, EHP_EOL_inc_elastomer, 'technosphere'),
        (EF_EOL_inc_plastic.key, EHP_EOL_inc_PVC, 'technosphere'),
        (EF_EOL_inc_electronic.key, EHP_EOL_inc_electronics, 'technosphere'),
        (EF_EOL_inc_lubricatingoil.key, EHP_EOL_inc_lubricatingoil, 'technosphere'),
        (EF_EOL_inc_PEHD.key, EHP_EOL_inc_PEwell, 'technosphere'),
        (EF_EOL_inc_hazardouswaste.key, EHP_EOL_inc_bentonitewell, 'technosphere'),
        (EF_EOL_inc_hazardouswaste.key, EHP_EOL_inc_glycolethylenewell, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in C4_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_EHP = C4_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        C4_EHP_acts[bd_name].new_exchange(
            input=C4_EHP_acts[bd_name].key, 
            output=C4_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_EHP.lci()
    mlca_C4_EHP.lcia()
    mlca_C4_EHP.scores

    dfresults_C4_EHP = pd.DataFrame.from_dict(mlca_C4_EHP.scores, orient='index')
    dfresults_C4_EHP.index = pd.MultiIndex.from_tuples(dfresults_C4_EHP.index, names=['Column', 'Row'])
    dfresults_C4_EHP = dfresults_C4_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_EHP.columns = dfresults_C4_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_EHP = dfresults_C4_EHP.mul(SFEHP, axis=0)

### 15.2.10 Module B4

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_EHP = pd.DataFrame(index=dfresults_A13_EHP.index)  # Start with the index from A13 for consistency   
    dfresults_B4_EHP = (
        dfresults_A13_EHP.add(dfresults_A4_EHP, fill_value=0)
        .add(dfresults_C3_EHP, fill_value=0)
        .add(dfresults_C4_EHP, fill_value=0)
        ).mul(EHPNumberReplacements, axis=0)

### 15.2.11 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_EHP_exchange_configs = [
        (EF_recycling_steel.key, EHP_EOL_recycling_reinforcingsteel, 'technosphere'),
        (EF_primary_steel.key, EHP_EOL_primary_reinforcingsteel, 'technosphere'),
        (EF_recycling_steel.key, EHP_EOL_recycling_lowalloyedsteel, 'technosphere'),
        (EF_primary_steel.key, EHP_EOL_primary_lowalloyedsteel, 'technosphere'),
        (EF_recycling_copper.key, EHP_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, EHP_EOL_primary_copper, 'technosphere'),
        (EF_recycling_steel.key, EHP_EOL_recycling_airfun, 'technosphere'),
        (EF_primary_steel.key, EHP_EOL_primary_airfun, 'technosphere'),
        (EF_recycling_steel.key, EHP_EOL_recycling_steelwell, 'technosphere'),
        (EF_primary_steel.key, EHP_EOL_primary_steelwell, 'technosphere'),
        (EF_recycling_PEHD.key, EHP_EOL_recycling_PEwell, 'technosphere'),
        (EF_primary_PEHD.key, EHP_EOL_primary_PEwell, 'technosphere'),
        (EF_recycling_concrete.key, EHP_EOL_recycling_cementwell, 'technosphere'),
        (EF_primary_concrete.key, EHP_EOL_primary_cementwell, 'technosphere'),
        (EF_recycling_iron.key, EHP_EOL_recycling_ironwell, 'technosphere'),
        (EF_primary_steel.key, EHP_EOL_primary_ironwell, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in D1_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_EHP = D1_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        D1_EHP_acts[bd_name].new_exchange(
            input=D1_EHP_acts[bd_name].key, 
            output=D1_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_EHP.lci()
    mlca_D1_EHP.lcia()
    mlca_D1_EHP.scores

    dfresults_D1_EHP = pd.DataFrame.from_dict(mlca_D1_EHP.scores, orient='index')
    dfresults_D1_EHP.index = pd.MultiIndex.from_tuples(dfresults_D1_EHP.index, names=['Column', 'Row'])
    dfresults_D1_EHP = dfresults_D1_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_EHP.columns = dfresults_D1_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_EHP = dfresults_D1_EHP.mul(SFEHP, axis=0)

### 15.2.12 Module D1 (B4 included)

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_EHP = dfresults_D1_EHP.add(dfresults_D1_EHP.mul(EHPNumberReplacements, axis=0))

### 15.2.13 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_EHP_exchange_configs = [
        (EF_heat_energyrecovery.key, EHP_EOL_energyrecoveryheat_elastomer, 'technosphere'),
        (EF_electricity_energyrecovery.key, EHP_EOL_energyrecoveryele_elastomer, 'technosphere'),
        (EF_heat_energyrecovery.key, EHP_EOL_energyrecoveryheat_PVC, 'technosphere'),
        (EF_electricity_energyrecovery.key, EHP_EOL_energyrecoveryele_PVC, 'technosphere'),
        (EF_heat_energyrecovery.key, EHP_EOL_energyrecoveryheat_electronics, 'technosphere'),
        (EF_electricity_energyrecovery.key, EHP_EOL_energyrecoveryele_electronics, 'technosphere'),
        (EF_heat_energyrecovery.key, EHP_EOL_energyrecoveryheat_lubricatingoil, 'technosphere'),
        (EF_electricity_energyrecovery.key, EHP_EOL_energyrecoveryele_lubricatingoil, 'technosphere'),
        (EF_heat_energyrecovery.key, EHP_EOL_energyrecoveryheat_PEwell, 'technosphere'),
        (EF_electricity_energyrecovery.key, EHP_EOL_energyrecoveryele_PEwell, 'technosphere'),
        (EF_heat_energyrecovery.key, EHP_EOL_energyrecoveryheat_bentonitewell, 'technosphere'),
        (EF_electricity_energyrecovery.key, EHP_EOL_energyrecoveryele_bentonitewell, 'technosphere'),
        (EF_heat_energyrecovery.key, EHP_EOL_energyrecoveryheat_glycolethylenewell, 'technosphere'),
        (EF_electricity_energyrecovery.key, EHP_EOL_energyrecoveryele_glycolethylenewell, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ElectricHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_EHP_exchange_list = []
        
        for input_key, data_array, exc_type in D3_EHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_EHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_EHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_EHP = D3_EHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_EHP.save()

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ElectricHeatPump_BdName:
        D3_EHP_acts[bd_name].new_exchange(
            input=D3_EHP_acts[bd_name].key, 
            output=D3_EHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_EHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_EHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_EHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ElectricHeatPump_BdName, ElectricHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_EHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_EHP.lci()
    mlca_D3_EHP.lcia()
    mlca_D3_EHP.scores

    dfresults_D3_EHP = pd.DataFrame.from_dict(mlca_D3_EHP.scores, orient='index')
    dfresults_D3_EHP.index = pd.MultiIndex.from_tuples(dfresults_D3_EHP.index, names=['Column', 'Row'])
    dfresults_D3_EHP = dfresults_D3_EHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_EHP.columns = dfresults_D3_EHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_EHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_EHP = dfresults_D3_EHP.mul(SFEHP, axis=0)

### 15.2.14 Module D3 (B4 included)

In [ ]:
sheet_name = "Electric Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_EHP = dfresults_D3_EHP.add(dfresults_D3_EHP.mul(EHPNumberReplacements, axis=0))

# 16. Gas-absorption Heat Pump

## 16.1 Activities

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_GAHP_{bd_name}',
        'name': f'A13_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            A13_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_GAHP_{bd_name}',
        'name': f'A4_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            A4_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_GAHP_{bd_name}',
        'name': f'A5_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            A5_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_GAHP_{bd_name}',
        'name': f'B2_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            B2_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_GAHP_{bd_name}',
        'name': f'B6_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            B6_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_GAHP_{bd_name}',
        'name': f'C2_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            C2_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_GAHP_{bd_name}',
        'name': f'C3_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            C3_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_GAHP_{bd_name}',
        'name': f'C4_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            C4_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_GAHP_{bd_name}',
        'name': f'D1_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            D1_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Absorption Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_GAHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasAbsorptionHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_GAHP_{bd_name}',
        'name': f'D3_GAHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_GAHP_acts[bd_name] = scenario_db.new_activity(**data)
            D3_GAHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 16.2 Exchanges

### 16.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_GAHP_exchange_configs = [
        (EF_Copper.key, GAHP_copper, 'technosphere'),
        (EF_Aluminium.key, GAHP_aluminium, 'technosphere'),
        (EF_steel.key, GAHP_lowalloyedsteel, 'technosphere'),
        (EF_StainlessSteel.key, GAHP_stainlessSteel, 'technosphere'),
        (EF_Ferrite.key, GAHP_ferrite, 'technosphere'),
        (EF_brass.key, GAHP_brass, 'technosphere'),
        (EF_Alumina.key, GAHP_alumina, 'technosphere'),
        (EF_NBR.key, GAHP_NBR, 'technosphere'),
        (EF_Elastomer.key, GAHP_syntheticRubber, 'technosphere'),
        (EF_AmmoniaA13.key, GAHP_ammonia, 'technosphere'),
        (EF_waterIn2.key, GAHP_water, 'biosphere'),
        (EF_Electronic.key, GAHP_electronics, 'technosphere'),
        (EF_mineralwool.key, GAHP_mineralwool, 'technosphere'),
        (EF_polyurethane.key, GAHP_PUfoam, 'technosphere'),
        (EF_Elastomer.key, GAHP_EPDM, 'technosphere'),
        (EF_steel.key, GAHP_batterySteel, 'technosphere'),
        (EF_Aluminium.key, GAHP_batteryAluminium, 'technosphere'),
        (EF_StainlessSteel.key, GAHP_airfun, 'technosphere'),
        (EF_StainlessSteel.key, GAHP_pump, 'technosphere'),
        (EF_Lubricating_oil.key, GAHP_lubricatingOil, 'technosphere'),
        (EF_weldingarcsteel.key, GAHP_welding, 'technosphere'),
        (EF_waterconsumption.key, GAHP_WaterConsumption, 'technosphere'),
        (EF_waterair.key, GAHP_EmissionsWaterAir, 'biosphere'),
        (EF_waterwater.key, GAHP_EmissionsWaterWater, 'biosphere'),
        (EF_Wastewater_treatment.key, GAHP_WastewaterTreatment, 'technosphere'),
        (EF_Electricity.key, GAHP_electricityA13, 'technosphere'),
        (EF_Heat.key, GAHP_heat, 'technosphere'),
        (EF_Hazardous.key, GAHP_hazardouswaste, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in A13_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_GAHP = A13_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        A13_GAHP_acts[bd_name].new_exchange(
            input=A13_GAHP_acts[bd_name].key, 
            output=A13_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_GAHP.lci()
    mlca_A13_GAHP.lcia()
    mlca_A13_GAHP.scores

    dfresults_A13_GAHP = pd.DataFrame.from_dict(mlca_A13_GAHP.scores, orient='index')
    dfresults_A13_GAHP.index = pd.MultiIndex.from_tuples(dfresults_A13_GAHP.index, names=['Column', 'Row'])
    dfresults_A13_GAHP = dfresults_A13_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_GAHP.columns = dfresults_A13_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_GAHP = dfresults_A13_GAHP.mul(SFGAHP, axis=0)

### 16.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_GAHP_exchange_configs = [
        (EF_Truk_16_32.key, GAHP_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, GAHP_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in A4_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_GAHP = A4_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        A4_GAHP_acts[bd_name].new_exchange(
            input=A4_GAHP_acts[bd_name].key, 
            output=A4_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_GAHP.lci()
    mlca_A4_GAHP.lcia()
    mlca_A4_GAHP.scores

    dfresults_A4_GAHP = pd.DataFrame.from_dict(mlca_A4_GAHP.scores, orient='index')
    dfresults_A4_GAHP.index = pd.MultiIndex.from_tuples(dfresults_A4_GAHP.index, names=['Column', 'Row'])
    dfresults_A4_GAHP = dfresults_A4_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_GAHP.columns = dfresults_A4_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_GAHP = dfresults_A4_GAHP.mul(SFGAHP, axis=0)

### 16.2.3 Module A5

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        A5_GAHP_acts[bd_name].new_exchange(
            input=A5_GAHP_acts[bd_name].key, 
            output=A5_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_GAHP.lci()
    mlca_A5_GAHP.lcia()
    mlca_A5_GAHP.scores

    dfresults_A5_GAHP = pd.DataFrame.from_dict(mlca_A5_GAHP.scores, orient='index')
    dfresults_A5_GAHP.index = pd.MultiIndex.from_tuples(dfresults_A5_GAHP.index, names=['Column', 'Row'])
    dfresults_A5_GAHP = dfresults_A5_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_GAHP.columns = dfresults_A5_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_GAHP = dfresults_A5_GAHP.mul(SFGAHP, axis=0)

### 16.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_GAHP_exchange_configs = [
        (EF_LightTruk.key, GAHP_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in B2_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_GAHP = B2_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        B2_GAHP_acts[bd_name].new_exchange(
            input=B2_GAHP_acts[bd_name].key, 
            output=B2_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_GAHP.lci()
    mlca_B2_GAHP.lcia()
    mlca_B2_GAHP.scores

    dfresults_B2_GAHP = pd.DataFrame.from_dict(mlca_B2_GAHP.scores, orient='index')
    dfresults_B2_GAHP.index = pd.MultiIndex.from_tuples(dfresults_B2_GAHP.index, names=['Column', 'Row'])
    dfresults_B2_GAHP = dfresults_B2_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_GAHP.columns = dfresults_B2_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_GAHP = dfresults_B2_GAHP.mul(SFGAHP, axis=0)

### 16.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_GAHP_exchange_configs = [
        (EF_Acetaldehyde.key, GAHP_acetaldehyde, 'biosphere'),
        (EF_Aceticacid.key, GAHP_aceticacid, 'biosphere'),
        (EF_Benzene.key, GAHP_benzene, 'biosphere'),
        (EF_Benzoapyrene.key, GAHP_benzoapyrene, 'biosphere'),
        (EF_Buthane.key, GAHP_butane, 'biosphere'),
        (EF_CarbondioxideFossil.key, GAHP_carbonDioxideFossil, 'biosphere'),
        (EF_CarbonmonoxideFossil.key, GAHP_carbonMonoxideFossil, 'biosphere'),
        (EF_DinitrogenMonoxide.key, GAHP_dinitrogenMonoxide, 'biosphere'),
        (EF_Formaldehyde.key, GAHP_formaldehyde, 'biosphere'),
        (EF_Mercury.key, GAHP_mercury, 'biosphere'),
        (EF_MethaneFossil.key, GAHP_methaneFossil, 'biosphere'),
        (EF_NitrogenOxides.key, GAHP_nitrogenOxides, 'biosphere'),
        (EF_PAH.key, GAHP_PAH, 'biosphere'),
        (EF_Particulates025.key, GAHP_particulates, 'biosphere'),
        (EF_Pentane.key, GAHP_pentane, 'biosphere'),
        (EF_PropaneB6.key, GAHP_propane, 'biosphere'),
        (EF_PropionicAcid.key, GAHP_propionicAcid, 'biosphere'),
        (EF_SulfurDioxide.key, GAHP_sulfurDioxide, 'biosphere'),
        (EF_Toluene.key, GAHP_toluene, 'biosphere'),
        (EF_Nitrate.key, GAHP_nitrate, 'biosphere'),
        (EF_Nitrite.key, GAHP_nitrite, 'biosphere'),
        (EF_Sulfate.key, GAHP_sulfate, 'biosphere'),
        (EF_Sulfite.key, GAHP_sulfite, 'biosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in B6_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_GAHP = B6_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        B6_GAHP_acts[bd_name].new_exchange(
            input=B6_GAHP_acts[bd_name].key, 
            output=B6_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_GAHP.lci()
    mlca_B6_GAHP.lcia()
    mlca_B6_GAHP.scores

    dfresults_B6_GAHP = pd.DataFrame.from_dict(mlca_B6_GAHP.scores, orient='index')
    dfresults_B6_GAHP.index = pd.MultiIndex.from_tuples(dfresults_B6_GAHP.index, names=['Column', 'Row'])
    dfresults_B6_GAHP = dfresults_B6_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_GAHP.columns = dfresults_B6_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_GAHP.rename(columns=method_lookup, inplace=True)

### 16.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_GAHP_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, GAHP_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in C2_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_GAHP = C2_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        C2_GAHP_acts[bd_name].new_exchange(
            input=C2_GAHP_acts[bd_name].key, 
            output=C2_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_GAHP.lci()
    mlca_C2_GAHP.lcia()
    mlca_C2_GAHP.scores

    dfresults_C2_GAHP = pd.DataFrame.from_dict(mlca_C2_GAHP.scores, orient='index')
    dfresults_C2_GAHP.index = pd.MultiIndex.from_tuples(dfresults_C2_GAHP.index, names=['Column', 'Row'])
    dfresults_C2_GAHP = dfresults_C2_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_GAHP.columns = dfresults_C2_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_GAHP = dfresults_C2_GAHP.mul(SFGAHP, axis=0)

### 16.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_GAHP_exchange_configs = [
        (EF_EOL_metal.key, GAHP_EOL_copper, 'technosphere'),
        (EF_EOL_Aluminium.key, GAHP_EOL_aluminium, 'technosphere'),
        (EF_EOL_metal.key, GAHP_EOL_lowalloyedsteel, 'technosphere'),
        (EF_EOL_metal.key, GAHP_EOL_stainlessSteel, 'technosphere'),
        (EF_EOL_metal.key, GAHP_EOL_ferrite, 'technosphere'),
        (EF_EOL_metal.key, GAHP_EOL_brass, 'technosphere'),
        (EF_EOL_Aluminium.key, GAHP_EOL_alumina, 'technosphere'),
        (EF_EOL_Elastomer.key, GAHP_EOL_NBR, 'technosphere'),
        (EF_EOL_Elastomer.key, GAHP_EOL_syntheticRubber, 'technosphere'),
        (EF_EOL_electronics.key, GAHP_EOL_electronics, 'technosphere'),
        (EF_EOL_stonewool.key, GAHP_EOL_mineralwool, 'technosphere'),
        (EF_EOL_PUfoam.key, GAHP_EOL_PUfoam, 'technosphere'),
        (EF_EOL_Elastomer.key, GAHP_EOL_EPDM, 'technosphere'),
        (EF_EOL_metal.key, GAHP_EOL_batterySteel, 'technosphere'),
        (EF_EOL_Aluminium.key, GAHP_EOL_batteryAluminium, 'technosphere'),
        (EF_EOL_metal.key, GAHP_EOL_airfun, 'technosphere'),
        (EF_EOL_metal.key, GAHP_EOL_pump, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in C3_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_GAHP = C3_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        C3_GAHP_acts[bd_name].new_exchange(
            input=C3_GAHP_acts[bd_name].key, 
            output=C3_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_GAHP.lci()
    mlca_C3_GAHP.lcia()
    mlca_C3_GAHP.scores

    dfresults_C3_GAHP = pd.DataFrame.from_dict(mlca_C3_GAHP.scores, orient='index')
    dfresults_C3_GAHP.index = pd.MultiIndex.from_tuples(dfresults_C3_GAHP.index, names=['Column', 'Row'])
    dfresults_C3_GAHP = dfresults_C3_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_GAHP.columns = dfresults_C3_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_GAHP = dfresults_C3_GAHP.mul(SFGAHP, axis=0)

### 16.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_GAHP_exchange_configs = [
        (EF_EOL_landfill_metal.key, GAHP_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, GAHP_EOL_landfill_aluminium, 'technosphere'),
        (EF_EOL_landfill_metal.key, GAHP_EOL_landfill_lowalloyedsteel, 'technosphere'),
        (EF_EOL_landfill_metal.key, GAHP_EOL_landfill_stainlessSteel, 'technosphere'),
        (EF_EOL_landfill_metal.key, GAHP_EOL_landfill_ferrite, 'technosphere'),
        (EF_EOL_landfill_metal.key, GAHP_EOL_landfill_brass, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, GAHP_EOL_landfill_alumina, 'technosphere'),
        (EF_EOL_inc_elastomer.key, GAHP_EOL_inc_NBR, 'technosphere'),
        (EF_EOL_landfill_plastic.key, GAHP_EOL_landfill_NBR, 'technosphere'),
        (EF_EOL_inc_elastomer.key, GAHP_EOL_inc_syntheticRubber, 'technosphere'),
        (EF_EOL_landfill_plastic.key, GAHP_EOL_landfill_syntheticRubber, 'technosphere'),
        (EF_EOL_inc_hazardouswaste.key, GAHP_EOL_landfill_ammonia, 'technosphere'),
        (EF_EOL_inc_electronic.key, GAHP_EOL_inc_electronics, 'technosphere'),
        (EF_EOL_landfill_electronic.key, GAHP_EOL_landfill_electronics, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, GAHP_EOL_landfill_mineralwool, 'technosphere'),
        (EF_EOL_landfill_inert.key, GAHP_EOL_landfill_PUfoam, 'technosphere'),
        (EF_EOL_inc_elastomer.key, GAHP_EOL_inc_EPDM, 'technosphere'),
        (EF_EOL_landfill_plastic.key, GAHP_EOL_landfill_EPDM, 'technosphere'),
        (EF_EOL_landfill_metal.key, GAHP_EOL_landfill_batterySteel, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, GAHP_EOL_landfill_batteryAluminium, 'technosphere'),
        (EF_EOL_landfill_metal.key, GAHP_EOL_landfill_airfun, 'technosphere'),
        (EF_EOL_landfill_metal.key, GAHP_EOL_landfill_pump, 'technosphere'),
        (EF_EOL_inc_hazardouswaste.key, GAHP_EOL_inc_hazardouswaste, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in C4_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_GAHP = C4_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        C4_GAHP_acts[bd_name].new_exchange(
            input=C4_GAHP_acts[bd_name].key, 
            output=C4_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_GAHP.lci()
    mlca_C4_GAHP.lcia()
    mlca_C4_GAHP.scores

    dfresults_C4_GAHP = pd.DataFrame.from_dict(mlca_C4_GAHP.scores, orient='index')
    dfresults_C4_GAHP.index = pd.MultiIndex.from_tuples(dfresults_C4_GAHP.index, names=['Column', 'Row'])
    dfresults_C4_GAHP = dfresults_C4_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_GAHP.columns = dfresults_C4_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_GAHP = dfresults_C4_GAHP.mul(SFGAHP, axis=0)

### 16.2.9 Module B4

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_GAHP = pd.DataFrame(index=dfresults_A13_GAHP.index)  # Start with the index from A13 for consistency   
    dfresults_B4_GAHP = (
        dfresults_A13_GAHP.add(dfresults_A4_GAHP, fill_value=0)
        .add(dfresults_C3_GAHP, fill_value=0)
        .add(dfresults_C4_GAHP, fill_value=0)
        ).mul(GAHPNumberReplacements, axis=0)

### 16.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_GAHP_exchange_configs = [
        (EF_recycling_copper.key, GAHP_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, GAHP_EOL_primary_copper, 'technosphere'),
        (EF_recycling_aluminium.key, GAHP_EOL_recycling_aluminium, 'technosphere'),
        (EF_primary_aluminium.key, GAHP_EOL_primary_aluminium, 'technosphere'),
        (EF_recycling_steel.key, GAHP_EOL_recycling_lowalloyedsteel, 'technosphere'),
        (EF_primary_steel.key, GAHP_EOL_primary_lowalloyedsteel, 'technosphere'),
        (EF_recycling_steel.key, GAHP_EOL_recycling_stainlessSteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, GAHP_EOL_primary_stainlessSteel, 'technosphere'),
        (EF_recycling_steel.key, GAHP_EOL_recycling_batterysteel, 'technosphere'),
        (EF_primary_steel.key, GAHP_EOL_primary_batterysteel, 'technosphere'),
        (EF_recycling_aluminium.key, GAHP_EOL_recycling_batteryAluminium, 'technosphere'),
        (EF_primary_aluminium.key, GAHP_EOL_primary_batteryAluminium, 'technosphere'),
        (EF_recycling_steel.key, GAHP_EOL_recycling_airfun, 'technosphere'),
        (EF_primary_steel.key, GAHP_EOL_primary_airfun, 'technosphere'),
        (EF_recycling_steel.key, GAHP_EOL_recycling_pump, 'technosphere'),
        (EF_primary_steel.key, GAHP_EOL_primary_pump, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in D1_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_GAHP = D1_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        D1_GAHP_acts[bd_name].new_exchange(
            input=D1_GAHP_acts[bd_name].key, 
            output=D1_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_GAHP.lci()
    mlca_D1_GAHP.lcia()
    mlca_D1_GAHP.scores

    dfresults_D1_GAHP = pd.DataFrame.from_dict(mlca_D1_GAHP.scores, orient='index')
    dfresults_D1_GAHP.index = pd.MultiIndex.from_tuples(dfresults_D1_GAHP.index, names=['Column', 'Row'])
    dfresults_D1_GAHP = dfresults_D1_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_GAHP.columns = dfresults_D1_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_GAHP = dfresults_D1_GAHP.mul(SFGAHP, axis=0)

### 16.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_GAHP = dfresults_D1_GAHP.add(dfresults_D1_GAHP.mul(GAHPNumberReplacements, axis=0))

### 16.2.12 Module D3 (related to the export of energy as results of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_GAHP_exchange_configs = [
        (EF_heat_energyrecovery.key, GAHP_EOL_energyrecoveryheat_NBR, 'technosphere'),
        (EF_electricity_energyrecovery.key, GAHP_EOL_energyrecoveryele_NBR, 'technosphere'),
        (EF_heat_energyrecovery.key, GAHP_EOL_energyrecoveryheat_syntheticRubber, 'technosphere'),
        (EF_electricity_energyrecovery.key, GAHP_EOL_energyrecoveryele_syntheticRubber, 'technosphere'),
        (EF_heat_energyrecovery.key, GAHP_EOL_energyrecoveryheat_electronics, 'technosphere'),
        (EF_electricity_energyrecovery.key, GAHP_EOL_energyrecoveryele_electronics, 'technosphere'),
        (EF_heat_energyrecovery.key, GAHP_EOL_energyrecoveryheat_EPDM, 'technosphere'),
        (EF_electricity_energyrecovery.key, GAHP_EOL_energyrecoveryele_EPDM, 'technosphere'),
        (EF_heat_energyrecovery.key, GAHP_EOL_energyrecoveryheat_hazardouswaste, 'technosphere'),
        (EF_electricity_energyrecovery.key, GAHP_EOL_energyrecoveryele_hazardouswaste, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasAbsorptionHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_GAHP_exchange_list = []
        
        for input_key, data_array, exc_type in D3_GAHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_GAHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_GAHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_GAHP = D3_GAHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_GAHP.save()

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasAbsorptionHeatPump_BdName:
        D3_GAHP_acts[bd_name].new_exchange(
            input=D3_GAHP_acts[bd_name].key, 
            output=D3_GAHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_GAHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_GAHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_GAHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasAbsorptionHeatPump_BdName, GasAbsorptionHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_GAHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_GAHP.lci()
    mlca_D3_GAHP.lcia()
    mlca_D3_GAHP.scores

    dfresults_D3_GAHP = pd.DataFrame.from_dict(mlca_D3_GAHP.scores, orient='index')
    dfresults_D3_GAHP.index = pd.MultiIndex.from_tuples(dfresults_D3_GAHP.index, names=['Column', 'Row'])
    dfresults_D3_GAHP = dfresults_D3_GAHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_GAHP.columns = dfresults_D3_GAHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_GAHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_GAHP = dfresults_D3_GAHP.mul(SFGAHP, axis=0)

### 16.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "Gas Absorption Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_GAHP = dfresults_D3_GAHP.add(dfresults_D3_GAHP.mul(GAHPNumberReplacements, axis=0))

# 17. Gas-engine Heat Pump

## 17.1 Activities

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_GEHP_{bd_name}',
        'name': f'A13_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            A13_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_GEHP_{bd_name}',
        'name': f'A4_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            A4_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_GEHP_{bd_name}',
        'name': f'A5_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            A5_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B1_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B1_GEHP_{bd_name}',
        'name': f'B1_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B1_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            B1_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_GEHP_{bd_name}',
        'name': f'B2_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            B2_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_GEHP_{bd_name}',
        'name': f'B6_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            B6_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_GEHPEle_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_GEHPEle_{bd_name}',
        'name': f'B6_GEHPEle_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_GEHPEle_acts[bd_name] = scenario_db.new_activity(**data)
            B6_GEHPEle_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_GEHP_{bd_name}',
        'name': f'C2_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            C2_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_GEHP_{bd_name}',
        'name': f'C3_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            C3_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_GEHP_{bd_name}',
        'name': f'C4_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            C4_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_GEHP_{bd_name}',
        'name': f'D1_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            D1_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Gas Engine Heat Pump"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_GEHP_acts = {}  # Dictionary to store activities by building name
    for bd_name in GasEngineHeatPump_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_GEHP_{bd_name}',
        'name': f'D3_GEHP_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_GEHP_acts[bd_name] = scenario_db.new_activity(**data)
            D3_GEHP_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 17.2 Exchanges

### 17.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_GEHP_exchange_configs = [
        (EF_Copper.key, GEHP_copper, 'technosphere'),
        (EF_Aluminium.key, GEHP_aluminium, 'technosphere'),
        (EF_steel.key, GEHP_lowalloyedsteel, 'technosphere'),
        (EF_StainlessSteel.key, GEHP_stainlessSteel, 'technosphere'),
        (EF_Ferrite.key, GEHP_ferrite, 'technosphere'),
        (EF_brass.key, GEHP_brass, 'technosphere'),
        (EF_Alumina.key, GEHP_alumina, 'technosphere'),
        (EF_Electronic.key, GEHP_electronics, 'technosphere'),
        (EF_Lubricating_oil.key, GEHP_lubricatingoil, 'technosphere'),
        (EF_weldingarcsteel.key, GEHP_welding, 'technosphere'),
        (EF_waterconsumption.key, GEHP_waterconsumption, 'technosphere'),
        (EF_waterair.key, GEHP_emissionwaterair, 'biosphere'),
        (EF_waterwater.key, GEHP_emissionwaterwater, 'biosphere'),
        (EF_Wastewater_treatment.key, GEHP_wastewatertreatment, 'technosphere'),
        (EF_Electricity.key, GEHP_electricityA13, 'technosphere'),
        (EF_Heat.key, GEHP_heat, 'technosphere'),
        (EF_Hazardous.key, GEHP_hazardouswaste, 'technosphere'),
        (EF_refrigerantA13.key, GEHP_refrigerantgas, 'technosphere'),
    ]
    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in A13_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_GEHP = A13_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        A13_GEHP_acts[bd_name].new_exchange(
            input=A13_GEHP_acts[bd_name].key, 
            output=A13_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_GEHP.lci()
    mlca_A13_GEHP.lcia()
    mlca_A13_GEHP.scores

    dfresults_A13_GEHP = pd.DataFrame.from_dict(mlca_A13_GEHP.scores, orient='index')
    dfresults_A13_GEHP.index = pd.MultiIndex.from_tuples(dfresults_A13_GEHP.index, names=['Column', 'Row'])
    dfresults_A13_GEHP = dfresults_A13_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_GEHP.columns = dfresults_A13_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_GEHP = dfresults_A13_GEHP.mul(SFGEHP, axis=0)

### 17.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_GEHP_exchange_configs = [
        (EF_Truk_16_32.key, GEHP_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, GEHP_Light_Truk, 'technosphere'),
    ]
    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in A4_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_GEHP = A4_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        A4_GEHP_acts[bd_name].new_exchange(
            input=A4_GEHP_acts[bd_name].key, 
            output=A4_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_GEHP.lci()
    mlca_A4_GEHP.lcia()
    mlca_A4_GEHP.scores

    dfresults_A4_GEHP = pd.DataFrame.from_dict(mlca_A4_GEHP.scores, orient='index')
    dfresults_A4_GEHP.index = pd.MultiIndex.from_tuples(dfresults_A4_GEHP.index, names=['Column', 'Row'])
    dfresults_A4_GEHP = dfresults_A4_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_GEHP.columns = dfresults_A4_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_GEHP = dfresults_A4_GEHP.mul(SFGEHP, axis=0)

### 17.2.3 Module A5

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        A5_GEHP_acts[bd_name].new_exchange(
            input=A5_GEHP_acts[bd_name].key, 
            output=A5_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_GEHP.lci()
    mlca_A5_GEHP.lcia()
    mlca_A5_GEHP.scores

    dfresults_A5_GEHP = pd.DataFrame.from_dict(mlca_A5_GEHP.scores, orient='index')
    dfresults_A5_GEHP.index = pd.MultiIndex.from_tuples(dfresults_A5_GEHP.index, names=['Column', 'Row'])
    dfresults_A5_GEHP = dfresults_A5_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_GEHP.columns = dfresults_A5_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_GEHP = dfresults_A5_GEHP.mul(SFGEHP, axis=0)

### 17.2.4 Module B1

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B1_GEHP_exchange_configs = [
        (EF_refrigerantA13.key, GEHP_maintenancerefrigerantgas, 'technosphere'),
        (EF_ethanepentafluoro.key, GEHP_maintenanceethanepentafluoro_methanedifluoro, 'biosphere'),
    ]
    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        B1_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in B1_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B1_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B1_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B1_GEHP = B1_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B1_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        B1_GEHP_acts[bd_name].new_exchange(
            input=B1_GEHP_acts[bd_name].key, 
            output=B1_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B1_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B1_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B1_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B1_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B1_GEHP.lci()
    mlca_B1_GEHP.lcia()
    mlca_B1_GEHP.scores

    dfresults_B1_GEHP = pd.DataFrame.from_dict(mlca_B1_GEHP.scores, orient='index')
    dfresults_B1_GEHP.index = pd.MultiIndex.from_tuples(dfresults_B1_GEHP.index, names=['Column', 'Row'])
    dfresults_B1_GEHP = dfresults_B1_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B1_GEHP.columns = dfresults_B1_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B1_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B1_GEHP = dfresults_B1_GEHP.mul(SFGEHP, axis=0)

### 17.2.5 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_GEHP_exchange_configs = [
        (EF_Light_Truk.key, GEHP_Light_Truk_B2, 'technosphere'),
    ]
    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in B2_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_GEHP = B2_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        B2_GEHP_acts[bd_name].new_exchange(
            input=B2_GEHP_acts[bd_name].key, 
            output=B2_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_GEHP.lci()
    mlca_B2_GEHP.lcia()
    mlca_B2_GEHP.scores

    dfresults_B2_GEHP = pd.DataFrame.from_dict(mlca_B2_GEHP.scores, orient='index')
    dfresults_B2_GEHP.index = pd.MultiIndex.from_tuples(dfresults_B2_GEHP.index, names=['Column', 'Row'])
    dfresults_B2_GEHP = dfresults_B2_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_GEHP.columns = dfresults_B2_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_GEHP = dfresults_B2_GEHP.mul(SFGEHP, axis=0)

### 17.2.6 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_GEHP_exchange_configs = [
        (EF_energyvectorNG.key, GEHP_EnergyVectorNG, 'technosphere'),
        (EF_energyvectorBiom.key, GEHP_EnergyVectorBiomethane, 'technosphere'),
        (EF_energyvectorHydrogen.key, GEHP_EnergyVectorHydrogen, 'technosphere'),
        (EF_energyvectorBiom.key, GEHP_EnergyVectorBiomethaneInput, 'technosphere'),
        (EF_LPG.key, GEHP_EnergyVectorLPGinput, 'technosphere'),
        (EF_energyvectorHydrogen.key, GEHP_EnergyVectorHydrogeninput, 'technosphere'),

        (EF_Acetaldehyde.key, GEHP_acetaldehyde, 'biosphere'),
        (EF_Aceticacid.key, GEHP_aceticacid, 'biosphere'),
        (EF_Benzene.key, GEHP_benzene, 'biosphere'),
        (EF_Benzoapyrene.key, GEHP_benzoapyrene, 'biosphere'),
        (EF_Buthane.key, GEHP_butane, 'biosphere'),
        (EF_CarbondioxideFossil.key, GEHP_carbonDioxideFossil, 'biosphere'),
        (EF_CarbonmonoxideFossil.key, GEHP_carbonMonoxideFossil, 'biosphere'),
        (EF_DinitrogenMonoxide.key, GEHP_dinitrogenMonoxide, 'biosphere'),
        (EF_Formaldehyde.key, GEHP_formaldehyde, 'biosphere'),
        (EF_Mercury.key, GEHP_mercury, 'biosphere'),
        (EF_MethaneFossil.key, GEHP_methaneFossil, 'biosphere'),
        (EF_NitrogenOxides.key, GEHP_nitrogenOxides, 'biosphere'),
        (EF_PAH.key, GEHP_PAH, 'biosphere'),
        (EF_Particulates025.key, GEHP_particulates, 'biosphere'),
        (EF_Pentane.key, GEHP_pentane, 'biosphere'),
        (EF_PropaneB6.key, GEHP_propane, 'biosphere'),
        (EF_PropionicAcid.key, GEHP_propionicAcid, 'biosphere'),
        (EF_SulfurDioxide.key, GEHP_sulfurDioxide, 'biosphere'),
        (EF_Toluene.key, GEHP_toluene, 'biosphere'),
        (EF_Nitrate.key, GEHP_nitrate, 'biosphere'),
        (EF_Nitrite.key, GEHP_nitrite, 'biosphere'),
        (EF_Sulfate.key, GEHP_sulfate, 'biosphere'),
        (EF_Sulfite.key, GEHP_sulfite, 'biosphere'),
        (EF_ethanepentafluoroB6.key, GEHP_usephaserefrigerantgas, 'biosphere'),
        (EF_methanedifluoroB6.key, GEHP_usephaseethanepentafluoro_methanedifluoro, 'biosphere'),
        (EF_electricity.key, GEHP_electricityB6, 'technosphere'),
    ]
    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in B6_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_GEHP = B6_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        B6_GEHP_acts[bd_name].new_exchange(
            input=B6_GEHP_acts[bd_name].key, 
            output=B6_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_GEHP.lci()
    mlca_B6_GEHP.lcia()
    mlca_B6_GEHP.scores

    dfresults_B6_GEHP = pd.DataFrame.from_dict(mlca_B6_GEHP.scores, orient='index')
    dfresults_B6_GEHP.index = pd.MultiIndex.from_tuples(dfresults_B6_GEHP.index, names=['Column', 'Row'])
    dfresults_B6_GEHP = dfresults_B6_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_GEHP.columns = dfresults_B6_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_GEHP.rename(columns=method_lookup, inplace=True)

### 17.2.7 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_GEHP_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, GEHP_Transport_to_treatment_plant, 'technosphere'),
    ]
    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in C2_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_GEHP = C2_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        C2_GEHP_acts[bd_name].new_exchange(
            input=C2_GEHP_acts[bd_name].key, 
            output=C2_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_GEHP.lci()
    mlca_C2_GEHP.lcia()
    mlca_C2_GEHP.scores

    dfresults_C2_GEHP = pd.DataFrame.from_dict(mlca_C2_GEHP.scores, orient='index')
    dfresults_C2_GEHP.index = pd.MultiIndex.from_tuples(dfresults_C2_GEHP.index, names=['Column', 'Row'])
    dfresults_C2_GEHP = dfresults_C2_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_GEHP.columns = dfresults_C2_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_GEHP = dfresults_C2_GEHP.mul(SFGEHP, axis=0)

### 17.2.8 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_GEHP_exchange_configs = [
        (EF_EOL_metal.key, GEHP_EOL_copper, 'technosphere'),
        (EF_EOL_Aluminium.key, GEHP_EOL_aluminium, 'technosphere'),
        (EF_EOL_metal.key, GEHP_EOL_lowalloyedsteel, 'technosphere'),
        (EF_EOL_metal.key, GEHP_EOL_stainlessSteel, 'technosphere'),
        (EF_EOL_metal.key, GEHP_EOL_ferrite, 'technosphere'),
        (EF_EOL_metal.key, GEHP_EOL_brass, 'technosphere'),
        (EF_EOL_Aluminium.key, GEHP_EOL_alumina, 'technosphere'),
        (EF_EOL_electronics.key, GEHP_EOL_electronics, 'technosphere'),
        (EF_EOL_refrigerantgas.key, GEHP_EOL_refrigerantgas, 'technosphere'),
    ]
    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in C3_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_GEHP = C3_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        C3_GEHP_acts[bd_name].new_exchange(
            input=C3_GEHP_acts[bd_name].key, 
            output=C3_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_GEHP.lci()
    mlca_C3_GEHP.lcia()
    mlca_C3_GEHP.scores

    dfresults_C3_GEHP = pd.DataFrame.from_dict(mlca_C3_GEHP.scores, orient='index')
    dfresults_C3_GEHP.index = pd.MultiIndex.from_tuples(dfresults_C3_GEHP.index, names=['Column', 'Row'])
    dfresults_C3_GEHP = dfresults_C3_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_GEHP.columns = dfresults_C3_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_GEHP = dfresults_C3_GEHP.mul(SFGEHP, axis=0)

### 17.2.9 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_GEHP_exchange_configs = [
        (EF_EOL_landfill_metal.key, GEHP_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_landfill_aluminium.key, GEHP_EOL_landfill_aluminium, 'technosphere'),
        (EF_EOL_landfill_metal.key,            GEHP_EOL_landfill_lowalloyedsteel, 'technosphere'),
        (EF_EOL_landfill_metal.key,            GEHP_EOL_landfill_stainlessSteel, 'technosphere'),
        (EF_EOL_landfill_metal.key,            GEHP_EOL_landfill_ferrite,         'technosphere'),
        (EF_EOL_landfill_metal.key,            GEHP_EOL_landfill_brass,           'technosphere'),
        (EF_EOL_landfill_aluminium.key,        GEHP_EOL_landfill_alumina,         'technosphere'),
        (EF_EOL_inc_electronic.key,            GEHP_EOL_inc_electronics,          'technosphere'),
        (EF_EOL_landfill_electronic.key,       GEHP_EOL_landfill_electronics,     'technosphere'),
        (EF_EOL_inc_lubricatingoil.key,        GEHP_EOL_inc_lubricatingoil,       'technosphere'),
        (EF_EOL_inc_hazardouswaste.key,        GEHP_EOL_inc_hazardouswaste,       'technosphere'),
        (EF_EOL_landfill_refrigerantgas.key,   GEHP_EOL_landfill_refrigerantgas,  'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in C4_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_GEHP = C4_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        C4_GEHP_acts[bd_name].new_exchange(
            input=C4_GEHP_acts[bd_name].key, 
            output=C4_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_GEHP.lci()
    mlca_C4_GEHP.lcia()
    mlca_C4_GEHP.scores

    dfresults_C4_GEHP = pd.DataFrame.from_dict(mlca_C4_GEHP.scores, orient='index')
    dfresults_C4_GEHP.index = pd.MultiIndex.from_tuples(dfresults_C4_GEHP.index, names=['Column', 'Row'])
    dfresults_C4_GEHP = dfresults_C4_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_GEHP.columns = dfresults_C4_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_GEHP = dfresults_C4_GEHP.mul(SFGEHP, axis=0)

### 17.2.10 Module B4

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_GEHP = pd.DataFrame(index=dfresults_A13_GEHP.index)  # Start with the index from A13 for consistency   
    dfresults_B4_GEHP = (
        dfresults_A13_GEHP.add(dfresults_A4_GEHP, fill_value=0)
        .add(dfresults_C3_GEHP, fill_value=0)
        .add(dfresults_C4_GEHP, fill_value=0)
        ).mul(GEHPNumberReplacements, axis=0)

### 17.2.11 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_GEHP_exchange_configs = [
        (EF_recycling_copper.key, GEHP_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, GEHP_EOL_primary_copper, 'technosphere'),
        (EF_recycling_aluminium.key, GEHP_EOL_recycling_aluminium, 'technosphere'),
        (EF_primary_aluminium.key, GEHP_EOL_primary_aluminium, 'technosphere'),
        (EF_recycling_steel.key, GEHP_EOL_recycling_lowalloyedsteel, 'technosphere'),
        (EF_primary_steel.key, GEHP_EOL_primary_lowalloyedsteel, 'technosphere'),
        (EF_recycling_steel.key, GEHP_EOL_recycling_stainlessSteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, GEHP_EOL_primary_stainlessSteel, 'technosphere'),
    ]    

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in D1_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_GEHP = D1_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        D1_GEHP_acts[bd_name].new_exchange(
            input=D1_GEHP_acts[bd_name].key, 
            output=D1_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_GEHP.lci()
    mlca_D1_GEHP.lcia()
    mlca_D1_GEHP.scores

    dfresults_D1_GEHP = pd.DataFrame.from_dict(mlca_D1_GEHP.scores, orient='index')
    dfresults_D1_GEHP.index = pd.MultiIndex.from_tuples(dfresults_D1_GEHP.index, names=['Column', 'Row'])
    dfresults_D1_GEHP = dfresults_D1_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_GEHP.columns = dfresults_D1_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_GEHP = dfresults_D1_GEHP.mul(SFGEHP, axis=0)

### 17.2.12 Module D1 (B4 included)

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_GEHP = dfresults_D1_GEHP.add(dfresults_D1_GEHP.mul(GEHPNumberReplacements, axis=0))

### 17.2.13 Module D3 (related to the export of energy as result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_GEHP_exchange_configs = [
        (EF_heat_energyrecovery.key, GEHP_EOL_energyrecoveryheat_electronics, 'technosphere'),
        (EF_electricity_energyrecovery.key, GEHP_EOL_energyrecoveryele_electronics, 'technosphere'),
        (EF_heat_energyrecovery.key, GEHP_EOL_energyrecoveryheat_lubricatingoil, 'technosphere'),
        (EF_electricity_energyrecovery.key, GEHP_EOL_energyrecoveryele_lubricatingoil, 'technosphere'),
        (EF_heat_energyrecovery.key, GEHP_EOL_energyrecoveryheat_hazardouswaste, 'technosphere'),
        (EF_electricity_energyrecovery.key, GEHP_EOL_energyrecoveryele_hazardouswaste, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(GasEngineHeatPump_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_GEHP_exchange_list = []
        
        for input_key, data_array, exc_type in D3_GEHP_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_GEHP_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_GEHP_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_GEHP = D3_GEHP_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_GEHP.save()

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in GasEngineHeatPump_BdName:
        D3_GEHP_acts[bd_name].new_exchange(
            input=D3_GEHP_acts[bd_name].key, 
            output=D3_GEHP_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_GEHP = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_GEHP['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_GEHP_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(GasEngineHeatPump_BdName, GasEngineHeatPump_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_GEHP = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_GEHP.lci()
    mlca_D3_GEHP.lcia()
    mlca_D3_GEHP.scores

    dfresults_D3_GEHP = pd.DataFrame.from_dict(mlca_D3_GEHP.scores, orient='index')
    dfresults_D3_GEHP.index = pd.MultiIndex.from_tuples(dfresults_D3_GEHP.index, names=['Column', 'Row'])
    dfresults_D3_GEHP = dfresults_D3_GEHP.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_GEHP.columns = dfresults_D3_GEHP.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_GEHP.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_GEHP = dfresults_D3_GEHP.mul(SFGEHP, axis=0)

### 17.2.14 Module D3 (B4 included)

In [ ]:
sheet_name = "Gas Engine Heat Pump"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_GEHP = dfresults_D3_GEHP.add(dfresults_D3_GEHP.mul(GEHPNumberReplacements, axis=0))

# 18. Concrete storages

## 18.1 Activities

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_ThermalConcreteStorage_{bd_name}',
        'name': f'A13_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            A13_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_ThermalConcreteStorage_{bd_name}',
        'name': f'A4_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            A4_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_ThermalConcreteStorage_{bd_name}',
        'name': f'A5_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            A5_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_ThermalConcreteStorage_{bd_name}',
        'name': f'B2_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            B2_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_ThermalConcreteStorage_{bd_name}',
        'name': f'B6_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            B6_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_ThermalConcreteStorage_{bd_name}',
        'name': f'C2_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            C2_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_ThermalConcreteStorage_{bd_name}',
        'name': f'C3_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            C3_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_ThermalConcreteStorage_{bd_name}',
        'name': f'C4_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            C4_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_ThermalConcreteStorage_{bd_name}',
        'name': f'D1_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            D1_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Concrete thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_ThermalConcreteStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in ConcreteStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_ThermalConcreteStorage_{bd_name}',
        'name': f'D3_ThermalConcreteStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_ThermalConcreteStorage_acts[bd_name] = scenario_db.new_activity(**data)
            D3_ThermalConcreteStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 18.2 Exchanges

### 18.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_ThermalConcreteStorage_exchange_configs = [
        (EF_steel.key, np.atleast_1d(ThermalConcreteStorage_steellowalloyed), 'technosphere'),
        (EF_steel.key, np.atleast_1d(ThermalConcreteStorage_steelunalloyed), 'technosphere'),
        (EF_concreteCH.key, np.atleast_1d(ThermalConcreteStorage_concrete), 'technosphere'),
        (EF_mineralwool.key, np.atleast_1d(ThermalConcreteStorage_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ConcreteStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_ThermalConcreteStorage_exchange_list = []
        
        for input_key, data_array, exc_type in A13_ThermalConcreteStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_ThermalConcreteStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_ThermalConcreteStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_ThermalConcreteStorage = A13_ThermalConcreteStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_ThermalConcreteStorage.save()

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        A13_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=A13_ThermalConcreteStorage_acts[bd_name].key, 
            output=A13_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_ThermalConcreteStorage.lci()
    mlca_A13_ThermalConcreteStorage.lcia()
    mlca_A13_ThermalConcreteStorage.scores

    dfresults_A13_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_A13_ThermalConcreteStorage.scores, orient='index')
    dfresults_A13_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_A13_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_A13_ThermalConcreteStorage = dfresults_A13_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_ThermalConcreteStorage.columns = dfresults_A13_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_ThermalConcreteStorage = dfresults_A13_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

### 18.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_ThermalConcreteStorage_exchange_configs = [
        (EF_Truk_16_32.key, ThermalConcreteStorage_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, ThermalConcreteStorage_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ConcreteStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_ThermalConcreteStorage_exchange_list = []
        
        for input_key, data_array, exc_type in A4_ThermalConcreteStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_ThermalConcreteStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_ThermalConcreteStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_ThermalConcreteStorage = A4_ThermalConcreteStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_ThermalConcreteStorage.save()

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        A4_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=A4_ThermalConcreteStorage_acts[bd_name].key, 
            output=A4_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_ThermalConcreteStorage.lci()
    mlca_A4_ThermalConcreteStorage.lcia()
    mlca_A4_ThermalConcreteStorage.scores

    dfresults_A4_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_A4_ThermalConcreteStorage.scores, orient='index')
    dfresults_A4_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_A4_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_A4_ThermalConcreteStorage = dfresults_A4_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_ThermalConcreteStorage.columns = dfresults_A4_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_ThermalConcreteStorage = dfresults_A4_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

### 18.2.3 Module A5

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        A5_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=A5_ThermalConcreteStorage_acts[bd_name].key, 
            output=A5_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_ThermalConcreteStorage.lci()
    mlca_A5_ThermalConcreteStorage.lcia()
    mlca_A5_ThermalConcreteStorage.scores

    dfresults_A5_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_A5_ThermalConcreteStorage.scores, orient='index')
    dfresults_A5_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_A5_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_A5_ThermalConcreteStorage = dfresults_A5_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_ThermalConcreteStorage.columns = dfresults_A5_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_ThermalConcreteStorage = dfresults_A5_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

### 18.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_ThermalConcreteStorage_exchange_configs = [
        (EF_LightTruk.key, ThermalConcreteStorage_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ConcreteStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_ThermalConcreteStorage_exchange_list = []
        
        for input_key, data_array, exc_type in B2_ThermalConcreteStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_ThermalConcreteStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_ThermalConcreteStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_ThermalConcreteStorage = B2_ThermalConcreteStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_ThermalConcreteStorage.save()

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        B2_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=B2_ThermalConcreteStorage_acts[bd_name].key, 
            output=B2_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_ThermalConcreteStorage.lci()
    mlca_B2_ThermalConcreteStorage.lcia()
    mlca_B2_ThermalConcreteStorage.scores

    dfresults_B2_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_B2_ThermalConcreteStorage.scores, orient='index')
    dfresults_B2_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_B2_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_B2_ThermalConcreteStorage = dfresults_B2_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_ThermalConcreteStorage.columns = dfresults_B2_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_ThermalConcreteStorage = dfresults_B2_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

### 18.2.5 Module B6

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        B6_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=B6_ThermalConcreteStorage_acts[bd_name].key, 
            output=B6_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_ThermalConcreteStorage.lci()
    mlca_B6_ThermalConcreteStorage.lcia()
    mlca_B6_ThermalConcreteStorage.scores

    dfresults_B6_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_B6_ThermalConcreteStorage.scores, orient='index')
    dfresults_B6_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_B6_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_B6_ThermalConcreteStorage = dfresults_B6_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_ThermalConcreteStorage.columns = dfresults_B6_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

### 18.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_ThermalConcreteStorage_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, ThermalConcreteStorage_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ConcreteStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_ThermalConcreteStorage_exchange_list = []
        
        for input_key, data_array, exc_type in C2_ThermalConcreteStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_ThermalConcreteStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_ThermalConcreteStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_ThermalConcreteStorage = C2_ThermalConcreteStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_ThermalConcreteStorage.save()

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        C2_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=C2_ThermalConcreteStorage_acts[bd_name].key, 
            output=C2_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_ThermalConcreteStorage.lci()
    mlca_C2_ThermalConcreteStorage.lcia()
    mlca_C2_ThermalConcreteStorage.scores

    dfresults_C2_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_C2_ThermalConcreteStorage.scores, orient='index')
    dfresults_C2_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_C2_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_C2_ThermalConcreteStorage = dfresults_C2_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_ThermalConcreteStorage.columns = dfresults_C2_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_ThermalConcreteStorage = dfresults_C2_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

### 18.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_ThermalConcreteStorage_exchange_configs = [
        (EF_EOL_metal.key, np.atleast_1d(ThermalConcreteStorage_EOL_steellowalloyed), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(ThermalConcreteStorage_EOL_steelunalloyed), 'technosphere'),
        (EF_EOL_concrete.key, np.atleast_1d(ThermalConcreteStorage_EOL_concrete), 'technosphere'),
        (EF_EOL_stonewool.key, np.atleast_1d(ThermalConcreteStorage_EOL_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ConcreteStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_ThermalConcreteStorage_exchange_list = []
        
        for input_key, data_array, exc_type in C3_ThermalConcreteStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_ThermalConcreteStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_ThermalConcreteStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_ThermalConcreteStorage = C3_ThermalConcreteStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_ThermalConcreteStorage.save()

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        C3_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=C3_ThermalConcreteStorage_acts[bd_name].key, 
            output=C3_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_ThermalConcreteStorage.lci()
    mlca_C3_ThermalConcreteStorage.lcia()
    mlca_C3_ThermalConcreteStorage.scores

    dfresults_C3_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_C3_ThermalConcreteStorage.scores, orient='index')
    dfresults_C3_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_C3_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_C3_ThermalConcreteStorage = dfresults_C3_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_ThermalConcreteStorage.columns = dfresults_C3_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_ThermalConcreteStorage = dfresults_C3_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

### 18.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_ThermalConcreteStorage_exchange_configs = [
        (EF_EOL_landfill_metal.key, np.atleast_1d(ThermalConcreteStorage_EOL_landfill_steellowalloyed), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(ThermalConcreteStorage_EOL_landfill_Steelunalloyed), 'technosphere'),
        (EF_EOL_landfill_concrete.key, np.atleast_1d(ThermalConcreteStorage_EOL_landfill_concrete), 'technosphere'),
        (EF_EOL_landfill_stonewool.key, np.atleast_1d(ThermalConcreteStorage_EOL_landfill_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ConcreteStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_ThermalConcreteStorage_exchange_list = []
        
        for input_key, data_array, exc_type in C4_ThermalConcreteStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_ThermalConcreteStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_ThermalConcreteStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_ThermalConcreteStorage = C4_ThermalConcreteStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_ThermalConcreteStorage.save()

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        C4_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=C4_ThermalConcreteStorage_acts[bd_name].key, 
            output=C4_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_ThermalConcreteStorage.lci()
    mlca_C4_ThermalConcreteStorage.lcia()
    mlca_C4_ThermalConcreteStorage.scores

    dfresults_C4_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_C4_ThermalConcreteStorage.scores, orient='index')
    dfresults_C4_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_C4_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_C4_ThermalConcreteStorage = dfresults_C4_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_ThermalConcreteStorage.columns = dfresults_C4_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_ThermalConcreteStorage = dfresults_C4_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

### 18.2.9 Module B4

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_ThermalConcreteStorage = pd.DataFrame(index=dfresults_A13_ThermalConcreteStorage.index)  # Start with the index from A13 for consistency   
    dfresults_B4_ThermalConcreteStorage = (
        dfresults_A13_ThermalConcreteStorage.add(dfresults_A4_ThermalConcreteStorage, fill_value=0)
        .add(dfresults_C3_ThermalConcreteStorage, fill_value=0)
        .add(dfresults_C4_ThermalConcreteStorage, fill_value=0)
        ).mul(ThermalConcreteStorageNumberReplacements, axis=0)

### 18.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_ThermalConcreteStorage_exchange_configs = [
        (EF_recycling_steel.key, np.atleast_1d(ThermalConcreteStorage_EOL_recycling_steellowalloyed), 'technosphere'),
        (EF_primary_steel.key, np.atleast_1d(ThermalConcreteStorage_EOL_primary_steellowalloyed), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(ThermalConcreteStorage_EOL_recycling_Steelunalloyed), 'technosphere'),
        (EF_primary_steel.key, np.atleast_1d(ThermalConcreteStorage_EOL_primary_Steelunalloyed), 'technosphere'),
        (EF_recycling_concrete.key, np.atleast_1d(ThermalConcreteStorage_EOL_recycling_concrete), 'technosphere'),
        (EF_primary_concrete.key, np.atleast_1d(ThermalConcreteStorage_EOL_primary_concrete), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(ConcreteStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_ThermalConcreteStorage_exchange_list = []
        
        for input_key, data_array, exc_type in D1_ThermalConcreteStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_ThermalConcreteStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_ThermalConcreteStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_ThermalConcreteStorage = D1_ThermalConcreteStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_ThermalConcreteStorage.save()

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        D1_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=D1_ThermalConcreteStorage_acts[bd_name].key, 
            output=D1_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_ThermalConcreteStorage.lci()
    mlca_D1_ThermalConcreteStorage.lcia()
    mlca_D1_ThermalConcreteStorage.scores

    dfresults_D1_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_D1_ThermalConcreteStorage.scores, orient='index')
    dfresults_D1_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_D1_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_D1_ThermalConcreteStorage = dfresults_D1_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_ThermalConcreteStorage.columns = dfresults_D1_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_ThermalConcreteStorage = dfresults_D1_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

### 18.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_ThermalConcreteStorage = dfresults_D1_ThermalConcreteStorage.add(dfresults_D1_ThermalConcreteStorage.mul(ThermalConcreteStorageNumberReplacements, axis=0))

### 18.2.12 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in ConcreteStorage_BdName:
        D3_ThermalConcreteStorage_acts[bd_name].new_exchange(
            input=D3_ThermalConcreteStorage_acts[bd_name].key, 
            output=D3_ThermalConcreteStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Concrete thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_ThermalConcreteStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_ThermalConcreteStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_ThermalConcreteStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(ConcreteStorage_BdName, ConcreteStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_ThermalConcreteStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_ThermalConcreteStorage.lci()
    mlca_D3_ThermalConcreteStorage.lcia()
    mlca_D3_ThermalConcreteStorage.scores

    dfresults_D3_ThermalConcreteStorage = pd.DataFrame.from_dict(mlca_D3_ThermalConcreteStorage.scores, orient='index')
    dfresults_D3_ThermalConcreteStorage.index = pd.MultiIndex.from_tuples(dfresults_D3_ThermalConcreteStorage.index, names=['Column', 'Row'])
    dfresults_D3_ThermalConcreteStorage = dfresults_D3_ThermalConcreteStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_ThermalConcreteStorage.columns = dfresults_D3_ThermalConcreteStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_ThermalConcreteStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_ThermalConcreteStorage = dfresults_D3_ThermalConcreteStorage.mul(SFThermalConcreteStorage, axis=0)

# 19. Steel thermal storages

## 19.1 Activities

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A13_ThermalSteelStorage_{bd_name}',
        'name': f'A13_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            A13_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A4_ThermalSteelStorage_{bd_name}',
        'name': f'A4_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            A4_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'A5_ThermalSteelStorage_{bd_name}',
        'name': f'A5_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            A5_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B2_ThermalSteelStorage_{bd_name}',
        'name': f'B2_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            B2_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'B6_ThermalSteelStorage_{bd_name}',
        'name': f'B6_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            B6_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C2_ThermalSteelStorage_{bd_name}',
        'name': f'C2_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            C2_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C3_ThermalSteelStorage_{bd_name}',
        'name': f'C3_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            C3_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'C4_ThermalSteelStorage_{bd_name}',
        'name': f'C4_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            C4_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D1_ThermalSteelStorage_{bd_name}',
        'name': f'D1_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            D1_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Steel thermal storage"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_ThermalSteelStorage_acts = {}  # Dictionary to store activities by building name
    for bd_name in SteelStorage_BdName:
        # Create a unique activity for each building's boiler
        data = {
        'code': f'D3_ThermalSteelStorage_{bd_name}',
        'name': f'D3_ThermalSteelStorage_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_ThermalSteelStorage_acts[bd_name] = scenario_db.new_activity(**data)
            D3_ThermalSteelStorage_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 19.2 Exchanges

### 19.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_ThermalSteelStorage_exchange_configs = [
        (EF_alkydpaint.key, np.atleast_1d(ThermalSteelStorage_alkydpaint), 'technosphere'),
        (EF_Aluminium.key, np.atleast_1d(ThermalSteelStorage_aluminium), 'technosphere'),
        (EF_steel.key, np.atleast_1d(ThermalSteelStorage_reinforcingsteel), 'technosphere'),
        (EF_sheetrollingaluminium.key, np.atleast_1d(ThermalSteelStorage_sheetrollingAluminium), 'technosphere'),
        (EF_sheetrollingsteel.key, np.atleast_1d(ThermalSteelStorage_sheetrollingSteel), 'technosphere'),
        (EF_StainlessSteel.key, np.atleast_1d(ThermalSteelStorage_chromiumsteel), 'technosphere'),
        (EF_mineralwool.key, np.atleast_1d(ThermalSteelStorage_stonewool), 'technosphere'),
        (EF_weldingarcaluminium.key, np.atleast_1d(ThermalSteelStorage_weldingarcaluminium), 'technosphere'),
        (EF_weldingarcsteel.key, np.atleast_1d(ThermalSteelStorage_weldingarcsteel), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SteelStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_ThermalSteelStorage_exchange_list = []
        
        for input_key, data_array, exc_type in A13_ThermalSteelStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_ThermalSteelStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_ThermalSteelStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_ThermalSteelStorage = A13_ThermalSteelStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_ThermalSteelStorage.save()

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        A13_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=A13_ThermalSteelStorage_acts[bd_name].key, 
            output=A13_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_ThermalSteelStorage.lci()
    mlca_A13_ThermalSteelStorage.lcia()
    mlca_A13_ThermalSteelStorage.scores

    dfresults_A13_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_A13_ThermalSteelStorage.scores, orient='index')
    dfresults_A13_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_A13_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_A13_ThermalSteelStorage = dfresults_A13_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_ThermalSteelStorage.columns = dfresults_A13_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_ThermalSteelStorage = dfresults_A13_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

### 19.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_ThermalSteelStorage_exchange_configs = [
        (EF_Truk_16_32.key, ThermalSteelStorage_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, ThermalSteelStorage_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SteelStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_ThermalSteelStorage_exchange_list = []
        
        for input_key, data_array, exc_type in A4_ThermalSteelStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_ThermalSteelStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_ThermalSteelStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_ThermalSteelStorage = A4_ThermalSteelStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_ThermalSteelStorage.save()

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        A4_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=A4_ThermalSteelStorage_acts[bd_name].key, 
            output=A4_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_ThermalSteelStorage.lci()
    mlca_A4_ThermalSteelStorage.lcia()
    mlca_A4_ThermalSteelStorage.scores

    dfresults_A4_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_A4_ThermalSteelStorage.scores, orient='index')
    dfresults_A4_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_A4_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_A4_ThermalSteelStorage = dfresults_A4_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_ThermalSteelStorage.columns = dfresults_A4_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_ThermalSteelStorage = dfresults_A4_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

### 19.2.3 Module A5

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        A5_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=A5_ThermalSteelStorage_acts[bd_name].key, 
            output=A5_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_ThermalSteelStorage.lci()
    mlca_A5_ThermalSteelStorage.lcia()
    mlca_A5_ThermalSteelStorage.scores

    dfresults_A5_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_A5_ThermalSteelStorage.scores, orient='index')
    dfresults_A5_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_A5_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_A5_ThermalSteelStorage = dfresults_A5_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_ThermalSteelStorage.columns = dfresults_A5_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_ThermalSteelStorage = dfresults_A5_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

### 19.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_ThermalSteelStorage_exchange_configs = [
        (EF_LightTruk.key, ThermalSteelStorage_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SteelStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_ThermalSteelStorage_exchange_list = []
        
        for input_key, data_array, exc_type in B2_ThermalSteelStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_ThermalSteelStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_ThermalSteelStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_ThermalSteelStorage = B2_ThermalSteelStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_ThermalSteelStorage.save()

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        B2_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=B2_ThermalSteelStorage_acts[bd_name].key, 
            output=B2_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_ThermalSteelStorage.lci()
    mlca_B2_ThermalSteelStorage.lcia()
    mlca_B2_ThermalSteelStorage.scores

    dfresults_B2_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_B2_ThermalSteelStorage.scores, orient='index')
    dfresults_B2_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_B2_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_B2_ThermalSteelStorage = dfresults_B2_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_ThermalSteelStorage.columns = dfresults_B2_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_ThermalSteelStorage = dfresults_B2_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

### 19.2.5 Module B6

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        B6_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=B6_ThermalSteelStorage_acts[bd_name].key, 
            output=B6_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_ThermalSteelStorage.lci()
    mlca_B6_ThermalSteelStorage.lcia()
    mlca_B6_ThermalSteelStorage.scores

    dfresults_B6_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_B6_ThermalSteelStorage.scores, orient='index')
    dfresults_B6_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_B6_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_B6_ThermalSteelStorage = dfresults_B6_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_ThermalSteelStorage.columns = dfresults_B6_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

### 19.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_ThermalSteelStorage_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, ThermalSteelStorage_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SteelStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_ThermalSteelStorage_exchange_list = []
        
        for input_key, data_array, exc_type in C2_ThermalSteelStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_ThermalSteelStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_ThermalSteelStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_ThermalSteelStorage = C2_ThermalSteelStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_ThermalSteelStorage.save()

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        C2_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=C2_ThermalSteelStorage_acts[bd_name].key, 
            output=C2_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_ThermalSteelStorage.lci()
    mlca_C2_ThermalSteelStorage.lcia()
    mlca_C2_ThermalSteelStorage.scores

    dfresults_C2_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_C2_ThermalSteelStorage.scores, orient='index')
    dfresults_C2_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_C2_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_C2_ThermalSteelStorage = dfresults_C2_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_ThermalSteelStorage.columns = dfresults_C2_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_ThermalSteelStorage = dfresults_C2_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

### 19.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_ThermalSteelStorage_exchange_configs = [
        (EF_EOL_alkydpaint.key, np.atleast_1d(ThermalSteelStorage_EOL_alkydpaint), 'technosphere'),
        (EF_EOL_Aluminium.key, np.atleast_1d(ThermalSteelStorage_EOL_aluminium), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(ThermalSteelStorage_EOL_reinforcingsteel), 'technosphere'),
        (EF_EOL_Aluminium.key, np.atleast_1d(ThermalSteelStorage_EOL_sheetrollingAluminium), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(ThermalSteelStorage_EOL_sheetrollingSteel), 'technosphere'),
        (EF_EOL_metal.key, np.atleast_1d(ThermalSteelStorage_EOL_chromiumsteel), 'technosphere'),
        (EF_EOL_stonewool.key, np.atleast_1d(ThermalSteelStorage_EOL_stonewool), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SteelStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_ThermalSteelStorage_exchange_list = []
        
        for input_key, data_array, exc_type in C3_ThermalSteelStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_ThermalSteelStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_ThermalSteelStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_ThermalSteelStorage = C3_ThermalSteelStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_ThermalSteelStorage.save()

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        C3_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=C3_ThermalSteelStorage_acts[bd_name].key, 
            output=C3_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_ThermalSteelStorage.lci()
    mlca_C3_ThermalSteelStorage.lcia()
    mlca_C3_ThermalSteelStorage.scores

    dfresults_C3_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_C3_ThermalSteelStorage.scores, orient='index')
    dfresults_C3_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_C3_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_C3_ThermalSteelStorage = dfresults_C3_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_ThermalSteelStorage.columns = dfresults_C3_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_ThermalSteelStorage = dfresults_C3_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

### 19.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_ThermalSteelStorage_exchange_configs = [
        (EF_EOL_landfill_aluminium.key, np.atleast_1d(ThermalSteelStorage_EOL_landfill_aluminium), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(ThermalSteelStorage_EOL_landfill_reinforcingsteel), 'technosphere'),
        (EF_EOL_landfill_aluminium.key, np.atleast_1d(ThermalSteelStorage_EOL_landfill_sheetrollingaluminium), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(ThermalSteelStorage_EOL_landfill_sheetrollingsteel), 'technosphere'),
        (EF_EOL_landfill_metal.key, np.atleast_1d(ThermalSteelStorage_EOL_landfill_chromiumsteel), 'technosphere'),
        (EF_EOL_landfill_stonewool.key, np.atleast_1d(ThermalSteelStorage_EOL_landfill_stonewool), 'technosphere'),
        (EF_EOL_inc_alkydpaint.key, np.atleast_1d(ThermalSteelStorage_EOL_inc_alkydpaint), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SteelStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_ThermalSteelStorage_exchange_list = []
        
        for input_key, data_array, exc_type in C4_ThermalSteelStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_ThermalSteelStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_ThermalSteelStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_ThermalSteelStorage = C4_ThermalSteelStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_ThermalSteelStorage.save()

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        C4_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=C4_ThermalSteelStorage_acts[bd_name].key, 
            output=C4_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_ThermalSteelStorage.lci()
    mlca_C4_ThermalSteelStorage.lcia()
    mlca_C4_ThermalSteelStorage.scores

    dfresults_C4_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_C4_ThermalSteelStorage.scores, orient='index')
    dfresults_C4_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_C4_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_C4_ThermalSteelStorage = dfresults_C4_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_ThermalSteelStorage.columns = dfresults_C4_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_ThermalSteelStorage = dfresults_C4_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

### 19.2.9 Module B4

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_ThermalSteelStorage = pd.DataFrame(index=dfresults_A13_ThermalSteelStorage.index)  # Start with the index from A13 for consistency   
    dfresults_B4_ThermalSteelStorage = (
        dfresults_A13_ThermalSteelStorage.add(dfresults_A4_ThermalSteelStorage, fill_value=0)
        .add(dfresults_C3_ThermalSteelStorage, fill_value=0)
        .add(dfresults_C4_ThermalSteelStorage, fill_value=0)
        ).mul(ThermalSteelStorageNumberReplacements, axis=0)

### 19.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_ThermalSteelStorage_exchange_configs = [
        (EF_recycling_aluminium.key, np.atleast_1d(ThermalSteelStorage_EOL_recycling_aluminium), 'technosphere'),
        (EF_primary_aluminium.key, np.atleast_1d(ThermalSteelStorage_EOL_primary_aluminium), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(ThermalSteelStorage_EOL_recycling_reinforcingsteel), 'technosphere'),
        (EF_primary_steel.key, np.atleast_1d(ThermalSteelStorage_EOL_primary_reinforcingsteel), 'technosphere'),
        (EF_recycling_aluminium.key, np.atleast_1d(ThermalSteelStorage_EOL_recycling_sheetrollingaluminium), 'technosphere'),
        (EF_primary_aluminium.key, np.atleast_1d(ThermalSteelStorage_EOL_primary_sheetrollingaluminium), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(ThermalSteelStorage_EOL_recycling_sheetrollingsteel), 'technosphere'),
        (EF_primary_steel.key, np.atleast_1d(ThermalSteelStorage_EOL_primary_sheetrollingsteel), 'technosphere'),
        (EF_recycling_steel.key, np.atleast_1d(ThermalSteelStorage_EOL_recycling_chromiumsteel), 'technosphere'),
        (EF_primary_stainlessSteel.key, np.atleast_1d(ThermalSteelStorage_EOL_primary_chromiumsteel), 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(SteelStorage_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_ThermalSteelStorage_exchange_list = []
        
        for input_key, data_array, exc_type in D1_ThermalSteelStorage_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_ThermalSteelStorage_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_ThermalSteelStorage_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_ThermalSteelStorage = D1_ThermalSteelStorage_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_ThermalSteelStorage.save()

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        D1_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=D1_ThermalSteelStorage_acts[bd_name].key, 
            output=D1_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_ThermalSteelStorage.lci()
    mlca_D1_ThermalSteelStorage.lcia()
    mlca_D1_ThermalSteelStorage.scores

    dfresults_D1_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_D1_ThermalSteelStorage.scores, orient='index')
    dfresults_D1_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_D1_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_D1_ThermalSteelStorage = dfresults_D1_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_ThermalSteelStorage.columns = dfresults_D1_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_ThermalSteelStorage = dfresults_D1_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

### 19.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_ThermalSteelStorage = dfresults_D1_ThermalSteelStorage.add(dfresults_D1_ThermalSteelStorage.mul(ThermalSteelStorageNumberReplacements, axis=0))

### 19.2.12 Module D3 (related to the export of energy as a results of waste incineration)

In [ ]:
sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in SteelStorage_BdName:
        D3_ThermalSteelStorage_acts[bd_name].new_exchange(
            input=D3_ThermalSteelStorage_acts[bd_name].key, 
            output=D3_ThermalSteelStorage_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Steel thermal storage"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_ThermalSteelStorage = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_ThermalSteelStorage['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_ThermalSteelStorage_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(SteelStorage_BdName, SteelStorage_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_ThermalSteelStorage = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_ThermalSteelStorage.lci()
    mlca_D3_ThermalSteelStorage.lcia()
    mlca_D3_ThermalSteelStorage.scores

    dfresults_D3_ThermalSteelStorage = pd.DataFrame.from_dict(mlca_D3_ThermalSteelStorage.scores, orient='index')
    dfresults_D3_ThermalSteelStorage.index = pd.MultiIndex.from_tuples(dfresults_D3_ThermalSteelStorage.index, names=['Column', 'Row'])
    dfresults_D3_ThermalSteelStorage = dfresults_D3_ThermalSteelStorage.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_ThermalSteelStorage.columns = dfresults_D3_ThermalSteelStorage.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_ThermalSteelStorage.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_ThermalSteelStorage = dfresults_D3_ThermalSteelStorage.mul(SFThermalSteelStorage, axis=0)

# 20. Geothermal

## 20.1 Activities

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_DEEPWELL_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A13_DEEPWELL_{bd_name}',
        'name': f'A13_DEEPWELL_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_DEEPWELL_acts[bd_name] = scenario_db.new_activity(**data)
            A13_DEEPWELL_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_GEOHRX_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A13_GEOHRX_{bd_name}',
        'name': f'A13_GEOHRX_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_GEOHRX_acts[bd_name] = scenario_db.new_activity(**data)
            A13_GEOHRX_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_DEEPWELL_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A4_DEEPWELL_{bd_name}',
        'name': f'A4_DEEPWELL_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_DEEPWELL_acts[bd_name] = scenario_db.new_activity(**data)
            A4_DEEPWELL_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_GEOHRX_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A4_GEOHRX_{bd_name}',
        'name': f'A4_GEOHRX_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_GEOHRX_acts[bd_name] = scenario_db.new_activity(**data)
            A4_GEOHRX_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_DEEPWELL_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A5_DEEPWELL_{bd_name}',
        'name': f'A5_DEEPWELL_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_DEEPWELL_acts[bd_name] = scenario_db.new_activity(**data)
            A5_DEEPWELL_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_GEOHRX_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A5_GEOHRX_{bd_name}',
        'name': f'A5_GEOHRX_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_GEOHRX_acts[bd_name] = scenario_db.new_activity(**data)
            A5_GEOHRX_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_GEO_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B2_GEO_{bd_name}',
        'name': f'B2_GEO_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_GEO_acts[bd_name] = scenario_db.new_activity(**data)
            B2_GEO_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_GEO_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B6_GEO_{bd_name}',
        'name': f'B6_GEO_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_GEO_acts[bd_name] = scenario_db.new_activity(**data)
            B6_GEO_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_DEEPWELL_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C2_DEEPWELL_{bd_name}',
        'name': f'C2_DEEPWELL_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_DEEPWELL_acts[bd_name] = scenario_db.new_activity(**data)
            C2_DEEPWELL_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_GEOHRX_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C2_GEOHRX_{bd_name}',
        'name': f'C2_GEOHRX_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_GEOHRX_acts[bd_name] = scenario_db.new_activity(**data)
            C2_GEOHRX_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_DEEPWELL_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C3_DEEPWELL_{bd_name}',
        'name': f'C3_DEEPWELL_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_DEEPWELL_acts[bd_name] = scenario_db.new_activity(**data)
            C3_DEEPWELL_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_GEOHRX_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C3_GEOHRX_{bd_name}',
        'name': f'C3_GEOHRX_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_GEOHRX_acts[bd_name] = scenario_db.new_activity(**data)
            C3_GEOHRX_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_DEEPWELL_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C4_DEEPWELL_{bd_name}',
        'name': f'C4_DEEPWELL_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_DEEPWELL_acts[bd_name] = scenario_db.new_activity(**data)
            C4_DEEPWELL_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_GEOHRX_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C4_GEOHRX_{bd_name}',
        'name': f'C4_GEOHRX_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_GEOHRX_acts[bd_name] = scenario_db.new_activity(**data)
            C4_GEOHRX_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_DEEPWELL_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D1_DEEPWELL_{bd_name}',
        'name': f'D1_DEEPWELL_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_DEEPWELL_acts[bd_name] = scenario_db.new_activity(**data)
            D1_DEEPWELL_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_GEOHRX_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D1_GEOHRX_{bd_name}',
        'name': f'D1_GEOHRX_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_GEOHRX_acts[bd_name] = scenario_db.new_activity(**data)
            D1_GEOHRX_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_DEEPWELL_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D3_DEEPWELL_{bd_name}',
        'name': f'D3_DEEPWELL_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_DEEPWELL_acts[bd_name] = scenario_db.new_activity(**data)
            D3_DEEPWELL_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Geothermal plant"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_GEOHRX_acts = {}  # Dictionary to store activities by building name
    for bd_name in Geothermal_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D3_GEOHRX_{bd_name}',
        'name': f'D3_GEOHRX_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_GEOHRX_acts[bd_name] = scenario_db.new_activity(**data)
            D3_GEOHRX_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 20.2 Exchanges

### 20.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_DEEPWELL_exchange_configs = [
        (EF_steel.key, Geothermal_lowalloyedsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_DEEPWELL_exchange_list = []
        
        for input_key, data_array, exc_type in A13_DEEPWELL_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_DEEPWELL_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_DEEPWELL_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_DEEPWELL = A13_DEEPWELL_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_DEEPWELL.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        A13_DEEPWELL_acts[bd_name].new_exchange(
            input=A13_DEEPWELL_acts[bd_name].key, 
            output=A13_DEEPWELL_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_DEEPWELL = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_DEEPWELL['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_DEEPWELL_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_DEEPWELL = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_DEEPWELL.lci()
    mlca_A13_DEEPWELL.lcia()
    mlca_A13_DEEPWELL.scores

    dfresults_A13_DEEPWELL = pd.DataFrame.from_dict(mlca_A13_DEEPWELL.scores, orient='index')
    dfresults_A13_DEEPWELL.index = pd.MultiIndex.from_tuples(dfresults_A13_DEEPWELL.index, names=['Column', 'Row'])
    dfresults_A13_DEEPWELL = dfresults_A13_DEEPWELL.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_DEEPWELL.columns = dfresults_A13_DEEPWELL.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_DEEPWELL.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_DEEPWELL = dfresults_A13_DEEPWELL.mul(SFwell, axis=0)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_GEOHRX_exchange_configs = [
        (EF_StainlessSteel.key, Geothermal_stainlessSteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_GEOHRX_exchange_list = []
        
        for input_key, data_array, exc_type in A13_GEOHRX_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_GEOHRX_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_GEOHRX_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_GEOHRX = A13_GEOHRX_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_GEOHRX.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        A13_GEOHRX_acts[bd_name].new_exchange(
            input=A13_GEOHRX_acts[bd_name].key, 
            output=A13_GEOHRX_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_GEOHRX = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_GEOHRX['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_GEOHRX_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_GEOHRX = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_GEOHRX.lci()
    mlca_A13_GEOHRX.lcia()
    mlca_A13_GEOHRX.scores

    dfresults_A13_GEOHRX = pd.DataFrame.from_dict(mlca_A13_GEOHRX.scores, orient='index')
    dfresults_A13_GEOHRX.index = pd.MultiIndex.from_tuples(dfresults_A13_GEOHRX.index, names=['Column', 'Row'])
    dfresults_A13_GEOHRX = dfresults_A13_GEOHRX.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_GEOHRX.columns = dfresults_A13_GEOHRX.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_GEOHRX.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_GEOHRX = dfresults_A13_GEOHRX.mul(SFHRX, axis=0)

In [ ]:
dfresults_A13_GEO = dfresults_A13_DEEPWELL.add(dfresults_A13_GEOHRX)

### 20.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_DEEPWELL_exchange_configs = [
        (EF_Truk_16_32.key, GeothermalDeepwell_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, GeothermalDeepwell_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_DEEPWELL_exchange_list = []
        
        for input_key, data_array, exc_type in A4_DEEPWELL_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_DEEPWELL_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_DEEPWELL_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_DEEPWELL = A4_DEEPWELL_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_DEEPWELL.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        A4_DEEPWELL_acts[bd_name].new_exchange(
            input=A4_DEEPWELL_acts[bd_name].key, 
            output=A4_DEEPWELL_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_DEEPWELL = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_DEEPWELL['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_DEEPWELL_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_DEEPWELL = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_DEEPWELL.lci()
    mlca_A4_DEEPWELL.lcia()
    mlca_A4_DEEPWELL.scores

    dfresults_A4_DEEPWELL = pd.DataFrame.from_dict(mlca_A4_DEEPWELL.scores, orient='index')
    dfresults_A4_DEEPWELL.index = pd.MultiIndex.from_tuples(dfresults_A4_DEEPWELL.index, names=['Column', 'Row'])
    dfresults_A4_DEEPWELL = dfresults_A4_DEEPWELL.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_DEEPWELL.columns = dfresults_A4_DEEPWELL.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_DEEPWELL.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_DEEPWELL = dfresults_A4_DEEPWELL.mul(SFwell, axis=0)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_GEOHRX_exchange_configs = [
        (EF_Truk_16_32.key, GeothermalHRX_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, GeothermalHRX_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_GEOHRX_exchange_list = []
        
        for input_key, data_array, exc_type in A4_GEOHRX_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_GEOHRX_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_GEOHRX_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_GEOHRX = A4_GEOHRX_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_GEOHRX.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        A4_GEOHRX_acts[bd_name].new_exchange(
            input=A4_GEOHRX_acts[bd_name].key, 
            output=A4_GEOHRX_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_GEOHRX = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_GEOHRX['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_GEOHRX_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_GEOHRX = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_GEOHRX.lci()
    mlca_A4_GEOHRX.lcia()
    mlca_A4_GEOHRX.scores

    dfresults_A4_GEOHRX = pd.DataFrame.from_dict(mlca_A4_GEOHRX.scores, orient='index')
    dfresults_A4_GEOHRX.index = pd.MultiIndex.from_tuples(dfresults_A4_GEOHRX.index, names=['Column', 'Row'])
    dfresults_A4_GEOHRX = dfresults_A4_GEOHRX.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_GEOHRX.columns = dfresults_A4_GEOHRX.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_GEOHRX.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_GEOHRX = dfresults_A4_GEOHRX.mul(SFHRX, axis=0)

In [ ]:
dfresults_A4_GEO = dfresults_A4_DEEPWELL.add(dfresults_A4_GEOHRX)

### 20.2.3 Module A5

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A5_DEEPWELL_exchange_configs = [
        (EF_deepwell.key, Geothermal_deepwell, 'technosphere'),
        (EF_diesel.key, Geothermal_dieselA5, 'technosphere'),
        (EF_Light_Truk.key, GeothermalDeepwell_Light_TrukA5, 'technosphere'),
        (EF_Truk_16_32.key, GeothermalDeepwell_Truk_16_32A5, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        A5_DEEPWELL_exchange_list = []
        
        for input_key, data_array, exc_type in A5_DEEPWELL_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A5_DEEPWELL_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A5_DEEPWELL_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A5_DEEPWELL = A5_DEEPWELL_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A5_DEEPWELL.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        A5_DEEPWELL_acts[bd_name].new_exchange(
            input=A5_DEEPWELL_acts[bd_name].key, 
            output=A5_DEEPWELL_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_DEEPWELL = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_DEEPWELL['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_DEEPWELL_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_DEEPWELL = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_DEEPWELL.lci()
    mlca_A5_DEEPWELL.lcia()
    mlca_A5_DEEPWELL.scores

    dfresults_A5_DEEPWELL = pd.DataFrame.from_dict(mlca_A5_DEEPWELL.scores, orient='index')
    dfresults_A5_DEEPWELL.index = pd.MultiIndex.from_tuples(dfresults_A5_DEEPWELL.index, names=['Column', 'Row'])
    dfresults_A5_DEEPWELL = dfresults_A5_DEEPWELL.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_DEEPWELL.columns = dfresults_A5_DEEPWELL.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_DEEPWELL.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_DEEPWELL = dfresults_A5_DEEPWELL.mul(SFwell, axis=0)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A5_GEOHRX_exchange_configs = [
        (EF_Light_Truk.key, GeothermalHRX_Light_TrukA5, 'technosphere'),
        (EF_Truk_16_32.key, GeothermalHRX_Truk_16_32A5, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        A5_GEOHRX_exchange_list = []
        
        for input_key, data_array, exc_type in A5_GEOHRX_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A5_GEOHRX_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A5_GEOHRX_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A5_GEOHRX = A5_GEOHRX_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A5_GEOHRX.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        A5_GEOHRX_acts[bd_name].new_exchange(
            input=A5_GEOHRX_acts[bd_name].key, 
            output=A5_GEOHRX_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_GEOHRX = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_GEOHRX['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_GEOHRX_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_GEOHRX = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_GEOHRX.lci()
    mlca_A5_GEOHRX.lcia()
    mlca_A5_GEOHRX.scores

    dfresults_A5_GEOHRX = pd.DataFrame.from_dict(mlca_A5_GEOHRX.scores, orient='index')
    dfresults_A5_GEOHRX.index = pd.MultiIndex.from_tuples(dfresults_A5_GEOHRX.index, names=['Column', 'Row'])
    dfresults_A5_GEOHRX = dfresults_A5_GEOHRX.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_GEOHRX.columns = dfresults_A5_GEOHRX.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_GEOHRX.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_GEOHRX = dfresults_A5_GEOHRX.mul(SFHRX, axis=0)

In [ ]:
dfresults_A5_GEO = dfresults_A5_DEEPWELL.add(dfresults_A5_GEOHRX)

### 20.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_GEO_exchange_configs = [
        (EF_Light_Truk.key, Geothermal_Light_Truk_B2_HRX, 'technosphere'),
        (EF_well.key, Geothermal_Light_Truk_B2_well, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_GEO_exchange_list = []
        
        for input_key, data_array, exc_type in B2_GEO_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_GEO_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_GEO_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_GEO = B2_GEO_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_GEO.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        B2_GEO_acts[bd_name].new_exchange(
            input=B2_GEO_acts[bd_name].key, 
            output=B2_GEO_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_GEO = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_GEO['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_GEO_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_GEO = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_GEO.lci()
    mlca_B2_GEO.lcia()
    mlca_B2_GEO.scores

    dfresults_B2_GEO = pd.DataFrame.from_dict(mlca_B2_GEO.scores, orient='index')
    dfresults_B2_GEO.index = pd.MultiIndex.from_tuples(dfresults_B2_GEO.index, names=['Column', 'Row'])
    dfresults_B2_GEO = dfresults_B2_GEO.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_GEO.columns = dfresults_B2_GEO.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_GEO.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_GEO = dfresults_B2_GEO.mul(SFwell, axis=0)

### 20.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_GEO_exchange_configs = [
        (EF_Stimulation.key, Geothermal_usephasestimulation, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_GEO_exchange_list = []
        
        for input_key, data_array, exc_type in B6_GEO_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_GEO_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_GEO_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_GEO = B6_GEO_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_GEO.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        B6_GEO_acts[bd_name].new_exchange(
            input=B6_GEO_acts[bd_name].key, 
            output=B6_GEO_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_GEO = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_GEO['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_GEO_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_GEO = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_GEO.lci()
    mlca_B6_GEO.lcia()
    mlca_B6_GEO.scores

    dfresults_B6_GEO = pd.DataFrame.from_dict(mlca_B6_GEO.scores, orient='index')
    dfresults_B6_GEO.index = pd.MultiIndex.from_tuples(dfresults_B6_GEO.index, names=['Column', 'Row'])
    dfresults_B6_GEO = dfresults_B6_GEO.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_GEO.columns = dfresults_B6_GEO.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_GEO.rename(columns=method_lookup, inplace=True)

### 20.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_DEEPWELL_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, GeothermalDeepwell_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_DEEPWELL_exchange_list = []
        
        for input_key, data_array, exc_type in C2_DEEPWELL_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_DEEPWELL_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in C2_DEEPWELL_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_DEEPWELL = C2_DEEPWELL_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_DEEPWELL.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        C2_DEEPWELL_acts[bd_name].new_exchange(
            input=C2_DEEPWELL_acts[bd_name].key, 
            output=C2_DEEPWELL_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_DEEPWELL = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_DEEPWELL['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_DEEPWELL_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_DEEPWELL = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_DEEPWELL.lci()
    mlca_C2_DEEPWELL.lcia()
    mlca_C2_DEEPWELL.scores

    dfresults_C2_DEEPWELL = pd.DataFrame.from_dict(mlca_C2_DEEPWELL.scores, orient='index')
    dfresults_C2_DEEPWELL.index = pd.MultiIndex.from_tuples(dfresults_C2_DEEPWELL.index, names=['Column', 'Row'])
    dfresults_C2_DEEPWELL = dfresults_C2_DEEPWELL.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_DEEPWELL.columns = dfresults_C2_DEEPWELL.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_DEEPWELL.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_DEEPWELL = dfresults_C2_DEEPWELL.mul(SFwell, axis=0)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_GEOHRX_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, GeothermalHRX_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_GEOHRX_exchange_list = []
        
        for input_key, data_array, exc_type in C2_GEOHRX_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_GEOHRX_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_GEOHRX_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_GEOHRX = C2_GEOHRX_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_GEOHRX.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        C2_GEOHRX_acts[bd_name].new_exchange(
            input=C2_GEOHRX_acts[bd_name].key, 
            output=C2_GEOHRX_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_GEOHRX = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_GEOHRX['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_GEOHRX_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_GEOHRX = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_GEOHRX.lci()
    mlca_C2_GEOHRX.lcia()
    mlca_C2_GEOHRX.scores

    dfresults_C2_GEOHRX = pd.DataFrame.from_dict(mlca_C2_GEOHRX.scores, orient='index')
    dfresults_C2_GEOHRX.index = pd.MultiIndex.from_tuples(dfresults_C2_GEOHRX.index, names=['Column', 'Row'])
    dfresults_C2_GEOHRX = dfresults_C2_GEOHRX.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_GEOHRX.columns = dfresults_C2_GEOHRX.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_GEOHRX.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_GEOHRX = dfresults_C2_GEOHRX.mul(SFHRX, axis=0)

In [ ]:
dfresults_C2_GEO = dfresults_C2_DEEPWELL.add(dfresults_C2_GEOHRX)

### 20.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_DEEPWELL_exchange_configs = [
        (EF_EOL_metal.key, Geothermal_EOL_lowalloyedsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_DEEPWELL_exchange_list = []
        
        for input_key, data_array, exc_type in C3_DEEPWELL_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_DEEPWELL_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_DEEPWELL_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_DEEPWELL = C3_DEEPWELL_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_DEEPWELL.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        C3_DEEPWELL_acts[bd_name].new_exchange(
            input=C3_DEEPWELL_acts[bd_name].key, 
            output=C3_DEEPWELL_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_DEEPWELL = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_DEEPWELL['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_DEEPWELL_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_DEEPWELL = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_DEEPWELL.lci()
    mlca_C3_DEEPWELL.lcia()
    mlca_C3_DEEPWELL.scores

    dfresults_C3_DEEPWELL = pd.DataFrame.from_dict(mlca_C3_DEEPWELL.scores, orient='index')
    dfresults_C3_DEEPWELL.index = pd.MultiIndex.from_tuples(dfresults_C3_DEEPWELL.index, names=['Column', 'Row'])
    dfresults_C3_DEEPWELL = dfresults_C3_DEEPWELL.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_DEEPWELL.columns = dfresults_C3_DEEPWELL.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_DEEPWELL.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_DEEPWELL = dfresults_C3_DEEPWELL.mul(SFwell, axis=0)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_GEOHRX_exchange_configs = [
        (EF_EOL_metal.key, Geothermal_EOL_lowalloyedsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_GEOHRX_exchange_list = []
        
        for input_key, data_array, exc_type in C3_GEOHRX_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_GEOHRX_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_GEOHRX_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_GEOHRX = C3_GEOHRX_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_GEOHRX.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        C3_GEOHRX_acts[bd_name].new_exchange(
            input=C3_GEOHRX_acts[bd_name].key, 
            output=C3_GEOHRX_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_GEOHRX = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_GEOHRX['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_GEOHRX_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_GEOHRX = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_GEOHRX.lci()
    mlca_C3_GEOHRX.lcia()
    mlca_C3_GEOHRX.scores

    dfresults_C3_GEOHRX = pd.DataFrame.from_dict(mlca_C3_GEOHRX.scores, orient='index')
    dfresults_C3_GEOHRX.index = pd.MultiIndex.from_tuples(dfresults_C3_GEOHRX.index, names=['Column', 'Row'])
    dfresults_C3_GEOHRX = dfresults_C3_GEOHRX.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_GEOHRX.columns = dfresults_C3_GEOHRX.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_GEOHRX.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_GEOHRX = dfresults_C3_GEOHRX.mul(SFHRX, axis=0)

In [ ]:
dfresults_C3_GEO = dfresults_C3_DEEPWELL.add(dfresults_C3_GEOHRX)

### 20.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_DEEPWELL_exchange_configs = [
        (EF_EOL_landfill_metal.key, Geothermal_EOL_landfill_lowalloyedsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_DEEPWELL_exchange_list = []
        
        for input_key, data_array, exc_type in C4_DEEPWELL_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_DEEPWELL_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_DEEPWELL_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_DEEPWELL = C4_DEEPWELL_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_DEEPWELL.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        C4_DEEPWELL_acts[bd_name].new_exchange(
            input=C4_DEEPWELL_acts[bd_name].key, 
            output=C4_DEEPWELL_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_DEEPWELL = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_DEEPWELL['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_DEEPWELL_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_DEEPWELL = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_DEEPWELL.lci()
    mlca_C4_DEEPWELL.lcia()
    mlca_C4_DEEPWELL.scores

    dfresults_C4_DEEPWELL = pd.DataFrame.from_dict(mlca_C4_DEEPWELL.scores, orient='index')
    dfresults_C4_DEEPWELL.index = pd.MultiIndex.from_tuples(dfresults_C4_DEEPWELL.index, names=['Column', 'Row'])
    dfresults_C4_DEEPWELL = dfresults_C4_DEEPWELL.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_DEEPWELL.columns = dfresults_C4_DEEPWELL.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_DEEPWELL.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_DEEPWELL = dfresults_C4_DEEPWELL.mul(SFwell, axis=0)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_GEOHRX_exchange_configs = [
        (EF_EOL_landfill_metal.key, Geothermal_EOL_landfill_stainlessSteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_GEOHRX_exchange_list = []
        
        for input_key, data_array, exc_type in C4_GEOHRX_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_GEOHRX_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_GEOHRX_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_GEOHRX = C4_GEOHRX_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_GEOHRX.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        C4_GEOHRX_acts[bd_name].new_exchange(
            input=C4_GEOHRX_acts[bd_name].key, 
            output=C4_GEOHRX_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_GEOHRX = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_GEOHRX['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_GEOHRX_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_GEOHRX = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_GEOHRX.lci()
    mlca_C4_GEOHRX.lcia()
    mlca_C4_GEOHRX.scores

    dfresults_C4_GEOHRX = pd.DataFrame.from_dict(mlca_C4_GEOHRX.scores, orient='index')
    dfresults_C4_GEOHRX.index = pd.MultiIndex.from_tuples(dfresults_C4_GEOHRX.index, names=['Column', 'Row'])
    dfresults_C4_GEOHRX = dfresults_C4_GEOHRX.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_GEOHRX.columns = dfresults_C4_GEOHRX.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_GEOHRX.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_GEOHRX = dfresults_C4_GEOHRX.mul(SFHRX, axis=0)

In [ ]:
dfresults_C4_GEO = dfresults_C4_DEEPWELL.add(dfresults_C4_GEOHRX)

### 20.2.9 Module B4

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_GEO = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_GEO['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_DEEPWELL = pd.DataFrame(index=dfresults_A13_DEEPWELL.index)  # Start with the index from A13 for consistency   
    dfresults_B4_DEEPWELL = (
        dfresults_A13_DEEPWELL.add(dfresults_A4_DEEPWELL, fill_value=0)
        .add(dfresults_C3_DEEPWELL, fill_value=0)
        .add(dfresults_C4_DEEPWELL, fill_value=0)
    ).mul(DeepwellNumberReplacements, axis=0)

    dfresults_B4_GEOHRX = pd.DataFrame(index=dfresults_A13_GEOHRX.index)  # Start with the index from A13 for consistency   
    dfresults_B4_GEOHRX = (
        dfresults_A13_GEOHRX.add(dfresults_A4_GEOHRX, fill_value=0)
        .add(dfresults_C3_GEOHRX, fill_value=0)
        .add(dfresults_C4_GEOHRX, fill_value=0)
    ).mul(HRXNumberReplacements, axis=0)

    dfresults_B4_GEO = dfresults_B4_DEEPWELL.add(dfresults_B4_GEOHRX)

### 20.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_DEEPWELL_exchange_configs = [
        (EF_recycling_steel.key, Geothermal_EOL_recycling_lowalloyedsteel, 'technosphere'),
        (EF_primary_steel.key, Geothermal_EOL_primary_lowalloyedsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_DEEPWELL_exchange_list = []
        
        for input_key, data_array, exc_type in D1_DEEPWELL_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_DEEPWELL_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_DEEPWELL_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_DEEPWELL = D1_DEEPWELL_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_DEEPWELL.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        D1_DEEPWELL_acts[bd_name].new_exchange(
            input=D1_DEEPWELL_acts[bd_name].key, 
            output=D1_DEEPWELL_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_DEEPWELL = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_DEEPWELL['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_DEEPWELL_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_DEEPWELL = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_DEEPWELL.lci()
    mlca_D1_DEEPWELL.lcia()
    mlca_D1_DEEPWELL.scores

    dfresults_D1_DEEPWELL = pd.DataFrame.from_dict(mlca_D1_DEEPWELL.scores, orient='index')
    dfresults_D1_DEEPWELL.index = pd.MultiIndex.from_tuples(dfresults_D1_DEEPWELL.index, names=['Column', 'Row'])
    dfresults_D1_DEEPWELL = dfresults_D1_DEEPWELL.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_DEEPWELL.columns = dfresults_D1_DEEPWELL.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_DEEPWELL.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_DEEPWELL = dfresults_D1_DEEPWELL.mul(SFwell, axis=0)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_GEOHRX_exchange_configs = [
        (EF_recycling_steel.key, Geothermal_EOL_recycling_stainlessSteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, Geothermal_EOL_primary_stainlessSteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Geothermal_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_GEOHRX_exchange_list = []
        
        for input_key, data_array, exc_type in D1_GEOHRX_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_GEOHRX_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_GEOHRX_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_GEOHRX = D1_GEOHRX_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_GEOHRX.save()

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        D1_GEOHRX_acts[bd_name].new_exchange(
            input=D1_GEOHRX_acts[bd_name].key, 
            output=D1_GEOHRX_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_GEOHRX = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_GEOHRX['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_GEOHRX_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_GEOHRX = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_GEOHRX.lci()
    mlca_D1_GEOHRX.lcia()
    mlca_D1_GEOHRX.scores

    dfresults_D1_GEOHRX = pd.DataFrame.from_dict(mlca_D1_GEOHRX.scores, orient='index')
    dfresults_D1_GEOHRX.index = pd.MultiIndex.from_tuples(dfresults_D1_GEOHRX.index, names=['Column', 'Row'])
    dfresults_D1_GEOHRX = dfresults_D1_GEOHRX.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_GEOHRX.columns = dfresults_D1_GEOHRX.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_GEOHRX.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_GEOHRX = dfresults_D1_GEOHRX.mul(SFHRX, axis=0)

### 20.2.10 Module D1 (B4 included)

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    dfresults_D1_GEO = dfresults_D1_DEEPWELL.add(dfresults_D1_GEOHRX)
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_DEEPWELL = dfresults_D1_DEEPWELL.add(dfresults_D1_DEEPWELL.mul(DeepwellNumberReplacements, axis=0))
    dfresults_D1_GEOHRX = dfresults_D1_GEOHRX.add(dfresults_D1_GEOHRX.mul(HRXNumberReplacements, axis=0))
    dfresults_D1_GEO = dfresults_D1_DEEPWELL.add(dfresults_D1_GEOHRX)

### 20.2.11 Module D3 (related to the export of energy as a results of waste incineration)

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        D3_DEEPWELL_acts[bd_name].new_exchange(
            input=D3_DEEPWELL_acts[bd_name].key, 
            output=D3_DEEPWELL_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_DEEPWELL = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_DEEPWELL['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_DEEPWELL_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_DEEPWELL = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_DEEPWELL.lci()
    mlca_D3_DEEPWELL.lcia()
    mlca_D3_DEEPWELL.scores

    dfresults_D3_DEEPWELL = pd.DataFrame.from_dict(mlca_D3_DEEPWELL.scores, orient='index')
    dfresults_D3_DEEPWELL.index = pd.MultiIndex.from_tuples(dfresults_D3_DEEPWELL.index, names=['Column', 'Row'])
    dfresults_D3_DEEPWELL = dfresults_D3_DEEPWELL.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_DEEPWELL.columns = dfresults_D3_DEEPWELL.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_DEEPWELL.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_DEEPWELL = dfresults_D3_DEEPWELL.mul(SFwell, axis=0)

In [ ]:
sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Geothermal_BdName:
        D3_GEOHRX_acts[bd_name].new_exchange(
            input=D3_GEOHRX_acts[bd_name].key, 
            output=D3_GEOHRX_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Geothermal plant"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_GEOHRX = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_GEOHRX['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_GEOHRX_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Geothermal_BdName, Geothermal_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_GEOHRX = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_GEOHRX.lci()
    mlca_D3_GEOHRX.lcia()
    mlca_D3_GEOHRX.scores

    dfresults_D3_GEOHRX = pd.DataFrame.from_dict(mlca_D3_GEOHRX.scores, orient='index')
    dfresults_D3_GEOHRX.index = pd.MultiIndex.from_tuples(dfresults_D3_GEOHRX.index, names=['Column', 'Row'])
    dfresults_D3_GEOHRX = dfresults_D3_GEOHRX.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_GEOHRX.columns = dfresults_D3_GEOHRX.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_GEOHRX.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_GEOHRX = dfresults_D3_GEOHRX.mul(SFHRX, axis=0)

In [ ]:
dfresults_D3_GEO = dfresults_D3_DEEPWELL.add(dfresults_D3_GEOHRX)

# 21. Cooling - Absorption chiller

## 21.1 Activities

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A13_AbsChiller_{bd_name}',
        'name': f'A13_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            A13_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A4_AbsChiller_{bd_name}',
        'name': f'A4_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            A4_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A5_AbsChiller_{bd_name}',
        'name': f'A5_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            A5_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B2_AbsChiller_{bd_name}',
        'name': f'B2_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            B2_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B6_AbsChiller_{bd_name}',
        'name': f'B6_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            B6_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C2_AbsChiller_{bd_name}',
        'name': f'C2_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            C2_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C3_AbsChiller_{bd_name}',
        'name': f'C3_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            C3_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C4_AbsChiller_{bd_name}',
        'name': f'C4_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            C4_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D1_AbsChiller_{bd_name}',
        'name': f'D1_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            D1_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Absorption chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_AbsChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in AbsorptionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D3_AbsChiller_{bd_name}',
        'name': f'D3_AbsChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_AbsChiller_acts[bd_name] = scenario_db.new_activity(**data)
            D3_AbsChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 21.2 Exchanges

### 21.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_AbsChiller_exchange_configs = [
        (EF_Aluminium.key, AbsChiller_aluminium, 'technosphere'),
        (EF_AmmoniaA13.key, AbsChiller_ammonia, 'technosphere'),
        (EF_Copper.key, AbsChiller_copper, 'technosphere'),
        (EF_ElectricityLow.key, AbsChiller_electricitylowvoltage, 'technosphere'),
        (EF_Electricity.key, AbsChiller_electricitymediumvoltage, 'technosphere'),
        (EF_Electronic.key, AbsChiller_electronics, 'technosphere'),
        (EF_glycole.key, AbsChiller_glycolethylene, 'technosphere'),
        (EF_HeatOther.key, AbsChiller_heatOther, 'technosphere'),
        (EF_Heat.key, AbsChiller_heatfromGas, 'technosphere'),
        (EF_polyethylene.key, AbsChiller_PEHD, 'technosphere'),
        (EF_steel.key, AbsChiller_reinforcingsteel, 'technosphere'),
        (EF_sheetrollingsteel.key, AbsChiller_sheetrollingsteel, 'technosphere'),
        (EF_sheetrollingaluminium.key, AbsChiller_sheetrollingaluminium, 'technosphere'),
        (EF_sheetrollingsteel.key, AbsChiller_sheetrollingStainlessSteel, 'technosphere'),
        (EF_mineralwool.key, AbsChiller_stonelwool, 'technosphere'),
        (EF_Elastomer.key, AbsChiller_elastomer, 'technosphere'),
        (EF_wiredrawingcopper.key, AbsChiller_wiredrawingcopper, 'technosphere'),
        (EF_waterconsumption.key, AbsChiller_water, 'technosphere'),
        (EF_waterwater.key, AbsChiller_waterwater, 'biosphere'),
        (EF_waterair.key, AbsChiller_waterunspec, 'biosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in A13_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_AbsChiller = A13_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        A13_AbsChiller_acts[bd_name].new_exchange(
            input=A13_AbsChiller_acts[bd_name].key, 
            output=A13_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_AbsChiller.lci()
    mlca_A13_AbsChiller.lcia()
    mlca_A13_AbsChiller.scores

    dfresults_A13_AbsChiller = pd.DataFrame.from_dict(mlca_A13_AbsChiller.scores, orient='index')
    dfresults_A13_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_A13_AbsChiller.index, names=['Column', 'Row'])
    dfresults_A13_AbsChiller = dfresults_A13_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_AbsChiller.columns = dfresults_A13_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_AbsChiller = dfresults_A13_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_AbsChiller_exchange_configs = [
        (EF_Truk_16_32.key, AbsChiller_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, AbsChiller_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in A4_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_AbsChiller = A4_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        A4_AbsChiller_acts[bd_name].new_exchange(
            input=A4_AbsChiller_acts[bd_name].key, 
            output=A4_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_AbsChiller.lci()
    mlca_A4_AbsChiller.lcia()
    mlca_A4_AbsChiller.scores

    dfresults_A4_AbsChiller = pd.DataFrame.from_dict(mlca_A4_AbsChiller.scores, orient='index')
    dfresults_A4_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_A4_AbsChiller.index, names=['Column', 'Row'])
    dfresults_A4_AbsChiller = dfresults_A4_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_AbsChiller.columns = dfresults_A4_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_AbsChiller = dfresults_A4_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.3 Module A5

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        A5_AbsChiller_acts[bd_name].new_exchange(
            input=A5_AbsChiller_acts[bd_name].key, 
            output=A5_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_AbsChiller.lci()
    mlca_A5_AbsChiller.lcia()
    mlca_A5_AbsChiller.scores

    dfresults_A5_AbsChiller = pd.DataFrame.from_dict(mlca_A5_AbsChiller.scores, orient='index')
    dfresults_A5_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_A5_AbsChiller.index, names=['Column', 'Row'])
    dfresults_A5_AbsChiller = dfresults_A5_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_AbsChiller.columns = dfresults_A5_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_AbsChiller = dfresults_A5_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_AbsChiller_exchange_configs = [
        (EF_InjectionMoulding.key, AbsChiller_injectionMoulding, 'technosphere'),
        (EF_LightTruk.key, AbsChiller_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in B2_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_AbsChiller = B2_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        B2_AbsChiller_acts[bd_name].new_exchange(
            input=B2_AbsChiller_acts[bd_name].key, 
            output=B2_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_AbsChiller.lci()
    mlca_B2_AbsChiller.lcia()
    mlca_B2_AbsChiller.scores

    dfresults_B2_AbsChiller = pd.DataFrame.from_dict(mlca_B2_AbsChiller.scores, orient='index')
    dfresults_B2_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_B2_AbsChiller.index, names=['Column', 'Row'])
    dfresults_B2_AbsChiller = dfresults_B2_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_AbsChiller.columns = dfresults_B2_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_AbsChiller = dfresults_B2_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_AbsChiller_exchange_configs = [
        (EF_electricity.key, AbsChiller_usephaseelectricity, 'technosphere'),
        (EF_electricityPV.key, AbsChiller_usephaseelectricityPV, 'technosphere'),
        # (EF_sun.key, AbsChiller_usephaseelectricityRECSPV, 'technosphere'),
        # (EF_hydro.key, AbsChiller_usephaseelectricityRECSHydro, 'technosphere'),
        # (EF_wind.key, AbsChiller_usephaseelectricityRECSEolic, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in B6_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_AbsChiller = B6_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        B6_AbsChiller_acts[bd_name].new_exchange(
            input=B6_AbsChiller_acts[bd_name].key, 
            output=B6_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_AbsChiller.lci()
    mlca_B6_AbsChiller.lcia()
    mlca_B6_AbsChiller.scores

    dfresults_B6_AbsChiller = pd.DataFrame.from_dict(mlca_B6_AbsChiller.scores, orient='index')
    dfresults_B6_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_B6_AbsChiller.index, names=['Column', 'Row'])
    dfresults_B6_AbsChiller = dfresults_B6_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_AbsChiller.columns = dfresults_B6_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_AbsChiller.rename(columns=method_lookup, inplace=True)

### 21.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_AbsChiller_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, AbsChiller_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in C2_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_AbsChiller = C2_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        C2_AbsChiller_acts[bd_name].new_exchange(
            input=C2_AbsChiller_acts[bd_name].key, 
            output=C2_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_AbsChiller.lci()
    mlca_C2_AbsChiller.lcia()
    mlca_C2_AbsChiller.scores

    dfresults_C2_AbsChiller = pd.DataFrame.from_dict(mlca_C2_AbsChiller.scores, orient='index')
    dfresults_C2_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_C2_AbsChiller.index, names=['Column', 'Row'])
    dfresults_C2_AbsChiller = dfresults_C2_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_AbsChiller.columns = dfresults_C2_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_AbsChiller = dfresults_C2_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_AbsChiller_exchange_configs = [
        (EF_EOL_Aluminium.key, AbsChiller_EOL_aluminium, 'technosphere'),
        (EF_EOL_metal.key, AbsChiller_EOL_copper, 'technosphere'),
        (EF_EOL_electronics.key, AbsChiller_EOL_electronics, 'technosphere'),
        (EF_EOL_plastic.key, AbsChiller_EOL_PEHD, 'technosphere'),
        (EF_EOL_metal.key, AbsChiller_EOL_reinforcingsteel, 'technosphere'),
        (EF_EOL_stonewool.key, AbsChiller_EOL_stonelwool, 'technosphere'),
        (EF_EOL_Elastomer.key, AbsChiller_EOL_elastomer, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in C3_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_AbsChiller = C3_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        C3_AbsChiller_acts[bd_name].new_exchange(
            input=C3_AbsChiller_acts[bd_name].key, 
            output=C3_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_AbsChiller.lci()
    mlca_C3_AbsChiller.lcia()
    mlca_C3_AbsChiller.scores

    dfresults_C3_AbsChiller = pd.DataFrame.from_dict(mlca_C3_AbsChiller.scores, orient='index')
    dfresults_C3_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_C3_AbsChiller.index, names=['Column', 'Row'])
    dfresults_C3_AbsChiller = dfresults_C3_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_AbsChiller.columns = dfresults_C3_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_AbsChiller = dfresults_C3_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_AbsChiller_exchange_configs = [
        (EF_EOL_landfill_aluminium.key, AbsChiller_EOL_landfill_aluminium, 'technosphere'),
        (EF_EOL_inc_hazardouswaste.key, AbsChiller_EOL_inc_ammonia, 'technosphere'),
        (EF_EOL_landfill_metal.key, AbsChiller_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_inc_electronic.key, AbsChiller_EOL_inc_electronics, 'technosphere'),
        (EF_EOL_landfill_electronic.key, AbsChiller_EOL_landfill_electronics, 'technosphere'),
        (EF_EOL_inc_PEHD.key, AbsChiller_EOL_inc_PEHD, 'technosphere'),
        (EF_EOL_landfill_PEHD.key, AbsChiller_EOL_landfill_PEHD, 'technosphere'),
        (EF_EOL_landfill_metal.key, AbsChiller_EOL_landfill_reinforcingsteel, 'technosphere'),
        (EF_EOL_landfill_stonewool.key, AbsChiller_EOL_landfill_stonelwool, 'technosphere'),
        (EF_EOL_inc_elastomer.key, AbsChiller_EOL_inc_elastomer, 'technosphere'),
        (EF_EOL_landfill_plastic.key, AbsChiller_EOL_landfill_elastomer, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in C4_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_AbsChiller = C4_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        C4_AbsChiller_acts[bd_name].new_exchange(
            input=C4_AbsChiller_acts[bd_name].key, 
            output=C4_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_AbsChiller.lci()
    mlca_C4_AbsChiller.lcia()
    mlca_C4_AbsChiller.scores

    dfresults_C4_AbsChiller = pd.DataFrame.from_dict(mlca_C4_AbsChiller.scores, orient='index')
    dfresults_C4_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_C4_AbsChiller.index, names=['Column', 'Row'])
    dfresults_C4_AbsChiller = dfresults_C4_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_AbsChiller.columns = dfresults_C4_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_AbsChiller = dfresults_C4_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.9 Module B4

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_AbsChiller = pd.DataFrame(index=dfresults_A13_AbsChiller.index)  # Start with the index from A13 for consistency   
    dfresults_B4_AbsChiller = (
        dfresults_A13_AbsChiller.add(dfresults_A4_AbsChiller, fill_value=0)
        .add(dfresults_C3_AbsChiller, fill_value=0)
        .add(dfresults_C4_AbsChiller, fill_value=0)
    ).mul(AbsChillerNumberReplacements, axis=0)

### 21.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_AbsChiller_exchange_configs = [
        (EF_recycling_PEHD.key, AbsChiller_EOL_recycling_PEHD, 'technosphere'),
        (EF_primary_PEHD.key, AbsChiller_EOL_primary_PEHD, 'technosphere'),
        (EF_recycling_steel.key, AbsChiller_EOL_recycling_steel, 'technosphere'),
        (EF_primary_steel.key, AbsChiller_EOL_primary_steel, 'technosphere'),
        (EF_recycling_aluminium.key, AbsChiller_EOL_recycling_aluminium, 'technosphere'),
        (EF_primary_aluminium.key, AbsChiller_EOL_primary_aluminium, 'technosphere'),
        (EF_recycling_copper.key, AbsChiller_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, AbsChiller_EOL_primary_copper, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in D1_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_AbsChiller = D1_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        D1_AbsChiller_acts[bd_name].new_exchange(
            input=D1_AbsChiller_acts[bd_name].key, 
            output=D1_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_AbsChiller.lci()
    mlca_D1_AbsChiller.lcia()
    mlca_D1_AbsChiller.scores

    dfresults_D1_AbsChiller = pd.DataFrame.from_dict(mlca_D1_AbsChiller.scores, orient='index')
    dfresults_D1_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_D1_AbsChiller.index, names=['Column', 'Row'])
    dfresults_D1_AbsChiller = dfresults_D1_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_AbsChiller.columns = dfresults_D1_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_AbsChiller = dfresults_D1_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_AbsChiller = dfresults_D1_AbsChiller.add(dfresults_D1_AbsChiller.mul(AbsChillerNumberReplacements, axis=0))

### 21.2.12 Module D3 (related to the export of energy as result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_AbsChiller_exchange_configs = [
        (EF_heat_energyrecovery.key, AbsChiller_EOL_energyrecoveryheat_ammonia, 'technosphere'),
        (EF_electricity_energyrecovery.key, AbsChiller_EOL_energyrecoveryele_ammonia, 'technosphere'),
        (EF_heat_energyrecovery.key, AbsChiller_EOL_energyrecoveryheat_electronics, 'technosphere'),
        (EF_electricity_energyrecovery.key, AbsChiller_EOL_energyrecoveryele_electronics, 'technosphere'),
        (EF_heat_energyrecovery.key, AbsChiller_EOL_energyrecoveryheat_PEHD, 'technosphere'),
        (EF_electricity_energyrecovery.key, AbsChiller_EOL_energyrecoveryele_PEHD, 'technosphere'),
        (EF_heat_energyrecovery.key, AbsChiller_EOL_energyrecoveryheat_elastomer, 'technosphere'),
        (EF_electricity_energyrecovery.key, AbsChiller_EOL_energyrecoveryele_elastomer, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(AbsorptionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_AbsChiller_exchange_list = []
        
        for input_key, data_array, exc_type in D3_AbsChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_AbsChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_AbsChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_AbsChiller = D3_AbsChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_AbsChiller.save()

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in AbsorptionChiller_BdName:
        D3_AbsChiller_acts[bd_name].new_exchange(
            input=D3_AbsChiller_acts[bd_name].key, 
            output=D3_AbsChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_AbsChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_AbsChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_AbsChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(AbsorptionChiller_BdName, AbsorptionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_AbsChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_AbsChiller.lci()
    mlca_D3_AbsChiller.lcia()
    mlca_D3_AbsChiller.scores

    dfresults_D3_AbsChiller = pd.DataFrame.from_dict(mlca_D3_AbsChiller.scores, orient='index')
    dfresults_D3_AbsChiller.index = pd.MultiIndex.from_tuples(dfresults_D3_AbsChiller.index, names=['Column', 'Row'])
    dfresults_D3_AbsChiller = dfresults_D3_AbsChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_AbsChiller.columns = dfresults_D3_AbsChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_AbsChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_AbsChiller = dfresults_D3_AbsChiller.mul(SFAbsChiller, axis=0)

### 21.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "Cooling - Absorption chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_AbsChiller = dfresults_D3_AbsChiller.add(dfresults_D3_AbsChiller.mul(AbsChillerNumberReplacements, axis=0))

# 22. Cooling - compression chiller

## 22.1 Activities

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A13_CompressionChiller_{bd_name}',
        'name': f'A13_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            A13_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A4_CompressionChiller_{bd_name}',
        'name': f'A4_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            A4_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A5_CompressionChiller_{bd_name}',
        'name': f'A5_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            A5_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B2_CompressionChiller_{bd_name}',
        'name': f'B2_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            B2_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B6_CompressionChiller_{bd_name}',
        'name': f'B6_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            B6_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C2_CompressionChiller_{bd_name}',
        'name': f'C2_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            C2_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C3_CompressionChiller_{bd_name}',
        'name': f'C3_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            C3_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C4_CompressionChiller_{bd_name}',
        'name': f'C4_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            C4_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D1_CompressionChiller_{bd_name}',
        'name': f'D1_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            D1_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Compression chiller"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_CompressionChiller_acts = {}  # Dictionary to store activities by building name
    for bd_name in CompressionChiller_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D3_CompressionChiller_{bd_name}',
        'name': f'D3_CompressionChiller_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_CompressionChiller_acts[bd_name] = scenario_db.new_activity(**data)
            D3_CompressionChiller_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 22.2 Exchanges

### 22.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_CompressionChiller_exchange_configs = [
        (EF_Aluminium.key, CompressionChiller_aluminium, 'technosphere'),
        (EF_castiron.key, CompressionChiller_castiron, 'technosphere'),
        (EF_Copper.key, CompressionChiller_copper, 'technosphere'),
        (EF_steel.key, CompressionChiller_lowalloyedsteel, 'technosphere'),
        (EF_sheetrollingsteel.key, CompressionChiller_sheetrollingsteel, 'technosphere'),
        (EF_sheetrollingaluminium.key, CompressionChiller_sheetrollingaluminium, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CompressionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_CompressionChiller_exchange_list = []
        
        for input_key, data_array, exc_type in A13_CompressionChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_CompressionChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_CompressionChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_CompressionChiller = A13_CompressionChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_CompressionChiller.save()

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        A13_CompressionChiller_acts[bd_name].new_exchange(
            input=A13_CompressionChiller_acts[bd_name].key, 
            output=A13_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_CompressionChiller.lci()
    mlca_A13_CompressionChiller.lcia()
    mlca_A13_CompressionChiller.scores

    dfresults_A13_CompressionChiller = pd.DataFrame.from_dict(mlca_A13_CompressionChiller.scores, orient='index')
    dfresults_A13_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_A13_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_A13_CompressionChiller = dfresults_A13_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_CompressionChiller.columns = dfresults_A13_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_CompressionChiller = dfresults_A13_CompressionChiller.mul(SFCompressionChiller, axis=0)

### 22.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_CompressionChiller_exchange_configs = [
        (EF_Truk_16_32.key, CompressionChiller_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, CompressionChiller_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CompressionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_CompressionChiller_exchange_list = []
        
        for input_key, data_array, exc_type in A4_CompressionChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_CompressionChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_CompressionChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_CompressionChiller = A4_CompressionChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_CompressionChiller.save()

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        A4_CompressionChiller_acts[bd_name].new_exchange(
            input=A4_CompressionChiller_acts[bd_name].key, 
            output=A4_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_CompressionChiller.lci()
    mlca_A4_CompressionChiller.lcia()
    mlca_A4_CompressionChiller.scores

    dfresults_A4_CompressionChiller = pd.DataFrame.from_dict(mlca_A4_CompressionChiller.scores, orient='index')
    dfresults_A4_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_A4_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_A4_CompressionChiller = dfresults_A4_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_CompressionChiller.columns = dfresults_A4_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_CompressionChiller = dfresults_A4_CompressionChiller.mul(SFCompressionChiller, axis=0)

### 22.2.3 Module A5

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        A5_CompressionChiller_acts[bd_name].new_exchange(
            input=A5_CompressionChiller_acts[bd_name].key, 
            output=A5_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_CompressionChiller.lci()
    mlca_A5_CompressionChiller.lcia()
    mlca_A5_CompressionChiller.scores

    dfresults_A5_CompressionChiller = pd.DataFrame.from_dict(mlca_A5_CompressionChiller.scores, orient='index')
    dfresults_A5_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_A5_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_A5_CompressionChiller = dfresults_A5_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_CompressionChiller.columns = dfresults_A5_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_CompressionChiller = dfresults_A5_CompressionChiller.mul(SFCompressionChiller, axis=0)

### 22.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_CompressionChiller_exchange_configs = [
        (EF_LightTruk.key, CompressionChiller_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CompressionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_CompressionChiller_exchange_list = []
        
        for input_key, data_array, exc_type in B2_CompressionChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_CompressionChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_CompressionChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_CompressionChiller = B2_CompressionChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_CompressionChiller.save()

In [ ]:
# B2exc = {}  # 
# for bd_name in CompressionChiller_BdName:
#     B2exc[bd_name] = B2_CompressionChiller_acts[bd_name].exchanges()
#     B2exc[bd_name].delete()

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        B2_CompressionChiller_acts[bd_name].new_exchange(
            input=B2_CompressionChiller_acts[bd_name].key, 
            output=B2_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_CompressionChiller.lci()
    mlca_B2_CompressionChiller.lcia()
    mlca_B2_CompressionChiller.scores

    dfresults_B2_CompressionChiller = pd.DataFrame.from_dict(mlca_B2_CompressionChiller.scores, orient='index')
    dfresults_B2_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_B2_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_B2_CompressionChiller = dfresults_B2_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_CompressionChiller.columns = dfresults_B2_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_CompressionChiller = dfresults_B2_CompressionChiller.mul(SFCompressionChiller, axis=0)

### 22.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_CompressionChiller_exchange_configs = [
        (EF_electricity.key, CompressionChiller_usephaseelectricity, 'technosphere'),
        (EF_electricityPV.key, CompressionChiller_usephaseelectricityPV, 'technosphere'),
        # (EF_sun.key, CompressionChiller_usephaseelectricityRECSPV, 'technosphere'),
        # (EF_hydro.key, CompressionChiller_usephaseelectricityRECSHydro, 'technosphere'),
        # (EF_wind.key, CompressionChiller_usephaseelectricityRECSEolic, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CompressionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_CompressionChiller_exchange_list = []
        
        for input_key, data_array, exc_type in B6_CompressionChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_CompressionChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_CompressionChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_CompressionChiller = B6_CompressionChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_CompressionChiller.save()

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        B6_CompressionChiller_acts[bd_name].new_exchange(
            input=B6_CompressionChiller_acts[bd_name].key, 
            output=B6_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_CompressionChiller.lci()
    mlca_B6_CompressionChiller.lcia()
    mlca_B6_CompressionChiller.scores

    dfresults_B6_CompressionChiller = pd.DataFrame.from_dict(mlca_B6_CompressionChiller.scores, orient='index')
    dfresults_B6_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_B6_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_B6_CompressionChiller = dfresults_B6_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_CompressionChiller.columns = dfresults_B6_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_CompressionChiller.rename(columns=method_lookup, inplace=True)

### 22.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_CompressionChiller_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, CompressionChiller_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CompressionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_CompressionChiller_exchange_list = []
        
        for input_key, data_array, exc_type in C2_CompressionChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_CompressionChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_CompressionChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_CompressionChiller = C2_CompressionChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_CompressionChiller.save()

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        C2_CompressionChiller_acts[bd_name].new_exchange(
            input=C2_CompressionChiller_acts[bd_name].key, 
            output=C2_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_CompressionChiller.lci()
    mlca_C2_CompressionChiller.lcia()
    mlca_C2_CompressionChiller.scores

    dfresults_C2_CompressionChiller = pd.DataFrame.from_dict(mlca_C2_CompressionChiller.scores, orient='index')
    dfresults_C2_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_C2_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_C2_CompressionChiller = dfresults_C2_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_CompressionChiller.columns = dfresults_C2_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_CompressionChiller = dfresults_C2_CompressionChiller.mul(SFCompressionChiller, axis=0)

### 22.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_CompressionChiller_exchange_configs = [
        (EF_EOL_Aluminium.key, CompressionChiller_EOL_aluminium, 'technosphere'),
        (EF_EOL_metal.key, CompressionChiller_EOL_copper, 'technosphere'),
        (EF_EOL_metal.key, CompressionChiller_EOL_castiron, 'technosphere'),
        (EF_EOL_metal.key, CompressionChiller_EOL_lowalloyedsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CompressionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_CompressionChiller_exchange_list = []
        
        for input_key, data_array, exc_type in C3_CompressionChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_CompressionChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_CompressionChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_CompressionChiller = C3_CompressionChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_CompressionChiller.save()

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        C3_CompressionChiller_acts[bd_name].new_exchange(
            input=C3_CompressionChiller_acts[bd_name].key, 
            output=C3_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_CompressionChiller.lci()
    mlca_C3_CompressionChiller.lcia()
    mlca_C3_CompressionChiller.scores

    dfresults_C3_CompressionChiller = pd.DataFrame.from_dict(mlca_C3_CompressionChiller.scores, orient='index')
    dfresults_C3_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_C3_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_C3_CompressionChiller = dfresults_C3_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_CompressionChiller.columns = dfresults_C3_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_CompressionChiller = dfresults_C3_CompressionChiller.mul(SFCompressionChiller, axis=0)

### 22.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_CompressionChiller_exchange_configs = [
        (EF_EOL_landfill_aluminium.key, CompressionChiller_EOL_landfill_aluminium, 'technosphere'),
        (EF_EOL_landfill_metal.key, CompressionChiller_EOL_landfill_copper, 'technosphere'),
        (EF_EOL_landfill_metal.key, CompressionChiller_EOL_landfill_lowalloyedsteel, 'technosphere'),
        (EF_EOL_landfill_metal.key, CompressionChiller_EOL_landfill_castiron, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CompressionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_CompressionChiller_exchange_list = []
        
        for input_key, data_array, exc_type in C4_CompressionChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_CompressionChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_CompressionChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_CompressionChiller = C4_CompressionChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_CompressionChiller.save()

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        C4_CompressionChiller_acts[bd_name].new_exchange(
            input=C4_CompressionChiller_acts[bd_name].key, 
            output=C4_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_CompressionChiller.lci()
    mlca_C4_CompressionChiller.lcia()
    mlca_C4_CompressionChiller.scores

    dfresults_C4_CompressionChiller = pd.DataFrame.from_dict(mlca_C4_CompressionChiller.scores, orient='index')
    dfresults_C4_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_C4_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_C4_CompressionChiller = dfresults_C4_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_CompressionChiller.columns = dfresults_C4_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_CompressionChiller = dfresults_C4_CompressionChiller.mul(SFCompressionChiller, axis=0)

### 22.2.9 Module B4

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_CompressionChiller = pd.DataFrame(index=dfresults_A13_CompressionChiller.index)  # Start with the index from A13 for consistency   
    dfresults_B4_CompressionChiller = (
        dfresults_A13_CompressionChiller.add(dfresults_A4_CompressionChiller, fill_value=0)
        .add(dfresults_C3_CompressionChiller, fill_value=0)
        .add(dfresults_C4_CompressionChiller, fill_value=0)
    ).mul(CompressionChillerNumberReplacements, axis=0)

### 22.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_CompressionChiller_exchange_configs = [
        (EF_recycling_steel.key, CompressionChiller_EOL_recycling_castiron, 'technosphere'),
        (EF_primary_castiron.key, CompressionChiller_EOL_primary_castiron, 'technosphere'),
        (EF_recycling_steel.key, CompressionChiller_EOL_recycling_steel, 'technosphere'),
        (EF_primary_steel.key, CompressionChiller_EOL_primary_steel, 'technosphere'),
        (EF_recycling_aluminium.key, CompressionChiller_EOL_recycling_aluminium, 'technosphere'),
        (EF_primary_aluminium.key, CompressionChiller_EOL_primary_aluminium, 'technosphere'),
        (EF_recycling_copper.key, CompressionChiller_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, CompressionChiller_EOL_primary_copper, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CompressionChiller_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_CompressionChiller_exchange_list = []
        
        for input_key, data_array, exc_type in D1_CompressionChiller_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_CompressionChiller_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_CompressionChiller_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_CompressionChiller = D1_CompressionChiller_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_CompressionChiller.save()

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        D1_CompressionChiller_acts[bd_name].new_exchange(
            input=D1_CompressionChiller_acts[bd_name].key, 
            output=D1_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_CompressionChiller.lci()
    mlca_D1_CompressionChiller.lcia()
    mlca_D1_CompressionChiller.scores

    dfresults_D1_CompressionChiller = pd.DataFrame.from_dict(mlca_D1_CompressionChiller.scores, orient='index')
    dfresults_D1_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_D1_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_D1_CompressionChiller = dfresults_D1_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_CompressionChiller.columns = dfresults_D1_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_CompressionChiller = dfresults_D1_CompressionChiller.mul(SFCompressionChiller, axis=0)

### 22.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_CompressionChiller = dfresults_D1_CompressionChiller.add(dfresults_D1_CompressionChiller.mul(CompressionChillerNumberReplacements, axis=0))

### 22.2.12 Module D3 (related to the export of energy as result of waste incineration)

In [ ]:
sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CompressionChiller_BdName:
        D3_CompressionChiller_acts[bd_name].new_exchange(
            input=D3_CompressionChiller_acts[bd_name].key, 
            output=D3_CompressionChiller_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Compression chiller"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_CompressionChiller = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_CompressionChiller['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_CompressionChiller_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CompressionChiller_BdName, CompressionChiller_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_CompressionChiller = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_CompressionChiller.lci()
    mlca_D3_CompressionChiller.lcia()
    mlca_D3_CompressionChiller.scores

    dfresults_D3_CompressionChiller = pd.DataFrame.from_dict(mlca_D3_CompressionChiller.scores, orient='index')
    dfresults_D3_CompressionChiller.index = pd.MultiIndex.from_tuples(dfresults_D3_CompressionChiller.index, names=['Column', 'Row'])
    dfresults_D3_CompressionChiller = dfresults_D3_CompressionChiller.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_CompressionChiller.columns = dfresults_D3_CompressionChiller.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_CompressionChiller.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_CompressionChiller = dfresults_D3_CompressionChiller.mul(SFCompressionChiller, axis=0)

# 23. Cooling - Dry cooler

## 23.1 Activities

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A13_DryCooler_{bd_name}',
        'name': f'A13_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            A13_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A4_DryCooler_{bd_name}',
        'name': f'A4_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            A4_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A5_DryCooler_{bd_name}',
        'name': f'A5_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            A5_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B2_DryCooler_{bd_name}',
        'name': f'B2_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            B2_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B6_DryCooler_{bd_name}',
        'name': f'B6_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            B6_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C2_DryCooler_{bd_name}',
        'name': f'C2_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            C2_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C3_DryCooler_{bd_name}',
        'name': f'C3_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            C3_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C4_DryCooler_{bd_name}',
        'name': f'C4_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            C4_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D1_DryCooler_{bd_name}',
        'name': f'D1_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            D1_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Dry coooler"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_DryCooler_acts = {}  # Dictionary to store activities by building name
    for bd_name in DryCooler_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D3_DryCooler_{bd_name}',
        'name': f'D3_DryCooler_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_DryCooler_acts[bd_name] = scenario_db.new_activity(**data)
            D3_DryCooler_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 23.2 Exchanges

### 23.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_DryCooler_exchange_configs = [
        (EF_steel.key,                DryCooler_lowalloyedsteel,      'technosphere'),
        (EF_Aluminium.key,            DryCooler_aluminium,            'technosphere'),
        (EF_StainlessSteel.key,       DryCooler_chromiumsteel,        'technosphere'),
        (EF_Copper.key,               DryCooler_copper,               'technosphere'),
        (EF_sheetrollingsteel.key,    DryCooler_sheetrollingsteel,    'technosphere'),
        (EF_sheetrollingaluminium.key,DryCooler_sheetrollingaluminium,'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(DryCooler_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_DryCooler_exchange_list = []
        
        for input_key, data_array, exc_type in A13_DryCooler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_DryCooler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_DryCooler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_DryCooler = A13_DryCooler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_DryCooler.save()

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        A13_DryCooler_acts[bd_name].new_exchange(
            input=A13_DryCooler_acts[bd_name].key, 
            output=A13_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_DryCooler.lci()
    mlca_A13_DryCooler.lcia()
    mlca_A13_DryCooler.scores

    dfresults_A13_DryCooler = pd.DataFrame.from_dict(mlca_A13_DryCooler.scores, orient='index')
    dfresults_A13_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_A13_DryCooler.index, names=['Column', 'Row'])
    dfresults_A13_DryCooler = dfresults_A13_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_DryCooler.columns = dfresults_A13_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_DryCooler = dfresults_A13_DryCooler.mul(SFDryCooler, axis=0)

### 23.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_DryCooler_exchange_configs = [
        (EF_Truk_16_32.key, DryCooler_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, DryCooler_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(DryCooler_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_DryCooler_exchange_list = []
        
        for input_key, data_array, exc_type in A4_DryCooler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_DryCooler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_DryCooler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_DryCooler = A4_DryCooler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_DryCooler.save()

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        A4_DryCooler_acts[bd_name].new_exchange(
            input=A4_DryCooler_acts[bd_name].key, 
            output=A4_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_DryCooler.lci()
    mlca_A4_DryCooler.lcia()
    mlca_A4_DryCooler.scores

    dfresults_A4_DryCooler = pd.DataFrame.from_dict(mlca_A4_DryCooler.scores, orient='index')
    dfresults_A4_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_A4_DryCooler.index, names=['Column', 'Row'])
    dfresults_A4_DryCooler = dfresults_A4_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_DryCooler.columns = dfresults_A4_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_DryCooler = dfresults_A4_DryCooler.mul(SFDryCooler, axis=0)

### 23.2.3 Module A5

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        A5_DryCooler_acts[bd_name].new_exchange(
            input=A5_DryCooler_acts[bd_name].key, 
            output=A5_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_DryCooler.lci()
    mlca_A5_DryCooler.lcia()
    mlca_A5_DryCooler.scores

    dfresults_A5_DryCooler = pd.DataFrame.from_dict(mlca_A5_DryCooler.scores, orient='index')
    dfresults_A5_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_A5_DryCooler.index, names=['Column', 'Row'])
    dfresults_A5_DryCooler = dfresults_A5_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_DryCooler.columns = dfresults_A5_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_DryCooler = dfresults_A5_DryCooler.mul(SFDryCooler, axis=0)

### 23.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_DryCooler_exchange_configs = [
        (EF_LightTruk.key, DryCooler_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(DryCooler_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_DryCooler_exchange_list = []
        
        for input_key, data_array, exc_type in B2_DryCooler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_DryCooler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_DryCooler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_DryCooler = B2_DryCooler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_DryCooler.save()

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        B2_DryCooler_acts[bd_name].new_exchange(
            input=B2_DryCooler_acts[bd_name].key, 
            output=B2_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_DryCooler.lci()
    mlca_B2_DryCooler.lcia()
    mlca_B2_DryCooler.scores

    dfresults_B2_DryCooler = pd.DataFrame.from_dict(mlca_B2_DryCooler.scores, orient='index')
    dfresults_B2_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_B2_DryCooler.index, names=['Column', 'Row'])
    dfresults_B2_DryCooler = dfresults_B2_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_DryCooler.columns = dfresults_B2_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_DryCooler = dfresults_B2_DryCooler.mul(SFDryCooler, axis=0)

### 23.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_DryCooler_exchange_configs = [
        (EF_electricity.key, DryCooler_usephaseelectricity, 'technosphere'),
        (EF_electricityPV.key, DryCooler_usephaseelectricityPV, 'technosphere'),
        # (EF_sun.key, DryCooler_usephaseelectricityRECSPV, 'technosphere'),
        # (EF_hydro.key, DryCooler_usephaseelectricityRECSHydro, 'technosphere'),
        # (EF_wind.key, DryCooler_usephaseelectricityRECSEolic, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(DryCooler_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_DryCooler_exchange_list = []
        
        for input_key, data_array, exc_type in B6_DryCooler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_DryCooler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_DryCooler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_DryCooler = B6_DryCooler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_DryCooler.save()

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        B6_DryCooler_acts[bd_name].new_exchange(
            input=B6_DryCooler_acts[bd_name].key, 
            output=B6_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_DryCooler.lci()
    mlca_B6_DryCooler.lcia()
    mlca_B6_DryCooler.scores

    dfresults_B6_DryCooler = pd.DataFrame.from_dict(mlca_B6_DryCooler.scores, orient='index')
    dfresults_B6_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_B6_DryCooler.index, names=['Column', 'Row'])
    dfresults_B6_DryCooler = dfresults_B6_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_DryCooler.columns = dfresults_B6_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_DryCooler.rename(columns=method_lookup, inplace=True)

### 23.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_DryCooler_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, DryCooler_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(DryCooler_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_DryCooler_exchange_list = []
        
        for input_key, data_array, exc_type in C2_DryCooler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_DryCooler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_DryCooler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_DryCooler = C2_DryCooler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_DryCooler.save()

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        C2_DryCooler_acts[bd_name].new_exchange(
            input=C2_DryCooler_acts[bd_name].key, 
            output=C2_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_DryCooler.lci()
    mlca_C2_DryCooler.lcia()
    mlca_C2_DryCooler.scores

    dfresults_C2_DryCooler = pd.DataFrame.from_dict(mlca_C2_DryCooler.scores, orient='index')
    dfresults_C2_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_C2_DryCooler.index, names=['Column', 'Row'])
    dfresults_C2_DryCooler = dfresults_C2_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_DryCooler.columns = dfresults_C2_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_DryCooler = dfresults_C2_DryCooler.mul(SFDryCooler, axis=0)

### 23.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_DryCooler_exchange_configs = [
        (EF_EOL_Aluminium.key, DryCooler_EOL_aluminium, 'technosphere'),
        (EF_EOL_metal.key, DryCooler_EOL_copper, 'technosphere'),
        (EF_EOL_metal.key, DryCooler_EOL_chromiumsteel, 'technosphere'),
        (EF_EOL_metal.key, DryCooler_EOL_lowalloyedsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(DryCooler_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_DryCooler_exchange_list = []
        
        for input_key, data_array, exc_type in C3_DryCooler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_DryCooler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C3_DryCooler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_DryCooler = C3_DryCooler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_DryCooler.save()

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        C3_DryCooler_acts[bd_name].new_exchange(
            input=C3_DryCooler_acts[bd_name].key, 
            output=C3_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_DryCooler.lci()
    mlca_C3_DryCooler.lcia()
    mlca_C3_DryCooler.scores

    dfresults_C3_DryCooler = pd.DataFrame.from_dict(mlca_C3_DryCooler.scores, orient='index')
    dfresults_C3_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_C3_DryCooler.index, names=['Column', 'Row'])
    dfresults_C3_DryCooler = dfresults_C3_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_DryCooler.columns = dfresults_C3_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_DryCooler = dfresults_C3_DryCooler.mul(SFDryCooler, axis=0)

### 23.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_DryCooler_exchange_configs = [
        (EF_EOL_landfill_aluminium.key,      DryCooler_EOL_landfill_aluminium,     'technosphere'),
        (EF_EOL_landfill_metal.key,          DryCooler_EOL_landfill_copper,        'technosphere'),
        (EF_EOL_landfill_metal.key,          DryCooler_EOL_landfill_lowalloyedsteel,'technosphere'),
        (EF_EOL_landfill_metal.key,          DryCooler_EOL_landfill_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(DryCooler_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_DryCooler_exchange_list = []
        
        for input_key, data_array, exc_type in C4_DryCooler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_DryCooler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_DryCooler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_DryCooler = C4_DryCooler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_DryCooler.save()

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        C4_DryCooler_acts[bd_name].new_exchange(
            input=C4_DryCooler_acts[bd_name].key, 
            output=C4_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_DryCooler.lci()
    mlca_C4_DryCooler.lcia()
    mlca_C4_DryCooler.scores

    dfresults_C4_DryCooler = pd.DataFrame.from_dict(mlca_C4_DryCooler.scores, orient='index')
    dfresults_C4_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_C4_DryCooler.index, names=['Column', 'Row'])
    dfresults_C4_DryCooler = dfresults_C4_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_DryCooler.columns = dfresults_C4_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_DryCooler = dfresults_C4_DryCooler.mul(SFDryCooler, axis=0)

### 23.2.9 Module B4

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_DryCooler = pd.DataFrame(index=dfresults_A13_DryCooler.index)  # Start with the index from A13 for consistency   
    dfresults_B4_DryCooler = (
        dfresults_A13_DryCooler.add(dfresults_A4_DryCooler, fill_value=0)
        .add(dfresults_C3_DryCooler, fill_value=0)
        .add(dfresults_C4_DryCooler, fill_value=0)
    ).mul(DryCoolerNumberReplacements, axis=0)

### 23.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_DryCooler_exchange_configs = [
        (EF_recycling_aluminium.key, DryCooler_EOL_recycling_aluminium, 'technosphere'),
        (EF_primary_aluminium.key, DryCooler_EOL_primary_aluminium, 'technosphere'),
        (EF_recycling_copper.key, DryCooler_EOL_recycling_copper, 'technosphere'),
        (EF_primary_copper.key, DryCooler_EOL_primary_copper, 'technosphere'),
        (EF_recycling_steel.key, DryCooler_EOL_recycling_lowalloyedsteel, 'technosphere'),
        (EF_primary_steel.key, DryCooler_EOL_primary_lowalloyedsteel, 'technosphere'),
        (EF_recycling_steel.key, DryCooler_EOL_recycling_chromiumsteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, DryCooler_EOL_primary_chromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(DryCooler_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_DryCooler_exchange_list = []
        
        for input_key, data_array, exc_type in D1_DryCooler_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_DryCooler_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_DryCooler_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_DryCooler = D1_DryCooler_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_DryCooler.save()

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        D1_DryCooler_acts[bd_name].new_exchange(
            input=D1_DryCooler_acts[bd_name].key, 
            output=D1_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_DryCooler.lci()
    mlca_D1_DryCooler.lcia()
    mlca_D1_DryCooler.scores

    dfresults_D1_DryCooler = pd.DataFrame.from_dict(mlca_D1_DryCooler.scores, orient='index')
    dfresults_D1_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_D1_DryCooler.index, names=['Column', 'Row'])
    dfresults_D1_DryCooler = dfresults_D1_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_DryCooler.columns = dfresults_D1_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_DryCooler = dfresults_D1_DryCooler.mul(SFDryCooler, axis=0)

### 23.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_DryCooler = dfresults_D1_DryCooler.add(dfresults_D1_DryCooler.mul(DryCoolerNumberReplacements, axis=0))

### 23.2.12 Module D3 (related to the export of energy as result of waste incineration)

In [ ]:
sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in DryCooler_BdName:
        D3_DryCooler_acts[bd_name].new_exchange(
            input=D3_DryCooler_acts[bd_name].key, 
            output=D3_DryCooler_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Dry coooler"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_DryCooler = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_DryCooler['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_DryCooler_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(DryCooler_BdName, DryCooler_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_DryCooler = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_DryCooler.lci()
    mlca_D3_DryCooler.lcia()
    mlca_D3_DryCooler.scores

    dfresults_D3_DryCooler = pd.DataFrame.from_dict(mlca_D3_DryCooler.scores, orient='index')
    dfresults_D3_DryCooler.index = pd.MultiIndex.from_tuples(dfresults_D3_DryCooler.index, names=['Column', 'Row'])
    dfresults_D3_DryCooler = dfresults_D3_DryCooler.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_DryCooler.columns = dfresults_D3_DryCooler.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_DryCooler.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_DryCooler = dfresults_A13_DryCooler.mul(SFDryCooler, axis=0)

# 24. Cooling - Evaporative cooling tower

## 24.1 Activities

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A13_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A13_CoolingTower_{bd_name}',
        'name': f'A13_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A13_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            A13_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A4_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A4_CoolingTower_{bd_name}',
        'name': f'A4_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            A4_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            A4_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    A5_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'A5_CoolingTower_{bd_name}',
        'name': f'A5_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            A5_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            A5_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B2_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B2_CoolingTower_{bd_name}',
        'name': f'B2_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            B2_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            B2_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B6_CoolingTower_{bd_name}',
        'name': f'B6_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            B6_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C2_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C2_CoolingTower_{bd_name}',
        'name': f'C2_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'ton kilometer / m2'
        }

        try:
            C2_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            C2_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C3_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C3_CoolingTower_{bd_name}',
        'name': f'C3_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C3_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            C3_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    C4_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'C4_CoolingTower_{bd_name}',
        'name': f'C4_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            C4_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            C4_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D1_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D1_CoolingTower_{bd_name}',
        'name': f'D1_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D1_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            D1_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

In [ ]:
sheet_name = "Cooling - Cooling tower"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    D3_CoolingTower_acts = {}  # Dictionary to store activities by building name
    for bd_name in CoolingTower_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'D3_CoolingTower_{bd_name}',
        'name': f'D3_CoolingTower_{bd_name}',
        'location': 'IT',
        'unit': 'unit / m2'
        }

        try:
            D3_CoolingTower_acts[bd_name] = scenario_db.new_activity(**data)
            D3_CoolingTower_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 24.2 Exchanges

### 24.2.1 Module A1-3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A13_CoolingTower_exchange_configs = [
        (EF_StainlessSteel.key, CoolingTower_stainlessSteel, 'technosphere'),
        (EF_PVC.key, CoolingTower_PVC, 'technosphere'),
        (EF_castiron.key, CoolingTower_castiron, 'technosphere'),
        (EF_steel.key, CoolingTower_airfun, 'technosphere'),
        (EF_sheetrollingsteel.key, CoolingTower_sheetrollingchromiumsteel, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        A13_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in A13_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A13_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A13_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A13_CoolingTower = A13_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A13_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        A13_CoolingTower_acts[bd_name].new_exchange(
            input=A13_CoolingTower_acts[bd_name].key, 
            output=A13_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A13_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A13_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A13_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A13_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A13_CoolingTower.lci()
    mlca_A13_CoolingTower.lcia()
    mlca_A13_CoolingTower.scores

    dfresults_A13_CoolingTower = pd.DataFrame.from_dict(mlca_A13_CoolingTower.scores, orient='index')
    dfresults_A13_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_A13_CoolingTower.index, names=['Column', 'Row'])
    dfresults_A13_CoolingTower = dfresults_A13_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A13_CoolingTower.columns = dfresults_A13_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A13_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A13_CoolingTower = dfresults_A13_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.2 Module A4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    A4_CoolingTower_exchange_configs = [
        (EF_Truk_16_32.key, CoolingTower_Truk_16_32, 'technosphere'),
        (EF_Light_Truk.key, CoolingTower_Light_Truk, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        A4_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in A4_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            A4_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in A4_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_A4_CoolingTower = A4_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_A4_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        A4_CoolingTower_acts[bd_name].new_exchange(
            input=A4_CoolingTower_acts[bd_name].key, 
            output=A4_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A4_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A4_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A4_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A4_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A4_CoolingTower.lci()
    mlca_A4_CoolingTower.lcia()
    mlca_A4_CoolingTower.scores

    dfresults_A4_CoolingTower = pd.DataFrame.from_dict(mlca_A4_CoolingTower.scores, orient='index')
    dfresults_A4_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_A4_CoolingTower.index, names=['Column', 'Row'])
    dfresults_A4_CoolingTower = dfresults_A4_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A4_CoolingTower.columns = dfresults_A4_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A4_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A4_CoolingTower = dfresults_A4_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.3 Module A5

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        A5_CoolingTower_acts[bd_name].new_exchange(
            input=A5_CoolingTower_acts[bd_name].key, 
            output=A5_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_A5_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_A5_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {A5_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_A5_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_A5_CoolingTower.lci()
    mlca_A5_CoolingTower.lcia()
    mlca_A5_CoolingTower.scores

    dfresults_A5_CoolingTower = pd.DataFrame.from_dict(mlca_A5_CoolingTower.scores, orient='index')
    dfresults_A5_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_A5_CoolingTower.index, names=['Column', 'Row'])
    dfresults_A5_CoolingTower = dfresults_A5_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_A5_CoolingTower.columns = dfresults_A5_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_A5_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_A5_CoolingTower = dfresults_A5_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.4 Module B2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B2_CoolingTower_exchange_configs = [
        (EF_LightTruk.key, CoolingTower_Light_Truk_B2, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        B2_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in B2_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B2_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B2_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B2_CoolingTower = B2_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B2_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        B2_CoolingTower_acts[bd_name].new_exchange(
            input=B2_CoolingTower_acts[bd_name].key, 
            output=B2_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B2_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B2_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B2_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B2_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B2_CoolingTower.lci()
    mlca_B2_CoolingTower.lcia()
    mlca_B2_CoolingTower.scores

    dfresults_B2_CoolingTower = pd.DataFrame.from_dict(mlca_B2_CoolingTower.scores, orient='index')
    dfresults_B2_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_B2_CoolingTower.index, names=['Column', 'Row'])
    dfresults_B2_CoolingTower = dfresults_B2_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B2_CoolingTower.columns = dfresults_B2_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B2_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_B2_CoolingTower = dfresults_B2_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.5 Module B6

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_CoolingTower_exchange_configs = [
        (EF_electricity.key, CoolingTower_usephaseelectricity, 'technosphere'),
        (EF_electricityPV.key, CoolingTower_usephaseelectricityPV, 'technosphere'),
        # (EF_sun.key, CoolingTower_usephaseelectricityRECSPV, 'technosphere'),
        # (EF_hydro.key, CoolingTower_usephaseelectricityRECSHydro, 'technosphere'),
        # (EF_wind.key, CoolingTower_usephaseelectricityRECSEolic, 'technosphere'),
        (EF_watercoolingtower.key, CoolingTower_waterdemand, 'technosphere'),
        (EF_Wastewater.key, CoolingTower_wastewater, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in B6_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_CoolingTower = B6_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        B6_CoolingTower_acts[bd_name].new_exchange(
            input=B6_CoolingTower_acts[bd_name].key, 
            output=B6_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_CoolingTower.lci()
    mlca_B6_CoolingTower.lcia()
    mlca_B6_CoolingTower.scores

    dfresults_B6_CoolingTower = pd.DataFrame.from_dict(mlca_B6_CoolingTower.scores, orient='index')
    dfresults_B6_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_B6_CoolingTower.index, names=['Column', 'Row'])
    dfresults_B6_CoolingTower = dfresults_B6_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_CoolingTower.columns = dfresults_B6_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_CoolingTower.rename(columns=method_lookup, inplace=True)

### 24.2.6 Module C2

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C2_CoolingTower_exchange_configs = [
        (EF_Transport_to_treatment_plant.key, CoolingTower_Transport_to_treatment_plant, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        C2_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in C2_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C2_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C2_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C2_CoolingTower = C2_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C2_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        C2_CoolingTower_acts[bd_name].new_exchange(
            input=C2_CoolingTower_acts[bd_name].key, 
            output=C2_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C2_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C2_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C2_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C2_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C2_CoolingTower.lci()
    mlca_C2_CoolingTower.lcia()
    mlca_C2_CoolingTower.scores

    dfresults_C2_CoolingTower = pd.DataFrame.from_dict(mlca_C2_CoolingTower.scores, orient='index')
    dfresults_C2_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_C2_CoolingTower.index, names=['Column', 'Row'])
    dfresults_C2_CoolingTower = dfresults_C2_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C2_CoolingTower.columns = dfresults_C2_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C2_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C2_CoolingTower = dfresults_C2_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.7 Module C3

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C3_CoolingTower_exchange_configs = [
        (EF_EOL_metal.key, CoolingTower_EOL_stainlessSteel, 'technosphere'),
        (EF_EOL_PVC.key, CoolingTower_EOL_PVC, 'technosphere'),
        (EF_EOL_metal.key, CoolingTower_EOL_castiron, 'technosphere'),
        (EF_EOL_metal.key, CoolingTower_EOL_airfun, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        C3_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in C3_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C3_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                    
        # Iterate through the list of exchange data
        for exchange_data in C3_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C3_CoolingTower = C3_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C3_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        C3_CoolingTower_acts[bd_name].new_exchange(
            input=C3_CoolingTower_acts[bd_name].key, 
            output=C3_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C3_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C3_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C3_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C3_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C3_CoolingTower.lci()
    mlca_C3_CoolingTower.lcia()
    mlca_C3_CoolingTower.scores

    dfresults_C3_CoolingTower = pd.DataFrame.from_dict(mlca_C3_CoolingTower.scores, orient='index')
    dfresults_C3_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_C3_CoolingTower.index, names=['Column', 'Row'])
    dfresults_C3_CoolingTower = dfresults_C3_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C3_CoolingTower.columns = dfresults_C3_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C3_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C3_CoolingTower = dfresults_C3_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.8 Module C4

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    C4_CoolingTower_exchange_configs = [
        (EF_EOL_landfill_metal.key, CoolingTower_EOL_landfill_stainlessSteel, 'technosphere'),
        (EF_EOL_inc_plastic.key, CoolingTower_EOL_inc_PVC, 'technosphere'),
        (EF_EOL_landfill_plastic.key, CoolingTower_EOL_landfill_PVC, 'technosphere'),
        (EF_EOL_landfill_metal.key, CoolingTower_EOL_landfill_castiron, 'technosphere'),
        (EF_EOL_landfill_metal.key, CoolingTower_EOL_landfill_airfun, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        C4_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in C4_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            C4_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in C4_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_C4_CoolingTower = C4_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_C4_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        C4_CoolingTower_acts[bd_name].new_exchange(
            input=C4_CoolingTower_acts[bd_name].key, 
            output=C4_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_C4_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_C4_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {C4_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_C4_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_C4_CoolingTower.lci()
    mlca_C4_CoolingTower.lcia()
    mlca_C4_CoolingTower.scores

    dfresults_C4_CoolingTower = pd.DataFrame.from_dict(mlca_C4_CoolingTower.scores, orient='index')
    dfresults_C4_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_C4_CoolingTower.index, names=['Column', 'Row'])
    dfresults_C4_CoolingTower = dfresults_C4_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_C4_CoolingTower.columns = dfresults_C4_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_C4_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_C4_CoolingTower = dfresults_C4_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.9 Module B4

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B4_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B4_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_B4_CoolingTower = pd.DataFrame(index=dfresults_A13_CoolingTower.index)  # Start with the index from A13 for consistency   
    dfresults_B4_CoolingTower = (
        dfresults_A13_CoolingTower.add(dfresults_A4_CoolingTower, fill_value=0)
        .add(dfresults_C3_CoolingTower, fill_value=0)
        .add(dfresults_C4_CoolingTower, fill_value=0)
        ).mul(CoolingTowerNumberReplacements, axis=0)
    

### 24.2.10 Module D1 (related to the export of secondary materials)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D1_CoolingTower_exchange_configs = [
        (EF_recycling_steel.key,        CoolingTower_EOL_recycling_stainlessSteel, 'technosphere'),
        (EF_primary_stainlessSteel.key, CoolingTower_EOL_primary_stainlessSteel,   'technosphere'),
        (EF_recycling_PEHD.key,         CoolingTower_EOL_inc_PVC,                  'technosphere'),
        (EF_primary_PEHD.key,           CoolingTower_EOL_landfill_PVC,             'technosphere'),
        (EF_recycling_steel.key,        CoolingTower_EOL_recycling_castiron,       'technosphere'),
        (EF_primary_steel.key,          CoolingTower_EOL_primary_castiron,         'technosphere'),
        (EF_recycling_steel.key,        CoolingTower_EOL_recycling_airfun,         'technosphere'),
        (EF_primary_steel.key,          CoolingTower_EOL_primary_airfun,           'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        D1_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in D1_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D1_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D1_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D1_CoolingTower = D1_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D1_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        D1_CoolingTower_acts[bd_name].new_exchange(
            input=D1_CoolingTower_acts[bd_name].key, 
            output=D1_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D1_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D1_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D1_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D1_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D1_CoolingTower.lci()
    mlca_D1_CoolingTower.lcia()
    mlca_D1_CoolingTower.scores

    dfresults_D1_CoolingTower = pd.DataFrame.from_dict(mlca_D1_CoolingTower.scores, orient='index')
    dfresults_D1_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_D1_CoolingTower.index, names=['Column', 'Row'])
    dfresults_D1_CoolingTower = dfresults_D1_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D1_CoolingTower.columns = dfresults_D1_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D1_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D1_CoolingTower = dfresults_D1_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.11 Module D1 (B4 included)

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D1_CoolingTower = dfresults_D1_CoolingTower.add(dfresults_D1_CoolingTower.mul(CoolingTowerNumberReplacements, axis=0))

### 24.2.12 Module D3 (related to the export of energy as result of waste incineration)

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    D3_CoolingTower_exchange_configs = [
        (EF_heat_energyrecovery.key, CoolingTower_EOL_energyrecoveryheat_PVC, 'technosphere'),
        (EF_electricity_energyrecovery.key, CoolingTower_EOL_energyrecoveryele_PVC, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(CoolingTower_BdName):
        
        # Create a list of dictionaries for exchanges
        D3_CoolingTower_exchange_list = []
        
        for input_key, data_array, exc_type in D3_CoolingTower_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            D3_CoolingTower_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in D3_CoolingTower_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_D3_CoolingTower = D3_CoolingTower_acts[bd_name].new_exchange(**exchange_data)
            new_exc_D3_CoolingTower.save()

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in CoolingTower_BdName:
        D3_CoolingTower_acts[bd_name].new_exchange(
            input=D3_CoolingTower_acts[bd_name].key, 
            output=D3_CoolingTower_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_D3_CoolingTower = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_D3_CoolingTower['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {D3_CoolingTower_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(CoolingTower_BdName, CoolingTower_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_D3_CoolingTower = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_D3_CoolingTower.lci()
    mlca_D3_CoolingTower.lcia()
    mlca_D3_CoolingTower.scores

    dfresults_D3_CoolingTower = pd.DataFrame.from_dict(mlca_D3_CoolingTower.scores, orient='index')
    dfresults_D3_CoolingTower.index = pd.MultiIndex.from_tuples(dfresults_D3_CoolingTower.index, names=['Column', 'Row'])
    dfresults_D3_CoolingTower = dfresults_D3_CoolingTower.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_D3_CoolingTower.columns = dfresults_D3_CoolingTower.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_D3_CoolingTower.rename(columns=method_lookup, inplace=True)

    # Scale the results to the Lifetime of the energy system and the Reference Study Period (SF=RSP/Lifetime)
    dfresults_D3_CoolingTower = dfresults_D3_CoolingTower.mul(SFCoolingTower, axis=0)

### 24.2.13 Module D3 (B4 included)

In [ ]:
sheet_name = "Cooling - Cooling tower"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
    dfresults_D3_CoolingTower = dfresults_D3_CoolingTower.add(dfresults_D3_CoolingTower.mul(CoolingTowerNumberReplacements, axis=0))
    

# 25. Appliances

## 25.1 Activities

In [ ]:
sheet_name = "Appliances demand"

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping activities creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with activities creation.")
    B6_Appliances_acts = {}  # Dictionary to store activities by building name
    for bd_name in Appliances_BdName:
        # Create a unique activity for each building's energy system
        data = {
        'code': f'B6_Appliances_{bd_name}',
        'name': f'B6_Appliances_{bd_name}',
        'location': 'IT',
        'unit': 'kilowatt hour / m2'
        }

        try:
            B6_Appliances_acts[bd_name] = scenario_db.new_activity(**data)
            B6_Appliances_acts[bd_name].save()
        except Exception as e:
            print(f"Node already exists or error during creation: {e}")

## 25.2 Exchanges

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Appliances demand"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")

    # 1. Define the mapping list (Tuples of: Input Key, Data Array, Type)
    B6_Appliances_exchange_configs = [
        (EF_electricity.key, Appliances_usephaseelectricity, 'technosphere'),
        (EF_electricityPV.key, Appliances_usephaseelectricityPV, 'technosphere'),
    ]

    # 2. Prepare the bulk update dictionary
    # We update the 'exchanges' list inside the activity object directly
    for i, bd_name in enumerate(Appliances_BdName):
        
        # Create a list of dictionaries for exchanges
        B6_Appliances_exchange_list = []
        
        for input_key, data_array, exc_type in B6_Appliances_exchange_configs:
            if float(data_array[i]) == 0:
                continue
            B6_Appliances_exchange_list.append({
                'amount': float(data_array[i]),
                'type': exc_type,
                'input': input_key
                })
                
        # Iterate through the list of exchange data
        for exchange_data in B6_Appliances_exchange_list:
            # Create a new exchange for each item in the list
            new_exc_B6_Appliances = B6_Appliances_acts[bd_name].new_exchange(**exchange_data)
            new_exc_B6_Appliances.save()

In [ ]:
sheet_name = "Appliances demand"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping exchanges creation.")
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with exchanges creation.")
    for bd_name in Appliances_BdName:
        B6_Appliances_acts[bd_name].new_exchange(
            input=B6_Appliances_acts[bd_name].key, 
            output=B6_Appliances_acts[bd_name].key,
            amount=1,
            type='production'
    ).save()

In [ ]:
# Check for the sheet BEFORE starting the heavy work

sheet_name = "Appliances demand"
xls = pd.ExcelFile(file_path)

if sheet_name not in xls.sheet_names:
    print(f"Sheet '{sheet_name}' not found. Skipping LCA calculations.")
    # Define the columns
    columns = ['Bd_Name', 'PM', 'A', 'CCbio', 'CCfossil', 'CCluc', 'CC', 
           'ERnren', 'EF', 'EM', 'ET', 'RUMM', 'OD', 'POF', 'WU', 'PENRT']

    # Create dataframe with zeros and the full column Bd_Name
    dfresults_B6_Appliances = pd.DataFrame(0, index=range(len(Bd_Name)), columns=columns)
    dfresults_B6_Appliances['Bd_Name'] = Bd_Name
else:
    print(f"Sheet '{sheet_name}' found. Proceeding with LCA calculations.")
        
    # 1. Create a dictionary of functional units
    # format: [{activity_key: amount}, {activity_key: amount}, ...]

    functional_units = {
        bd_name: {B6_Appliances_acts[bd_name].id: 1 / bd_area} 
        for bd_name, bd_area in zip(Appliances_BdName, Appliances_BdArea)
    }

    my_methods = list(methods.values())
    methods_config = {"impact_categories": my_methods}

    data_objs = bd.get_multilca_data_objs(functional_units=functional_units, method_config=methods_config)

    mlca_B6_Appliances = bc.MultiLCA(
        demands=functional_units,
        method_config=methods_config,
        data_objs=data_objs,
    )
    mlca_B6_Appliances.lci()
    mlca_B6_Appliances.lcia()
    mlca_B6_Appliances.scores

    dfresults_B6_Appliances = pd.DataFrame.from_dict(mlca_B6_Appliances.scores, orient='index')
    dfresults_B6_Appliances.index = pd.MultiIndex.from_tuples(dfresults_B6_Appliances.index, names=['Column', 'Row'])
    dfresults_B6_Appliances = dfresults_B6_Appliances.unstack(level=0)

    # Create a reverse lookup for methods O(1) lookup
    # Maps the long tuple back to 'CC', 'A', etc.
    method_lookup = {v: k for k, v in methods.items()}

    # Clean up column index (remove '0' or whatever the value column name was)
    dfresults_B6_Appliances.columns = dfresults_B6_Appliances.columns.droplevel(0)

    # Map the columns using your existing lookup dictionary
    # The keys of your columns are the long method tuples
    dfresults_B6_Appliances.rename(columns=method_lookup, inplace=True)
    

# 26. TOTAL AND OUTPUT

## 26.1 Heating (per building, /m2)

### 26.1.1 Module A13

In [ ]:
dfresults_A13_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A13_Heating[cols_to_exclude]
df_data = dfresults_A13_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A13_Heating[cols_to_exclude]
df_data = dfresults_A13_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A13_Heating[cols_to_exclude]
df_data = dfresults_A13_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A13_Heating[cols_to_exclude]
df_data = dfresults_A13_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A13_Heating[cols_to_exclude]
df_data = dfresults_A13_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A13_Heating[cols_to_exclude]
df_data = dfresults_A13_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A13_Heating[cols_to_exclude]
df_data = dfresults_A13_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Heating = dfresults_A13_Heating.merge(dfresults_A13_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A13_Heating[cols_to_exclude]
df_data = dfresults_A13_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.2 Module A4

In [ ]:
dfresults_A4_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A4_Heating[cols_to_exclude]
df_data = dfresults_A4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A4_Heating[cols_to_exclude]
df_data = dfresults_A4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A4_Heating[cols_to_exclude]
df_data = dfresults_A4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A4_Heating[cols_to_exclude]
df_data = dfresults_A4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A4_Heating[cols_to_exclude]
df_data = dfresults_A4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A4_Heating[cols_to_exclude]
df_data = dfresults_A4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A4_Heating[cols_to_exclude]
df_data = dfresults_A4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Heating = dfresults_A4_Heating.merge(dfresults_A4_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A4_Heating[cols_to_exclude]
df_data = dfresults_A4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.3 Module A5

In [ ]:
dfresults_A5_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A5_Heating[cols_to_exclude]
df_data = dfresults_A5_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A5_Heating[cols_to_exclude]
df_data = dfresults_A5_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A5_Heating[cols_to_exclude]
df_data = dfresults_A5_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A5_Heating[cols_to_exclude]
df_data = dfresults_A5_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A5_Heating[cols_to_exclude]
df_data = dfresults_A5_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A5_Heating[cols_to_exclude]
df_data = dfresults_A5_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A5_Heating[cols_to_exclude]
df_data = dfresults_A5_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Heating = dfresults_A5_Heating.merge(dfresults_A5_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_A5_Heating[cols_to_exclude]
df_data = dfresults_A5_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.4 Module B1

In [ ]:
dfresults_B1_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B1_Heating = dfresults_B1_Heating.merge(dfresults_B2_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B1_Heating = dfresults_B1_Heating.merge(dfresults_B2_EHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B1_Heating = dfresults_B1_Heating.merge(dfresults_B1_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B1_Heating = dfresults_B1_Heating.merge(dfresults_B1_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B1_Heating[cols_to_exclude]
df_data = dfresults_B1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B1_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.5 Module B2

In [ ]:
dfresults_B2_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B2_Heating[cols_to_exclude]
df_data = dfresults_B2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B2_Heating[cols_to_exclude]
df_data = dfresults_B2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B2_Heating[cols_to_exclude]
df_data = dfresults_B2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B2_Heating[cols_to_exclude]
df_data = dfresults_B2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B2_Heating[cols_to_exclude]
df_data = dfresults_B2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B2_Heating[cols_to_exclude]
df_data = dfresults_B2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B2_Heating[cols_to_exclude]
df_data = dfresults_B2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Heating = dfresults_B2_Heating.merge(dfresults_B2_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B2_Heating[cols_to_exclude]
df_data = dfresults_B2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.6 Module B6

In [ ]:
dfresults_B6_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B6_Heating[cols_to_exclude]
df_data = dfresults_B6_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B6_Heating[cols_to_exclude]
df_data = dfresults_B6_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B6_Heating[cols_to_exclude]
df_data = dfresults_B6_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B6_Heating[cols_to_exclude]
df_data = dfresults_B6_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B6_Heating[cols_to_exclude]
df_data = dfresults_B6_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B6_Heating[cols_to_exclude]
df_data = dfresults_B6_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B6_Heating[cols_to_exclude]
df_data = dfresults_B6_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Heating = dfresults_B6_Heating.merge(dfresults_B6_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B6_Heating[cols_to_exclude]
df_data = dfresults_B6_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.7 Module C2

In [ ]:
dfresults_C2_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C2_Heating[cols_to_exclude]
df_data = dfresults_C2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C2_Heating[cols_to_exclude]
df_data = dfresults_C2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C2_Heating[cols_to_exclude]
df_data = dfresults_C2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C2_Heating[cols_to_exclude]
df_data = dfresults_C2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C2_Heating[cols_to_exclude]
df_data = dfresults_C2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C2_Heating[cols_to_exclude]
df_data = dfresults_C2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C2_Heating[cols_to_exclude]
df_data = dfresults_C2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Heating = dfresults_C2_Heating.merge(dfresults_C2_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C2_Heating[cols_to_exclude]
df_data = dfresults_C2_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.8 Module C3

In [ ]:
dfresults_C3_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C3_Heating[cols_to_exclude]
df_data = dfresults_C3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C3_Heating[cols_to_exclude]
df_data = dfresults_C3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C3_Heating[cols_to_exclude]
df_data = dfresults_C3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C3_Heating[cols_to_exclude]
df_data = dfresults_C3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C3_Heating[cols_to_exclude]
df_data = dfresults_C3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C3_Heating[cols_to_exclude]
df_data = dfresults_C3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C3_Heating[cols_to_exclude]
df_data = dfresults_C3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Heating = dfresults_C3_Heating.merge(dfresults_C3_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C3_Heating[cols_to_exclude]
df_data = dfresults_C3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.9 Module C4

In [ ]:
dfresults_C4_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C4_Heating[cols_to_exclude]
df_data = dfresults_C4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C4_Heating[cols_to_exclude]
df_data = dfresults_C4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C4_Heating[cols_to_exclude]
df_data = dfresults_C4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C4_Heating[cols_to_exclude]
df_data = dfresults_C4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C4_Heating[cols_to_exclude]
df_data = dfresults_C4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C4_Heating[cols_to_exclude]
df_data = dfresults_C4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C4_Heating[cols_to_exclude]
df_data = dfresults_C4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Heating = dfresults_C4_Heating.merge(dfresults_C4_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_C4_Heating[cols_to_exclude]
df_data = dfresults_C4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.10 Module B4

In [ ]:
dfresults_B4_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B4_Heating[cols_to_exclude]
df_data = dfresults_B4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B4_Heating[cols_to_exclude]
df_data = dfresults_B4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B4_Heating[cols_to_exclude]
df_data = dfresults_B4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B4_Heating[cols_to_exclude]
df_data = dfresults_B4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B4_Heating[cols_to_exclude]
df_data = dfresults_B4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B4_Heating[cols_to_exclude]
df_data = dfresults_B4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B4_Heating[cols_to_exclude]
df_data = dfresults_B4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Heating = dfresults_B4_Heating.merge(dfresults_B4_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_B4_Heating[cols_to_exclude]
df_data = dfresults_B4_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.11 Module D1

In [ ]:
dfresults_D1_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D1_Heating[cols_to_exclude]
df_data = dfresults_D1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D1_Heating[cols_to_exclude]
df_data = dfresults_D1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D1_Heating[cols_to_exclude]
df_data = dfresults_D1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D1_Heating[cols_to_exclude]
df_data = dfresults_D1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D1_Heating[cols_to_exclude]
df_data = dfresults_D1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D1_Heating[cols_to_exclude]
df_data = dfresults_D1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D1_Heating[cols_to_exclude]
df_data = dfresults_D1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Heating = dfresults_D1_Heating.merge(dfresults_D1_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D1_Heating[cols_to_exclude]
df_data = dfresults_D1_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.12 Module D3

In [ ]:
dfresults_D3_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})

sheet_name = "Boiler"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_Boiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_Boiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Biomass boiler"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_Biomassboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_Biomassboiler, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Fuel oil boiler"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_Fueloilboiler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_Fueloilboiler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D3_Heating[cols_to_exclude]
df_data = dfresults_D3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Internal Combustion Engine"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_RICE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_RICE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Organic Rankine Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_ORC, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_ORC, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D3_Heating[cols_to_exclude]
df_data = dfresults_D3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Waste-to-Energy"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_WtE, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_WtE, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "CHP-Combined Cycle"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_CombinedCycle, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_CombinedCycle, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D3_Heating[cols_to_exclude]
df_data = dfresults_D3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "CHP-Turbogas"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_GasTurbine, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_GasTurbine, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Heat recovery"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_HeatRecovery, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_HeatRecovery, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D3_Heating[cols_to_exclude]
df_data = dfresults_D3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Solar thermal system"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_SolarThermal, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_SolarThermal, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Electric Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_EHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_EHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D3_Heating[cols_to_exclude]
df_data = dfresults_D3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Gas Absorption Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_GAHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_GAHP, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Gas Engine Heat Pump"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_GEHP, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_GEHP, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D3_Heating[cols_to_exclude]
df_data = dfresults_D3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Concrete thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_ThermalConcreteStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_ThermalConcreteStorage, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Steel thermal storage"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_ThermalSteelStorage, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_ThermalSteelStorage, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D3_Heating[cols_to_exclude]
df_data = dfresults_D3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Heating = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Geothermal plant"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_GEO, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Heating = dfresults_D3_Heating.merge(dfresults_D3_GEO, left_on='Bd_Name', right_index=True, how='left')   

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Heating Energy Systems']
df_ids = dfresults_D3_Heating[cols_to_exclude]
df_data = dfresults_D3_Heating.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Heating = pd.concat([df_ids, df_summed], axis=1)

### 26.1.13 Whole Life Cycle

In [ ]:
dfresults_WLC_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_A13_Heating[cols_to_include]
                .add(dfresults_A4_Heating[cols_to_include])
                .add(dfresults_A5_Heating[cols_to_include])
                .add(dfresults_B1_Heating[cols_to_include])
                .add(dfresults_B2_Heating[cols_to_include])
                .add(dfresults_B6_Heating[cols_to_include])
                .add(dfresults_C2_Heating[cols_to_include])
                .add(dfresults_C3_Heating[cols_to_include])
                .add(dfresults_C4_Heating[cols_to_include])
                .add(dfresults_B4_Heating[cols_to_include])
                .add(dfresults_D1_Heating[cols_to_include])
                .add(dfresults_D3_Heating[cols_to_include]))
dfresults_WLC_Heating = pd.concat([dfresults_WLC_Heating, df_data], axis=1)

In [ ]:
dfresults_WLC_noD_Heating = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_A13_Heating[cols_to_include]
                .add(dfresults_A4_Heating[cols_to_include])
                .add(dfresults_A5_Heating[cols_to_include])
                .add(dfresults_B1_Heating[cols_to_include])
                .add(dfresults_B2_Heating[cols_to_include])
                .add(dfresults_B6_Heating[cols_to_include])
                .add(dfresults_C2_Heating[cols_to_include])
                .add(dfresults_C3_Heating[cols_to_include])
                .add(dfresults_C4_Heating[cols_to_include])
                .add(dfresults_B4_Heating[cols_to_include]))
dfresults_WLC_noD_Heating = pd.concat([dfresults_WLC_noD_Heating, df_data], axis=1)

## 26.2 Cooling (per building, /m2)

### 26.2.1 Module A13

In [ ]:
dfresults_A13_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Cooling = dfresults_A13_Cooling.merge(dfresults_A13_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Cooling = dfresults_A13_Cooling.merge(dfresults_A13_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Cooling = dfresults_A13_Cooling.merge(dfresults_A13_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Cooling = dfresults_A13_Cooling.merge(dfresults_A13_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Cooling = dfresults_A13_Cooling.merge(dfresults_A13_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Cooling = dfresults_A13_Cooling.merge(dfresults_A13_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_A13_Cooling[cols_to_exclude]
df_data = dfresults_A13_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_A13_Cooling = dfresults_A13_Cooling.merge(dfresults_A13_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A13_Cooling = dfresults_A13_Cooling.merge(dfresults_A13_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_A13_Cooling[cols_to_exclude]
df_data = dfresults_A13_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A13_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.2 Module A4

In [ ]:
dfresults_A4_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Cooling = dfresults_A4_Cooling.merge(dfresults_A4_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Cooling = dfresults_A4_Cooling.merge(dfresults_A4_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Cooling = dfresults_A4_Cooling.merge(dfresults_A4_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Cooling = dfresults_A4_Cooling.merge(dfresults_A4_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Cooling = dfresults_A4_Cooling.merge(dfresults_A4_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Cooling = dfresults_A4_Cooling.merge(dfresults_A4_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_A4_Cooling[cols_to_exclude]
df_data = dfresults_A4_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_A4_Cooling = dfresults_A4_Cooling.merge(dfresults_A4_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A4_Cooling = dfresults_A4_Cooling.merge(dfresults_A4_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_A4_Cooling[cols_to_exclude]
df_data = dfresults_A4_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A4_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.3 Module A5

In [ ]:
dfresults_A5_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Cooling = dfresults_A5_Cooling.merge(dfresults_A5_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Cooling = dfresults_A5_Cooling.merge(dfresults_A5_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Cooling = dfresults_A5_Cooling.merge(dfresults_A5_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Cooling = dfresults_A5_Cooling.merge(dfresults_A5_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Cooling = dfresults_A5_Cooling.merge(dfresults_A5_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Cooling = dfresults_A5_Cooling.merge(dfresults_A5_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_A5_Cooling[cols_to_exclude]
df_data = dfresults_A5_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_A5_Cooling = dfresults_A5_Cooling.merge(dfresults_A5_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_A5_Cooling = dfresults_A5_Cooling.merge(dfresults_A5_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_A5_Cooling[cols_to_exclude]
df_data = dfresults_A5_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_A5_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.4 Module B2

In [ ]:
dfresults_B2_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Cooling = dfresults_B2_Cooling.merge(dfresults_B2_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Cooling = dfresults_B2_Cooling.merge(dfresults_B2_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Cooling = dfresults_B2_Cooling.merge(dfresults_B2_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Cooling = dfresults_B2_Cooling.merge(dfresults_B2_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Cooling = dfresults_B2_Cooling.merge(dfresults_B2_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Cooling = dfresults_B2_Cooling.merge(dfresults_B2_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_B2_Cooling[cols_to_exclude]
df_data = dfresults_B2_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_B2_Cooling = dfresults_B2_Cooling.merge(dfresults_B2_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B2_Cooling = dfresults_B2_Cooling.merge(dfresults_B2_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_B2_Cooling[cols_to_exclude]
df_data = dfresults_B2_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B2_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.5 Module B6

In [ ]:
dfresults_B6_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Cooling = dfresults_B6_Cooling.merge(dfresults_B6_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Cooling = dfresults_B6_Cooling.merge(dfresults_B6_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Cooling = dfresults_B6_Cooling.merge(dfresults_B6_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Cooling = dfresults_B6_Cooling.merge(dfresults_B6_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Cooling = dfresults_B6_Cooling.merge(dfresults_B6_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Cooling = dfresults_B6_Cooling.merge(dfresults_B6_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_B6_Cooling[cols_to_exclude]
df_data = dfresults_B6_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Cooling = dfresults_B6_Cooling.merge(dfresults_B6_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Cooling = dfresults_B6_Cooling.merge(dfresults_B6_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_B6_Cooling[cols_to_exclude]
df_data = dfresults_B6_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B6_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.6 Module C2

In [ ]:
dfresults_C2_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Cooling = dfresults_C2_Cooling.merge(dfresults_C2_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Cooling = dfresults_C2_Cooling.merge(dfresults_C2_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Cooling = dfresults_C2_Cooling.merge(dfresults_C2_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Cooling = dfresults_C2_Cooling.merge(dfresults_C2_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Cooling = dfresults_C2_Cooling.merge(dfresults_C2_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Cooling = dfresults_C2_Cooling.merge(dfresults_C2_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_C2_Cooling[cols_to_exclude]
df_data = dfresults_C2_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_C2_Cooling = dfresults_C2_Cooling.merge(dfresults_C2_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C2_Cooling = dfresults_C2_Cooling.merge(dfresults_C2_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_C2_Cooling[cols_to_exclude]
df_data = dfresults_C2_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C2_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.7 Module C3

In [ ]:
dfresults_C3_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Cooling = dfresults_C3_Cooling.merge(dfresults_C3_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Cooling = dfresults_C3_Cooling.merge(dfresults_C3_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Cooling = dfresults_C3_Cooling.merge(dfresults_C3_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Cooling = dfresults_C3_Cooling.merge(dfresults_C3_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Cooling = dfresults_C3_Cooling.merge(dfresults_C3_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Cooling = dfresults_C3_Cooling.merge(dfresults_C3_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_C3_Cooling[cols_to_exclude]
df_data = dfresults_C3_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_C3_Cooling = dfresults_C3_Cooling.merge(dfresults_C3_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C3_Cooling = dfresults_C3_Cooling.merge(dfresults_C3_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_C3_Cooling[cols_to_exclude]
df_data = dfresults_C3_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C3_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.8 Module C4

In [ ]:
dfresults_C4_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Cooling = dfresults_C4_Cooling.merge(dfresults_C4_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Cooling = dfresults_C4_Cooling.merge(dfresults_C4_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Cooling = dfresults_C4_Cooling.merge(dfresults_C4_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Cooling = dfresults_C4_Cooling.merge(dfresults_C4_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Cooling = dfresults_C4_Cooling.merge(dfresults_C4_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Cooling = dfresults_C4_Cooling.merge(dfresults_C4_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_C4_Cooling[cols_to_exclude]
df_data = dfresults_C4_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_C4_Cooling = dfresults_C4_Cooling.merge(dfresults_C4_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_C4_Cooling = dfresults_C4_Cooling.merge(dfresults_C4_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_C4_Cooling[cols_to_exclude]
df_data = dfresults_C4_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_C4_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.9 Module B4

In [ ]:
dfresults_B4_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Cooling = dfresults_B4_Cooling.merge(dfresults_B4_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Cooling = dfresults_B4_Cooling.merge(dfresults_B4_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Cooling = dfresults_B4_Cooling.merge(dfresults_B4_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Cooling = dfresults_B4_Cooling.merge(dfresults_B4_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Cooling = dfresults_B4_Cooling.merge(dfresults_B4_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Cooling = dfresults_B4_Cooling.merge(dfresults_B4_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_B4_Cooling[cols_to_exclude]
df_data = dfresults_B4_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_B4_Cooling = dfresults_B4_Cooling.merge(dfresults_B4_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B4_Cooling = dfresults_B4_Cooling.merge(dfresults_B4_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_B4_Cooling[cols_to_exclude]
df_data = dfresults_B4_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_B4_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.10 Module D1

In [ ]:
dfresults_D1_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Cooling = dfresults_D1_Cooling.merge(dfresults_D1_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Cooling = dfresults_D1_Cooling.merge(dfresults_D1_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Cooling = dfresults_D1_Cooling.merge(dfresults_D1_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Cooling = dfresults_D1_Cooling.merge(dfresults_D1_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Cooling = dfresults_D1_Cooling.merge(dfresults_D1_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Cooling = dfresults_D1_Cooling.merge(dfresults_D1_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_D1_Cooling[cols_to_exclude]
df_data = dfresults_D1_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_D1_Cooling = dfresults_D1_Cooling.merge(dfresults_D1_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D1_Cooling = dfresults_D1_Cooling.merge(dfresults_D1_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_D1_Cooling[cols_to_exclude]
df_data = dfresults_D1_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D1_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.11 Module D3

In [ ]:
dfresults_D3_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})

sheet_name = "Cooling - Absorption chiller"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Cooling = dfresults_D3_Cooling.merge(dfresults_D3_AbsChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Cooling = dfresults_D3_Cooling.merge(dfresults_D3_AbsChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Compression chiller"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Cooling = dfresults_D3_Cooling.merge(dfresults_D3_CompressionChiller, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Cooling = dfresults_D3_Cooling.merge(dfresults_D3_CompressionChiller, left_on='Bd_Name', right_index=True, how='left')

sheet_name = "Cooling - Cooling tower"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Cooling = dfresults_D3_Cooling.merge(dfresults_D3_CoolingTower, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Cooling = dfresults_D3_Cooling.merge(dfresults_D3_CoolingTower, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_D3_Cooling[cols_to_exclude]
df_data = dfresults_D3_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Cooling = pd.concat([df_ids, df_summed], axis=1)

sheet_name = "Cooling - Dry cooler"
if sheet_name not in xls.sheet_names:
    dfresults_D3_Cooling = dfresults_D3_Cooling.merge(dfresults_D3_DryCooler, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_D3_Cooling = dfresults_D3_Cooling.merge(dfresults_D3_DryCooler, left_on='Bd_Name', right_index=True, how='left')

# Sum the merged columns up to now, by grouping on the part of the column name before the last underscore
cols_to_exclude = ['Bd_Name', 'Bd_Area', 'Cooling Energy Systems']
df_ids = dfresults_D3_Cooling[cols_to_exclude]
df_data = dfresults_D3_Cooling.drop(columns=cols_to_exclude)
df_summed = df_data.T.groupby(lambda x: x.rsplit('_', 1)[0]).sum().T
dfresults_D3_Cooling = pd.concat([df_ids, df_summed], axis=1)

### 26.2.12 Whole Life Cycle

In [ ]:
dfresults_WLC_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_A13_Cooling[cols_to_include]
                .add(dfresults_A4_Cooling[cols_to_include])
                .add(dfresults_A5_Cooling[cols_to_include])
                .add(dfresults_B2_Cooling[cols_to_include])
                .add(dfresults_B6_Cooling[cols_to_include])
                .add(dfresults_C2_Cooling[cols_to_include])
                .add(dfresults_C3_Cooling[cols_to_include])
                .add(dfresults_C4_Cooling[cols_to_include])
                .add(dfresults_B4_Cooling[cols_to_include])
                .add(dfresults_D1_Cooling[cols_to_include])
                .add(dfresults_D3_Cooling[cols_to_include]))
dfresults_WLC_Cooling = pd.concat([dfresults_WLC_Cooling, df_data], axis=1)

In [ ]:
dfresults_WLC_noD_Cooling = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_A13_Cooling[cols_to_include]
                .add(dfresults_A4_Cooling[cols_to_include])
                .add(dfresults_A5_Cooling[cols_to_include])
                .add(dfresults_B2_Cooling[cols_to_include])
                .add(dfresults_B6_Cooling[cols_to_include])
                .add(dfresults_C2_Cooling[cols_to_include])
                .add(dfresults_C3_Cooling[cols_to_include])
                .add(dfresults_C4_Cooling[cols_to_include])
                .add(dfresults_B4_Cooling[cols_to_include]))
dfresults_WLC_noD_Cooling = pd.concat([dfresults_WLC_noD_Cooling, df_data], axis=1)

## 26.3 Appliances (per building, /m2)

### 26.3.1 Module B6

In [ ]:
dfresults_B6_Appliances_IND = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area})

sheet_name = "Appliances demand"
if sheet_name not in xls.sheet_names:
    dfresults_B6_Appliances = dfresults_B6_Appliances_IND.merge(dfresults_B6_Appliances, left_on='Bd_Name', right_on='Bd_Name', how='left')
else:
    dfresults_B6_Appliances = dfresults_B6_Appliances_IND.merge(dfresults_B6_Appliances, left_on='Bd_Name', right_index=True, how='left')

### 26.3.2 Whole Life Cycle

In [ ]:
dfresults_WLC_Appliances = dfresults_B6_Appliances.copy()

## 26.4 Total (per building, /m2)

In [ ]:
dfresults_A13 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_A13_Heating[cols_to_include].add(dfresults_A13_Cooling[cols_to_include]))
dfresults_A13 = pd.concat([dfresults_A13, df_data], axis=1)

In [ ]:
dfresults_A4 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_A4_Heating[cols_to_include].add(dfresults_A4_Cooling[cols_to_include]))
dfresults_A4 = pd.concat([dfresults_A4, df_data], axis=1)

In [ ]:
dfresults_A5 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_A5_Heating[cols_to_include].add(dfresults_A5_Cooling[cols_to_include]))
dfresults_A5 = pd.concat([dfresults_A5, df_data], axis=1)

In [ ]:
dfresults_B1 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = dfresults_B1_Heating[cols_to_include]
dfresults_B1 = pd.concat([dfresults_B1, df_data], axis=1)

In [ ]:
dfresults_B2 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_B2_Heating[cols_to_include].add(dfresults_B2_Cooling[cols_to_include]))
dfresults_B2 = pd.concat([dfresults_B2, df_data], axis=1)

In [ ]:
dfresults_B6 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_B6_Heating[cols_to_include]
                .add(dfresults_B6_Cooling[cols_to_include])
                .add(dfresults_B6_Appliances[cols_to_include]))
dfresults_B6 = pd.concat([dfresults_B6, df_data], axis=1)

In [ ]:
dfresults_C2 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_C2_Heating[cols_to_include].add(dfresults_C2_Cooling[cols_to_include]))
dfresults_C2 = pd.concat([dfresults_C2, df_data], axis=1)

In [ ]:
dfresults_C3 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_C3_Heating[cols_to_include].add(dfresults_C3_Cooling[cols_to_include]))
dfresults_C3 = pd.concat([dfresults_C3, df_data], axis=1)

In [ ]:
dfresults_C4 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_C4_Heating[cols_to_include].add(dfresults_C4_Cooling[cols_to_include]))
dfresults_C4 = pd.concat([dfresults_C4, df_data], axis=1)

In [ ]:
dfresults_B4 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_B4_Heating[cols_to_include].add(dfresults_B4_Cooling[cols_to_include]))
dfresults_B4 = pd.concat([dfresults_B4, df_data], axis=1)

In [ ]:
dfresults_D1 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_D1_Heating[cols_to_include].add(dfresults_D1_Cooling[cols_to_include]))
dfresults_D1 = pd.concat([dfresults_D1, df_data], axis=1)

In [ ]:
dfresults_D3 = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_D3_Heating[cols_to_include].add(dfresults_D3_Cooling[cols_to_include]))
dfresults_D3 = pd.concat([dfresults_D3, df_data], axis=1)

In [ ]:
dfresults_WLC = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_WLC_Heating[cols_to_include]
                .add(dfresults_WLC_Cooling[cols_to_include])
                .add(dfresults_WLC_Appliances[cols_to_include]))
dfresults_WLC = pd.concat([dfresults_WLC, df_data], axis=1)

In [ ]:
dfresults_WLC_noD = pd.DataFrame({'Bd_Name': Bd_Name, 'Bd_Area': Bd_Area, 'Heating Energy Systems': Heating_systems, 'Cooling Energy Systems': Cooling_systems})
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
df_data = (dfresults_WLC_noD_Heating[cols_to_include]
                .add(dfresults_WLC_noD_Cooling[cols_to_include]))
dfresults_WLC_noD = pd.concat([dfresults_WLC_noD, df_data], axis=1)

## 26.5 Total (per neighbourhood, /m2)

In [ ]:
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
dfresults_WLC_hood = dfresults_WLC[cols_to_include].sum(axis=0)

In [ ]:
cols_to_include = ['A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']
dfresults_WLC_noD_hood = dfresults_WLC_noD[cols_to_include].sum(axis=0)

## 26.6 Excel output export

In [ ]:
rename_dict = {
    "Bd_Name": "Building Name",
    "Bd_Area": "Building Area [m²]",
    "CC": "Climate Change: Total [kg CO₂ eq / m²]",
    "A": "Acidification [mol H⁺ eq / m²]",
    "ERnren": "Energy Resources: Non-Renewable [MJₗₕᵥ / m²]",
    "EF": "Eutrophication: Freshwater [kg P eq / m²]",
    "EM": "Eutrophication: Marine [kg N eq / m²]",
    "ET": "Eutrophication: Terrestrial [mol N eq / m²]",
    "RUMM": "Material Resources: Metals/Minerals [kg Sb eq / m²]",
    "OD": "Ozone Depletion [kg CFC-11 eq / m²]",
    "PM": "Particulate Matter Formation [disease incidence / m²]",
    "POF": "Photochemical Oxidant Formation: Human Health [kg NMVOC eq / m²]",
    "WU": "Water Use [m³ / m²]",
    "CCbio": "Climate Change: Biogenic [kg CO₂ eq / m²]",
    "CCfossil": "Climate Change: Fossil [kg CO₂ eq / m²]",
    "CCluc": "Climate Change: Land Use And Land Use Change [kg CO₂ eq / m²]",
    "PENRT": "Total Use Of Non-Renewable Primary Energy Resources [MJₚₑ / m²]"
}

cols_to_rename = ['Bd_Name', 'Bd_Area', 'A', 'CC', 'CCbio', 'CCfossil', 'CCluc', 'EF', 'EM', 'ERnren', 'ET', 'OD', 'PENRT', 'PM', 'POF', 'RUMM', 'WU']

### 26.6.1 Heating (per building, /m2)

In [ ]:
dfresults_A13_Heating.rename(columns=rename_dict, inplace=True)
dfresults_A4_Heating.rename(columns=rename_dict, inplace=True)
dfresults_A5_Heating.rename(columns=rename_dict, inplace=True)
dfresults_B1_Heating.rename(columns=rename_dict, inplace=True)
dfresults_B2_Heating.rename(columns=rename_dict, inplace=True)
dfresults_B6_Heating.rename(columns=rename_dict, inplace=True)
dfresults_C2_Heating.rename(columns=rename_dict, inplace=True)
dfresults_C3_Heating.rename(columns=rename_dict, inplace=True)
dfresults_C4_Heating.rename(columns=rename_dict, inplace=True)
dfresults_B4_Heating.rename(columns=rename_dict, inplace=True)
dfresults_D1_Heating.rename(columns=rename_dict, inplace=True)
dfresults_D3_Heating.rename(columns=rename_dict, inplace=True)
dfresults_WLC_Heating.rename(columns=rename_dict, inplace=True)
dfresults_WLC_noD_Heating.rename(columns=rename_dict, inplace=True)

In [ ]:
output_filename = f'{scenario_db_name}_LCA_results_heating.xlsx'
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        dfresults_WLC_Heating.to_excel(writer, sheet_name='Whole Lifecycle', index=False)
        dfresults_WLC_noD_Heating.to_excel(writer, sheet_name='Whole Lifecycle (no D)', index=False)
        dfresults_A13_Heating.to_excel(writer, sheet_name='Module A13', index=False)
        dfresults_A4_Heating.to_excel(writer, sheet_name='Module A4', index=False)
        dfresults_A5_Heating.to_excel(writer, sheet_name='Module A5', index=False)
        dfresults_B1_Heating.to_excel(writer, sheet_name='Module B1', index=False)
        dfresults_B2_Heating.to_excel(writer, sheet_name='Module B2', index=False)
        dfresults_B6_Heating.to_excel(writer, sheet_name='Module B6', index=False)
        dfresults_C2_Heating.to_excel(writer, sheet_name='Module C2', index=False)
        dfresults_C3_Heating.to_excel(writer, sheet_name='Module C3', index=False)
        dfresults_C4_Heating.to_excel(writer, sheet_name='Module C4', index=False)
        dfresults_B4_Heating.to_excel(writer, sheet_name='Module B4', index=False)
        dfresults_D1_Heating.to_excel(writer, sheet_name='Module D1', index=False)
        dfresults_D3_Heating.to_excel(writer, sheet_name='Module D3', index=False)
        

        # --- Define Styles ---
        number_format = '0.000E+00'
        center_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        thin_border = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'), bottom=Side(style='thin')
        )

        # --- Apply formatting to all sheets ---
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Iterate through all cells once to apply styles and calculate widths
            max_lengths = {}
            for row in worksheet.iter_rows():
                for cell in row:
                    # Apply alignment and border to all cells
                    cell.alignment = center_alignment
                    cell.border = thin_border

                    # Apply number format conditionally
                    if cell.row > 1 and cell.column > 2 and isinstance(cell.value, (int, float)):
                        cell.number_format = number_format
                    
                # Set a standard width for columns (e.g., width of 20)
                STANDARD_WIDTH = 25

                # Iterate through all columns that have data
                for col_idx in range(1, worksheet.max_column + 1):
                    col_letter = get_column_letter(col_idx)
                    worksheet.column_dimensions[col_letter].width = STANDARD_WIDTH

### 26.6.2 Cooliing (per building, /m2)

In [ ]:
dfresults_A13_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_A4_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_A5_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_B2_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_B6_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_C2_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_C3_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_C4_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_B4_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_D1_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_D3_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_WLC_Cooling.rename(columns=rename_dict, inplace=True)
dfresults_WLC_noD_Cooling.rename(columns=rename_dict, inplace=True)

In [ ]:
output_filename = f'{scenario_db_name}_LCA_results_cooling.xlsx'
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        dfresults_WLC_Cooling.to_excel(writer, sheet_name='Whole Lifecycle', index=False)
        dfresults_WLC_noD_Cooling.to_excel(writer, sheet_name='Whole Lifecycle (no D)', index=False)
        dfresults_A13_Cooling.to_excel(writer, sheet_name='Module A13', index=False)
        dfresults_A4_Cooling.to_excel(writer, sheet_name='Module A4', index=False)
        dfresults_A5_Cooling.to_excel(writer, sheet_name='Module A5', index=False)
        dfresults_B2_Cooling.to_excel(writer, sheet_name='Module B2', index=False)
        dfresults_B6_Cooling.to_excel(writer, sheet_name='Module B6', index=False)
        dfresults_C2_Cooling.to_excel(writer, sheet_name='Module C2', index=False)
        dfresults_C3_Cooling.to_excel(writer, sheet_name='Module C3', index=False)
        dfresults_C4_Cooling.to_excel(writer, sheet_name='Module C4', index=False)
        dfresults_B4_Cooling.to_excel(writer, sheet_name='Module B4', index=False)
        dfresults_D1_Cooling.to_excel(writer, sheet_name='Module D1', index=False)
        dfresults_D3_Cooling.to_excel(writer, sheet_name='Module D3', index=False)
        

        # --- Define Styles ---
        number_format = '0.000E+00'
        center_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        thin_border = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'), bottom=Side(style='thin')
        )

        # --- Apply formatting to all sheets ---
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Iterate through all cells once to apply styles and calculate widths
            max_lengths = {}
            for row in worksheet.iter_rows():
                for cell in row:
                    # Apply alignment and border to all cells
                    cell.alignment = center_alignment
                    cell.border = thin_border

                    # Apply number format conditionally
                    if cell.row > 1 and cell.column > 2 and isinstance(cell.value, (int, float)):
                        cell.number_format = number_format
                    
                # Set a standard width for columns (e.g., width of 20)
                STANDARD_WIDTH = 25

                # Iterate through all columns that have data
                for col_idx in range(1, worksheet.max_column + 1):
                    col_letter = get_column_letter(col_idx)
                    worksheet.column_dimensions[col_letter].width = STANDARD_WIDTH

### 26.6.3 Appliances (per building, /m2)

In [ ]:
dfresults_B6_Appliances.rename(columns=rename_dict, inplace=True)
dfresults_WLC_Appliances.rename(columns=rename_dict, inplace=True)

In [ ]:
output_filename = f'{scenario_db_name}_LCA_results_appliances.xlsx'
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        dfresults_WLC_Appliances.to_excel(writer, sheet_name='Whole Lifecycle (only B6)', index=False)
        dfresults_B6_Appliances.to_excel(writer, sheet_name='Module B6', index=False)
        
        # --- Define Styles ---
        number_format = '0.000E+00'
        center_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        thin_border = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'), bottom=Side(style='thin')
        )

        # --- Apply formatting to all sheets ---
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Iterate through all cells once to apply styles and calculate widths
            max_lengths = {}
            for row in worksheet.iter_rows():
                for cell in row:
                    # Apply alignment and border to all cells
                    cell.alignment = center_alignment
                    cell.border = thin_border

                    # Apply number format conditionally
                    if cell.row > 1 and cell.column > 2 and isinstance(cell.value, (int, float)):
                        cell.number_format = number_format
                    
                # Set a standard width for columns (e.g., width of 20)
                STANDARD_WIDTH = 25

                # Iterate through all columns that have data
                for col_idx in range(1, worksheet.max_column + 1):
                    col_letter = get_column_letter(col_idx)
                    worksheet.column_dimensions[col_letter].width = STANDARD_WIDTH

### 26.6.4 Total (per building, /m2)

In [ ]:
dfresults_A13.rename(columns=rename_dict, inplace=True)
dfresults_A4.rename(columns=rename_dict, inplace=True)
dfresults_A5.rename(columns=rename_dict, inplace=True)
dfresults_B1.rename(columns=rename_dict, inplace=True)
dfresults_B2.rename(columns=rename_dict, inplace=True)
dfresults_B6.rename(columns=rename_dict, inplace=True)
dfresults_C2.rename(columns=rename_dict, inplace=True)
dfresults_C3.rename(columns=rename_dict, inplace=True)
dfresults_C4.rename(columns=rename_dict, inplace=True)
dfresults_B4.rename(columns=rename_dict, inplace=True)
dfresults_D1.rename(columns=rename_dict, inplace=True)
dfresults_D3.rename(columns=rename_dict, inplace=True)
dfresults_WLC.rename(columns=rename_dict, inplace=True)
dfresults_WLC_noD.rename(columns=rename_dict, inplace=True)

In [ ]:
output_filename = f'{scenario_db_name}_LCA_results.xlsx'
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        dfresults_WLC.to_excel(writer, sheet_name='Whole Lifecycle', index=False)
        dfresults_WLC_noD.to_excel(writer, sheet_name='Whole Lifecycle (no D)', index=False)
        dfresults_A13.to_excel(writer, sheet_name='Module A13', index=False)
        dfresults_A4.to_excel(writer, sheet_name='Module A4', index=False)
        dfresults_A5.to_excel(writer, sheet_name='Module A5', index=False)
        dfresults_B1.to_excel(writer, sheet_name='Module B1', index=False)
        dfresults_B2.to_excel(writer, sheet_name='Module B2', index=False)
        dfresults_B6.to_excel(writer, sheet_name='Module B6', index=False)
        dfresults_C2.to_excel(writer, sheet_name='Module C2', index=False)
        dfresults_C3.to_excel(writer, sheet_name='Module C3', index=False)
        dfresults_C4.to_excel(writer, sheet_name='Module C4', index=False)
        dfresults_B4.to_excel(writer, sheet_name='Module B4', index=False)
        dfresults_D1.to_excel(writer, sheet_name='Module D1', index=False)
        dfresults_D3.to_excel(writer, sheet_name='Module D3', index=False)
        

        # --- Define Styles ---
        number_format = '0.000E+00'
        center_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        thin_border = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'), bottom=Side(style='thin')
        )

        # --- Apply formatting to all sheets ---
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Iterate through all cells once to apply styles and calculate widths
            max_lengths = {}
            for row in worksheet.iter_rows():
                for cell in row:
                    # Apply alignment and border to all cells
                    cell.alignment = center_alignment
                    cell.border = thin_border

                    # Apply number format conditionally
                    if cell.row > 1 and cell.column > 2 and isinstance(cell.value, (int, float)):
                        cell.number_format = number_format
            
            # Set a standard width for columns (e.g., width of 20)
            STANDARD_WIDTH = 25

            # Iterate through all columns that have data
            for col_idx in range(1, worksheet.max_column + 1):
                col_letter = get_column_letter(col_idx)
                worksheet.column_dimensions[col_letter].width = STANDARD_WIDTH

### 26.6.5 Total (per neighbourhood, /m2)

In [ ]:
rename_dict = {
    "CC": "Climate Change: Total [kg CO₂ eq / m²]",
    "A": "Acidification [mol H⁺ eq / m²]",
    "ERnren": "Energy Resources: Non-Renewable [MJₗₕᵥ / m²]",
    "EF": "Eutrophication: Freshwater [kg P eq / m²]",
    "EM": "Eutrophication: Marine [kg N eq / m²]",
    "ET": "Eutrophication: Terrestrial [mol N eq / m²]",
    "RUMM": "Material Resources: Metals/Minerals [kg Sb eq / m²]",
    "OD": "Ozone Depletion [kg CFC-11 eq / m²]",
    "PM": "Particulate Matter Formation [disease incidence / m²]",
    "POF": "Photochemical Oxidant Formation: Human Health [kg NMVOC eq / m²]",
    "WU": "Water Use [m³ / m²]",
    "CCbio": "Climate Change: Biogenic [kg CO₂ eq / m²]",
    "CCfossil": "Climate Change: Fossil [kg CO₂ eq / m²]",
    "CCluc": "Climate Change: Land Use And Land Use Change [kg CO₂ eq / m²]",
    "PENRT": "Total Use Of Non-Renewable Primary Energy Resources [MJₚₑ / m²]"
}

dfresults_WLC_hood = dfresults_WLC_hood.to_frame().T
dfresults_WLC_hood.rename(columns=rename_dict, inplace=True)
dfresults_WLC_noD_hood = dfresults_WLC_noD_hood.to_frame().T
dfresults_WLC_noD_hood.rename(columns=rename_dict, inplace=True)

In [ ]:
output_filename = f'{scenario_db_name}_LCA_results_wholeneighbourhood.xlsx'
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        dfresults_WLC_hood.to_excel(writer, sheet_name='Whole Lifecycle (hood)', index=False)
        dfresults_WLC_noD_hood.to_excel(writer, sheet_name='Whole Lifecycle (no D, hood)', index=False)

        # --- Define Styles ---
        number_format = '0.00E+00'
        center_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        thin_border = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'), bottom=Side(style='thin')
        )

        # --- Apply formatting to all sheets ---
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Iterate through all cells once to apply styles and calculate widths
            max_lengths = {}
            for row in worksheet.iter_rows():
                for cell in row:
                    # Apply alignment and border to all cells
                    cell.alignment = center_alignment
                    cell.border = thin_border

                    # Apply number format conditionally
                    if cell.row > 1 and cell.column > 2 and isinstance(cell.value, (int, float)):
                        cell.number_format = number_format
                    
                # 1. Set a standard width for columns
                STANDARD_WIDTH = 25

                # Iterate through all columns that have data
                for col_idx in range(1, worksheet.max_column + 1):
                    col_letter = get_column_letter(col_idx)
                    worksheet.column_dimensions[col_letter].width = STANDARD_WIDTH